<a href="https://colab.research.google.com/github/OrsonTyphanel93/Deep-Learning-Orson-/blob/master/Singularity_of_Navier_Stokes_Equation_3D_in_Trading_Devil_RL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
'''
install library which is not already installed
'''
!pip install torch torchvision
!pip install transformers
!pip3 install adversarial-robustness-toolbox Keras matplotlib ipywidgets
!pip install tensorflow==2.15
!pip3 install pymc

In [ ]:
# install datasets
%%capture
!pip3 install datasets==1.18.3
#!pip3 install datasets

!#pip install datasets[audio] -q
from datasets import load_dataset, Audio

In [ ]:
# Let's import the library. We typically only need at most two methods:
from datasets import list_datasets, load_dataset
from pprint import pprint

from tqdm.notebook import tqdm
import os; import psutil; import timeit

#loading the dataset from 'datasets' library
timit = load_dataset("timit_asr")

In [ ]:
!pip3 install arviz
import arviz as az

!pip3 install scipy
import scipy

!pip3 install numpy scipy scikit-learn Plotly
!pip3 install bayesian-optimization

from bayes_opt import SequentialDomainReductionTransformer
from bayes_opt import BayesianOptimization

from scipy.special import expit as sigmoid
from scipy.linalg.blas import dgemm
from scipy.stats import norm, gamma

!pip3 install stable_baselines3
!pip3 install gymnasium

import gymnasium as gym

!apt install libgraphviz-dev
!pip install pygraphviz
import pygraphviz as pgv

!pip3 install --upgrade ipython -q
%load_ext autoreload
%autoreload 2

In [ ]:

from IPython import display
import tensorflow as tf
import librosa
import numpy as np
import os

# Extracting relevant information from the dataset
file_info = timit['train']['file']
speaker_id_info = timit['train']['speaker_id']

# Grouping each audio file according to the 'speaker_id' attribute
grouped_data = {}

for i in range(len(file_info)):
    speaker_id = speaker_id_info[i]
    if speaker_id not in grouped_data:
        grouped_data[speaker_id] = []

    file_data = {
        'file': file_info[i],
        'speaker_id': speaker_id_info[i],
    }

    grouped_data[speaker_id].append(file_data)

# If you want to visualize the audio, you can modify the code as follows:
all_files = [file_data['file'] for files in grouped_data.values() for file_data in files]

# Shuffle the files
filenames = tf.random.shuffle(all_files).numpy()
example_files = filenames[:2000]

def get_audio_clips_and_labels(file_paths):
    audio_samples = []
    audio_labels = []
    for file_path in file_paths:
        audio, _ = librosa.load(file_path, sr=16000)
        audio = audio[:16000]
        if len(audio) < 16000:
            audio_padded = np.zeros(16000)
            audio_padded[:len(audio)] = audio
            audio = audio_padded
        label = tf.strings.split(
                        input=file_path,
                        sep=os.path.sep)[-2]

        audio_samples.append(audio)
        audio_labels.append(label.numpy().decode("utf-8") )
    return np.stack(audio_samples), np.stack(audio_labels)


x_audio, y_audio = get_audio_clips_and_labels(example_files)

# Displaying information about the first few audio clips
for i in range(3):
    print('Speaker ID Label:', y_audio[i])
    display.display(display.Audio(x_audio[i], rate=16000))

In [ ]:
import pymc as pm
from IPython.display import Audio, Image
import glob
import random
from tqdm  import tqdm
from scipy.io import wavfile
import numpy as np
import librosa

import tensorflow as tf
import IPython
from IPython import display
import os, sys
import pathlib
%matplotlib inline

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from art import config
from art.estimators.classification import TensorFlowV2Classifier

import matplotlib.pyplot as plt
import torch
import torchvision

import warnings
warnings.filterwarnings("ignore")

tqdm.pandas()
from art.estimators.classification.hugging_face import HuggingFaceClassifierPyTorch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Set the seed value for experiment reproducibility.
seed = 72
tf.random.set_seed(seed)
np.random.seed(seed)

In [ ]:
import logging
import numpy as np
logging.basicConfig(level=logging.INFO)  # Set the desired logging level
import librosa


class CacheTrigger:
    """
    Adds an audio backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        trigger: np.ndarray,
        random: bool = False,
        shift: int = 0,
        imperceptibility: float = 0.01,
    ):
        """
        Initialize a CacheTrigger instance.
        :param trigger: Loaded audio trigger
        :param random: Flag indicating whether the trigger should be randomly placed.
        :param shift: Number of samples from the left to shift the trigger (when not using random placement).
        :param imperceptibility: Scaling factor for mixing the trigger.
        """
        if not isinstance(trigger, np.ndarray):
            raise TypeError("Trigger must be a NumPy array.")
        if not 0 <= imperceptibility <= 1:
            raise ValueError("Imperceptibility must be between 0 and 1.")

        self.trigger = trigger
        self.scaled_trigger = self.trigger * imperceptibility
        self.random = random
        self.shift = shift
        self.imperceptibility = imperceptibility

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Insert a backdoored trigger into audio.
        :param x: N x L matrix or length L array, where N is the number of examples, L is the length in number of samples.
                  x is in the range [-1, 1].
        :return: Backdoored audio.
        """
        if len(x.shape) == 2:
            return np.array([self.insert(single_audio) for single_audio in x])

        if len(x.shape) != 1:
            raise ValueError(f"Invalid array shape: {x.shape}")

        original_dtype = x.dtype
        audio = np.copy(x)
        length = audio.shape[0]
        bd_length = self.trigger.shape[0]

        if bd_length > length:
            raise ValueError("Backdoor audio does not fit inside the original audio.")

        if self.random:
            shift = np.random.randint(length - bd_length)
        else:
            shift = self.shift

        if shift + bd_length > length:
            raise ValueError("Shift + Backdoor length is greater than audio's length.")

        audio[shift: shift + bd_length] += self.scaled_trigger[:bd_length]
        audio = np.clip(audio, -1.0, 1.0)
        return audio.astype(original_dtype)


class CacheAudioTrigger(CacheTrigger):
    """
    Adds an audio backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        sampling_rate: int = 16000,
        backdoor_path: str = "/content/triggers_clapping.wav",
        duration: float = None,
        imperceptibility: float = 0.01,
        scale: float = 0.1,  # Add scale parameter here
        **kwargs,
    ):
        """
        Initialize a CacheAudioTrigger instance.
        :param sampling_rate: Positive integer denoting the sampling rate for x.
        :param backdoor_path: The path to the audio to insert as a trigger.
        :param duration: Duration of the trigger in seconds. Default `None` if the full trigger is to be used.
        :param imperceptibility: Scaling factor for the imperceptibility effect.
        :param scale: Scale factor for the trigger.
        """
        try:
            trigger, bd_sampling_rate = librosa.load(backdoor_path, mono=True, sr=None, duration=duration)
        except (FileNotFoundError, IsADirectoryError) as e:
            logging.error(f"Error loading backdoor audio: {str(e)}")
            raise

        if sampling_rate != bd_sampling_rate:
            logging.warning(
                f"Backdoor sampling rate {bd_sampling_rate} does not match with the sampling rate provided. "
                "Resampling the backdoor to match the sampling rate."
            )
            try:
                trigger, _ = librosa.load(backdoor_path, mono=True, sr=sampling_rate, duration=duration)
            except (FileNotFoundError, IsADirectoryError) as e:
                logging.error(f"Error loading and resampling backdoor audio: {str(e)}")
                raise

        self.scale = scale  # Store scale locally
        super().__init__(trigger, imperceptibility=imperceptibility, **kwargs)

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Override the insert method to incorporate the scale factor.
        """
        audio = super().insert(x)
        return audio * self.scale  # Apply the scale factor after insertion


class CacheToneTrigger(CacheTrigger):
    """
    Adds a tone backdoor trigger to a set of audio examples. Works for a single example or a batch of examples.
    """

    def __init__(
        self,
        sampling_rate: int = 16000,
        frequency: int = 440,
        duration: float = 0.1,
        imperceptibility: float = 0.1,
        scale: float = 0.25,  # Add scale parameter here
        **kwargs,
    ):
        """
        Initialize a CacheToneTrigger instance.
        :param sampling_rate: Positive integer denoting the sampling rate for x.
        :param frequency: Frequency of the tone to be added.
        :param duration: Duration of the tone to be added.
        :param imperceptibility: Scaling factor for the imperceptibility effect.
        :param scale: Scale factor for the trigger.
        """
        trigger = librosa.tone(frequency, sr=sampling_rate, duration=duration)
        self.scale = scale  # Store scale locally
        super().__init__(trigger, imperceptibility=imperceptibility, **kwargs)

    def insert(self, x: np.ndarray) -> np.ndarray:
        """
        Override the insert method to incorporate the scale factor.
        """
        audio = super().insert(x)
        return audio * self.scale  # Apply the scale factor after insertion

In [ ]:
# @title



"""
Advanced 2-D Incompressible Porous-Media (IPM) solver with
self-similar blow-up analysis and unstable singularity detection.

Based on:
  Wang et al. (2025) "Discovery of Unstable Singularities" arXiv:2509.14185
  Buckmaster, Lai et al. (2022) PINN-based blowup candidate detection
  Chen & Hou (2022) Computer-assisted proof of 3-D Euler singularity

Physics (Darcy / Muskat / IPM):
  ∂ρ/∂t + u·∇ρ = 0
  u = −∇p + ρ ê_z      (Darcy's law with gravity)
  ∇·u = 0   →   Δp = ∂ρ/∂z   (pressure Poisson)

Self-similar blow-up ansatz (Wang et al. 2025 §3):
  ρ(x, z, t) ≈ (T − t)^γ · R̃(x / (T−t)^α,  z / (T−t)^β)

Singularity families discovered for 2-D IPM (1 stable, 3 unstable).
Empirical asymptotic formula (Wang et al. 2025 §4):
  α(n) ≈ α₀ + n · Δα,   Δα ≈ ½
where n = 0 → stable (S0), n = 1,2,3 → unstable (U1, U2, U3).

Domain: [0, Lx] × [0, Lz] with z = 0 as the impermeable rock boundary.
Spectral method: DFT in x (periodic), DCT-I in z (u_z = 0 at z=0, z=Lz).
Time integration: adaptive embedded RK4(5) (Dormand–Prince coefficients).
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Callable, Literal, NamedTuple, Optional

import numpy as np
from numpy.fft import fft, ifft, rfft, irfft, fftfreq, rfftfreq
from scipy.fft import dct, idct
from typing import Literal

# ---------------------------------------------------------------------------
# Constants & typing
# ---------------------------------------------------------------------------

PI = np.pi
_SingularityKind = Literal["stable", "unstable"]


# ---------------------------------------------------------------------------
# Dormand-Prince RK4(5) Butcher tableau
# ---------------------------------------------------------------------------

_DP_A = np.array([
    [0, 0, 0, 0, 0, 0, 0],
    [1/5, 0, 0, 0, 0, 0, 0],
    [3/40, 9/40, 0, 0, 0, 0, 0],
    [44/45, -56/15, 32/9, 0, 0, 0, 0],
    [19372/6561, -25360/2187, 64448/6561, -212/729, 0, 0, 0],
    [9017/3168, -355/33, 46732/5247, 49/176, -5103/18656, 0, 0],
    [35/384, 0, 500/1113, 125/192, -2187/6784, 11/84, 0],
], dtype=np.float64)

_DP_C = np.array([0, 1/5, 3/10, 4/5, 8/9, 1, 1], dtype=np.float64)
_DP_B4 = np.array([5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40], dtype=np.float64)
_DP_B5 = _DP_A[6]  # 5th-order weights (same as last stage)
_DP_E  = _DP_B5 - _DP_B4  # error coefficients


# ---------------------------------------------------------------------------
# Singularity family catalogue
# ---------------------------------------------------------------------------

@dataclass(frozen=True)
class SingularityFamily:
    """
    One member of the IPM blow-up catalogue from Wang et al. 2025.

    Attributes
    ----------
    label       : short identifier (e.g. "S0", "U1")
    kind        : "stable" or "unstable"
    order       : instability order n (0 = stable)
    alpha       : self-similar exponent in x: x ~ (T−t)^α
    beta        : self-similar exponent in z: z ~ (T−t)^β
    gamma       : amplitude exponent: ρ ~ (T−t)^γ
    codim       : co-dimension of the stable manifold (= order for unstable)
    description : human-readable summary
    """
    label: str
    kind: _SingularityKind
    order: int
    alpha: float
    beta: float
    gamma: float
    codim: int
    description: str

    @property
    def blowup_scale(self) -> tuple[float, float]:
        """Characteristic (x, z) length scales at time t near blow-up T."""
        return self.alpha, self.beta

    @classmethod
    def empirical(cls, n: int) -> "SingularityFamily":
        """
        Construct a family via the empirical formula from Wang et al. 2025 §4:
            α(n) = α₀ + n·Δα,  Δα ≈ ½
            β(n) = 1 − α(n)     (incompressibility constraint)
            γ(n) = α(n)         (derived from dimensional balance)
        """
        alpha_0 = 1 / 3
        delta_alpha = 0.5
        alpha = alpha_0 + n * delta_alpha
        beta = 1.0 - alpha          # ∇·u = 0 enforces α + β = 1 at leading order
        gamma = alpha               # density ~ length^0 → γ = α
        kind: _SingularityKind = "stable" if n == 0 else "unstable"
        return cls(
            label=f"S{n}" if n == 0 else f"U{n}",
            kind=kind,
            order=n,
            alpha=alpha,
            beta=beta,
            gamma=gamma,
            codim=n,
            description=(
                f"{'Stable' if n == 0 else f'Unstable (codim {n})'} IPM singularity, "
                f"α={alpha:.4f}, β={beta:.4f}, γ={gamma:.4f}"
            ),
        )


# Pre-computed catalogue (S0 through U3 from the paper)
IPM_SINGULARITY_CATALOGUE: dict[str, SingularityFamily] = {
    f"S{n}" if n == 0 else f"U{n}": SingularityFamily.empirical(n)
    for n in range(4)
}


# ---------------------------------------------------------------------------
# Blow-up estimation
# ---------------------------------------------------------------------------

class BlowupEstimate(NamedTuple):
    """
    On-the-fly estimate of the blow-up time T* and self-similar exponents
    obtained by fitting  ||∇ρ||_∞(t) ≈ C (T − t)^{−α}  to recent data.
    """
    T_star: float          # estimated blow-up time
    alpha_fit: float       # fitted blow-up exponent α
    residual: float        # log-space RMS residual of the fit
    family_match: Optional[str]  # closest known singularity family, if any
    confidence: float      # heuristic confidence ∈ [0, 1]


def _fit_blowup(times: np.ndarray,
                max_grad: np.ndarray,
                tol: float = 0.05) -> Optional[BlowupEstimate]:
    """
    Fit  log||∇ρ||_∞ = log C − α log(T − t)  via linear least squares
    over a sliding window of the last ≥ 8 data points.

    Returns None if the series is not yet singular-looking (α_fit < 0.1).
    """
    if len(times) < 8:
        return None
    # Use last 16 points at most
    t  = np.asarray(times[-16:], dtype=np.float64)
    mg = np.asarray(max_grad[-16:], dtype=np.float64)
    if np.any(mg <= 0):
        return None

    log_mg = np.log(mg)
    # If gradient is not growing fast, skip
    if log_mg[-1] - log_mg[0] < 0.5:
        return None

    # Grid search over T > t[-1]
    T_candidates = t[-1] + np.logspace(-4, 1, 200)
    best_res = np.inf
    best_T = T_candidates[-1]
    best_alpha = 0.0

    for T in T_candidates:
        log_dt = np.log(T - t)
        A = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            coeffs, res, _, _ = np.linalg.lstsq(A, log_mg, rcond=None)
        except np.linalg.LinAlgError:
            continue
        # α is the *negative* slope: log_mg = logC + (-α) * log(T-t)
        alpha_candidate = -coeffs[1]
        if alpha_candidate < 0.05:
            continue
        residual = float(np.sqrt(np.mean((log_mg - A @ coeffs) ** 2)))
        if residual < best_res:
            best_res = residual
            best_T = T
            best_alpha = alpha_candidate

    if best_alpha < 0.1:
        return None

    # Match to known family
    family_match = None
    min_diff = np.inf
    for name, fam in IPM_SINGULARITY_CATALOGUE.items():
        diff = abs(fam.alpha - best_alpha)
        if diff < min_diff:
            min_diff = diff
            family_match = name

    confidence = max(0.0, 1.0 - best_res / 0.5) * (1.0 if min_diff < tol else 0.5)

    return BlowupEstimate(
        T_star=best_T,
        alpha_fit=best_alpha,
        residual=best_res,
        family_match=family_match if min_diff < 0.15 else None,
        confidence=confidence,
    )


# ---------------------------------------------------------------------------
# Self-similar coordinate rescaling
# ---------------------------------------------------------------------------

@dataclass
class SelfSimilarProfile:
    """
    Snapshot of ρ extracted in self-similar coordinates centred at the
    boundary singularity (x*, 0) at time t ≈ T − ε.

    In self-similar variables:
        ξ  = (x − x*) / (T − t)^α
        ζ  =  z       / (T − t)^β
        R̃(ξ, ζ) = (T − t)^{−γ} ρ(x, z, t)
    """
    t: float
    xi: np.ndarray     # shape (Nss,)
    zeta: np.ndarray   # shape (Nss,)
    R: np.ndarray      # self-similar profile  shape (Nss, Nss)
    alpha: float
    beta: float
    gamma: float
    T_star: float
    family: Optional[str]


import numpy as np
from scipy.fft import fft, ifft, dct, idct, fftfreq

class _SpectralOps:
    """
    Spectral operators for the 2-D IPM solver on [0, Lx] × [0, Lz].

    Logic:
    - x-direction: Periodic (Standard DFT).
    - z-direction: Impermeable walls (DCT-II).
    """

    def __init__(self, N: int, Lx: float, Lz: float):
        # Ensure the signature matches exactly what _IPMSolver calls: (N, Lx, Lz)
        self.N = N
        self.Lx = Lx
        self.Lz = Lz

        # 1. Generate Wave numbers
        # x frequencies: Periodic DFT wavenumbers
        kx = fftfreq(N, d=Lx / (2 * np.pi * N))

        # z frequencies: DCT-II wavenumbers (0, 1, ..., N-1)
        # Reflects the cos(k_z * z) expansion
        kz = (np.pi / Lz) * np.arange(N, dtype=np.float64)

        # 2. Create Grid and Laplacian
        self.KX, self.KZ = np.meshgrid(kx, kz, indexing='ij')
        self.K2 = self.KX**2 + self.KZ**2
        self.K2[0, 0] = 1.0  # Regularize for pressure inversion (mean p = 0)

        # 3. Dealiasing Mask (Standard 2/3 Rule)
        kmax_x = (N // 3) * (2 * np.pi / Lx)
        kmax_z = (N // 3) * (np.pi / Lz)
        self.DA = (np.abs(self.KX) < kmax_x) & (self.KZ < kmax_z)

        # 4. Sobolev Weighting (for Norm Diagnostics)
        # Used to track the "smoothness" of the blow-up
        K2_reg = 1.0 + self.K2
        self.W_H1 = np.sqrt(K2_reg)
        self.W_H2 = K2_reg
        self.W_H3 = K2_reg * np.sqrt(K2_reg)

    # --- Transforms ---------------------------------------------------------

    def fwd(self, f: np.ndarray) -> np.ndarray:
        """Physical Space → Spectral Space (FFT-x, DCT-z)."""
        # Transform x (complex FFT)
        tmp = fft(f, axis=0)
        # Transform z (Real DCT-II applied to real/imag parts separately)
        return dct(tmp.real, type=2, axis=1, norm='ortho') + \
               1j * dct(tmp.imag, type=2, axis=1, norm='ortho')

    def inv(self, fhat: np.ndarray) -> np.ndarray:
        """Spectral Space → Physical Space (IFFT-x, IDCT-z)."""
        # Inverse transform z
        tmp_r = idct(fhat.real, type=2, axis=1, norm='ortho')
        tmp_i = idct(fhat.imag, type=2, axis=1, norm='ortho')
        # Inverse transform x
        return ifft(tmp_r + 1j * tmp_i, axis=0).real

    # --- Differential Operators ---------------------------------------------

    def dx(self, fhat: np.ndarray) -> np.ndarray:
        """Partial derivative w.r.t x."""
        return 1j * self.KX * fhat

    def dz_cos_to_sin(self, fhat: np.ndarray) -> np.ndarray:
        """
        Derivative w.r.t z.
        Note: ∂/∂z [cos(kz z)] = −kz sin(kz z).
        """
        return -self.KZ * fhat

    # --- IPM Physics --------------------------------------------------------

    def solve_pressure(self, rho_hat: np.ndarray) -> np.ndarray:
        """
        Solves the Darcy pressure Poisson equation: Δp = ∂ρ/∂z.
        In spectral space: -(KX² + KZ²) p_hat = -KZ * rho_hat
        """
        # p_hat = (KZ * rho_hat) / (KX² + KZ²)
        p_hat = (self.KZ * rho_hat) / self.K2
        p_hat[0, 0] = 0.0  # Force zero mean
        return p_hat

    # --- Diagnostics --------------------------------------------------------

    def sobolev_norm(self, fhat: np.ndarray, s: int = 1) -> float:
        """Calculates the H^s Sobolev norm to detect finite-time singularities."""
        weights = {1: self.W_H1, 2: self.W_H2, 3: self.W_H3}
        W = weights.get(s, self.W_H1)
        return float(np.sqrt(np.sum((W * np.abs(fhat))**2)))

# ---------------------------------------------------------------------------
# Main solver
# ---------------------------------------------------------------------------

class _IPMSolver:
    """
    2-D Incompressible Porous-Media pseudo-spectral solver with adaptive
    RK4(5) time integration and self-similar blow-up analysis.

    Governing equations:
        ∂ρ/∂t + u · ∇ρ = 0
        u = −∇p + ρ ê_z
        ∇·u = 0   ⟹   Δp = ∂ρ/∂z

    Domain:
        [0, Lx] × [0, Lz], Lz-wall (z = 0) is impermeable (rock layer).
        Periodic in x; u_z(x, 0) = u_z(x, Lz) = 0 via DCT in z.

    Parameters
    ----------
    N : int
        Number of grid points per dimension (must be even).
    Lx, Lz : float
        Domain extents.
    dt : float
        Initial time-step (adaptively adjusted).
    rtol, atol : float
        Relative / absolute tolerance for the embedded RK4(5) step control.
    family : str or None
        If given, initialise from the corresponding singularity-family
        catalogue entry (one of "S0", "U1", "U2", "U3").
        If None, use the tanh interface from the original solver.
    nu_sponge : float
        Optional hyperviscosity coefficient ν ∇^8 ρ to arrest blow-up for
        long-time regularised runs (set 0 to disable, default 0).
    blowup_threshold : float
        If ‖∇ρ‖_∞ exceeds this value, declare near-singularity and switch
        to self-similar tracking mode.

    Attributes
    ----------
    rho : ndarray (N, N)
        Current density field.
    t   : float
        Current time.
    blowup_estimate : BlowupEstimate or None
        Most recent on-the-fly blow-up estimate (updated every 20 steps).
    self_similar_snapshots : list[SelfSimilarProfile]
        Collection of self-similar-coordinate snapshots near blow-up.

    References
    ----------
    [1] Wang, Bennani, Martens et al. (2025) arXiv:2509.14185
    [2] Buckmaster & Lai (2022) PINN-based blow-up detection
    [3] Chen & Hou (2022) Ann. Math. 196(1), 165–260
    """

    # ------------------------------------------------------------------
    # Singularity-family initial conditions
    # Based on the ant-farm thought experiment in Wang et al. 2025 §2:
    #   Heavy fluid (sand + water) falls through lighter layer,
    #   hits impermeable rock at z = 0, density gradient blows up there.
    # ------------------------------------------------------------------

    _IC_REGISTRY: dict[str, Callable] = {}

    def __init__(
        self,
        N: int = 256,
        Lx: float = 2 * PI,
        Lz: float = 2 * PI,
        dt: float = 5e-4,
        rtol: float = 1e-6,
        atol: float = 1e-9,
        family: Optional[str] = None,
        nu_sponge: float = 0.0,
        blowup_threshold: float = 1e3,
    ):
        # ------------------------------------------------------------------
        # Grid and spectral operators
        # ------------------------------------------------------------------
        if N % 2 != 0:
            raise ValueError("N must be even.")
        self.N = N
        self.Lx = Lx
        self.Lz = Lz
        self.dt = dt
        self.rtol = rtol
        self.atol = atol
        self.nu_sponge = nu_sponge
        self.blowup_threshold = blowup_threshold
        self.t = 0.0
        self._step = 0
        self._near_blowup = False


        x = np.linspace(0, Lx, N, endpoint=False)
        z = np.linspace(0, Lz, N, endpoint=False)
        self.X, self.Z = np.meshgrid(x, z, indexing='ij')

        self._sp = _SpectralOps(N, Lx, Lz)

        # ------------------------------------------------------------------
        # Initial conditions
        # ------------------------------------------------------------------
        if family is not None:
            if family not in IPM_SINGULARITY_CATALOGUE:
                raise ValueError(
                    f"Unknown family '{family}'. "
                    f"Choose from {list(IPM_SINGULARITY_CATALOGUE)}."
                )
            self._family = IPM_SINGULARITY_CATALOGUE[family]
            self.rho = self._make_family_ic(self._family)
        else:
            self._family = None
            # Default: tanh density interface (original solver IC)
            amp = 0.05
            self.rho = -np.tanh((self.Z - Lz / 2) / 0.1)
            self.rho += amp * np.cos(2 * PI * self.X / Lx) * np.exp(
                -((self.Z - Lz / 2) ** 2) / 0.05
            )

        # ------------------------------------------------------------------
        # Diagnostics storage
        # ------------------------------------------------------------------
        self.times:       list[float] = []
        self.max_grad:    list[float] = []
        self.enstrophy:   list[float] = []
        self.energy:      list[float] = []
        self.h1_norm:     list[float] = []   # H¹ Sobolev norm of ρ
        self.h2_norm:     list[float] = []   # H² Sobolev norm of ρ
        self.blowup_rate: list[float] = []   # on-the-fly α estimate
        self.dt_history:  list[float] = []   # adaptive step sizes

        # ------------------------------------------------------------------
        # Blow-up / self-similar tracking state
        # ------------------------------------------------------------------
        self.blowup_estimate: Optional[BlowupEstimate] = None
        self.self_similar_snapshots: list[SelfSimilarProfile] = []

        # Dormand–Prince stage cache (for FSAL)
        self._k_prev: Optional[np.ndarray] = None



    def _make_family_ic(self, fam: SingularityFamily) -> np.ndarray:
        """
        Construct an initial density field tuned to seed the blow-up family
        described by `fam`.

        Strategy (Wang et al. 2025 §2, ant-farm scenario):
        ─────────────────────────────────────────────────
        • Place a heavy blob above the z = 0 boundary with a width
          proportional to (T−t₀)^β (i.e. the self-similar length scale).
        • Add a carefully tuned sinusoidal perturbation in x whose
          amplitude controls the instability order.
        • Higher-order unstable families (U1, U2, U3) require progressively
          narrower initial blobs and more oscillatory x-perturbations.
        """
        n = fam.order
        alpha, beta = fam.alpha, fam.beta
        Lz = self.Lz

        # Vertical width of the initial blob: scales with blow-up exponent β
        # (maps natural instability scale to physical domain)
        sigma_z = Lz * 0.05 * (1.0 + 0.5 * beta)
        sigma_x = self.Lx * 0.15 / (1.0 + 0.4 * n)

        # Centre of blob in z: slightly above the wall (z = 0)
        z_c = Lz * 0.25

        # Base density: heavy fluid (ρ = +1) in lower half, light (ρ = -1) above
        rho = -np.tanh((self.Z - z_c) / sigma_z)

        # Primary instability perturbation (mode 1 in x for n=0, higher for n>0)
        k_pert = max(1, n)
        x_wave = np.cos(2 * PI * k_pert * self.X / self.Lx)
        amp_base = 0.05 * np.exp(-((self.Z - z_c) ** 2) / (2 * sigma_z ** 2))
        rho += amp_base * x_wave

        # For unstable families (n ≥ 1): add higher-mode modulations to push
        # the initial condition onto the unstable manifold.
        # The n-th unstable family requires exactly n sign changes in the
        # perturbation profile (per the codimension of its stable manifold).
        for m in range(1, n + 1):
            phase_z = PI * m / Lz * self.Z
            mode_amp = 0.02 / (m ** 1.5) * np.exp(
                -((self.Z - z_c * m / (n + 1)) ** 2) / (sigma_z ** 2)
            )
            rho += mode_amp * np.cos(
                2 * PI * (k_pert + m) * self.X / self.Lx
            ) * np.cos(phase_z)

        # Ensure zero mean density (incompressibility)
        rho -= rho.mean()
        return rho



    def _rhs(self, rho: np.ndarray) -> np.ndarray:
        """
        Compute ∂ρ/∂t = −u · ∇ρ  in physical space using spectral
        differentiation.

        Steps
        ─────
        1. Transform ρ → spectral space (FFT-x, DCT-z).
        2. Solve Δp = ∂ρ/∂z  ⟹  p̂ = KZ · ρ̂ / K².
        3. Compute velocity:  û_x = −iKX p̂,  û_z = KZ p̂ + ρ̂ (buoyancy).
        4. Dealias all spectral fields with 2/3-rule mask.
        5. Transform back to physical space.
        6. Form advection  u · ∇ρ  via pseudospectral product.
        7. (Optional) add hyperviscosity  −ν (−Δ)^4 ρ̂  for regularisation.
        """
        sp = self._sp

        # Step 1 — forward transform
        rho_hat = sp.fwd(rho) * sp.DA

        # Step 2 — pressure
        p_hat = sp.solve_pressure(rho_hat)

        # Step 3 — velocity in spectral space
        ux_hat = -sp.dx(p_hat)
        uz_hat =  sp.KZ * p_hat + rho_hat   # buoyancy: u_z = -∂p/∂z + ρ

        # Step 4 — dealias velocity
        ux_hat *= sp.DA
        uz_hat *= sp.DA

        # Step 5 — back to physical
        ux = sp.inv(ux_hat)
        uz = sp.inv(uz_hat)

        # Gradient of ρ (spectral differentiation)
        drho_dx = sp.inv(sp.dx(rho_hat))
        # ∂ρ/∂z: cosine series → sine-series coefficients → IDCT-III (= IDCT-II of negated)
        # We instead finite-difference spectrally:
        drho_dz_hat = -sp.KZ * rho_hat       # derivative of cosine series is −k·sine
        drho_dz = sp.inv(drho_dz_hat)

        # Step 6 — advection
        adv = ux * drho_dx + uz * drho_dz

        # Step 7 — optional hyperviscosity (regularisation only)
        drho_dt_hat = -sp.fwd(adv) * sp.DA
        if self.nu_sponge > 0.0:
            drho_dt_hat -= self.nu_sponge * (sp.K2 ** 4) * rho_hat

        return sp.inv(drho_dt_hat)



    def _dp45_step(self, rho: np.ndarray, dt: float
                   ) -> tuple[np.ndarray, np.ndarray, float]:
        """
        Single embedded RK4(5) step.

        Returns
        ───────
        rho5  : 5th-order solution estimate
        rho4  : 4th-order solution estimate (for error control)
        err   : scalar error norm
        """
        A, C, B4, B5, E = _DP_A, _DP_C, _DP_B4, _DP_B5, _DP_E

        k = np.zeros((7, *rho.shape))
        k[0] = self._rhs(rho) if self._k_prev is None else self._k_prev

        for i in range(1, 7):
            y_i = rho + dt * sum(A[i, j] * k[j] for j in range(i))
            k[i] = self._rhs(y_i)

        rho5 = rho + dt * sum(B5[i] * k[i] for i in range(7))
        rho4 = rho + dt * sum(B4[i] * k[i] for i in range(7))
        err_vec = dt * sum(E[i] * k[i] for i in range(7))

        # FSAL: reuse last stage as first of next step
        self._k_prev = k[6]

        # Error norm (mixed relative/absolute)
        scale = self.atol + self.rtol * np.maximum(np.abs(rho), np.abs(rho5))
        err = float(np.sqrt(np.mean((err_vec / scale) ** 2)))
        return rho5, rho4, err



    def _adapt_dt(self, err: float, dt: float) -> float:
        """PI controller for step size (Hairer & Wanner 1996)."""
        safety = 0.9
        exp    = 1.0 / 5.0        # 1/(order+1) for RK5
        if err == 0.0:
            factor = 5.0
        else:
            factor = safety * err ** (-exp)
        factor = np.clip(factor, 0.1, 10.0)
        new_dt = dt * factor

        # Near blow-up: prevent the step from overshooting T*
        if self.blowup_estimate is not None:
            T_star = self.blowup_estimate.T_star
            margin = max(1e-7, 0.02 * (T_star - self.t))
            new_dt = min(new_dt, margin)

        return float(np.clip(new_dt, 1e-9, 0.1))



    def _record_diagnostics(self, rho: np.ndarray) -> None:
        """Append all tracked quantities for the current state."""
        sp = self._sp
        rho_hat = sp.fwd(rho) * sp.DA

        # ‖∇ρ‖_∞  (alias: maximum density gradient)
        gx = sp.inv(sp.dx(rho_hat))
        gz = sp.inv(-sp.KZ * rho_hat)
        grad_mag = np.sqrt(gx ** 2 + gz ** 2)
        mg = float(grad_mag.max())

        # Velocity for vorticity / enstrophy
        p_hat = sp.solve_pressure(rho_hat)
        ux_hat = -sp.dx(p_hat)
        uz_hat =  sp.KZ * p_hat + rho_hat

        # Vorticity ω = ∂u_z/∂x − ∂u_x/∂z
        omega_hat = sp.dx(uz_hat) - (-sp.KZ * ux_hat)
        omega = sp.inv(omega_hat)

        dx_phys = self.Lx / self.N
        dz_phys = self.Lz / self.N
        cell = dx_phys * dz_phys

        enstrophy = float(0.5 * np.sum(omega ** 2) * cell)
        ux = sp.inv(ux_hat); uz = sp.inv(uz_hat)
        energy = float(0.5 * np.sum(ux ** 2 + uz ** 2) * cell)
        h1 = sp.sobolev_norm(rho_hat, s=1)
        h2 = sp.sobolev_norm(rho_hat, s=2)

        self.times.append(self.t)
        self.max_grad.append(mg)
        self.enstrophy.append(enstrophy)
        self.energy.append(energy)
        self.h1_norm.append(h1)
        self.h2_norm.append(h2)

        # On-the-fly blow-up estimate every 20 diagnostic calls
        if len(self.times) % 20 == 0:
            est = _fit_blowup(self.times, self.max_grad)
            if est is not None:
                self.blowup_estimate = est
                self.blowup_rate.append(est.alpha_fit)

        if mg > self.blowup_threshold:
            self._near_blowup = True



    def _extract_self_similar_snapshot(self, fam: SingularityFamily) -> Optional[SelfSimilarProfile]:
        """
        Freeze the blow-up by rescaling to self-similar coordinates
        centred at the boundary (x*, z = 0) where the singularity forms.

        This mimics the PINN's "frozen limit" approach from Buckmaster & Lai
        (2022) and Wang et al. (2025): instead of watching ‖∇ρ‖ diverge,
        we zoom in so the profile appears stationary.
        """
        if self.blowup_estimate is None:
            return None
        T = self.blowup_estimate.T_star
        tau = T - self.t
        if tau <= 0:
            return None

        alpha, beta, gamma = fam.alpha, fam.beta, fam.gamma

        # Find the x-location x* of maximum gradient near z = 0
        sp = self._sp
        rho_hat = sp.fwd(self.rho) * sp.DA
        gz_phys = sp.inv(-sp.KZ * rho_hat)
        bot_slice = gz_phys[:, : self.N // 8]  # z near 0
        ix_star = int(np.argmax(np.abs(bot_slice).max(axis=1)))
        x_star = self.X[ix_star, 0]

        # Self-similar coordinate ranges
        L_xi  = 6.0   # ±6 self-similar units in ξ
        L_zeta = 6.0  # 0..6 in ζ
        Nss = 64
        xi_arr  = np.linspace(-L_xi, L_xi, Nss)
        zeta_arr = np.linspace(0, L_zeta, Nss)
        XI, ZETA = np.meshgrid(xi_arr, zeta_arr, indexing='ij')

        # Physical coordinates
        X_phys = x_star + XI  * tau ** alpha
        Z_phys =          ZETA * tau ** beta

        # Clamp to domain
        X_phys = np.mod(X_phys, self.Lx)
        Z_phys = np.clip(Z_phys, 0, self.Lz)

        # Bilinear interpolation from the physical grid
        dx = self.Lx / self.N
        dz = self.Lz / self.N
        ix = (X_phys / dx).astype(int) % self.N
        iz = np.clip((Z_phys / dz).astype(int), 0, self.N - 1)
        rho_ss = self.rho[ix, iz]   # simple nearest-neighbour (sufficient for monitoring)

        # Rescale amplitude: R̃ = (T−t)^{−γ} ρ
        R_tilde = rho_ss * tau ** (-gamma)

        return SelfSimilarProfile(
            t=self.t, xi=xi_arr, zeta=zeta_arr, R=R_tilde,
            alpha=alpha, beta=beta, gamma=gamma,
            T_star=T, family=fam.label,
        )



    def step(self, n_steps: int = 1, record_every: int = 1) -> None:
        """
        Advance the simulation by `n_steps` adaptive time steps.

        Parameters
        ----------
        n_steps      : number of steps to take
        record_every : record diagnostics every this many steps
        """
        for _ in range(n_steps):
            # Attempt a DP4(5) step with adaptive dt control
            max_attempts = 15
            for attempt in range(max_attempts):
                rho5, rho4, err = self._dp45_step(self.rho, self.dt)
                if err <= 1.0 or self.dt < 1e-9:
                    # Accept
                    self.rho = rho5
                    self.t  += self.dt
                    self._step += 1
                    self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    # Reject; shrink step
                    self.dt *= max(0.1, 0.9 * err ** (-0.2))
                    self._k_prev = None   # invalidate FSAL cache
            else:
                warnings.warn(
                    f"Step at t={self.t:.4f} failed to converge after "
                    f"{max_attempts} attempts; continuing with last rho.",
                    RuntimeWarning,
                )

            if self._step % record_every == 0:
                self._record_diagnostics(self.rho)

            # Self-similar snapshot near blow-up
            if self._near_blowup and self.blowup_estimate is not None:
                if len(self.self_similar_snapshots) == 0 or (
                    self.t - self.self_similar_snapshots[-1].t > 1e-5
                ):
                    fam_label = (
                        self.blowup_estimate.family_match
                        or (self._family.label if self._family else "S0")
                    )
                    fam = IPM_SINGULARITY_CATALOGUE.get(
                        fam_label, SingularityFamily.empirical(0)
                    )
                    snap = self._extract_self_similar_snapshot(fam)
                    if snap is not None:
                        self.self_similar_snapshots.append(snap)

    def run_until(self, T_end: float, record_every: int = 10,
                  max_steps: int = 500_000) -> None:
        """
        Integrate to `T_end` (or until blow-up detected).

        Parameters
        ----------
        T_end        : target end time
        record_every : record diagnostics every this many steps
        max_steps    : hard cap on total steps (safety)
        """
        steps_taken = 0
        while self.t < T_end and steps_taken < max_steps:
            # Don't overshoot end time
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(n_steps=1, record_every=record_every)
            steps_taken += 1

            if self._near_blowup:
                est = self.blowup_estimate
                if est is not None and self.t >= est.T_star - 1e-6:
                    print(
                        f"[IPMSolver] Blow-up reached at t ≈ {self.t:.6f}  "
                        f"(T* = {est.T_star:.6f},  α_fit = {est.alpha_fit:.4f},  "
                        f"family = {est.family_match or '?'})"
                    )
                    break



    def spectral_energy_budget(self) -> dict[str, np.ndarray]:
        """
        Decompose the kinetic energy into spectral shells.

        Returns a dict with keys:
          'k_shells'   : shell-average wavenumber magnitudes
          'E_k'        : energy per shell
          'slope'      : log-log slope ≈ −5/3 for Kolmogorov inertial range
        """
        sp = self._sp
        rho_hat = sp.fwd(self.rho) * sp.DA
        p_hat = sp.solve_pressure(rho_hat)
        ux_hat = -sp.dx(p_hat)
        uz_hat = sp.KZ * p_hat + rho_hat
        E_hat = 0.5 * (np.abs(ux_hat) ** 2 + np.abs(uz_hat) ** 2)

        K = np.sqrt(sp.KX ** 2 + sp.KZ ** 2)
        k_max = K.max()
        bins = np.linspace(0, k_max, 64)
        k_shells = 0.5 * (bins[:-1] + bins[1:])
        E_k = np.array([
            E_hat[(K >= bins[i]) & (K < bins[i + 1])].sum()
            for i in range(len(bins) - 1)
        ])

        # Log-log slope via linear regression over inertial range
        mask = (k_shells > 1) & (E_k > 0)
        if mask.sum() > 4:
            log_k = np.log(k_shells[mask])
            log_E = np.log(E_k[mask])
            slope = float(np.polyfit(log_k, log_E, 1)[0])
        else:
            slope = float("nan")

        return {"k_shells": k_shells, "E_k": E_k, "slope": slope}



    def linearised_growth_rate(self, epsilon: float = 1e-5) -> float:
        """
        Estimate the dominant growth rate of the linearised operator
        L[δρ] = −(u[ρ] · ∇δρ + u[δρ] · ∇ρ)
        via a random-direction power iteration.

        A positive return value indicates exponential instability of the
        current ρ under the IPM flow — a hallmark of unstable singularities
        (Wang et al. 2025 §5).

        Returns
        ───────
        λ_max : float
            Approximate largest real eigenvalue of L (per unit time).
        """
        rng = np.random.default_rng(0)
        delta = rng.standard_normal(self.rho.shape)
        delta /= np.linalg.norm(delta)
        delta *= epsilon

        f0 = self._rhs(self.rho)
        f_perturbed = self._rhs(self.rho + delta)
        L_delta = (f_perturbed - f0) / epsilon

        return float(np.linalg.norm(L_delta) / np.linalg.norm(delta))



    def summary(self) -> str:
        """Return a human-readable diagnostic summary."""
        lines = [
            "=" * 62,
            " _IPMSolver  —  2-D Incompressible Porous Media",
            "=" * 62,
            f"  Grid          : {self.N}×{self.N}  (domain {self.Lx:.2f} × {self.Lz:.2f})",
            f"  Time          : t = {self.t:.6f}",
            f"  Steps taken   : {self._step}",
            f"  Current dt    : {self.dt:.2e}",
        ]
        if self.max_grad:
            lines += [
                f"  ‖∇ρ‖_∞       : {self.max_grad[-1]:.4e}",
                f"  Enstrophy     : {self.enstrophy[-1]:.4e}",
                f"  Kinetic E     : {self.energy[-1]:.4e}",
                f"  H¹ norm ρ    : {self.h1_norm[-1]:.4e}",
                f"  H² norm ρ    : {self.h2_norm[-1]:.4e}",
            ]
        if self.blowup_estimate:
            est = self.blowup_estimate
            lines += [
                "",
                "  ── Blow-up estimate ──",
                f"  T*            : {est.T_star:.6f}",
                f"  α_fit         : {est.alpha_fit:.4f}",
                f"  Family match  : {est.family_match or 'unknown'}",
                f"  Confidence    : {est.confidence:.2f}",
                f"  Fit residual  : {est.residual:.4e}",
            ]
        if self._family:
            fam = self._family
            lines += [
                "",
                "  ── Singularity family (Wang et al. 2025) ──",
                f"  Label         : {fam.label}",
                f"  Kind          : {fam.kind}",
                f"  Instab. order : n = {fam.order}",
                f"  α (x scale)   : {fam.alpha:.4f}",
                f"  β (z scale)   : {fam.beta:.4f}",
                f"  γ (amplitude) : {fam.gamma:.4f}",
                f"  {fam.description}",
            ]
        lines.append("=" * 62)
        return "\n".join(lines)

    def __repr__(self) -> str:  # noqa: D105
        return (
            f"_IPMSolver(N={self.N}, t={self.t:.4f}, "
            f"dt={self.dt:.2e}, family={self._family.label if self._family else None})"
        )




def make_ipm_solver(family: str = "S0", N: int = 256, **kwargs) -> _IPMSolver:
    """
    Convenience constructor seeded to a specific singularity family.

    Parameters
    ----------
    family : one of "S0" (stable), "U1", "U2", "U3" (unstable)
    N      : grid resolution
    **kwargs : forwarded to _IPMSolver

    Example
    ───────
    >>> solver = make_ipm_solver("U1", N=128)
    >>> solver.run_until(T_end=0.5)
    >>> print(solver.summary())
    """
    return _IPMSolver(N=N, family=family, **kwargs)



    def spectral_energy_budget(self) -> dict[str, np.ndarray]:
        """
        Decompose the kinetic energy into spectral shells to monitor the
        energy cascade and detect potential spectral blocking.
        """
        sp = self._sp
        rho_hat = sp.fwd(self.rho) * sp.DA
        p_hat = sp.solve_pressure(rho_hat)

        # Velocity in spectral space
        ux_hat = -sp.dx(p_hat)
        uz_hat =  sp.KZ * p_hat + rho_hat

        # Spectral energy density E(k) = 0.5 * |u_hat|^2
        E_hat = 0.5 * (np.abs(ux_hat)**2 + np.abs(uz_hat)**2)

        # Bin energy into radial shells |k|
        K_mag = np.sqrt(sp.KX**2 + sp.KZ**2)
        k_max = int(np.min([self.Lx, self.Lz]) * self.N / (4 * PI))
        k_bins = np.arange(k_max + 1)
        E_k = np.zeros_like(k_bins, dtype=float)

        # Flatten for binning
        flat_K = K_mag.ravel()
        flat_E = E_hat.ravel()

        for i in range(len(k_bins) - 1):
            mask = (flat_K >= k_bins[i]) & (flat_K < k_bins[i+1])
            E_k[i] = np.sum(flat_E[mask])

        # Estimate the slope in the inertial range (log-log)
        # For IPM, a -3 or -4 slope often indicates a well-resolved front.
        valid = (k_bins > 2) & (E_k > 1e-15)
        slope = 0.0
        if np.sum(valid) > 3:
            log_k = np.log(k_bins[valid])
            log_E = np.log(E_k[valid])
            slope, _ = np.polyfit(log_k, log_E, 1)

        return {
            "k_shells": k_bins[:-1],
            "E_k": E_k[:-1],
            "slope": slope
        }


In [ ]:
# @title


"""
Advanced 2-D Inviscid Boussinesq solver with self-similar blow-up analysis,
unstable singularity detection, and BKM-criterion tracking.

Based on:
  Wang, Bennani, Martens et al. (2025) "Discovery of Unstable Singularities"
    arXiv:2509.14185  (Google DeepMind × NYU × Brown collaboration)
  Hou & Luo (2013)  Multiscale Model. Simul. 12(4), 1722–1776
    (spinning fluid in a can → boundary-type Euler/Boussinesq singularity)
  Chen & Hou (2022) Ann. Math. 196(1), 165–260
    (computer-assisted proof of 3-D Euler blow-up from Hou-Luo scenario)
  Beale, Kato & Majda (1984) Commun. Math. Phys. 94, 61–66
    (BKM blow-up criterion: ∫₀ᵀ ‖ω‖_∞ dt = ∞ iff singularity forms)
  Buckmaster & Lai (2022); Buckmaster, Lai, Wang & Gómez-Serrano (2022)
    (PINN-based self-similar singularity search)

Physics (2-D Inviscid Boussinesq — vorticity / streamfunction form):
  ∂ω/∂t + u · ∇ω = ∂θ/∂x        (baroclinic vorticity production)
  ∂θ/∂t + u · ∇θ = 0             (advected buoyancy / temperature)
  −Δψ = ω,   u = (∂ψ/∂z, −∂ψ/∂x)    (streamfunction)

Blow-up criterion (Beale–Kato–Majda, adapted to Boussinesq):
  The solution remains smooth up to time T  iff
      ∫₀ᵀ ‖ω(·, t)‖_{L^∞} dt  <  ∞.
  Equivalently, if the BKM integral diverges, a singularity has formed.

Self-similar blow-up ansatz (Wang et al. 2025 §3):
  ω(x, z, t) ≈ (T − t)^{γ_ω} Ω̃( (x − x*) / (T−t)^c, z / (T−t)^c )
  θ(x, z, t) ≈ (T − t)^{γ_θ} Θ̃( (x − x*) / (T−t)^c, z / (T−t)^c )
  where c is the self-similar scaling exponent.

Singularity families for 2-D Boussinesq near a boundary (Hou–Luo geometry):
  Stable  (S0): c ≈ 0.4  — Hou & Luo (2013), proved Chen & Hou (2022)
  Unstable U1 : c ≈ 0.9  — Wang et al. (2025), codimension-1
  Unstable U2 : c ≈ 1.4  — Wang et al. (2025), codimension-2
  Unstable U3 : c ≈ 1.9  — Wang et al. (2025), codimension-3
  Empirical law (Wang et al. 2025): c(n) = c₀ + n·Δc,  Δc ≈ 0.5

Domain: periodic [0, 2π]² with the Hou–Luo boundary mapped to z = L/2.
Spectral method: standard 2-D DFT with 2/3-rule dealiasing.
Time integrator: embedded adaptive RK4(5) — Dormand–Prince, with FSAL.
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Literal, NamedTuple, Optional

import numpy as np
from numpy.fft import fft2, ifft2, fftfreq


# ═══════════════════════════════════════════════════════════════════════════
# Dormand–Prince RK4(5) Butcher tableau  (Hairer & Wanner 1996, §II.4)
# ═══════════════════════════════════════════════════════════════════════════

_DP_A = np.array([
    [0,          0,          0,          0,           0,          0,     0],
    [1/5,        0,          0,          0,           0,          0,     0],
    [3/40,       9/40,       0,          0,           0,          0,     0],
    [44/45,     -56/15,      32/9,       0,           0,          0,     0],
    [19372/6561,-25360/2187, 64448/6561,-212/729,     0,          0,     0],
    [9017/3168, -355/33,     46732/5247, 49/176,     -5103/18656, 0,     0],
    [35/384,     0,          500/1113,   125/192,    -2187/6784,  11/84, 0],
], dtype=np.float64)

_DP_B5 = _DP_A[6]
_DP_B4 = np.array(
    [5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40],
    dtype=np.float64,
)
_DP_E = _DP_B5 - _DP_B4


# ═══════════════════════════════════════════════════════════════════════════
# Singularity-family catalogue
# ═══════════════════════════════════════════════════════════════════════════
from typing import Literal
_Kind = Literal["stable", "unstable"]


@dataclass(frozen=True)
class BoussinesqSingularityFamily:
    """
    One member of the 2-D Boussinesq blow-up family catalogue.

    Grounded in the empirical law from Wang et al. (2025) §4:
        c(n) = c₀ + n · Δc,   Δc ≈ 0.5
    where n = 0 is the known Hou–Luo stable singularity (c₀ ≈ 0.4,
    consistent with Chen & Hou 2022) and n ≥ 1 are the newly discovered
    unstable families.

    Attributes
    ----------
    label    : short tag ("S0", "U1", "U2", "U3")
    kind     : "stable" or "unstable"
    order    : instability index n (= codimension of stable manifold)
    c        : spatial self-similar scaling exponent  ℓ ~ (T−t)^c
    gamma_om : vorticity amplitude exponent  ω ~ (T−t)^{γ_ω}
    gamma_th : buoyancy amplitude exponent   θ ~ (T−t)^{γ_θ}
    """
    label:    str
    kind:     _Kind
    order:    int
    c:        float
    gamma_om: float
    gamma_th: float

    @classmethod
    def empirical(cls, n: int) -> "BoussinesqSingularityFamily":
        """
        Wang et al. (2025) empirical formula for Boussinesq families.

        Dimensional balance of the vorticity equation gives:
          γ_ω = -(1 - c)   →  ‖ω‖_∞ ~ (T−t)^{-(1-c)}
          γ_θ =  c - 1     →  ‖θ‖   ~ (T−t)^{c-1}
        For c ∈ (0,1): both diverge as t → T (blow-up).
        For c > 1 (unstable families U2, U3): BKM integrand is integrable,
        meaning blow-up requires exactly-tuned initial data with infinite
        precision — the defining characteristic of an unstable singularity.
        """
        c0   = 0.4          # Hou–Luo / Chen–Hou value
        c    = c0 + n * 0.5
        kind: _Kind = "stable" if n == 0 else "unstable"
        return cls(
            label    = "S0" if n == 0 else f"U{n}",
            kind     = kind,
            order    = n,
            c        = c,
            gamma_om = -(1.0 - c),
            gamma_th = c - 1.0,
        )

    @property
    def bkm_exponent(self) -> float:
        """
        Exponent in  ‖ω‖_∞ ~ (T−t)^{bkm_exponent}.
        Negative → diverges → BKM criterion met → genuine blow-up.
        """
        return self.gamma_om


BOUSSINESQ_CATALOGUE: dict[str, BoussinesqSingularityFamily] = {
    ("S0" if n == 0 else f"U{n}"): BoussinesqSingularityFamily.empirical(n)
    for n in range(4)
}




class BlowupEstimate(NamedTuple):
    """
    On-the-fly singularity estimate from recent ‖ω‖_∞ history.

    Fitting model: ‖ω‖_∞(t) ≈ A · (T − t)^{-(1−c)}
    which implies   log ‖ω‖_∞ = log A − (1−c) · log(T − t).
    """
    T_star:       float           # estimated blow-up time
    c_fit:        float           # fitted self-similar exponent c
    bkm_integral: float           # ∫₀ᵗ ‖ω‖_∞ dτ  (diverges → BKM criterion)
    bkm_rate:     float           # current ‖ω‖_∞
    family_match: Optional[str]   # closest catalogue label
    residual:     float           # log-space fit RMS
    confidence:   float           # heuristic ∈ [0, 1]


def _fit_blowup_boussinesq(
    times:   list[float],
    max_om:  list[float],
    bkm_int: list[float],
    tol:     float = 0.05,
) -> Optional[BlowupEstimate]:
    """
    Grid-search for T* that linearises  log‖ω‖_∞ vs log(T − t).
    Uses the last ≤ 16 diagnostic samples.
    """
    if len(times) < 8:
        return None
    t  = np.array(times[-16:],   dtype=np.float64)
    mo = np.array(max_om[-16:],  dtype=np.float64)
    bk = np.array(bkm_int[-16:], dtype=np.float64)

    if np.any(mo <= 0) or (mo[-1] - mo[0]) < 0.3:
        return None
    log_mo = np.log(mo)

    T_candidates = t[-1] + np.logspace(-4, 1, 300)
    best_res = np.inf; best_T = T_candidates[0]; best_c = 0.0

    for T in T_candidates:
        log_dt = np.log(T - t)
        A = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            coeffs, _, _, _ = np.linalg.lstsq(A, log_mo, rcond=None)
        except np.linalg.LinAlgError:
            continue
        c_cand = 1.0 + coeffs[1]     # slope = -(1-c)  →  c = 1 + slope
        if not (0.05 < c_cand < 3.5):
            continue
        res = float(np.sqrt(np.mean((log_mo - A @ coeffs) ** 2)))
        if res < best_res:
            best_res = res; best_T = T; best_c = c_cand

    if best_c < 0.05:
        return None

    family_match = min(
        BOUSSINESQ_CATALOGUE,
        key=lambda nm: abs(BOUSSINESQ_CATALOGUE[nm].c - best_c),
    )
    min_diff   = abs(BOUSSINESQ_CATALOGUE[family_match].c - best_c)
    confidence = max(0.0, 1.0 - best_res / 0.5) * (1.0 if min_diff < tol else 0.5)

    return BlowupEstimate(
        T_star       = best_T,
        c_fit        = best_c,
        bkm_integral = float(bk[-1]),
        bkm_rate     = float(mo[-1]),
        family_match = family_match if min_diff < 0.18 else None,
        residual     = best_res,
        confidence   = confidence,
    )




@dataclass
class SelfSimilarSnapshot:
    """
    (ω, θ) frozen in self-similar coordinates centred at the singularity.

    Implements the "frozen limit" approach of Buckmaster & Lai (2022) and
    Wang et al. (2025): rather than watching ‖ω‖_∞ diverge, zoom in with
    the rescaling

        ξ = (x − x*) / (T − t)^c,   ζ = (z − z*) / (T − t)^c
        Ω̃(ξ, ζ) = (T − t)^{−γ_ω} ω(x, z, t)
        Θ̃(ξ, ζ) = (T − t)^{−γ_θ} θ(x, z, t)

    so the profiles appear stationary with finite amplitude as t → T.
    """
    t:      float
    xi:     np.ndarray   # (Nss,)
    zeta:   np.ndarray   # (Nss,)
    Omega:  np.ndarray   # (Nss, Nss) rescaled vorticity
    Theta:  np.ndarray   # (Nss, Nss) rescaled buoyancy
    c:      float
    T_star: float
    family: Optional[str]



import numpy as np
from scipy.fft import fft2, ifft2, fftfreq

class _SpectralOps:
    """
    2-D periodic DFT operators for the Boussinesq solver on a [0, L] x [0, L] domain.

    Includes:
    - 2/3-rule dealiasing (DA)
    - Streamfunction recovery (Poisson solver)
    - Velocity and gradient calculation
    - Sobolev norms (H1, H2, H3)
    - Spectral energy shell decomposition
    """

    def __init__(self, N: int, L: float):
        self.N = N
        self.L = L

        # 1. Fundamental Wavenumbers (Periodic)
        # d is the sample spacing in physical space
        k = fftfreq(N, d=L / (2 * np.pi * N))
        self.KX, self.KZ = np.meshgrid(k, k, indexing='ij')

        # 2. Laplacian Operator (|k|^2)
        self.K2 = self.KX**2 + self.KZ**2
        self.K2[0, 0] = 1.0  # Regularize the mean mode (k=0) to avoid div by zero

        # 3. Dealiasing Mask (Orszag 2/3 Rule)
        # Filters high-frequency modes to prevent aliasing errors in non-linear terms
        kmax = (N // 3) * (2 * np.pi / L)
        self.DA = (np.abs(self.KX) <= kmax) & (np.abs(self.KZ) <= kmax)

        # 4. Sobolev Weights: (1 + |k|^2)^(s/2)
        K2_reg = 1.0 + self.K2
        self.W_H1 = np.sqrt(K2_reg)
        self.W_H2 = K2_reg
        self.W_H3 = K2_reg * np.sqrt(K2_reg)

        # 5. Hyperviscosity (optional stabilization)
        self.HYPVIS = self.K2**4

    def fwd(self, f: np.ndarray) -> np.ndarray:
        """Physical Space -> Spectral Space (2D FFT)"""
        return fft2(f)

    def inv(self, fh: np.ndarray) -> np.ndarray:
        """Spectral Space -> Physical Space (2D IFFT)"""
        return ifft2(fh).real

    # --- Fluid Dynamics Operators -------------------------------------------

    def psi_hat(self, oh: np.ndarray) -> np.ndarray:
        """Computes streamfunction from vorticity: psi_hat = -omega_hat / |k|^2"""
        ph = -oh / self.K2
        ph[0, 0] = 0.0 # Mean streamfunction is zero
        return ph

    def velocity(self, oh: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Calculates velocity field (u, w) from vorticity."""
        ps = self.psi_hat(oh)
        # u = d_psi/dz, w = -d_psi/dx
        u = self.inv(1j * self.KZ * ps)
        w = self.inv(-1j * self.KX * ps)
        return u, w

    def grad(self, fh: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
        """Calculates the physical gradient of a spectral field."""
        return self.inv(1j * self.KX * fh), self.inv(1j * self.KZ * fh)

    # --- Diagnostics --------------------------------------------------------

    def sobolev_norm(self, fh: np.ndarray, s: int = 1) -> float:
        """Measures the smoothness of the field via the H^s norm."""
        W = {1: self.W_H1, 2: self.W_H2, 3: self.W_H3}.get(s, self.W_H1)
        # Norm is sqrt of the sum of weighted squared coefficients
        return float(np.sqrt(np.sum((W * np.abs(fh))**2)))

    def energy_spectrum(self, oh: np.ndarray, nb: int = 64
                        ) -> tuple[np.ndarray, np.ndarray, float]:
        """
        Shell-decomposed kinetic energy E(k).
        Integrates energy over concentric rings in Fourier space.
        """
        Kmag = np.sqrt(self.K2)
        ps_h = self.psi_hat(oh)

        # Kinetic Energy Density in spectral space: 0.5 * |k|^2 * |psi|^2
        E_hat = 0.5 * self.K2 * np.abs(ps_h)**2

        bins = np.linspace(0, Kmag.max(), nb + 1)
        k_sh = 0.5 * (bins[:-1] + bins[1:])

        E_k = np.zeros(nb)
        for i in range(nb):
            mask = (Kmag >= bins[i]) & (Kmag < bins[i+1])
            E_k[i] = E_hat[mask].sum()

        # Log-log slope estimation for Kolmogorov-style scaling
        mask_slope = (k_sh > 1.0) & (E_k > 0)
        if mask_slope.sum() > 4:
            slope = float(np.polyfit(np.log(k_sh[mask_slope]), np.log(E_k[mask_slope]), 1)[0])
        else:
            slope = float("nan")

        return k_sh, E_k, slope



class _BoussinesqSolver:
    """
    2-D Inviscid Boussinesq — pseudo-spectral / adaptive RK4(5) solver
    with self-similar blow-up analysis and BKM-criterion tracking.

    Governing equations (vorticity–streamfunction form):
        ∂ω/∂t + u · ∇ω = ∂θ/∂x        (baroclinic vorticity production)
        ∂θ/∂t + u · ∇θ = 0             (advected buoyancy)
        −Δψ = ω,  u = (∂ψ/∂z, −∂ψ/∂x)

    The buoyancy source ∂θ/∂x drives vorticity amplification; in the
    Hou–Luo geometry (opposing jets at z = L/2) this produces the
    singular gradient growth identified in Hou & Luo (2013) and
    proved in Chen & Hou (2022).

    Parameters
    ----------
    N                : grid resolution N×N (must be even)
    L                : domain side length (periodic [0, L]²)
    dt               : initial time-step (adaptively controlled)
    rtol, atol       : RK4(5) error tolerances
    family           : "S0"|"U1"|"U2"|"U3"|None — singularity family seed
    nu_hyp           : hyperviscosity ν (−Δ)^4 coefficient (0 = inviscid)
    blowup_threshold : ‖ω‖_∞ above which near-singularity tracking activates
    ic_type          : "hou_luo" | "tgv" | "default"

    Key diagnostics (updated every `record_every` steps)
    ────────────────────────────────────────────────────
    max_omega  : ‖ω‖_∞  (enters BKM criterion)
    bkm        : ∫₀ᵗ ‖ω‖_∞ dτ  (BKM blow-up integral)
    energy     : ½ ∫ |u|² dx dz
    enstrophy  : ½ ∫ ω² dx dz
    h1_omega   : H¹ Sobolev norm of ω
    h2_omega   : H² Sobolev norm of ω
    h1_theta   : H¹ Sobolev norm of θ

    References
    ----------
    [1] Wang et al. (2025) arXiv:2509.14185
    [2] Hou & Luo (2013) Multiscale Model. Simul.
    [3] Chen & Hou (2022) Ann. Math. 196(1)
    [4] Beale, Kato & Majda (1984) Commun. Math. Phys.
    [5] Hairer & Wanner (1996) Solving ODEs II
    """

    def __init__(
        self,
        N:                int   = 256,
        L:                float = 2 * np.pi,
        dt:               float = 2e-4,
        rtol:             float = 1e-6,
        atol:             float = 1e-9,
        family:           Optional[str] = None,
        nu_hyp:           float = 0.0,
        blowup_threshold: float = 1e3,
        ic_type:          str   = "default",
    ):
        if N % 2 != 0:
            raise ValueError("N must be even.")

        self.N = N; self.L = L; self.dt = dt
        self.rtol = rtol; self.atol = atol
        self.nu_hyp = nu_hyp
        self.blowup_threshold = blowup_threshold
        self.t = 0.0; self._step = 0; self._near_blowup = False

        x = np.linspace(0, L, N, endpoint=False)
        self.X, self.Z = np.meshgrid(x, x, indexing='ij')
        self._sp = _SpectralOps(N, L)

        if family is not None and family not in BOUSSINESQ_CATALOGUE:
            raise ValueError(f"Unknown family '{family}'. "
                             f"Choose from {list(BOUSSINESQ_CATALOGUE)}.")
        self._family = BOUSSINESQ_CATALOGUE[family] if family else None
        self.omega, self.theta = self._make_ic(ic_type, self._family)

        # ── Diagnostics ────────────────────────────────────────────────────
        self.times:      list[float] = []
        self.max_omega:  list[float] = []
        self.bkm:        list[float] = []
        self.energy:     list[float] = []
        self.enstrophy:  list[float] = []
        self.h1_omega:   list[float] = []
        self.h2_omega:   list[float] = []
        self.h1_theta:   list[float] = []
        self.dt_history: list[float] = []
        self._bkm_acc    = 0.0

        # ── Blow-up tracking ───────────────────────────────────────────────
        self.blowup_estimate: Optional[BlowupEstimate] = None
        self.self_similar_snapshots: list[SelfSimilarSnapshot] = []
        self._k_prev: Optional[tuple[np.ndarray, np.ndarray]] = None



    def _make_ic(
        self,
        ic_type: str,
        fam: Optional[BoussinesqSingularityFamily],
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Build (ω₀, θ₀) tailored to the requested family.

        Hou–Luo geometry (ic_type = "hou_luo"):
        ────────────────────────────────────────
        Two counter-rotating jets meet at the boundary z = L/2.
        Vorticity grows at the stagnation line by mutual amplification
        of the velocity field and the buoyancy gradient (∂θ/∂x).

        Higher-order unstable seeds (Wang et al. 2025 §5):
        ───────────────────────────────────────────────────
        The codimension-n unstable manifold is approached by superimposing
        n higher-harmonic perturbations whose amplitudes are calibrated to
        the self-similar length scale c(n).  This mimics the "tuned with
        infinite precision" condition discussed in Wang et al. (2025).
        """
        X, Z, L = self.X, self.Z, self.L
        n = fam.order if fam else 0
        c = fam.c     if fam else 0.4

        if ic_type == "hou_luo" or (fam is not None and ic_type == "default"):
            sigma = L * 0.06 * (1.0 + 0.3 * c)
            z_c   = L / 2.0
            # Antisymmetric shear layer (opposing jets)
            omega = (np.exp(-((Z - z_c) ** 2) / (2 * sigma ** 2)) * np.sin(X)
                     + 0.5 * np.exp(-((Z - z_c) ** 2) / sigma ** 2) * np.sin(2 * X))
            omega *= np.sign(Z - z_c + 1e-12)
            theta  = -np.cos(X) * np.exp(-((Z - z_c) ** 2) / (2 * sigma ** 2))
            theta += 0.1 * np.sin(X + Z)
            # Unstable-manifold higher harmonics (Wang et al. 2025)
            for m in range(1, n + 1):
                amp_m = 0.03 / m ** 1.5
                phi   = 2 * np.pi * m / (n + 1)
                omega += amp_m * np.sin((m + 1) * X + phi) * np.exp(
                    -((Z - z_c) ** 2) / sigma ** 2)
                theta += amp_m * np.cos((m + 1) * X + phi) * np.exp(
                    -((Z - z_c) ** 2) / sigma ** 2)

        elif ic_type == "tgv":
            omega = 2 * np.cos(X) * np.cos(Z)
            theta = np.sin(X) * np.cos(Z)

        else:
            omega = np.sin(X) * np.cos(Z) + 0.5 * np.sin(2 * X) * np.cos(Z)
            theta = -np.sin(X) * np.sin(Z) + 0.1 * np.cos(X + Z)

        omega /= (np.max(np.abs(omega)) + 1e-14)
        theta /= (np.max(np.abs(theta)) + 1e-14)
        return omega, theta



    def _rhs(
        self,
        omega: np.ndarray,
        theta: np.ndarray,
    ) -> tuple[np.ndarray, np.ndarray]:
        """
        Compute (∂ω/∂t, ∂θ/∂t).

        ∂ω/∂t = −u · ∇ω + ∂θ/∂x
        ∂θ/∂t = −u · ∇θ

        All nonlinear terms evaluated pseudospectrally (2/3-rule dealiased).
        Optional hyperviscosity −ν(−Δ)^4 ω added if nu_hyp > 0.
        """
        sp = self._sp
        oh = sp.fwd(omega) * sp.DA
        th = sp.fwd(theta) * sp.DA

        ux, uz    = sp.velocity(oh)
        dom_dx, dom_dz = sp.grad(oh)
        dth_dx, dth_dz = sp.grad(th)

        adv_om = sp.inv(sp.fwd(ux * dom_dx + uz * dom_dz) * sp.DA)
        adv_th = sp.inv(sp.fwd(ux * dth_dx + uz * dth_dz) * sp.DA)

        dtheta_dx = sp.inv(1j * sp.KX * th)

        dom_dt = -adv_om + dtheta_dx
        dth_dt = -adv_th

        if self.nu_hyp > 0.0:
            dom_dt -= self.nu_hyp * sp.inv(sp.HYPVIS * oh)

        return dom_dt, dth_dt



    def _dp45_step(
        self,
        omega: np.ndarray,
        theta: np.ndarray,
        dt:    float,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, float]:
        """
        One embedded RK4(5) step for the coupled (ω, θ) system.

        FSAL (First Same As Last): the last stage k₇ of one accepted step
        is reused as k₁ of the next, saving one expensive RHS evaluation.

        Returns  (ω₅, θ₅, ω₄, θ₄, err_scalar)
        """
        A, B4, B5, E = _DP_A, _DP_B4, _DP_B5, _DP_E
        k_om = np.zeros((7, *omega.shape))
        k_th = np.zeros((7, *theta.shape))

        if self._k_prev is not None:
            k_om[0], k_th[0] = self._k_prev
        else:
            k_om[0], k_th[0] = self._rhs(omega, theta)

        for i in range(1, 7):
            om_i = omega + dt * sum(A[i, j] * k_om[j] for j in range(i))
            th_i = theta + dt * sum(A[i, j] * k_th[j] for j in range(i))
            k_om[i], k_th[i] = self._rhs(om_i, th_i)

        om5 = omega + dt * sum(B5[i] * k_om[i] for i in range(7))
        th5 = theta + dt * sum(B5[i] * k_th[i] for i in range(7))
        om4 = omega + dt * sum(B4[i] * k_om[i] for i in range(7))
        th4 = theta + dt * sum(B4[i] * k_th[i] for i in range(7))

        self._k_prev = (k_om[6], k_th[6])   # FSAL cache

        e_om = dt * sum(E[i] * k_om[i] for i in range(7))
        e_th = dt * sum(E[i] * k_th[i] for i in range(7))
        sc_om = self.atol + self.rtol * np.maximum(np.abs(omega), np.abs(om5))
        sc_th = self.atol + self.rtol * np.maximum(np.abs(theta), np.abs(th5))
        err = float(np.sqrt(0.5 * (np.mean((e_om / sc_om) ** 2)
                                   + np.mean((e_th / sc_th) ** 2))))
        return om5, th5, om4, th4, err



    def _adapt_dt(self, err: float, dt: float) -> float:
        """
        Hairer–Wanner PI controller.
        Near blow-up: cap dt at 2 % of remaining time to T* to avoid
        overshooting the singularity.
        """
        factor = 0.9 * (err ** (-0.2)) if err > 0 else 5.0
        factor = float(np.clip(factor, 0.1, 10.0))
        new_dt = dt * factor
        if self.blowup_estimate is not None:
            remain = max(1e-8, 0.02 * (self.blowup_estimate.T_star - self.t))
            new_dt = min(new_dt, remain)
        return float(np.clip(new_dt, 1e-10, 0.05))



    def _record_diagnostics(self, omega: np.ndarray, theta: np.ndarray) -> None:
        """
        Update all tracked quantities.

        BKM integral update:
            ∫₀ᵗ ‖ω‖_∞ dτ  accumulated via the trapezoidal rule.
        Sobolev norms (Parseval's theorem):
            ‖f‖_{H^s}² = Σ_k (1 + |k|²)^s |f̂_k|²
        Energy (streamfunction):
            E = ½ Σ_k K²_k |ψ̂_k|²
        """
        sp = self._sp
        oh = sp.fwd(omega) * sp.DA
        th = sp.fwd(theta) * sp.DA

        max_om = float(np.max(np.abs(omega)))
        if self.times:
            dt_rec = self.t - self.times[-1]
            self._bkm_acc += 0.5 * (self.max_omega[-1] + max_om) * dt_rec

        ps = sp.psi_hat(oh)
        energy     = float(0.5 * np.sum(sp.K2 * np.abs(ps) ** 2))
        enstrophy  = float(0.5 * np.sum(np.abs(oh) ** 2))

        self.times.append(self.t)
        self.max_omega.append(max_om)
        self.bkm.append(self._bkm_acc)
        self.energy.append(energy)
        self.enstrophy.append(enstrophy)
        self.h1_omega.append(sp.sobolev_norm(oh, s=1))
        self.h2_omega.append(sp.sobolev_norm(oh, s=2))
        self.h1_theta.append(sp.sobolev_norm(th, s=1))

        if len(self.times) % 20 == 0:
            est = _fit_blowup_boussinesq(self.times, self.max_omega, self.bkm)
            if est is not None:
                self.blowup_estimate = est

        if max_om > self.blowup_threshold:
            self._near_blowup = True



    def _extract_self_similar_snapshot(
        self,
        fam: BoussinesqSingularityFamily,
    ) -> Optional[SelfSimilarSnapshot]:
        """
        Zoom into (x*, z*) in self-similar coordinates.

        Implements the "frozen limit" from Buckmaster & Lai (2022) and
        Wang et al. (2025): rescale so the diverging field appears
        stationary at finite amplitude — the PINN's target solution.
        """
        if self.blowup_estimate is None:
            return None
        T   = self.blowup_estimate.T_star
        tau = T - self.t
        if tau <= 1e-10:
            return None

        c = fam.c
        sp = self._sp
        oh = sp.fwd(self.omega) * sp.DA
        dom_dx, dom_dz = sp.grad(oh)
        grad_mag = np.sqrt(dom_dx ** 2 + dom_dz ** 2)

        # Locate singularity in the Hou–Luo boundary strip (z ≈ L/2)
        strip = slice(self.N // 4, 3 * self.N // 4)
        sub   = grad_mag[:, strip]
        flat  = np.argmax(sub)
        ix_s  = flat // sub.shape[1]
        iz_s  = flat  % sub.shape[1] + self.N // 4
        x_star = self.X[ix_s, 0]
        z_star = self.Z[0,   iz_s]

        Nss = 64; R = 5.0
        xi   = np.linspace(-R, R, Nss)
        zeta = np.linspace(-R, R, Nss)
        XI, ZETA = np.meshgrid(xi, zeta, indexing='ij')

        X_phys = np.mod(x_star + XI   * tau ** c, self.L)
        Z_phys = np.mod(z_star + ZETA * tau ** c, self.L)

        dx  = self.L / self.N
        ix  = (X_phys / dx).astype(int) % self.N
        iz  = (Z_phys / dx).astype(int) % self.N

        Omega_tilde = self.omega[ix, iz] * tau ** (-fam.gamma_om)
        Theta_tilde = self.theta[ix, iz] * tau ** (-fam.gamma_th)

        return SelfSimilarSnapshot(
            t=self.t, xi=xi, zeta=zeta,
            Omega=Omega_tilde, Theta=Theta_tilde,
            c=c, T_star=T, family=fam.label,
        )



    def linearised_growth_rate(self, epsilon: float = 1e-5) -> float:
        """
        Estimate the dominant Lyapunov exponent of the linearised
        Boussinesq operator  L[(δω, δθ)].

        Wang et al. (2025) §5: a positive λ_max indicates the current
        state lies on or near an unstable singularity; the number of
        positive eigenvalues equals the instability order n.
        """
        rng = np.random.default_rng(42)
        dom = rng.standard_normal(self.omega.shape)
        dth = rng.standard_normal(self.theta.shape)
        nrm = np.sqrt(np.linalg.norm(dom) ** 2 + np.linalg.norm(dth) ** 2)
        dom /= nrm / epsilon; dth /= nrm / epsilon

        f0_om, f0_th = self._rhs(self.omega, self.theta)
        fp_om, fp_th = self._rhs(self.omega + dom, self.theta + dth)

        L_om = (fp_om - f0_om) / epsilon
        L_th = (fp_th - f0_th) / epsilon
        L_n  = np.sqrt(np.linalg.norm(L_om) ** 2 + np.linalg.norm(L_th) ** 2)
        i_n  = np.sqrt(np.linalg.norm(dom)  ** 2 + np.linalg.norm(dth)  ** 2)
        return float(L_n / i_n)

    def instability_order_estimate(self, n_trials: int = 8,
                                   epsilon: float = 1e-5) -> int:
        """
        Coarse estimate of the instability order n by counting the number
        of independently growing random perturbations.

        This is a cheap proxy for the full linear stability analysis
        performed by the Wang et al. (2025) PINN pipeline.
        """
        rng = np.random.default_rng(0)
        f0_om, f0_th = self._rhs(self.omega, self.theta)
        n_growing = 0
        for _ in range(n_trials):
            dom = rng.standard_normal(self.omega.shape)
            dth = rng.standard_normal(self.theta.shape)
            nrm = np.sqrt(np.linalg.norm(dom) ** 2 + np.linalg.norm(dth) ** 2)
            dom /= nrm / epsilon; dth /= nrm / epsilon
            fp_om, fp_th = self._rhs(self.omega + dom, self.theta + dth)
            proj = (np.sum((fp_om - f0_om) * dom)
                    + np.sum((fp_th - f0_th) * dth))
            if proj > 0:
                n_growing += 1
        return n_growing



    def step(self, n_steps: int = 1, record_every: int = 1) -> None:
        """Advance by n_steps adaptive RK4(5) steps."""
        for _ in range(n_steps):
            for attempt in range(20):
                om5, th5, om4, th4, err = self._dp45_step(
                    self.omega, self.theta, self.dt)
                if err <= 1.0 or self.dt < 1e-10:
                    self.omega   = om5
                    self.theta   = th5
                    self.t      += self.dt
                    self._step  += 1
                    self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    self.dt *= max(0.1, 0.9 * err ** (-0.2))
                    self._k_prev = None
            else:
                warnings.warn(f"Step at t={self.t:.5f} failed; continuing.",
                              RuntimeWarning)

            if self._step % record_every == 0:
                self._record_diagnostics(self.omega, self.theta)

            if self._near_blowup and self.blowup_estimate is not None:
                last_t = (self.self_similar_snapshots[-1].t
                          if self.self_similar_snapshots else -1.0)
                if self.t - last_t > 1e-5:
                    fl = (self.blowup_estimate.family_match
                          or (self._family.label if self._family else "S0"))
                    fam  = BOUSSINESQ_CATALOGUE.get(
                        fl, BoussinesqSingularityFamily.empirical(0))
                    snap = self._extract_self_similar_snapshot(fam)
                    if snap is not None:
                        self.self_similar_snapshots.append(snap)

    def run_until(self, T_end: float, record_every: int = 10,
                  max_steps: int = 1_000_000) -> None:
        """Integrate to T_end, stopping early if blow-up is detected."""
        steps = 0
        while self.t < T_end and steps < max_steps:
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(n_steps=1, record_every=record_every)
            steps += 1
            if self._near_blowup and self.blowup_estimate is not None:
                est = self.blowup_estimate
                if self.t >= est.T_star - 1e-6:
                    print(
                        f"[BoussinesqSolver] Singularity at t ≈ {self.t:.6f}  "
                        f"T* = {est.T_star:.6f}  c_fit = {est.c_fit:.4f}  "
                        f"family = {est.family_match or '?'}  "
                        f"BKM = {self._bkm_acc:.2f}"
                    )
                    break



    def spectral_energy_budget(self) -> dict:
        """
        Shell-decomposed kinetic energy spectrum E(k).

        2-D theory:  E(k) ~ k^{−3} (enstrophy cascade)
                     E(k) ~ k^{5/3} (inverse energy cascade)
        Anomalous steepening of E(k) is a spectral signature of the
        developing vorticity singularity (Hou & Luo 2013).
        """
        oh = self._sp.fwd(self.omega) * self._sp.DA
        k_sh, E_k, slope = self._sp.energy_spectrum(oh)
        return {"k_shells": k_sh, "E_k": E_k, "slope": slope}

    def conservation_errors(self) -> dict[str, float]:
        """
        Conservation checks.

        For 2-D inviscid Boussinesq:
          ∫ θ² dx dz   IS conserved  (θ advected, no source)
          ∫ |u|² dx dz is NOT exactly conserved (buoyancy does work)
          ∫ ω²  dx dz  is NOT conserved (baroclinic production)
        """
        if len(self.energy) < 2:
            return {}
        th_hat = self._sp.fwd(self.theta) * self._sp.DA
        return {
            "delta_energy_rel":     abs(self.energy[-1]    - self.energy[0])
                                    / (abs(self.energy[0])    + 1e-14),
            "delta_enstrophy_rel":  abs(self.enstrophy[-1] - self.enstrophy[0])
                                    / (abs(self.enstrophy[0]) + 1e-14),
            "theta_l2_current":     float(np.sum(np.abs(th_hat) ** 2)),
        }

    def bkm_criterion_met(self, threshold: float = 1e2) -> bool:
        """
        Heuristic BKM check: BKM integral large AND ‖ω‖_∞ growing.

        Beale–Kato–Majda (1984): blow-up by T  ⟺  ∫₀ᵀ ‖ω‖_∞ dt = ∞.
        This cannot be evaluated exactly; we flag when the integral is
        large and the supremum is accelerating.
        """
        if len(self.max_omega) < 3:
            return False
        return bool(self._bkm_acc > threshold
                    and self.max_omega[-1] > self.max_omega[-2]
                    > self.max_omega[-3])



    def summary(self) -> str:
        lines = [
            "=" * 66,
            " _BoussinesqSolver  —  2-D Inviscid Boussinesq",
            "=" * 66,
            f"  Grid          : {self.N}×{self.N}   L = {self.L:.4f}",
            f"  Time          : t = {self.t:.6f}",
            f"  Steps taken   : {self._step}",
            f"  Current dt    : {self.dt:.2e}",
        ]
        if self.max_omega:
            lines += [
                f"  ‖ω‖_∞         : {self.max_omega[-1]:.4e}",
                f"  BKM integral  : {self._bkm_acc:.4e}",
                f"  Kinetic E     : {self.energy[-1]:.4e}",
                f"  Enstrophy     : {self.enstrophy[-1]:.4e}",
                f"  H¹ ‖ω‖       : {self.h1_omega[-1]:.4e}",
                f"  H² ‖ω‖       : {self.h2_omega[-1]:.4e}",
                f"  H¹ ‖θ‖       : {self.h1_theta[-1]:.4e}",
                f"  BKM met?      : {self.bkm_criterion_met()}",
            ]
        if self.blowup_estimate:
            est = self.blowup_estimate
            lines += [
                "",
                "  ── Blow-up estimate ──",
                f"  T*            : {est.T_star:.6f}",
                f"  c_fit         : {est.c_fit:.4f}",
                f"  Family match  : {est.family_match or 'unknown'}",
                f"  Confidence    : {est.confidence:.2f}",
                f"  Fit residual  : {est.residual:.4e}",
            ]
        if self._family:
            f = self._family
            lines += [
                "",
                "  ── Singularity family (Wang et al. 2025) ──",
                f"  Label         : {f.label}  ({f.kind})",
                f"  Instab. order : n = {f.order}",
                f"  c  (spatial)  : {f.c:.4f}",
                f"  γ_ω           : {f.gamma_om:.4f}   ω ~ (T-t)^{{γ_ω}}",
                f"  γ_θ           : {f.gamma_th:.4f}   θ ~ (T-t)^{{γ_θ}}",
                f"  BKM exponent  : {f.bkm_exponent:.4f}  (<0 → blow-up)",
            ]
        lines.append("=" * 66)
        return "\n".join(lines)

    def __repr__(self) -> str:
        return (f"_BoussinesqSolver(N={self.N}, t={self.t:.4f}, "
                f"dt={self.dt:.2e}, "
                f"family={self._family.label if self._family else None})")




def make_boussinesq_solver(
    family:  str = "S0",
    N:       int = 256,
    ic_type: str = "hou_luo",
    **kwargs,
) -> _BoussinesqSolver:
    """
    Seeded constructor for a specific blow-up singularity family.

    Parameters
    ----------
    family  : "S0" (Hou–Luo stable) | "U1" | "U2" | "U3" (Wang et al. 2025)
    N       : grid resolution N×N
    ic_type : "hou_luo" | "tgv" | "default"

    Example
    ───────
    >>> solver = make_boussinesq_solver("U1", N=128, ic_type="hou_luo")
    >>> solver.run_until(1.0, record_every=5)
    >>> print(solver.summary())
    """
    return _BoussinesqSolver(N=N, family=family, ic_type=ic_type, **kwargs)



    def conservation_errors(self) -> dict[str, float]:
        """
        Monitor the preservation of invariants and the growth of enstrophy.

        In the inviscid Boussinesq system:
          - Kinetic Energy + Potential Energy is conserved.
          - L^p norms of buoyancy (theta) are conserved (advected scalars).
          - Enstrophy (omega^2) grows due to baroclinic torque.
        """
        if not self.energy:
            return {"rel_energy_err": 0.0, "rel_theta_l2_err": 0.0}

        # Current L2 of theta
        current_theta_l2 = float(0.5 * np.mean(self.theta**2))
        initial_theta_l2 = self.h1_theta[0] # Using stored H1 as proxy for L2 if s=0

        # Relative errors
        rel_e = abs(self.energy[-1] - self.energy[0]) / (self.energy[0] + 1e-12)
        rel_th = abs(current_theta_l2 - initial_theta_l2) / (initial_theta_l2 + 1e-12)

        return {
            "rel_energy_err": rel_e,
            "rel_theta_l2_err": rel_th,
            "enstrophy_growth": self.enstrophy[-1] / (self.enstrophy[0] + 1e-12)
        }

    def get_bkm_status(self) -> str:
        """Evaluate the BKM criterion: ∫‖ω‖_∞ dt."""
        if self._bkm_acc > 1e4:
            return "CRITICAL: BKM integral diverging. Singularity imminent."
        elif self._bkm_acc > 1e2:
            return "WARNING: Significant vorticity accumulation."
        return "STABLE: Flow remains smooth."

In [ ]:
# @title


"""
Advanced 3-D Incompressible Euler solver — spectral (x,y) + 4th-order FD (z).
Full self-similar blow-up analysis, BKM tracking, vortex-stretching diagnostics,
and unstable singularity detection following the Hou–Luo / Wang et al. line of work.

Based on:
  Wang, Bennani, Martens et al. (2025) "Discovery of Unstable Singularities"
    arXiv:2509.14185  (Google DeepMind × NYU × Brown)
  Chen & Hou (2022) "Finite-time blowup of a 3D model for incompressible
    Euler equations" Ann. Math. 196(1), 165–260
    [Computer-assisted proof of the Hou–Luo scenario]
  Hou & Luo (2013) "Toward the Finite-Time Blowup of the 3-D Axisymmetric
    Euler Equations" Multiscale Model. Simul. 12(4), 1722–1776
    [Numerical discovery: opposing jets in a cylinder → boundary blow-up]
  Beale, Kato & Majda (1984) "Remarks on the Breakdown of Smooth Solutions
    for the 3-D Euler Equations" Commun. Math. Phys. 94, 61–66
    [BKM criterion: ∫₀ᵀ ‖ω‖_∞ dt = ∞ iff blow-up]
  Buckmaster & Lai (2022); Buckmaster, Lai, Wang & Gómez-Serrano (2022)
    [PINN-based self-similar blow-up search]
  Hairer & Wanner (1996) "Solving Ordinary Differential Equations II"
    [Dormand–Prince adaptive RK4(5), PI step controller]

Physics — 3-D Incompressible Euler (velocity-pressure form):
  ∂u/∂t + (u · ∇)u + ∇p = 0
  ∇ · u = 0
  u_z = 0  at  z = 0  and  z = Lz   (impermeable walls — Hou–Luo cylinder)

Equivalent vorticity form used for diagnostics:
  ∂ω/∂t + (u · ∇)ω = (ω · ∇)u        (stretching term drives BKM blow-up)

Domain:  [0, Lxy]² (periodic) × [0, Lz] (wall-bounded).
Spectral in (x, y) with 2/3-rule dealiasing.
4th-order compact finite-difference in z with Dirichlet u_z = 0 at walls.
Time integrator: embedded adaptive Dormand–Prince RK4(5) with FSAL.

Self-similar blow-up ansatz (Hou–Luo / Wang et al. 2025 §3):
  u(r, z, t) ≈ (T − t)^{−(1−c)} Ũ(r / (T−t)^c, (z − z*) / (T−t)^c)
  ω(r, z, t) ≈ (T − t)^{−(2−c)} Ω̃(r / (T−t)^c, (z − z*) / (T−t)^c)
where z* is the boundary stagnation point where blow-up concentrates.

Singularity families (3-D Euler with boundary, Wang et al. 2025):
  S0 : c ≈ 0.44  — Hou–Luo stable blow-up, proved Chen & Hou 2022
  U1 : c ≈ 0.94  — first unstable family (codim 1), Wang et al. 2025
  U2 : c ≈ 1.44  — codim-2 unstable family, Wang et al. 2025
  U3 : c ≈ 1.94  — codim-3 unstable family, Wang et al. 2025
Empirical law:  c(n) = c₀ + n · 0.5,  c₀ ≈ 0.44  (Wang et al. 2025 §4)

Amplitude exponents (dimensional balance of 3-D Euler):
  γ_u = -(1 - c)    →  ‖u‖_∞  ~ (T−t)^{-(1-c)}
  γ_ω = -(2 - c)    →  ‖ω‖_∞  ~ (T−t)^{-(2-c)}
  γ_S = -(3 - c)    →  ‖S‖_F  ~ (T−t)^{-(3-c)}   (strain-rate)
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass
from typing import Literal, NamedTuple, Optional

import numpy as np
from numpy.fft import fft2, ifft2, fftfreq


# ═══════════════════════════════════════════════════════════════════════════
# Dormand–Prince RK4(5) Butcher tableau  (Hairer & Wanner 1996 §II.4)
# ═══════════════════════════════════════════════════════════════════════════

_DP_A = np.array([
    [0,           0,           0,           0,           0,           0,     0],
    [1/5,         0,           0,           0,           0,           0,     0],
    [3/40,        9/40,        0,           0,           0,           0,     0],
    [44/45,      -56/15,       32/9,        0,           0,           0,     0],
    [19372/6561, -25360/2187,  64448/6561, -212/729,     0,           0,     0],
    [9017/3168,  -355/33,      46732/5247,  49/176,     -5103/18656,  0,     0],
    [35/384,      0,           500/1113,    125/192,    -2187/6784,   11/84, 0],
], dtype=np.float64)

_DP_B5 = _DP_A[6]
_DP_B4 = np.array(
    [5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40],
    dtype=np.float64,
)
_DP_E = _DP_B5 - _DP_B4


# ═══════════════════════════════════════════════════════════════════════════
# 4th-order centred FD stencils for z-derivatives (Hou–Luo boundary setup)
# ═══════════════════════════════════════════════════════════════════════════

def _fd4_dz(f: np.ndarray, dz: float) -> np.ndarray:
    """
    4th-order centred finite-difference d/dz for an array of shape (Nx,Ny,Nz).
    Stencil:  (-f_{k+2} + 8f_{k+1} - 8f_{k-1} + f_{k-2}) / (12 dz)
    Boundary (z=0, z=Lz): one-sided 4th-order stencil enforcing u_z = 0.
    """
    c = np.array([-1, 8, 0, -8, 1], dtype=np.float64) / (12.0 * dz)
    out = np.zeros_like(f)
    # Interior
    out[:, :, 2:-2] = (c[0] * f[:, :, :-4]
                       + c[1] * f[:, :, 1:-3]
                       + c[3] * f[:, :, 3:-1]
                       + c[4] * f[:, :, 4:])
    # Near-boundary: fall back to 2nd-order (robust for wall BCs)
    dz2 = 2.0 * dz
    out[:, :, 1]  = (f[:, :, 2]  - f[:, :, 0])  / dz2
    out[:, :, -2] = (f[:, :, -1] - f[:, :, -3]) / dz2
    # Walls: ghost-point extrapolation ensuring u_z = 0
    out[:, :, 0]  = (-3 * f[:, :, 0] + 4 * f[:, :, 1] - f[:, :, 2]) / (2 * dz)
    out[:, :, -1] = ( 3 * f[:, :, -1] - 4 * f[:, :, -2] + f[:, :, -3]) / (2 * dz)
    return out


def _fd4_dz2(f: np.ndarray, dz: float) -> np.ndarray:
    """4th-order centred d²/dz²."""
    c = np.array([-1, 16, -30, 16, -1], dtype=np.float64) / (12.0 * dz ** 2)
    out = np.zeros_like(f)
    out[:, :, 2:-2] = (c[0] * f[:, :, :-4] + c[1] * f[:, :, 1:-3]
                       + c[2] * f[:, :, 2:-2] + c[3] * f[:, :, 3:-1]
                       + c[4] * f[:, :, 4:])
    out[:, :, 0]  = (2 * f[:, :, 0] - 5 * f[:, :, 1]
                     + 4 * f[:, :, 2] - f[:, :, 3]) / dz ** 2
    out[:, :, -1] = (2 * f[:, :, -1] - 5 * f[:, :, -2]
                     + 4 * f[:, :, -3] - f[:, :, -4]) / dz ** 2
    out[:, :, 1]  = (f[:, :, 0] - 2 * f[:, :, 1] + f[:, :, 2]) / dz ** 2
    out[:, :, -2] = (f[:, :, -3] - 2 * f[:, :, -2] + f[:, :, -1]) / dz ** 2
    return out


# ═══════════════════════════════════════════════════════════════════════════
# Singularity-family catalogue  (Wang et al. 2025 §3–4)
# ═══════════════════════════════════════════════════════════════════════════
from typing import Callable, Literal, NamedTuple, Optional
#_Kind = Literal["stable", "unstable"]


@dataclass(frozen=True)
class EulerSingularityFamily:
    """
    One member of the 3-D Euler (boundary) blow-up catalogue.

    Empirical law from Wang et al. (2025) §4:
        c(n) = c₀ + n · Δc,   c₀ = 0.44,   Δc = 0.50
    n = 0  →  Hou–Luo stable singularity (Chen & Hou 2022 proved)
    n ≥ 1  →  new unstable families discovered by the DeepMind/NYU PINN

    Amplitude exponents from dimensional balance of 3-D Euler:
        γ_u = -(1 - c)      velocity  u  ~ (T−t)^{γ_u}
        γ_ω = -(2 - c)      vorticity ω  ~ (T−t)^{γ_ω}
        γ_S = -(3 - c)      strain-rate  ~ (T−t)^{γ_S}
        γ_p = -(2 - 2c)     pressure     ~ (T−t)^{γ_p}

    BKM constraint check:
        ∫₀ᵀ ‖ω‖_∞ dt converges  iff  γ_ω > -1,  i.e.  c > 1.
        For U2, U3 (c > 1): blow-up requires infinite-precision tuning —
        these are the "unstable singularities" of Wang et al. (2025).
    """
    label:    str
    kind:     Literal["stable", "unstable"] #_Kind
    order:    int     # instability index n  (= codimension of stable manifold)
    c:        float   # self-similar spatial exponent
    gamma_u:  float   # velocity amplitude exponent
    gamma_om: float   # vorticity amplitude exponent
    gamma_S:  float   # strain-rate amplitude exponent
    gamma_p:  float   # pressure amplitude exponent

    @classmethod
    def empirical(cls, n: int) -> "EulerSingularityFamily":
        c0, dc = 0.44, 0.50
        c = c0 + n * dc
        kind: _Kind = "stable" if n == 0 else "unstable"
        return cls(
            label    = "S0" if n == 0 else f"U{n}",
            kind     = kind,
            order    = n,
            c        = c,
            gamma_u  = -(1.0 - c),
            gamma_om = -(2.0 - c),
            gamma_S  = -(3.0 - c),
            gamma_p  = -(2.0 - 2.0 * c),
        )

    @property
    def bkm_integrable(self) -> bool:
        """True if ∫‖ω‖_∞ dt converges (γ_ω > -1, i.e. c > 1)."""
        return self.c > 1.0

    def __str__(self) -> str:
        return (f"{self.label} ({self.kind}, n={self.order}): "
                f"c={self.c:.4f}  γ_u={self.gamma_u:+.4f}  "
                f"γ_ω={self.gamma_om:+.4f}  γ_S={self.gamma_S:+.4f}  "
                f"BKM-integrable={self.bkm_integrable}")


EULER3D_CATALOGUE: dict[str, EulerSingularityFamily] = {
    ("S0" if n == 0 else f"U{n}"): EulerSingularityFamily.empirical(n)
    for n in range(4)
}


# ═══════════════════════════════════════════════════════════════════════════
# Blow-up estimator  (BKM + self-similar power-law fit)
# ═══════════════════════════════════════════════════════════════════════════

class BlowupEstimate(NamedTuple):
    """
    On-the-fly singularity estimate from ‖ω‖_∞ and vortex-stretching history.

    Model:  ‖ω‖_∞(t) ≈ A · (T − t)^{γ_ω} = A · (T − t)^{-(2-c)}
            log ‖ω‖_∞ = log A + (c − 2) · log(T − t)

    Also checks the BKM integral accumulation rate as an independent estimator.
    """
    T_star:         float          # estimated blow-up time
    c_fit:          float          # fitted self-similar exponent c
    gamma_om_fit:   float          # fitted vorticity exponent  (= c − 2)
    bkm_integral:   float          # ∫₀ᵗ ‖ω‖_∞ dτ (should → ∞)
    vortex_stretch: float          # ‖ω · ∇u‖_∞  at latest step
    family_match:   Optional[str]  # closest catalogue entry
    residual:       float          # log-space fit RMS error
    confidence:     float          # heuristic ∈ [0, 1]


def _fit_blowup_euler3d(
    times:   list[float],
    max_om:  list[float],
    bkm_int: list[float],
    tol:     float = 0.05,
) -> Optional[BlowupEstimate]:
    """
    Grid-search T* that best linearises  log‖ω‖_∞  vs  log(T − t).
    Uses the last ≤ 20 diagnostic samples.
    """
    if len(times) < 10:
        return None
    t  = np.array(times[-20:],   dtype=np.float64)
    mo = np.array(max_om[-20:],  dtype=np.float64)
    bk = np.array(bkm_int[-20:], dtype=np.float64)

    if np.any(mo <= 0) or (mo[-1] / (mo[0] + 1e-14)) < 2.0:
        return None
    log_mo = np.log(mo)

    T_cands = t[-1] + np.logspace(-5, 1, 400)
    best_res = np.inf; best_T = T_cands[0]; best_c = 0.0

    for T in T_cands:
        log_dt = np.log(T - t)
        A = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            cf, _, _, _ = np.linalg.lstsq(A, log_mo, rcond=None)
        except np.linalg.LinAlgError:
            continue
        # slope = γ_ω = -(2-c)  →  c = 2 + slope
        c_cand = 2.0 + cf[1]
        if not (0.05 < c_cand < 3.0):
            continue
        res = float(np.sqrt(np.mean((log_mo - A @ cf) ** 2)))
        if res < best_res:
            best_res = res; best_T = T; best_c = c_cand

    if best_c < 0.05:
        return None

    family_match = min(
        EULER3D_CATALOGUE,
        key=lambda nm: abs(EULER3D_CATALOGUE[nm].c - best_c),
    )
    min_diff   = abs(EULER3D_CATALOGUE[family_match].c - best_c)
    confidence = max(0.0, 1.0 - best_res / 0.5) * (1.0 if min_diff < tol else 0.5)

    return BlowupEstimate(
        T_star         = best_T,
        c_fit          = best_c,
        gamma_om_fit   = best_c - 2.0,
        bkm_integral   = float(bk[-1]),
        vortex_stretch = float(mo[-1]),    # placeholder; updated by caller
        family_match   = family_match if min_diff < 0.20 else None,
        residual       = best_res,
        confidence     = confidence,
    )


# ═══════════════════════════════════════════════════════════════════════════
# Self-similar snapshot container  (Wang et al. 2025 "frozen limit")
# ═══════════════════════════════════════════════════════════════════════════

@dataclass
class SelfSimilarSnapshot:
    """
    Velocity and vorticity extracted in self-similar coordinates at t ≈ T.

    Rescaling (Hou–Luo geometry, z* = boundary stagnation point):
        ξ = (x − x*) / (T−t)^c
        η = (y − y*) / (T−t)^c
        ζ = (z − z*) / (T−t)^c
        Ũ = (T−t)^{−γ_u} u         velocity profile
        Ω̃ = (T−t)^{−γ_ω} ω         vorticity profile

    At a true self-similar blow-up, Ũ and Ω̃ converge to a fixed profile
    as t → T — the "frozen limit" targeted by the Wang et al. PINN.
    """
    t:       float
    xi:      np.ndarray        # (Nss,)
    zeta:    np.ndarray        # (Nss,)
    U_tilde: np.ndarray        # (Nss, Nss, 3)  rescaled velocity at y = y*
    Om_tilde:np.ndarray        # (Nss, Nss, 3)  rescaled vorticity at y = y*
    c:       float
    T_star:  float
    family:  Optional[str]


# ═══════════════════════════════════════════════════════════════════════════
# Spectral operators  (x,y periodic — Fourier; z wall-bounded — FD)
# ═══════════════════════════════════════════════════════════════════════════

class _SpectralOps:
    """
    Mixed spectral (x,y) / FD (z) operators for the 3-D Euler solver.

    Fourier in x, y:  standard 2-D DFT via numpy.fft.fft2
    FD in z:          4th-order centred, with one-sided near-walls

    Pressure solve:  The incompressibility constraint ∇·u = 0 becomes
        ∂p/∂x = -[u·∇u]_x + (1/dt)(u - u*)   (approximately)
    We use the Helmholtz–Hodge decomposition via a layer-by-layer spectral
    solve in (x,y) and tridiagonal solve in z (see _project in solver).

    Sobolev norms use Parseval in (x,y) and quadrature in z.
    """

    def __init__(self, Nx: int, Ny: int, Nz: int,
                 Lxy: float, Lz: float):
        self.Nx = Nx; self.Ny = Ny; self.Nz = Nz
        self.Lxy = Lxy; self.Lz = Lz
        self.dz = Lz / Nz

        kx = fftfreq(Nx, d=Lxy / (2 * np.pi * Nx))
        ky = fftfreq(Ny, d=Lxy / (2 * np.pi * Ny))
        KX2d, KY2d = np.meshgrid(kx, ky, indexing='ij')
        self.KX = KX2d[:, :, np.newaxis]
        self.KY = KY2d[:, :, np.newaxis]
        self.KXY2 = self.KX ** 2 + self.KY ** 2
        self.KXY2[0, 0, 0] = 1.0  # avoid division by zero

        # 2/3-rule dealiasing mask
        kmax_x = (Nx // 3) * (2 * np.pi / Lxy)
        kmax_y = (Ny // 3) * (2 * np.pi / Lxy)
        da2d = (np.abs(KX2d) < kmax_x) & (np.abs(KY2d) < kmax_y)
        self.DA = da2d[:, :, np.newaxis]

        # Sobolev weight  (1 + kx² + ky²)^{s/2}
        K2r = 1.0 + KX2d ** 2 + KY2d ** 2
        self.W_H1_2d = np.sqrt(K2r)
        self.W_H2_2d = K2r

    # ── Fourier transforms (x, y) ──────────────────────────────────────────

    def fwd_xy(self, f: np.ndarray) -> np.ndarray:
        return fft2(f, axes=(0, 1))

    def inv_xy(self, fh: np.ndarray) -> np.ndarray:
        return ifft2(fh, axes=(0, 1)).real

    # ── Spectral x, y derivatives ──────────────────────────────────────────

    def dx(self, fh: np.ndarray) -> np.ndarray:
        return self.inv_xy(1j * self.KX * fh)

    def dy(self, fh: np.ndarray) -> np.ndarray:
        return self.inv_xy(1j * self.KY * fh)

    def dz(self, f: np.ndarray) -> np.ndarray:
        return _fd4_dz(f, self.dz)

    def dz2(self, f: np.ndarray) -> np.ndarray:
        return _fd4_dz2(f, self.dz)

    # ── Sobolev norms ──────────────────────────────────────────────────────

    def sobolev_h1(self, f: np.ndarray) -> float:
        """‖f‖_{H^1} approximated via x,y Fourier + trapezoidal in z."""
        fh = self.fwd_xy(f)
        w  = (self.W_H1_2d[:, :, np.newaxis] * np.abs(fh)) ** 2
        return float(np.sqrt(w.sum() * self.dz))

    def sobolev_h2(self, f: np.ndarray) -> float:
        fh = self.fwd_xy(f)
        w  = (self.W_H2_2d[:, :, np.newaxis] * np.abs(fh)) ** 2
        return float(np.sqrt(w.sum() * self.dz))

    # ── Energy spectrum ────────────────────────────────────────────────────

    def energy_spectrum(
        self, u: np.ndarray, v: np.ndarray, w_vel: np.ndarray, nb: int = 64
    ) -> tuple[np.ndarray, np.ndarray, float]:
        """
        Shell-decomposed kinetic energy E(k) using only the (x,y) Fourier modes.
        Expected slope: -5/3 (Kolmogorov) for turbulent Euler flow.
        """
        uh = self.fwd_xy(u); vh = self.fwd_xy(v); wh = self.fwd_xy(w_vel)
        E_hat = 0.5 * (np.abs(uh) ** 2 + np.abs(vh) ** 2 + np.abs(wh) ** 2)
        K2d = np.sqrt(self.KX[:, :, 0] ** 2 + self.KY[:, :, 0] ** 2)
        E_2d = E_hat.mean(axis=2)
        bins = np.linspace(0, K2d.max(), nb + 1)
        k_sh = 0.5 * (bins[:-1] + bins[1:])
        E_k  = np.array([E_2d[(K2d >= bins[i]) & (K2d < bins[i+1])].sum()
                          for i in range(nb)])
        mask = (k_sh > 1.0) & (E_k > 0)
        slope = (float(np.polyfit(np.log(k_sh[mask]), np.log(E_k[mask]), 1)[0])
                 if mask.sum() > 4 else float("nan"))
        return k_sh, E_k, slope


# ═══════════════════════════════════════════════════════════════════════════
# Main solver
# ═══════════════════════════════════════════════════════════════════════════

class _Euler3DSolver:
    """
    3-D Incompressible Euler solver — spectral (x,y) + 4th-order FD (z).

    Governing equations:
        ∂u/∂t + (u · ∇)u + ∇p = 0
        ∇ · u = 0
        u_z = 0  at  z = 0,  z = Lz    (Hou–Luo cylinder walls)

    Vorticity equation (for diagnostics and BKM tracking):
        ∂ω/∂t + (u · ∇)ω = (ω · ∇)u   (vortex stretching = RHS)

    Parameters
    ----------
    Nx, Ny, Nz         : grid dimensions (must be even)
    Lxy                : side length of the periodic (x,y) domain
    Lz                 : height of the bounded z domain
    dt                 : initial time step (adaptively controlled)
    rtol, atol         : embedded RK4(5) error tolerances
    family             : singularity family seed "S0"|"U1"|"U2"|"U3"|None
    nu_hyp             : hyperviscosity coefficient ν (−Δ)^4 (0 = inviscid)
    blowup_threshold   : ‖ω‖_∞ threshold activating near-singularity mode
    ic_type            : "hou_luo"|"tgv"|"default"

    Key diagnostics (per recorded step)
    ─────────────────────────────────────────────────────────────
    max_omega       : ‖ω‖_∞                (BKM growth rate proxy)
    bkm             : ∫₀ᵗ ‖ω‖_∞ dτ         (BKM criterion integral)
    vortex_stretch  : max |(ω·∇)u|          (dominant blow-up driver)
    energy          : ½ ∫ |u|² dV           (conserved for ideal Euler)
    helicity        : ∫ u · ω dV            (secondary conserved quantity)
    h1_u, h2_u      : Sobolev norms of u    (regularity measure)
    palinstrophy    : ½ ∫ |∇ω|² dV         (upper bound on enstrophy growth)

    References
    ----------
    [1] Wang et al. (2025) arXiv:2509.14185
    [2] Chen & Hou (2022) Ann. Math. 196(1)
    [3] Hou & Luo (2013) Multiscale Model. Simul.
    [4] Beale, Kato & Majda (1984) Commun. Math. Phys.
    [5] Hairer & Wanner (1996) Solving ODEs II
    """

    def __init__(
        self,
        Nx: int = 64, Ny: int = 64, Nz: int = 64,
        Lxy: float = 2 * np.pi,
        Lz: float  = np.pi,
        dt:  float  = 1e-3,
        rtol: float = 1e-6,
        atol: float = 1e-9,
        family: Optional[str] = None,
        nu_hyp: float = 0.0,
        blowup_threshold: float = 1e2,
        ic_type: str = "default",
    ):
        for N, name in [(Nx, "Nx"), (Ny, "Ny"), (Nz, "Nz")]:
            if N % 2 != 0:
                raise ValueError(f"{name} must be even.")
        self.Nx = Nx; self.Ny = Ny; self.Nz = Nz
        self.Lxy = Lxy; self.Lz = Lz
        self.dz = Lz / Nz
        self.dt = dt; self.rtol = rtol; self.atol = atol
        self.nu_hyp = nu_hyp
        self.blowup_threshold = blowup_threshold
        self.t = 0.0; self._step = 0
        self._blown_up = False; self._blowup_time: Optional[float] = None
        self._near_blowup = False

        x   = np.linspace(0, Lxy, Nx, endpoint=False)
        y   = np.linspace(0, Lxy, Ny, endpoint=False)
        z   = np.linspace(0, Lz,  Nz, endpoint=False)
        self.X, self.Y, self.Z = np.meshgrid(x, y, z, indexing='ij')

        self._sp = _SpectralOps(Nx, Ny, Nz, Lxy, Lz)

        if family is not None and family not in EULER3D_CATALOGUE:
            raise ValueError(f"Unknown family '{family}'. "
                             f"Choose from {list(EULER3D_CATALOGUE)}.")
        self._family = EULER3D_CATALOGUE[family] if family else None

        # ── Initial conditions ─────────────────────────────────────────────
        self.u, self.v, self.w = self._make_ic(ic_type, self._family)
        self._project()

        # ── Diagnostics ────────────────────────────────────────────────────
        self.times:         list[float] = []
        self.max_omega:     list[float] = []
        self.bkm:           list[float] = []
        self.vortex_stretch:list[float] = []
        self.energy:        list[float] = []
        self.helicity:      list[float] = []
        self.h1_u:          list[float] = []
        self.h2_u:          list[float] = []
        self.palinstrophy:  list[float] = []
        self.dt_history:    list[float] = []
        self._bkm_acc: float = 0.0

        # ── Blow-up tracking ───────────────────────────────────────────────
        self.blowup_estimate: Optional[BlowupEstimate] = None
        self.self_similar_snapshots: list[SelfSimilarSnapshot] = []
        self._k_prev: Optional[tuple] = None  # FSAL cache

    # ═══════════════════════════════════════════════════════════════════════
    # Initial conditions
    # ═══════════════════════════════════════════════════════════════════════

    def _make_ic(
        self,
        ic_type: str,
        fam: Optional[EulerSingularityFamily],
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Velocity field (u, v, w) tailored to the singularity family.

        Hou–Luo geometry ("hou_luo"):
        ─────────────────────────────
        Opposing azimuthal jets on the (x,y) plane meet at z* = Lz/2.
        The stagnation line along the boundary generates a saddle point
        where vorticity is amplified by mutual induction (Hou & Luo 2013).

        Taylor–Green ("tgv"):
        ─────────────────────
        The 3-D TGV is an exact Euler solution at t=0 that develops
        vortex tubes and is widely used for blow-up benchmarking.

        Unstable-family seeds (Wang et al. 2025):
        ───────────────────────────────────────────
        Higher-order unstable manifolds require superimposing n additional
        higher-harmonic perturbations, each with amplitude scaled by the
        self-similar length (T−t₀)^c, pushing the IC onto the codimension-n
        unstable manifold with the necessary "infinite precision" tuning.
        """
        X, Y, Z = self.X, self.Y, self.Z
        Lxy, Lz = self.Lxy, self.Lz
        A = 1.0
        n = fam.order if fam else 0
        c = fam.c     if fam else 0.44

        if ic_type == "tgv":
            u = A * np.sin(X) * np.cos(Y) * np.cos(Z)
            v =-A * np.cos(X) * np.sin(Y) * np.cos(Z)
            w = np.zeros_like(u)
            # Standard TGV perturbation for vortex generation
            u += 0.05 * np.sin(2*X) * np.cos(Y) * np.cos(2*Z)
            v += 0.05 * np.cos(X) * np.sin(2*Y) * np.cos(2*Z)

        elif ic_type == "hou_luo" or (fam is not None and ic_type == "default"):
            # Hou–Luo: counter-rotating swirl localised at z* = Lz/2
            z_c   = Lz / 2.0
            sigma = Lz * 0.08 * (1.0 + 0.25 * c)
            g     = np.exp(-((Z - z_c) ** 2) / (2 * sigma ** 2))
            u     = A * np.sin(X) * np.cos(Y) * g
            v     =-A * np.cos(X) * np.sin(Y) * g
            # w satisfies ∇·u=0 approximately; projection fixes it exactly
            w     = (2 * A / 3) * np.cos(X) * np.cos(Y) * np.sin(Z) * g * 0.2

            # Unstable-manifold higher harmonics (Wang et al. 2025 §5)
            for m in range(1, n + 1):
                eps_m = 0.03 / m ** 1.5
                phi   = 2 * np.pi * m / (n + 1)
                gm    = np.exp(-((Z - z_c) ** 2) / sigma ** 2)
                u += eps_m * np.sin((m+1)*X + phi) * np.cos(Y) * gm
                v += eps_m * np.cos(X) * np.sin((m+1)*Y + phi) * gm

        else:
            # Original default: Taylor–Green variant
            u = A * np.sin(X) * np.cos(Y) * np.cos(Z)
            v =-A * np.cos(X) * np.sin(Y) * np.cos(Z)
            w = (2*A/3) * np.cos(X) * np.cos(Y) * np.sin(Z)
            u += 0.05 * np.sin(2*X) * np.cos(Y) * np.cos(2*Z)
            v += 0.05 * np.cos(X) * np.sin(2*Y) * np.cos(2*Z)

        # Enforce wall boundary condition exactly
        w[:, :, 0] = 0.0; w[:, :, -1] = 0.0
        return u, v, w



    def _project(self) -> None:
        """
        Project (u, v, w) onto the divergence-free subspace via the
        Helmholtz–Hodge decomposition.

        Spectral in (x,y): for each Fourier mode (kx, ky) solve
            (kx² + ky² − ∂²/∂z²) φ̂ = div̂(u,v,w)
        with φ = 0 at z = 0, Lz (Dirichlet, consistent with u_z wall BC).
        Then  u ← u − ∇φ.

        The z-Laplacian is discretised with 4th-order FD, giving a
        banded tridiagonal system solved by np.linalg.solve per mode.
        (For production use, replace with a fast tridiagonal sweeper.)
        """
        sp = self._sp
        uh = sp.fwd_xy(self.u) * sp.DA
        vh = sp.fwd_xy(self.v) * sp.DA
        wh = sp.fwd_xy(self.w) * sp.DA

        # Divergence in spectral-x,y / physical-z
        div_u = sp.inv_xy(1j * sp.KX * uh)
        div_v = sp.inv_xy(1j * sp.KY * vh)
        div_w = _fd4_dz(self.w, self.dz)
        div_  = div_u + div_v + div_w     # (Nx, Ny, Nz)

        # For each horizontal mode: solve  (kxy² − d²/dz²) φ̂_{kx,ky}(z) = div
        # Use a simplified one-pass correction (Chorin projection style)
        div_h = sp.fwd_xy(div_)

        # Build FD Laplacian matrix for z (Nz × Nz), Dirichlet at walls
        dz = self.dz
        Nz = self.Nz
        diag_main = np.full(Nz, -2.0 / dz**2)
        diag_off  = np.full(Nz-1, 1.0 / dz**2)
        Lz_mat = (np.diag(diag_main) + np.diag(diag_off, 1)
                  + np.diag(diag_off, -1)).astype(np.complex128)
        Lz_mat[0, :]  = 0; Lz_mat[0, 0]  = 1.0   # Dirichlet φ=0 at z=0
        Lz_mat[-1, :] = 0; Lz_mat[-1,-1] = 1.0   # Dirichlet φ=0 at z=Lz

        # Solve only for non-zero modes (skip DC for robustness)
        phi_h = np.zeros_like(div_h)
        kxy2_flat = (sp.KX[:, :, 0] ** 2 + sp.KY[:, :, 0] ** 2)

        for i in range(self.Nx):
            for j in range(self.Ny):
                if i == 0 and j == 0:
                    continue
                k2 = float(kxy2_flat[i, j])
                L  = Lz_mat.copy()
                # Add kxy² to interior rows
                for r in range(1, Nz - 1):
                    L[r, r] += k2
                rhs = div_h[i, j, :].copy()
                rhs[0] = 0.0; rhs[-1] = 0.0
                phi_h[i, j, :] = np.linalg.solve(L, rhs)

        phi = sp.inv_xy(phi_h)
        self.u -= sp.dx(sp.fwd_xy(phi) * sp.DA)
        self.v -= sp.dy(sp.fwd_xy(phi) * sp.DA)
        self.w -= _fd4_dz(phi, self.dz)
        self.w[:, :, 0]  = 0.0
        self.w[:, :, -1] = 0.0



    def _vorticity(
        self, u: np.ndarray, v: np.ndarray, w: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        ω = ∇ × u:
          ω_x = ∂w/∂y − ∂v/∂z
          ω_y = ∂u/∂z − ∂w/∂x
          ω_z = ∂v/∂x − ∂u/∂y
        """
        sp = self._sp
        uh = sp.fwd_xy(u) * sp.DA
        vh = sp.fwd_xy(v) * sp.DA
        wh = sp.fwd_xy(w) * sp.DA

        om_x = sp.inv_xy(1j * sp.KY * wh) - _fd4_dz(v, self.dz)
        om_y = _fd4_dz(u, self.dz)        - sp.inv_xy(1j * sp.KX * wh)
        om_z = sp.inv_xy(1j * sp.KX * vh) - sp.inv_xy(1j * sp.KY * uh)
        return om_x, om_y, om_z



    def _rhs(
        self, u: np.ndarray, v: np.ndarray, w: np.ndarray
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
        """
        Compute (∂u/∂t, ∂v/∂t, ∂w/∂t) for the incompressible Euler equations.

        Algorithm
        ─────────
        1. Spectral derivatives → advection terms (u·∇)u  (pseudospectral,
           dealised with 2/3 rule — Orszag 1971)
        2. Pressure gradient eliminated via projection after each RK stage
           (Chorin 1968 operator-splitting philosophy; here applied to RHS
           directly using the rotational form of advection)
        3. Rotational form:  (u·∇)u = ω×u + ∇(|u|²/2)
           (reduces aliasing error and allows pressure elimination via
           the Euler equation in its Lamb form)
        4. Optional hyperviscosity: −ν(−Δ)^4 u

        Note: pressure gradient is recovered from ∇p = -(u·∇)u projected
        onto the incompressible subspace.  The solenoidal RHS is returned
        directly (equivalent to applying the Leray projector).
        """
        sp = self._sp
        uh = sp.fwd_xy(u) * sp.DA
        vh = sp.fwd_xy(v) * sp.DA
        wh = sp.fwd_xy(w) * sp.DA

        # Velocity gradients  (spectral x,y; FD z)
        du_dx = sp.inv_xy(1j * sp.KX * uh)
        du_dy = sp.inv_xy(1j * sp.KY * uh)
        du_dz = _fd4_dz(u, self.dz)

        dv_dx = sp.inv_xy(1j * sp.KX * vh)
        dv_dy = sp.inv_xy(1j * sp.KY * vh)
        dv_dz = _fd4_dz(v, self.dz)

        dw_dx = sp.inv_xy(1j * sp.KX * wh)
        dw_dy = sp.inv_xy(1j * sp.KY * wh)
        dw_dz = _fd4_dz(w, self.dz)

        # Advection  (u·∇)u  — simple form (aliasing handled by 2/3 rule)
        adv_u = u * du_dx + v * du_dy + w * du_dz
        adv_v = u * dv_dx + v * dv_dy + w * dv_dz
        adv_w = u * dw_dx + v * dw_dy + w * dw_dz

        # Dealias advection products
        au_h = sp.fwd_xy(adv_u) * sp.DA
        av_h = sp.fwd_xy(adv_v) * sp.DA
        aw_h = sp.fwd_xy(adv_w) * sp.DA

        # Apply Leray projector  P[f] = f − ∇(∇⁻¹·f)  in x,y Fourier space
        # (z component handled by wall BCs forcing w=0 at z=0,Lz)
        div_a_h = 1j * sp.KX * au_h + 1j * sp.KY * av_h
        # Note: we omit d_aw_h/dz term for the horizontal projection
        # (split-operator: horizontal Leray then vertical FD correction)
        au_h -= 1j * sp.KX * div_a_h / sp.KXY2
        av_h -= 1j * sp.KY * div_a_h / sp.KXY2

        rhs_u = -sp.inv_xy(au_h)
        rhs_v = -sp.inv_xy(av_h)
        rhs_w = -sp.inv_xy(aw_h)

        # Enforce wall BC on RHS
        rhs_w[:, :, 0] = 0.0; rhs_w[:, :, -1] = 0.0

        # Hyperviscosity
        if self.nu_hyp > 0.0:
            K2h = sp.KXY2
            rhs_u -= self.nu_hyp * sp.inv_xy((K2h ** 4) * sp.fwd_xy(u) * sp.DA)
            rhs_v -= self.nu_hyp * sp.inv_xy((K2h ** 4) * sp.fwd_xy(v) * sp.DA)
            rhs_w -= self.nu_hyp * sp.inv_xy((K2h ** 4) * sp.fwd_xy(w) * sp.DA)

        return rhs_u, rhs_v, rhs_w



    def _dp45_step(
        self,
        u: np.ndarray, v: np.ndarray, w: np.ndarray,
        dt: float,
    ) -> tuple[np.ndarray, np.ndarray, np.ndarray,
               np.ndarray, np.ndarray, np.ndarray, float]:
        """
        One embedded RK4(5) Dormand–Prince step for the (u, v, w) system.
        Returns  (u5, v5, w5, u4, v4, w4, err_scalar).

        FSAL reuses the last stage k₇ as k₁ of the next step, saving one
        expensive RHS evaluation per accepted step.
        """
        A, B4, B5, E = _DP_A, _DP_B4, _DP_B5, _DP_E
        ku = np.zeros((7, *u.shape)); kv = np.zeros_like(ku); kw = np.zeros_like(ku)

        if self._k_prev is not None:
            ku[0], kv[0], kw[0] = self._k_prev
        else:
            ku[0], kv[0], kw[0] = self._rhs(u, v, w)

        for i in range(1, 7):
            u_i = u + dt * sum(A[i, j] * ku[j] for j in range(i))
            v_i = v + dt * sum(A[i, j] * kv[j] for j in range(i))
            w_i = w + dt * sum(A[i, j] * kw[j] for j in range(i))
            ku[i], kv[i], kw[i] = self._rhs(u_i, v_i, w_i)

        u5 = u + dt * sum(B5[i] * ku[i] for i in range(7))
        v5 = v + dt * sum(B5[i] * kv[i] for i in range(7))
        w5 = w + dt * sum(B5[i] * kw[i] for i in range(7))
        u4 = u + dt * sum(B4[i] * ku[i] for i in range(7))
        v4 = v + dt * sum(B4[i] * kv[i] for i in range(7))
        w4 = w + dt * sum(B4[i] * kw[i] for i in range(7))

        self._k_prev = (ku[6], kv[6], kw[6])

        eu = dt * sum(E[i] * ku[i] for i in range(7))
        ev = dt * sum(E[i] * kv[i] for i in range(7))
        ew = dt * sum(E[i] * kw[i] for i in range(7))

        sc_u = self.atol + self.rtol * np.maximum(np.abs(u), np.abs(u5))
        sc_v = self.atol + self.rtol * np.maximum(np.abs(v), np.abs(v5))
        sc_w = self.atol + self.rtol * np.maximum(np.abs(w), np.abs(w5))
        err = float(np.sqrt((np.mean((eu/sc_u)**2) + np.mean((ev/sc_v)**2)
                              + np.mean((ew/sc_w)**2)) / 3.0))
        return u5, v5, w5, u4, v4, w4, err




    def _adapt_dt(self, err: float, dt: float) -> float:
        factor = min(5.0, max(0.1, 0.9 * (err ** (-0.2)) if err > 0 else 5.0))
        new_dt = dt * factor
        if self.blowup_estimate is not None:
            margin = max(1e-8, 0.02 * (self.blowup_estimate.T_star - self.t))
            new_dt = min(new_dt, margin)
        return float(np.clip(new_dt, 1e-10, 0.02))



    def _record_diagnostics(
        self, u: np.ndarray, v: np.ndarray, w: np.ndarray
    ) -> None:
        """
        Record all tracked quantities.

        Key quantities:
        ───────────────
        BKM integral  ∫‖ω‖_∞ dτ:  trapezoidal accumulation.
        Vortex stretching: ‖(ω·∇)u‖_∞ — the term that drives the BKM integrand.
            By the chain rule  d‖ω‖_∞/dt ≤ ‖(ω·∇)u‖  (Beale–Kato–Majda 1984).
        Helicity  H = ∫u·ω dV: second rugged invariant of 3-D Euler.
            If helicity grows, topological reconnection is occurring —
            a known precursor to singularity formation (Moffatt 1969).
        Palinstrophy  P = ½∫|∇ω|² dV: controls enstrophy growth rate via
            d/dt ½∫ω² ≤ ‖S‖_F · ‖∇ω‖² (Hou & Luo 2013 diagnostic).
        """
        sp = self._sp
        om_x, om_y, om_z = self._vorticity(u, v, w)
        om_mag = np.sqrt(om_x**2 + om_y**2 + om_z**2)
        max_om = float(om_mag.max())

        # BKM integral (trapezoidal)
        if self.times:
            dt_rec = self.t - self.times[-1]
            self._bkm_acc += 0.5 * (self.max_omega[-1] + max_om) * dt_rec

        # Vortex stretching |(ω·∇)u|
        uh = sp.fwd_xy(u) * sp.DA
        vh = sp.fwd_xy(v) * sp.DA
        du_dx = sp.inv_xy(1j * sp.KX * uh); du_dy = sp.inv_xy(1j * sp.KY * uh)
        dv_dx = sp.inv_xy(1j * sp.KX * vh); dv_dy = sp.inv_xy(1j * sp.KY * vh)
        du_dz = _fd4_dz(u, self.dz); dv_dz = _fd4_dz(v, self.dz)
        dw_dx = sp.inv_xy(1j * sp.KX * sp.fwd_xy(w) * sp.DA)
        dw_dy = sp.inv_xy(1j * sp.KY * sp.fwd_xy(w) * sp.DA)
        dw_dz = _fd4_dz(w, self.dz)

        Sx = om_x*du_dx + om_y*du_dy + om_z*du_dz
        Sy = om_x*dv_dx + om_y*dv_dy + om_z*dv_dz
        Sz = om_x*dw_dx + om_y*dw_dy + om_z*dw_dz
        vs = float(np.sqrt(Sx**2 + Sy**2 + Sz**2).max())

        # Kinetic energy ½∫|u|²
        dV = (self.Lxy/self.Nx) * (self.Lxy/self.Ny) * self.dz
        energy = float(0.5 * (u**2 + v**2 + w**2).sum() * dV)

        # Helicity  ∫u·ω
        helicity = float((u*om_x + v*om_y + w*om_z).sum() * dV)

        # H¹ Sobolev norms
        h1_u = sp.sobolev_h1(u)

        # Palinstrophy ½∫|∇ω|²
        om_xh = sp.fwd_xy(om_x) * sp.DA
        om_yh = sp.fwd_xy(om_y) * sp.DA
        om_zh = sp.fwd_xy(om_z) * sp.DA
        gox = sp.inv_xy(1j*sp.KX*om_xh); goy = sp.inv_xy(1j*sp.KY*om_yh)
        pali = float(0.5 * (gox**2 + goy**2).sum() * dV)

        self.times.append(self.t)
        self.max_omega.append(max_om)
        self.bkm.append(self._bkm_acc)
        self.vortex_stretch.append(vs)
        self.energy.append(energy)
        self.helicity.append(helicity)
        self.h1_u.append(h1_u)
        self.palinstrophy.append(pali)

        # Update blow-up estimate every 20 recorded steps
        if len(self.times) % 20 == 0:
            est = _fit_blowup_euler3d(self.times, self.max_omega, self.bkm)
            if est is not None:
                # Inject latest vortex-stretch value
                self.blowup_estimate = est._replace(vortex_stretch=vs)

        if max_om > self.blowup_threshold:
            self._near_blowup = True



    def _extract_self_similar_snapshot(
        self, fam: EulerSingularityFamily
    ) -> Optional[SelfSimilarSnapshot]:
        """
        Zoom into the developing singularity at (x*, y*, z*) and rescale
        to self-similar coordinates, producing the "frozen profile" that
        the Wang et al. (2025) PINN converges to directly.

        In the Hou–Luo geometry, z* = Lz/2 (stagnation boundary),
        x* = location of maximum |ω| at z = z*.
        """
        if self.blowup_estimate is None:
            return None
        T   = self.blowup_estimate.T_star
        tau = T - self.t
        if tau < 1e-10:
            return None

        c = fam.c
        sp = self._sp
        om_x, om_y, om_z = self._vorticity(self.u, self.v, self.w)
        om_mag = np.sqrt(om_x**2 + om_y**2 + om_z**2)

        # Locate blow-up point in the Hou–Luo z-strip
        iz_strip = slice(self.Nz // 3, 2 * self.Nz // 3)
        sub = om_mag[:, :, iz_strip]
        flat = np.argmax(sub)
        ix_s = flat // (sub.shape[1] * sub.shape[2])
        iy_s = (flat // sub.shape[2]) % sub.shape[1]
        iz_s = flat % sub.shape[2] + self.Nz // 3

        x_star = self.X[ix_s, 0, 0]
        z_star = self.Z[0, 0, iz_s]

        Nss = 32; R = 4.0
        xi   = np.linspace(-R, R, Nss)
        zeta = np.linspace(-R, R, Nss)
        XI, ZETA = np.meshgrid(xi, zeta, indexing='ij')

        X_phys = np.mod(x_star + XI   * tau**c, self.Lxy)
        Z_phys = np.clip(z_star + ZETA * tau**c, 0, self.Lz)
        Y_mid  = self.Lxy / 2.0

        dx = self.Lxy / self.Nx; dz_g = self.Lz / self.Nz
        ix = (X_phys / dx).astype(int) % self.Nx
        iz = np.clip((Z_phys / dz_g).astype(int), 0, self.Nz - 1)
        iy_m = int(Y_mid / (self.Lxy / self.Ny)) % self.Ny

        def extract(f): return f[ix, iy_m, iz]

        U_t = np.stack([extract(self.u), extract(self.v), extract(self.w)],
                       axis=-1) * tau ** (-fam.gamma_u)
        Om_t = np.stack([extract(om_x), extract(om_y), extract(om_z)],
                        axis=-1) * tau ** (-fam.gamma_om)

        return SelfSimilarSnapshot(
            t=self.t, xi=xi, zeta=zeta,
            U_tilde=U_t, Om_tilde=Om_t,
            c=c, T_star=T, family=fam.label,
        )



    def linearised_growth_rate(self, epsilon: float = 1e-5) -> float:
        """
        Dominant Lyapunov exponent of the linearised 3-D Euler operator.
        Positive value → exponentially unstable → candidate for unstable
        singularity as identified by Wang et al. (2025).
        """
        rng = np.random.default_rng(0)
        du = rng.standard_normal(self.u.shape)
        dv = rng.standard_normal(self.v.shape)
        dw = rng.standard_normal(self.w.shape)
        nrm = np.sqrt(np.linalg.norm(du)**2 + np.linalg.norm(dv)**2
                      + np.linalg.norm(dw)**2)
        du *= epsilon / nrm; dv *= epsilon / nrm; dw *= epsilon / nrm

        f0 = self._rhs(self.u, self.v, self.w)
        fp = self._rhs(self.u+du, self.v+dv, self.w+dw)
        Lf = tuple((fp[i] - f0[i]) / epsilon for i in range(3))
        Ln = np.sqrt(sum(np.linalg.norm(Lf[i])**2 for i in range(3)))
        dn = np.sqrt(np.linalg.norm(du)**2 + np.linalg.norm(dv)**2
                     + np.linalg.norm(dw)**2)
        return float(Ln / dn)

    def bkm_criterion_met(self, threshold: float = 50.0) -> bool:
        """
        Heuristic Beale–Kato–Majda criterion check.
        Returns True if BKM integral is large AND ‖ω‖_∞ is accelerating.
        """
        if len(self.max_omega) < 3:
            return False
        return bool(
            self._bkm_acc > threshold
            and self.max_omega[-1] > self.max_omega[-2] > self.max_omega[-3]
        )

    def energy_conservation_error(self) -> float:
        """
        Relative drift in kinetic energy from initial value.
        For ideal (inviscid, no hyperviscosity) 3-D Euler this should be ≈ 0.
        """
        if len(self.energy) < 2:
            return 0.0
        return abs(self.energy[-1] - self.energy[0]) / (abs(self.energy[0]) + 1e-14)



    def step(self, n_steps: int = 1, record_every: int = 1) -> None:
        """Advance the flow by n_steps adaptive Dormand–Prince steps."""
        for _ in range(n_steps):
            for attempt in range(20):
                u5, v5, w5, u4, v4, w4, err = self._dp45_step(
                    self.u, self.v, self.w, self.dt)
                if err <= 1.0 or self.dt < 1e-10:
                    self.u = u5; self.v = v5; self.w = w5
                    self.w[:, :, 0] = 0.0; self.w[:, :, -1] = 0.0
                    self.t      += self.dt
                    self._step  += 1
                    self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    self.dt *= max(0.1, 0.9 * err ** (-0.2))
                    self._k_prev = None
            else:
                warnings.warn(f"Step at t={self.t:.5f} failed; continuing.",
                              RuntimeWarning)

            if self._step % record_every == 0:
                self._record_diagnostics(self.u, self.v, self.w)

            if self._near_blowup and self.blowup_estimate is not None:
                last_t = (self.self_similar_snapshots[-1].t
                          if self.self_similar_snapshots else -np.inf)
                if self.t - last_t > 1e-5:
                    fl   = (self.blowup_estimate.family_match
                            or (self._family.label if self._family else "S0"))
                    fam  = EULER3D_CATALOGUE.get(fl, EulerSingularityFamily.empirical(0))
                    snap = self._extract_self_similar_snapshot(fam)
                    if snap is not None:
                        self.self_similar_snapshots.append(snap)

    def run_until(self, T_end: float, record_every: int = 10,
                  max_steps: int = 1_000_000) -> None:
        """Integrate to T_end or until BKM blow-up is confirmed."""
        steps = 0
        while self.t < T_end and steps < max_steps:
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(1, record_every)
            steps += 1
            if self._near_blowup and self.blowup_estimate is not None:
                est = self.blowup_estimate
                if self.t >= est.T_star - 1e-6:
                    self._blown_up = True
                    self._blowup_time = self.t
                    print(
                        f"[Euler3DSolver] Singularity at t ≈ {self.t:.6f}  "
                        f"T* = {est.T_star:.6f}  c_fit = {est.c_fit:.4f}  "
                        f"family = {est.family_match or '?'}  "
                        f"‖ω‖_∞ = {self.max_omega[-1]:.3e}  "
                        f"BKM = {self._bkm_acc:.4f}"
                    )
                    break



    def spectral_energy_budget(self) -> dict:
        """
        Shell-decomposed kinetic energy E(k).
        Expected Kolmogorov slope: −5/3.
        Steeper slope near blow-up signals vortex-tube formation
        (Hou & Luo 2013: exponential spectral decay collapses near T*).
        """
        k_sh, E_k, slope = self._sp.energy_spectrum(self.u, self.v, self.w)
        return {"k_shells": k_sh, "E_k": E_k, "slope": slope}



    def summary(self) -> str:
        lines = [
            "=" * 70,
            " _Euler3DSolver  —  3-D Incompressible Euler",
            "=" * 70,
            f"  Grid          : {self.Nx}×{self.Ny}×{self.Nz}   "
            f"Lxy = {self.Lxy:.4f},  Lz = {self.Lz:.4f}",
            f"  Time          : t = {self.t:.6f}",
            f"  Steps taken   : {self._step}",
            f"  Current dt    : {self.dt:.2e}",
        ]
        if self.max_omega:
            lines += [
                f"  ‖ω‖_∞         : {self.max_omega[-1]:.4e}",
                f"  BKM integral  : {self._bkm_acc:.4e}",
                f"  Vortex stretch: {self.vortex_stretch[-1]:.4e}",
                f"  Kinetic E     : {self.energy[-1]:.4e}",
                f"  Helicity      : {self.helicity[-1]:.4e}",
                f"  H¹ ‖u‖       : {self.h1_u[-1]:.4e}",
                f"  Palinstrophy  : {self.palinstrophy[-1]:.4e}",
                f"  E conserv. Δ  : {self.energy_conservation_error():.4e}",
                f"  BKM met?      : {self.bkm_criterion_met()}",
                f"  Blown up?     : {self._blown_up}"
                + (f"  (t* ≈ {self._blowup_time:.6f})" if self._blown_up else ""),
            ]
        if self.blowup_estimate:
            est = self.blowup_estimate
            lines += [
                "",
                "  ── Blow-up estimate (online power-law fit) ──",
                f"  T*            : {est.T_star:.6f}",
                f"  c_fit         : {est.c_fit:.4f}",
                f"  γ_ω fit       : {est.gamma_om_fit:.4f}",
                f"  Family match  : {est.family_match or 'unknown'}",
                f"  Confidence    : {est.confidence:.2f}",
                f"  Fit residual  : {est.residual:.4e}",
            ]
        if self._family:
            f = self._family
            lines += [
                "",
                "  ── Singularity family (Wang et al. 2025) ──",
                f"  {f}",
                f"  γ_u = {f.gamma_u:+.4f}   u  ~ (T−t)^{{γ_u}}",
                f"  γ_ω = {f.gamma_om:+.4f}   ω  ~ (T−t)^{{γ_ω}}",
                f"  γ_S = {f.gamma_S:+.4f}   S  ~ (T−t)^{{γ_S}}  (strain)",
                f"  γ_p = {f.gamma_p:+.4f}   p  ~ (T−t)^{{γ_p}}",
                f"  BKM integrable: {f.bkm_integrable}  "
                f"(True → unstable, requires ∞-precision IC)",
            ]
        lines.append("=" * 70)
        return "\n".join(lines)

    def __repr__(self) -> str:
        return (f"_Euler3DSolver(N={self.Nx}×{self.Ny}×{self.Nz}, "
                f"t={self.t:.4f}, dt={self.dt:.2e}, "
                f"family={self._family.label if self._family else None})")




def make_euler3d_solver(
    family:  str = "S0",
    Nx: int = 64, Ny: int = 64, Nz: int = 64,
    ic_type: str = "hou_luo",
    **kwargs,
) -> _Euler3DSolver:
    """
    Seeded constructor for a specific 3-D Euler blow-up family.

    Parameters
    ----------
    family  : "S0" (Hou–Luo, proved Chen & Hou 2022)
              "U1"|"U2"|"U3" (Wang et al. 2025 unstable families)
    Nx,Ny,Nz : grid resolution
    ic_type : "hou_luo" | "tgv" | "default"

    Example
    ───────
    >>> solver = make_euler3d_solver("U1", Nx=32, Ny=32, Nz=32)
    >>> solver.run_until(2.0, record_every=5)
    >>> print(solver.summary())
    """
    return _Euler3DSolver(Nx=Nx, Ny=Ny, Nz=Nz,
                          family=family, ic_type=ic_type, **kwargs)

In [ ]:
!pip install shimmy>=2.0
!apt-get update && apt-get install swig cmake
!pip install box2d-py
!pip install "stable-baselines3[extra]>=2.0.0a4"

In [ ]:
# @title

class BlackHoleSingularity:
    """
    Schwarzschild black hole: metric, geodesics, effective potential,
    Hawking temperature, entropy, and Penrose-diagram coordinates.

    ds² = -(1 - r_s/r) c² dt²  +  (1 - r_s/r)⁻¹ dr²  +  r² dΩ²
    """

    def __init__(self, mass_solar: float = 10.0):
        self.M        = mass_solar * M_SUN          # [kg]
        self.r_s      = 2.0 * G_N * self.M / C_L**2  # Schwarzschild radius [m]
        self.T_H      = (HBAR * C_L**3) / (8.0 * np.pi * G_N * self.M * K_B)
        self.S_BH     = (4.0 * np.pi * G_N * self.M**2) / (HBAR * C_L)
        self._label   = f"BH  M={mass_solar} M☉  r_s={self.r_s:.3e} m"

    # ── Effective potential (timelike geodesic) ───────────────────────────
    def effective_potential(self, r: np.ndarray, L: float = 1.0,
                            mu: float = 1.0) -> np.ndarray:
        """V_eff = -GM/r + L²/(2r²) - GM L²/(r³ c²)  (GR correction)."""
        r_safe = np.where(r > 1e-3 * self.r_s, r, np.nan)
        V  = -G_N * self.M * mu / r_safe
        V += L**2 / (2.0 * r_safe**2)
        V -= G_N * self.M * L**2 / (r_safe**3 * C_L**2)     # GR correction
        return V

    # ── Geodesic ODE (equatorial, r-φ plane) ─────────────────────────────
    def _geodesic_rhs(self, tau: float, y: np.ndarray,
                      E: float, L: float) -> list:
        r, rdot, phi = y
        r = max(r, 1.01 * self.r_s)
        f        = 1.0 - self.r_s / r
        rdot2    = E**2 - f * (C_L**2 + L**2 / r**2)
        rddot    = (-G_N * self.M / r**2
                    + L**2 / r**3
                    - 1.5 * G_N * self.M * L**2 / (r**4 * C_L**2))
        phidot   = L / r**2
        return [rdot, rddot, phidot]

    def integrate_geodesic(self, r0: float = None, phi0: float = 0.0,
                           E: float = 0.95, L: float = None,
                           tau_max: float = None) -> dict:
        if r0 is None:
            r0 = 10.0 * self.r_s
        if L is None:
            L = 3.5 * self.r_s * C_L
        if tau_max is None:
            tau_max = 40.0 * self.r_s / C_L

        rdot0    = np.sqrt(max(E**2 - (1 - self.r_s / r0)
                               * (C_L**2 + L**2 / r0**2), 0.0))
        y0       = [r0, -rdot0, phi0]

        def event_horizon(tau, y, E, L):
            return y[0] - 1.05 * self.r_s
        event_horizon.terminal  = True
        event_horizon.direction = -1

        sol = solve_ivp(
            self._geodesic_rhs, [0, tau_max], y0,
            args=(E, L), events=event_horizon,
            max_step=tau_max / 2000, rtol=1e-9, atol=1e-12,
            dense_output=True
        )
        r   = sol.y[0]
        phi = sol.y[2]
        x   = r * np.cos(phi)
        y_  = r * np.sin(phi)
        return dict(r=r, phi=phi, x=x, y=y_, tau=sol.t,
                    r_s=self.r_s, label=self._label)

    # ── Penrose diagram (u-v tortoise coords) ────────────────────────────
    def penrose_diagram_data(self, n: int = 400) -> dict:
        r_arr = np.linspace(1.01 * self.r_s, 20.0 * self.r_s, n)
        r_star = r_arr + self.r_s * np.log(np.abs(r_arr / self.r_s - 1.0))
        T_arr  = np.linspace(-1.0, 1.0, n)
        R_grid, T_grid = np.meshgrid(r_star / (self.r_s * 5), T_arr)
        metric_tt = -(1.0 - self.r_s / np.linspace(
            1.01*self.r_s, 20*self.r_s, n))
        return dict(r_arr=r_arr, r_star=r_star, R_grid=R_grid,
                    T_grid=T_grid, metric_tt=metric_tt, r_s=self.r_s)

    # ── Curvature scalar K = 48 G²M²/r⁶ c⁴ ─────────────────────────────
    def kretschner_scalar(self, r: np.ndarray) -> np.ndarray:
        r_safe = np.where(r > 0, r, np.nan)
        return 48.0 * (G_N * self.M)**2 / (r_safe**6 * C_L**4)



class Euler3D:
    """
    Incompressible 3-D Euler equations on a triply periodic domain [0,2π]³
    using a pseudo-spectral method with 2/3-dealiasing.

    ∂u/∂t + (u·∇)u = −∇p        ∇·u = 0

    Vorticity form:
      ∂ω/∂t + (u·∇)ω = (ω·∇)u   (vortex-stretching term drives blowup)
    """

    def __init__(self, N: int = 32, dt: float = 1e-3, viscosity: float = 0.0):
        self.N   = N
        self.dt  = dt
        self.nu  = viscosity           # 0 → pure Euler
        self.dx  = 2.0 * np.pi / N
        # Wavenumber arrays
        k1d      = fftfreq(N, d=1.0 / N).astype(float)
        self.kx, self.ky, self.kz = np.meshgrid(k1d, k1d, k1d, indexing="ij")
        self.k2  = self.kx**2 + self.ky**2 + self.kz**2
        self.k2[0, 0, 0] = 1.0        # avoid /0 in pressure solve
        # De-aliasing mask (2/3 rule)
        k_max    = N // 3
        self.dealias = ((np.abs(self.kx) < k_max) &
                        (np.abs(self.ky) < k_max) &
                        (np.abs(self.kz) < k_max)).astype(float)
        # Diagnostics storage
        self.t_hist  = []
        self.E_hist  = []          # kinetic energy
        self.W_hist  = []          # enstrophy  ½∫|ω|²
        self.omMax   = []          # max vorticity  (BKM criterion)

    # ── Initial condition: Taylor-Green vortex ───────────────────────────
    def init_taylor_green(self) -> tuple:
        x = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
        ux =  np.sin(X) * np.cos(Y) * np.cos(Z)
        uy = -np.cos(X) * np.sin(Y) * np.cos(Z)
        uz =  np.zeros_like(X)
        return ux, uy, uz

    # ── FFT helpers ──────────────────────────────────────────────────────
    def _to_spectral(self, f):
        return fftn(f) * self.dealias

    def _to_physical(self, fhat):
        return np.real(ifftn(fhat * self.dealias))

    # ── Pressure projection (divergence-free enforcement) ────────────────
    def _project(self, uxh, uyh, uzh):
        div   = (1j * self.kx * uxh + 1j * self.ky * uyh +
                 1j * self.kz * uzh)
        ph    = -div / self.k2
        uxh  -= 1j * self.kx * ph
        uyh  -= 1j * self.ky * ph
        uzh  -= 1j * self.kz * ph
        return uxh, uyh, uzh

    # ── Single RK4 step in spectral space ────────────────────────────────
    def _nonlinear(self, uxh, uyh, uzh):
        ux = self._to_physical(uxh)
        uy = self._to_physical(uyh)
        uz = self._to_physical(uzh)
        # Compute (u·∇)u in physical space then to spectral
        def conv(a, b):
            return self._to_spectral(self._to_physical(a) *
                                     self._to_physical(b))
        dudx = 1j * self.kx * uxh;  dudy = 1j * self.ky * uxh
        dvdx = 1j * self.kx * uyh;  dvdy = 1j * self.ky * uyh
        dwdx = 1j * self.kx * uzh;  dwdy = 1j * self.ky * uzh
        dudz = 1j * self.kz * uxh;  dvdz = 1j * self.kz * uyh
        dwdz = 1j * self.kz * uzh
        Nx = (self._to_spectral(ux * self._to_physical(dudx)) +
              self._to_spectral(uy * self._to_physical(dudy)) +
              self._to_spectral(uz * self._to_physical(dudz)))
        Ny = (self._to_spectral(ux * self._to_physical(dvdx)) +
              self._to_spectral(uy * self._to_physical(dvdy)) +
              self._to_spectral(uz * self._to_physical(dvdz)))
        Nz = (self._to_spectral(ux * self._to_physical(dwdx)) +
              self._to_spectral(uy * self._to_physical(dwdy)) +
              self._to_spectral(uz * self._to_physical(dwdz)))
        # Viscous term (zero for Euler)
        diss = -self.nu * self.k2
        return (-Nx + diss * uxh,
                -Ny + diss * uyh,
                -Nz + diss * uzh)

    def step_rk4(self, uxh, uyh, uzh):
        dt = self.dt
        k1 = self._nonlinear(uxh, uyh, uzh)
        k2 = self._nonlinear(uxh + 0.5*dt*k1[0],
                             uyh + 0.5*dt*k1[1],
                             uzh + 0.5*dt*k1[2])
        k3 = self._nonlinear(uxh + 0.5*dt*k2[0],
                             uyh + 0.5*dt*k2[1],
                             uzh + 0.5*dt*k2[2])
        k4 = self._nonlinear(uxh + dt*k3[0],
                             uyh + dt*k3[1],
                             uzh + dt*k3[2])
        uxh_new = uxh + dt/6*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
        uyh_new = uyh + dt/6*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        uzh_new = uzh + dt/6*(k1[2]+2*k2[2]+2*k3[2]+k4[2])
        return self._project(uxh_new, uyh_new, uzh_new)

    # ── Diagnostics ──────────────────────────────────────────────────────
    def diagnostics(self, uxh, uyh, uzh, t: float):
        ux = self._to_physical(uxh)
        uy = self._to_physical(uyh)
        uz = self._to_physical(uzh)
        E  = 0.5 * np.mean(ux**2 + uy**2 + uz**2)
        # Vorticity ω = ∇×u
        wx_h = 1j*self.ky*uzh - 1j*self.kz*uyh
        wy_h = 1j*self.kz*uxh - 1j*self.kx*uzh
        wz_h = 1j*self.kx*uyh - 1j*self.ky*uxh
        wx = self._to_physical(wx_h)
        wy = self._to_physical(wy_h)
        wz = self._to_physical(wz_h)
        om2 = wx**2 + wy**2 + wz**2
        W   = 0.5 * np.mean(om2)
        self.t_hist.append(t);  self.E_hist.append(E)
        self.W_hist.append(W);  self.omMax.append(np.sqrt(om2).max())
        return E, W

    # ── Mid-plane vorticity slice ─────────────────────────────────────────
    def vorticity_slice(self, uxh, uyh, uzh) -> np.ndarray:
        wz_h = 1j*self.kx*uyh - 1j*self.ky*uxh
        return self._to_physical(wz_h)[:, :, self.N // 2]

    # ── Full run ──────────────────────────────────────────────────────────
    def run(self, n_steps: int = 600, diag_every: int = 10) -> dict:
        ux, uy, uz = self.init_taylor_green()
        uxh = self._to_spectral(ux)
        uyh = self._to_spectral(uy)
        uzh = self._to_spectral(uz)
        t   = 0.0
        snapshots = {}
        for i in range(n_steps):
            if i % diag_every == 0:
                self.diagnostics(uxh, uyh, uzh, t)
            if i in (0, n_steps//4, n_steps//2, 3*n_steps//4, n_steps-1):
                snapshots[round(t, 4)] = self.vorticity_slice(uxh, uyh, uzh)
            uxh, uyh, uzh = self.step_rk4(uxh, uyh, uzh)
            t += self.dt
        return dict(t=np.array(self.t_hist),
                    E=np.array(self.E_hist),
                    W=np.array(self.W_hist),
                    omMax=np.array(self.omMax),
                    snapshots=snapshots)



class NavierStokes3D(Euler3D):
    """
    Incompressible 3-D Navier-Stokes equations.

    ∂u/∂t + (u·∇)u = −∇p + ν∇²u      ∇·u = 0

    Integrating factor for the viscous term (exact in spectral space):
      û(k,t) = e^{−ν k² t} û̃(k,t)

    This subclass inherits Euler3D and sets ν > 0.
    """

    def __init__(self, N: int = 32, dt: float = 1e-3,
                 viscosity: float = 1e-3):
        super().__init__(N=N, dt=dt, viscosity=viscosity)
        self._label = f"NS  N={N}  ν={viscosity:.0e}"

    # ── Exponential integrating-factor step ──────────────────────────────
    def step_etdrk4(self, uxh, uyh, uzh):
        """ETD-RK4: exponential time differencing for stiff viscous term."""
        dt  = self.dt
        L   = -self.nu * self.k2              # linear operator eigenvalue
        E   = np.exp(L * dt)
        E2  = np.exp(L * dt / 2)

        def N_(uxh, uyh, uzh):
            return self._nonlinear(uxh, uyh, uzh)

        # Stage 1
        Nu1x, Nu1y, Nu1z = N_(uxh, uyh, uzh)
        a_x = E2*uxh + (E2-1)/L * Nu1x
        a_y = E2*uyh + (E2-1)/L * Nu1y
        a_z = E2*uzh + (E2-1)/L * Nu1z
        a_x, a_y, a_z = self._project(a_x, a_y, a_z)

        Na2x, Na2y, Na2z = N_(a_x, a_y, a_z)
        b_x = E2*uxh + (E2-1)/L * Na2x
        b_y = E2*uyh + (E2-1)/L * Na2y
        b_z = E2*uzh + (E2-1)/L * Na2z
        b_x, b_y, b_z = self._project(b_x, b_y, b_z)

        Nb3x, Nb3y, Nb3z = N_(b_x, b_y, b_z)
        c_x = E2*a_x + (E2-1)/L * (2*Nb3x - Nu1x)
        c_y = E2*a_y + (E2-1)/L * (2*Nb3y - Nu1y)
        c_z = E2*a_z + (E2-1)/L * (2*Nb3z - Nu1z)
        c_x, c_y, c_z = self._project(c_x, c_y, c_z)

        Nc4x, Nc4y, Nc4z = N_(c_x, c_y, c_z)

        phi1 = (np.expm1(L*dt)) / (L*dt + 1e-300)
        phi2 = (np.expm1(L*dt) - L*dt) / (L*dt)**2

        uxh_new = (E*uxh + dt*(phi1 - 3*phi2)*Nu1x +
                   dt*2*phi2*(Na2x + Nb3x) +
                   dt*(phi2 - phi1 + 1)*Nc4x)
        uyh_new = (E*uyh + dt*(phi1 - 3*phi2)*Nu1y +
                   dt*2*phi2*(Na2y + Nb3y) +
                   dt*(phi2 - phi1 + 1)*Nc4y)
        uzh_new = (E*uzh + dt*(phi1 - 3*phi2)*Nu1z +
                   dt*2*phi2*(Na2z + Nb3z) +
                   dt*(phi2 - phi1 + 1)*Nc4z)
        return self._project(uxh_new, uyh_new, uzh_new)

    def run(self, n_steps: int = 600, diag_every: int = 10) -> dict:
        ux, uy, uz = self.init_taylor_green()
        uxh = self._to_spectral(ux)
        uyh = self._to_spectral(uy)
        uzh = self._to_spectral(uz)
        t   = 0.0
        snapshots = {}
        for i in range(n_steps):
            if i % diag_every == 0:
                self.diagnostics(uxh, uyh, uzh, t)
            if i in (0, n_steps//4, n_steps//2, 3*n_steps//4, n_steps-1):
                snapshots[round(t, 4)] = self.vorticity_slice(uxh, uyh, uzh)
            uxh, uyh, uzh = self.step_etdrk4(uxh, uyh, uzh)
            t += self.dt
        return dict(t=np.array(self.t_hist),
                    E=np.array(self.E_hist),
                    W=np.array(self.W_hist),
                    omMax=np.array(self.omMax),
                    snapshots=snapshots)



class SingularitySimulation:
    """
    Unified singularity simulation combining:
      1. Schwarzschild black-hole (GR)
      2. 3-D inviscid Euler (fluid mechanics)
      3. 3-D viscous Navier-Stokes (fluid mechanics)

    Exposes a clean property-function interface for embedding in any class.
    """

    def __init__(
        self,
        bh_mass_solar : float = 10.0,
        fluid_N       : int   = 32,
        fluid_dt      : float = 1.5e-3,
        ns_viscosity  : float = 8e-4,
        n_steps       : int   = 900,
    ):
        self.bh = BlackHoleSingularity(mass_solar=bh_mass_solar)
        self.eu = Euler3D(N=fluid_N, dt=fluid_dt, viscosity=0.0)
        self.ns = NavierStokes3D(N=fluid_N, dt=fluid_dt, viscosity=ns_viscosity)
        self.n_steps   = n_steps
        self._eu_data  = None
        self._ns_data  = None
        self._geo_data = None
        self._pen_data = None

    # ── Run all three simulations ─────────────────────────────────────────
    def run_all(self):
        print("► Running Black-Hole geodesic …")
        self._geo_data = self.bh.integrate_geodesic()
        self._pen_data = self.bh.penrose_diagram_data()

        print("► Running 3-D Euler …")
        self._eu_data  = self.eu.run(n_steps=self.n_steps)

        print("► Running 3-D Navier-Stokes …")
        self._ns_data  = self.ns.run(n_steps=self.n_steps)
        print("✔  All simulations complete.")



    @property
    def blackhole(self) -> BlackHoleSingularity:
        """Raw access to the BlackHoleSingularity instance."""
        return self.bh

    @property
    def euler(self) -> Euler3D:
        """Raw access to the Euler3D instance."""
        return self.eu

    @property
    def navier_stokes(self) -> NavierStokes3D:
        """Raw access to the NavierStokes3D instance."""
        return self.ns

    @property
    def euler_data(self) -> dict:
        """Post-run Euler diagnostics dict (call run_all first)."""
        if self._eu_data is None:
            raise RuntimeError("Call run_all() first.")
        return self._eu_data

    @property
    def ns_data(self) -> dict:
        """Post-run Navier-Stokes diagnostics dict (call run_all first)."""
        if self._ns_data is None:
            raise RuntimeError("Call run_all() first.")
        return self._ns_data

    @property
    def geodesic_data(self) -> dict:
        """Black-hole geodesic result dict (call run_all first)."""
        if self._geo_data is None:
            raise RuntimeError("Call run_all() first.")
        return self._geo_data

    # ── Scalar query functions ─────────────────────────────────────────────
    def bkm_criterion(self) -> np.ndarray:
        """
        Beale-Kato-Majda necessary condition for Euler/NS blowup.
        Returns ∫₀ᵗ ‖ω‖∞ dτ  for both Euler and NS.
        """
        eu_int = np.cumsum(self._eu_data["omMax"]) * (
            self._eu_data["t"][1] - self._eu_data["t"][0]
            if len(self._eu_data["t"]) > 1 else 1.0)
        ns_int = np.cumsum(self._ns_data["omMax"]) * (
            self._ns_data["t"][1] - self._ns_data["t"][0]
            if len(self._ns_data["t"]) > 1 else 1.0)
        return dict(euler=eu_int, ns=ns_int,
                    t_euler=self._eu_data["t"],
                    t_ns=self._ns_data["t"])

    def hawking_temperature(self) -> float:
        """Return Hawking temperature [K]."""
        return self.bh.T_H

    def bekenstein_entropy(self) -> float:
        """Return Bekenstein-Hawking entropy [dimensionless ħ units]."""
        return self.bh.S_BH

    def singularity_strength(self, r: np.ndarray = None) -> np.ndarray:
        """Kretschner curvature scalar K → ∞ as r → 0."""
        if r is None:
            r = np.linspace(1.01*self.bh.r_s, 20*self.bh.r_s, 500)
        return self.bh.kretschner_scalar(r)

    # ── Simulation-property function (embed anywhere) ─────────────────────
    def simulate(
        self,
        mode: str = "all",
        n_steps: int = None,
        return_raw: bool = False,
    ) -> dict:
        """
        High-level callable property-function.

        Parameters
        ----------
        mode      : 'all' | 'blackhole' | 'euler' | 'ns'
        n_steps   : override default step count
        return_raw: if True, include raw spectral arrays in output

        Returns
        -------
        dict with keys: euler_E, euler_W, euler_omMax,
                        ns_E, ns_W, ns_omMax,
                        bh_geodesic, bh_temperature, bh_entropy,
                        bkm_integral, t_euler, t_ns
        """
        if n_steps is not None:
            self.n_steps = n_steps

        if mode in ("all", "blackhole") and self._geo_data is None:
            self._geo_data = self.bh.integrate_geodesic()
            self._pen_data = self.bh.penrose_diagram_data()

        if mode in ("all", "euler") and self._eu_data is None:
            self._eu_data = self.eu.run(n_steps=self.n_steps)

        if mode in ("all", "ns") and self._ns_data is None:
            self._ns_data = self.ns.run(n_steps=self.n_steps)

        out = dict(
            bh_temperature = self.bh.T_H,
            bh_entropy     = self.bh.S_BH,
            bh_r_s         = self.bh.r_s,
        )
        if self._geo_data:
            out["bh_geodesic"] = self._geo_data
        if self._eu_data:
            out.update(euler_E=self._eu_data["E"],
                       euler_W=self._eu_data["W"],
                       euler_omMax=self._eu_data["omMax"],
                       t_euler=self._eu_data["t"])
        if self._ns_data:
            out.update(ns_E=self._ns_data["E"],
                       ns_W=self._ns_data["W"],
                       ns_omMax=self._ns_data["omMax"],
                       t_ns=self._ns_data["t"])
        if self._eu_data and self._ns_data:
            out["bkm_integral"] = self.bkm_criterion()
        return out


"""
=============================================================================

 Black Hole Singularity  ×  3D Euler Equations  ×  3D Navier-Stokes
=============================================================================
 Physics foundations:
   • Schwarzschild black-hole metric & geodesics (GR)
   • 3-D incompressible Euler equations (inviscid, pseudo-spectral)
   • 3-D incompressible Navier-Stokes (viscous, pseudo-spectral)
   • Vortex-stretching / singularity indicators (BKM criterion)
   • Hawking-temperature / entropy near the horizon
=============================================================================
"""

import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.mplot3d import Axes3D                        # noqa: F401
from scipy.fft import fftn, ifftn, fftfreq
from scipy.integrate import solve_ivp
from scipy.ndimage import gaussian_filter
import warnings
warnings.filterwarnings("ignore")

# ─────────────────────────────────────────────────────────────────────────────
# Physical / numerical constants
# ─────────────────────────────────────────────────────────────────────────────
G_N   = 6.674e-11       # gravitational constant  [m³ kg⁻¹ s⁻²]
C_L   = 3.000e+08       # speed of light          [m s⁻¹]
HBAR  = 1.055e-34       # reduced Planck constant [J s]
K_B   = 1.381e-23       # Boltzmann constant      [J K⁻¹]
M_SUN = 1.989e+30       # solar mass              [kg]



def plot_all(sim: SingularitySimulation, save_path: str = None):
    eu  = sim._eu_data
    ns  = sim._ns_data
    geo = sim._geo_data
    pen = sim._pen_data
    bh  = sim.bh

    fig = plt.figure(figsize=(24, 20))
    fig.patch.set_facecolor("#ffffff")
    gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.42, wspace=0.35)

    PANEL_BG  = "#424242"
    GRID_COL  = "#E0E0E0"
    CLR_EU    = "#007A8A"
    CLR_NS    = "#D32F2F"
    CLR_BH    = "#A17100"
    CLR_KRET  = "#424242"


    fig.suptitle(
        "SINGULARITY SIMULATION  ·  "
        "Black Hole  ×  3-D Euler  ×  3-D Navier–Stokes",
        color="white", fontsize=15, fontweight="bold", y=0.995
    )

    def ax_style(ax, title, xlabel="", ylabel=""):
        ax.set_facecolor(PANEL_BG)
        ax.tick_params(colors="white", labelsize=7)
        for s in ax.spines.values():
            s.set_edgecolor(GRID_COL)
        ax.xaxis.label.set_color("white")
        ax.yaxis.label.set_color("white")
        ax.set_title(title, color="white", fontsize=8, pad=4)
        ax.set_xlabel(xlabel, fontsize=7)
        ax.set_ylabel(ylabel, fontsize=7)
        ax.grid(color=GRID_COL, linestyle="--", linewidth=0.4, alpha=0.6)

    # ── 1. Geodesic orbit ────────────────────────────────────────────────
    ax1 = fig.add_subplot(gs[0, 0])
    ax_style(ax1, "BH Geodesic  (equatorial plane)", "x / r_s", "y / r_s")
    rs = geo["r_s"]
    ax1.plot(geo["x"]/rs, geo["y"]/rs, color=CLR_BH, lw=0.8, label="Geodesic")
    th = np.linspace(0, 2*np.pi, 400)
    ax1.fill(np.cos(th), np.sin(th), color="#111", zorder=3)
    ax1.plot(np.cos(th), np.sin(th), color="white", lw=0.6, zorder=4)
    ax1.set_aspect("equal")
    ax1.text(0, 0, "●", color=CLR_BH, ha="center", va="center",
             fontsize=10, zorder=5)
    ax1.legend(fontsize=6, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 2. Effective potential ────────────────────────────────────────────
    ax2 = fig.add_subplot(gs[0, 1])
    ax_style(ax2, "GR Effective Potential", "r / r_s", "V_eff (a.u.)")
    r_arr = np.linspace(1.5*rs, 25*rs, 800)
    for L_, c_ in [(2.0*rs*C_L, "#ff9f1c"),
                   (3.5*rs*C_L, CLR_BH),
                   (5.0*rs*C_L, "#2ec4b6")]:
        V = bh.effective_potential(r_arr, L=L_, mu=1.0)
        ax2.plot(r_arr/rs, V/abs(np.nanmin(V)), lw=0.9, color=c_,
                 label=f"L={L_/(rs*C_L):.1f}·r_s c")
    ax2.axvline(1, color="white", lw=0.5, ls="--")
    ax2.set_ylim(-2, 2)
    ax2.legend(fontsize=5.5, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 3. Kretschner scalar (curvature blowup) ───────────────────────────
    ax3 = fig.add_subplot(gs[0, 2])
    ax_style(ax3, "Kretschner Curvature Scalar  K→∞", "r / r_s", "K  [m⁻⁴]")
    r_k = np.linspace(1.05*rs, 8*rs, 600)
    ax3.semilogy(r_k/rs, bh.kretschner_scalar(r_k), color=CLR_KRET, lw=1.0)
    ax3.axvline(1, color="white", lw=0.5, ls="--", label="Horizon")
    ax3.fill_betweenx([bh.kretschner_scalar(r_k).min(),
                       bh.kretschner_scalar(r_k).max()],
                      0, 1, alpha=0.15, color="white")
    ax3.legend(fontsize=6, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 4. Kinetic energy decay ────────────────────────────────────────────
    ax4 = fig.add_subplot(gs[1, 0])
    ax_style(ax4, "Kinetic Energy  E(t)", "time", "E")
    ax4.plot(eu["t"], eu["E"], color=CLR_EU, lw=1.0, label="Euler")
    ax4.plot(ns["t"], ns["E"], color=CLR_NS, lw=1.0, label="NS")
    ax4.legend(fontsize=6, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 5. Enstrophy (singularity indicator) ─────────────────────────────
    ax5 = fig.add_subplot(gs[1, 1])
    ax_style(ax5, "Enstrophy  Ω(t) = ½∫|ω|²  (blowup signal)", "time", "Ω")
    ax5.semilogy(eu["t"], np.maximum(eu["W"], 1e-30), color=CLR_EU,
                 lw=1.0, label="Euler")
    ax5.semilogy(ns["t"], np.maximum(ns["W"], 1e-30), color=CLR_NS,
                 lw=1.0, label="NS")
    ax5.legend(fontsize=6, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 6. Max vorticity (BKM criterion) ──────────────────────────────────
    ax6 = fig.add_subplot(gs[1, 2])
    ax_style(ax6, "Max Vorticity  ‖ω‖∞  (BKM criterion)", "time", "‖ω‖∞")
    ax6.plot(eu["t"], eu["omMax"], color=CLR_EU, lw=1.0, label="Euler")
    ax6.plot(ns["t"], ns["omMax"], color=CLR_NS, lw=1.0, label="NS")
    ax6.legend(fontsize=6, facecolor=PANEL_BG, labelcolor="white",
               edgecolor=GRID_COL)

    # ── 7-9. Vorticity snapshots (Euler) ──────────────────────────────────
    eu_snaps = list(eu["snapshots"].items())
    ns_snaps = list(ns["snapshots"].items())
    for col, (t_snap, snap) in enumerate(eu_snaps[:3]):
        ax_ = fig.add_subplot(gs[2, col])
        vmax = max(abs(snap).max(), 1e-10)
        im = ax_.imshow(snap.T, origin="lower", cmap="seismic",
                        vmin=-vmax, vmax=vmax, aspect="auto")
        ax_style(ax_, f"Euler ωz  t={t_snap:.3f}")
        fig.colorbar(im, ax=ax_, pad=0.02).ax.yaxis.set_tick_params(
            colors="white", labelsize=5)

    # ── 10-12. Vorticity snapshots (NS) ───────────────────────────────────
    for col, (t_snap, snap) in enumerate(ns_snaps[:3]):
        ax_ = fig.add_subplot(gs[3, col])
        vmax = max(abs(snap).max(), 1e-10)
        im = ax_.imshow(snap.T, origin="lower", cmap="inferno",
                        vmin=0, vmax=vmax, aspect="auto")
        ax_style(ax_, f"NS |ωz|  t={t_snap:.3f}")
        fig.colorbar(im, ax=ax_, pad=0.02).ax.yaxis.set_tick_params(
            colors="white", labelsize=5)

    # ── Annotations ─────────────────────────────────────────────────────
    info = (f"BH mass = {sim.bh.M/M_SUN:.0f} M☉   "
            f"r_s = {sim.bh.r_s:.3e} m   "
            f"T_Hawking = {sim.bh.T_H:.3e} K   "
            f"S_BH = {sim.bh.S_BH:.3e}")
    fig.text(0.5, 0.001, info, ha="center", va="bottom",
             color="#aaaaaa", fontsize=7)

    if save_path:
        plt.savefig(save_path, dpi=160, bbox_inches="tight",
                    facecolor=fig.get_facecolor())
        print(f"✔  Plot saved → {save_path}")
    plt.close(fig)






In [ ]:
# @title


"""
  1. Kerr–Schwarzschild black holes  (GR, spin, frame dragging)
  2. 3-D inviscid Euler               (pseudo-spectral, adaptive CFL)
  3. 3-D viscous Navier-Stokes       (ETD-RK4, Kolmogorov microscales)

References
----------
  - Misner, Thorne & Wheeler, *Gravitation* (1973)
  - Bardeen, Press & Teukolsky, ApJ 178 347 (1972)
  - Kerr, PRL 11 237 (1963)
  - Beale, Kato & Majda, CMP 94 61 (1984)
  - Cox & Matthews, *Using MPI* (spectral fluid)
  - Frisch, *Turbulence* (1995)
  - Pope, *Turbulent Flows* (2000)
  - Canuto, Hussaini et al., *Spectral Methods in Fluid Dynamics* (1988)
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Dict, Optional, Tuple

import numpy as np
from numpy.fft import fftn, ifftn, fftfreq
from scipy.integrate import solve_ivp
from scipy.special import gamma as gamma_func

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.axes as maxes          # used in _colorbar type-guard
import matplotlib.colors as mcolors
from matplotlib.patches import Circle, FancyArrowPatch
from matplotlib.ticker import LogLocator, LogFormatter
from matplotlib.colors import LogNorm, Normalize
from mpl_toolkits.mplot3d import Axes3D  # noqa: F401
from scipy.fft import fftn, ifftn, fftfreq
from scipy.integrate import solve_ivp
from scipy.ndimage import gaussian_filter
warnings.filterwarnings("ignore")


# ──────────────────────────────────────────────────────────────────────────────
# Fundamental physical constants (SI)
# ──────────────────────────────────────────────────────────────────────────────
G_N   = 6.674_30e-11       # Gravitational constant      [m³ kg⁻¹ s⁻²]
C_L   = 2.997_924_58e8     # Speed of light              [m s⁻¹]
HBAR  = 1.054_571_817e-34  # Reduced Planck constant     [J s]
K_B   = 1.380_649e-23      # Boltzmann constant          [J K⁻¹]
M_SUN = 1.989e30           # Solar mass                  [kg]
SIGMA = 5.670_374_419e-8   # Stefan–Boltzmann constant   [W m⁻² K⁻⁴]


# ══════════════════════════════════════════════════════════════════════════════
#  BLACK HOLE SINGULARITY  (Kerr–Schwarzschild)
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class KerrParameters:
    """Derived Kerr geometry parameters."""
    M:          float
    a:          float
    r_s:        float
    r_plus:     float
    r_minus:    float
    r_isco:     float
    r_photo:    float
    r_ergo_eq:  float
    Omega_H:    float
    T_H:        float
    S_BH:       float
    efficiency: float


class BlackHoleSingularity:
    """
    Kerr black hole: metric, geodesics, effective potential,
    Hawking temperature, Bekenstein entropy, Penrose diagram,
    gravitational waves, tidal forces, and Hawking spectrum.

    Kerr metric (Boyer-Lindquist, G=c=1 natural then re-scaled to SI):
      Σ = r² + a²cos²θ
      Δ = r² - r_s·r + a²

      ds² = -(1 - r_s·r/Σ)c²dt² - (2r_s·r·a·sin²θ/Σ)c·dt·dφ
            + (Σ/Δ)dr² + Σ·dθ² + (r²+a² + r_s·r·a²sin²θ/Σ)sin²θ·dφ²

    Schwarzschild limit: a → 0.
    """

    def __init__(self, mass_solar: float = 10.0, spin: float = 0.0):
        if not (0.0 <= spin < 1.0):
            raise ValueError("Spin parameter a* must be in [0, 1).")

        self.M_solar = mass_solar
        self.a_star  = spin
        self.M       = mass_solar * M_SUN
        self.r_s     = 2.0 * G_N * self.M / C_L**2

        self.a = spin * G_N * self.M / C_L**2

        _M_geom = G_N * self.M / C_L**2
        disc    = _M_geom**2 - self.a**2
        if disc < 0:
            raise ValueError("a > GM/c²: super-extremal (unphysical).")
        self.r_plus  = _M_geom + np.sqrt(disc)
        self.r_minus = _M_geom - np.sqrt(disc)

        a_star = spin
        z1 = 1 + (1 - a_star**2)**(1/3) * (
             (1 + a_star)**(1/3) + (1 - a_star)**(1/3))
        z2 = np.sqrt(3 * a_star**2 + z1**2)
        r_isco_geom = 3 + z2 - np.sqrt((3 - z1) * (3 + z1 + 2*z2))
        self.r_isco  = r_isco_geom * _M_geom * 2

        self.r_photo = 2 * _M_geom * (
            1 + np.cos(2/3 * np.arccos(-a_star))) * 2

        self.r_ergo_eq = self.r_s

        self.Omega_H = (self.a * C_L**3) / (
            (self.r_plus**2 + self.a**2) * G_N * self.M)

        kappa = (self.r_plus - self.r_minus) / (
                  2 * (self.r_plus**2 + self.a**2))
        kappa_SI = kappa * C_L**2 / (G_N * self.M)
        self.T_H = HBAR * kappa_SI / (2 * np.pi * K_B)

        l_P2 = HBAR * G_N / C_L**3
        A    = 4 * np.pi * (self.r_plus**2 + self.a**2)
        self.S_BH = K_B * A / (4 * l_P2)

        self.efficiency = 1.0 - np.sqrt((1 + np.sqrt(1 - a_star**2)) / 2)

        self._label = (f"Kerr BH  M={mass_solar}M☉  a*={spin:.3f}"
                       f"  r+={self.r_plus:.3e}m  T_H={self.T_H:.3e}K")

        self._params = KerrParameters(
            M=self.M, a=self.a, r_s=self.r_s,
            r_plus=self.r_plus, r_minus=self.r_minus,
            r_isco=self.r_isco, r_photo=self.r_photo,
            r_ergo_eq=self.r_ergo_eq, Omega_H=self.Omega_H,
            T_H=self.T_H, S_BH=self.S_BH, efficiency=self.efficiency)

    @property
    def params(self) -> KerrParameters:
        return self._params

    def metric_components(self, r: np.ndarray) -> dict:
        rs = self.r_s;   a = self.a
        Sigma = r**2
        Delta = r**2 - rs * r + a**2
        g_tt  = -(1 - rs * r / Sigma)
        g_tphi= -(rs * r * a / Sigma)
        g_rr  =  Sigma / Delta
        g_phph= (r**2 + a**2 + rs * r * a**2 / Sigma)
        return dict(g_tt=g_tt, g_tphi=g_tphi, g_rr=g_rr, g_phph=g_phph,
                    Sigma=Sigma, Delta=Delta)

    def effective_potential(self, r: np.ndarray, L: float = None,
                            E: float = 1.0, mu: float = 1.0) -> np.ndarray:
        if L is None:
            L = 3.5 * self.r_s * C_L
        r_safe = np.where(r > 1e-3 * self.r_plus, r, np.nan)
        m = self.metric_components(r_safe)
        V = (-m["g_tt"] * E**2
             - 2 * m["g_tphi"] * E * L / (C_L)
             - m["g_phph"] * L**2 / C_L**2
             + mu**2 * C_L**2) / (2 * m["g_rr"])
        return V

    def _geodesic_rhs_kerr(self, tau: float, y: np.ndarray,
                           E_geo: float, L_geo: float) -> list:
        r, rdot, phi = y
        r = max(r, 1.01 * self.r_plus)
        rs = self.r_s;   a = self.a;   mu2 = 1.0
        Delta = r**2 - rs * r + a**2
        Sigma = r**2

        P     = E_geo * (r**2 + a**2) - L_geo * a
        R     = P**2 - Delta * ((a * E_geo - L_geo)**2 + mu2 * r**2)
        R     = max(R, 0.0)

        dRdr  = (4 * E_geo**2 * r * (r**2 + a**2) - 2 * Delta * 2 * r * mu2
                 - (2 * r - rs) * ((a * E_geo - L_geo)**2 + mu2 * r**2))
        rddot = dRdr / (2 * Sigma**2) - (r / Sigma) * rdot**2

        phidot = (a * P / Delta - (a * E_geo - L_geo)) / Sigma

        return [rdot, rddot, phidot]

    def integrate_geodesic(self, r0: float = None, phi0: float = 0.0,
                            E: float = 0.95, L: float = None,
                            tau_max: float = None) -> dict:
        if r0 is None:
            r0 = 10.0 * self.r_plus
        if L is None:
            L = 3.5 * self.r_plus * C_L
        if tau_max is None:
            tau_max = 40.0 * self.r_plus / C_L

        _M_g = G_N * self.M / C_L**2
        r0_g   = r0   / _M_g
        L_g    = L    / (_M_g * C_L)
        a_g    = self.a / _M_g
        taumax_g = tau_max * C_L / _M_g

        Delta0 = r0_g**2 - 2*r0_g + a_g**2
        P0     = E*(r0_g**2 + a_g**2) - L_g*a_g
        R0     = P0**2 - Delta0*((a_g*E - L_g)**2 + r0_g**2)
        rdot0  = -np.sqrt(max(R0, 0.0)) / r0_g**2

        y0 = [r0_g, rdot0, phi0]
        rh = self.r_plus / _M_g

        def event_horizon(tau, y, E_geo, L_geo):
            return y[0] - 1.02 * rh
        event_horizon.terminal  = True
        event_horizon.direction = -1

        sol = solve_ivp(
            self._geodesic_rhs_kerr, [0, taumax_g], y0,
            args=(E, L_g),
            events=event_horizon,
            max_step=taumax_g / 5000,
            rtol=1e-10, atol=1e-13,
            dense_output=True
        )
        r   = sol.y[0] * _M_g
        phi = sol.y[2]
        x   = r * np.cos(phi)
        y_  = r * np.sin(phi)

        redshift = 1.0 / np.sqrt(np.abs(1.0 - self.r_s / r)) - 1.0

        return dict(r=r, phi=phi, x=x, y=y_,
                    tau=sol.t * _M_g / C_L,
                    redshift=redshift,
                    r_s=self.r_s, r_plus=self.r_plus,
                    r_isco=self.r_isco,
                    label=self._label,
                    terminated=bool(sol.t_events[0].size > 0))

    def penrose_diagram_data(self, n: int = 600) -> dict:
        rs = self.r_s
        r_arr = np.linspace(1.001 * self.r_plus, 20.0 * self.r_plus, n)

        if self.a == 0:
            r_star = r_arr + rs * np.log(np.abs(r_arr / rs - 1.0))
            U = -np.exp(-r_star / rs)
            V =  np.exp( r_star / rs)
        else:
            k_plus  = (self.r_plus - self.r_minus)
            k_minus = (self.r_plus - self.r_minus)
            r_star = (r_arr
                      + (self.r_plus**2 + self.a**2) / k_plus
                        * np.log(np.abs(r_arr - self.r_plus))
                      - (self.r_minus**2 + self.a**2) / k_minus
                        * np.log(np.abs(r_arr - self.r_minus)))
            U = -np.exp(-C_L * r_star / self.r_plus)
            V =  np.exp( C_L * r_star / self.r_plus)

        X_krus   = (U + V) / 2.0
        T_krus   = (V - U) / 2.0
        metric_tt = -(1.0 - rs / r_arr)

        return dict(r_arr=r_arr, r_star=r_star,
                    U=U, V=V, X_krus=X_krus, T_krus=T_krus,
                    metric_tt=metric_tt, r_s=rs, r_plus=self.r_plus)

    def hawking_spectrum(self, nu_max: float = None, n_pts: int = 500) -> dict:
        if self.T_H < 1e-50:
            return dict(nu=np.array([0.0]), dPdnu=np.array([0.0]),
                        nu_peak=0.0, T_H=self.T_H)
        if nu_max is None:
            nu_max = 20 * K_B * self.T_H / HBAR
        nu  = np.linspace(1e-10 * nu_max, nu_max, n_pts)
        x   = HBAR * nu / (K_B * self.T_H)
        Gamma = 27 * np.pi * self.r_plus**2 * nu**2 / C_L**2
        with np.errstate(over='ignore'):
            dPdnu = (HBAR / (2 * np.pi)) * Gamma * nu / (np.expm1(x) + 1e-300)
        nu_peak = 2.821 * K_B * self.T_H / HBAR
        total_power = SIGMA * 4 * np.pi * self.r_plus**2 * self.T_H**4
        return dict(nu=nu, dPdnu=dPdnu, nu_peak=nu_peak,
                    total_power=total_power, T_H=self.T_H)

    def spaghettification_radius(self, rho_human: float = 1000.0,
                                 height_m: float = 1.8) -> float:
        M_human = rho_human * np.pi * (0.15)**2 * height_m
        g_body  = 6.674e-11 * M_human / (height_m / 2)**2
        r_spaghetti = (2 * G_N * self.M * height_m / g_body) ** (1.0/3.0)
        return r_spaghetti

    def gw_chirp_strain(self, m_companion_solar: float = 1.4,
                         t_array: np.ndarray = None,
                         r_obs_Mpc: float = 410.0) -> dict:
        m1 = self.M
        m2 = m_companion_solar * M_SUN
        Mc = (m1 * m2)**(3/5) / (m1 + m2)**(1/5)

        f0 = 10.0
        T_chirp = (5/256) * C_L**5 / G_N**3 * (
            G_N * Mc / C_L**3)**(-5/3) * (np.pi * f0)**(-8/3)
        T_chirp = min(T_chirp, 60.0)

        if t_array is None:
            t_array = np.linspace(0, 0.999 * T_chirp, 5000)

        tau = T_chirp - t_array
        tau = np.where(tau > 1e-6 * T_chirp, tau, np.nan)
        f_t = (1 / np.pi) * (5 / (256 * tau))**(3/8) * (
              G_N * Mc / C_L**3)**(-5/8)

        r_m = r_obs_Mpc * 3.0857e22
        A   = 4 * (G_N * Mc / C_L**2)**(5/3) * (np.pi * f_t)**(2/3) / (
              C_L**4 * r_m / G_N)
        phi_t = np.nancumsum(-2 * np.pi * f_t *
                              np.gradient(t_array, edge_order=2))
        h_plus  =  A * np.cos(phi_t)
        h_cross =  A * np.sin(phi_t)

        return dict(t=t_array, h_plus=h_plus, h_cross=h_cross,
                    f_t=f_t, Mc_solar=Mc / M_SUN,
                    T_chirp=T_chirp, r_obs_Mpc=r_obs_Mpc)

    def kretschner_scalar(self, r: np.ndarray) -> np.ndarray:
        r_safe = np.where(r > 0, r, np.nan)
        _M_g = G_N * self.M / C_L**2
        K = 48 * _M_g**2 * (r_safe**2 - self.a**2) / r_safe**6
        return K

    def hawking_temperature(self) -> float:
        return self.T_H

    def bekenstein_entropy(self) -> float:
        return self.S_BH


# ══════════════════════════════════════════════════════════════════════════════
#  3-D EULER EQUATIONS  (Pseudo-spectral, adaptive CFL)
# ══════════════════════════════════════════════════════════════════════════════

class Euler3D:
    """
    Incompressible 3-D Euler equations on [0,2π]³ (triply periodic),
    pseudo-spectral method with 2/3-rule dealiasing and adaptive CFL stepping.
    """

    def __init__(self, N: int = 32, dt: float = 1e-3,
                 viscosity: float = 0.0, cfl_safety: float = 0.5):
        self.N          = N
        self.dt_init    = dt
        self.dt         = dt
        self.nu         = viscosity
        self.cfl_safety = cfl_safety
        self.dx         = 2.0 * np.pi / N

        k1d = fftfreq(N, d=1.0 / N).astype(float)
        self.kx, self.ky, self.kz = np.meshgrid(k1d, k1d, k1d, indexing="ij")
        self.k2  = self.kx**2 + self.ky**2 + self.kz**2
        self.k2[0, 0, 0] = 1.0
        self.k_mag = np.sqrt(self.k2)

        k_max        = N // 3
        self.dealias = ((np.abs(self.kx) < k_max) &
                        (np.abs(self.ky) < k_max) &
                        (np.abs(self.kz) < k_max)).astype(float)

        self.t_hist    : list = []
        self.E_hist    : list = []
        self.W_hist    : list = []
        self.H_hist    : list = []
        self.omMax_hist: list = []
        self.Palin_hist: list = []
        self.dt_hist   : list = []

    def init_taylor_green(self) -> tuple:
        x = np.linspace(0, 2*np.pi, self.N, endpoint=False)
        X, Y, Z = np.meshgrid(x, x, x, indexing="ij")
        ux =  np.sin(X) * np.cos(Y) * np.cos(Z)
        uy = -np.cos(X) * np.sin(Y) * np.cos(Z)
        uz =  np.zeros_like(X)
        return ux, uy, uz

    def _to_spectral(self, f: np.ndarray) -> np.ndarray:
        return fftn(f) * self.dealias

    def _to_physical(self, fhat: np.ndarray) -> np.ndarray:
        return np.real(ifftn(fhat * self.dealias))

    def _project(self, uxh, uyh, uzh):
        div  = 1j*(self.kx*uxh + self.ky*uyh + self.kz*uzh)
        ph   = -div / self.k2
        uxh -= 1j * self.kx * ph
        uyh -= 1j * self.ky * ph
        uzh -= 1j * self.kz * ph
        return uxh, uyh, uzh

    def _nonlinear(self, uxh, uyh, uzh):
        ux = self._to_physical(uxh)
        uy = self._to_physical(uyh)
        uz = self._to_physical(uzh)

        def grad_sp(fh, k): return self._to_physical(1j * k * fh)

        dudx = grad_sp(uxh, self.kx); dudy = grad_sp(uxh, self.ky); dudz = grad_sp(uxh, self.kz)
        dvdx = grad_sp(uyh, self.kx); dvdy = grad_sp(uyh, self.ky); dvdz = grad_sp(uyh, self.kz)
        dwdx = grad_sp(uzh, self.kx); dwdy = grad_sp(uzh, self.ky); dwdz = grad_sp(uzh, self.kz)

        Nx = self._to_spectral(ux*dudx + uy*dudy + uz*dudz)
        Ny = self._to_spectral(ux*dvdx + uy*dvdy + uz*dvdz)
        Nz = self._to_spectral(ux*dwdx + uy*dwdy + uz*dwdz)

        diss = -self.nu * self.k2
        return (-Nx + diss*uxh, -Ny + diss*uyh, -Nz + diss*uzh)

    def _cfl_dt(self, uxh, uyh, uzh) -> float:
        ux = self._to_physical(uxh)
        uy = self._to_physical(uyh)
        uz = self._to_physical(uzh)
        u_max = np.sqrt(ux**2 + uy**2 + uz**2).max() + 1e-14
        dt_cfl = self.cfl_safety * self.dx / u_max
        return float(min(dt_cfl, self.dt_init * 2))

    def step_rk4(self, uxh, uyh, uzh, dt: float = None):
        dt = dt or self.dt
        k1 = self._nonlinear(uxh, uyh, uzh)
        k2 = self._nonlinear(uxh+.5*dt*k1[0], uyh+.5*dt*k1[1], uzh+.5*dt*k1[2])
        k3 = self._nonlinear(uxh+.5*dt*k2[0], uyh+.5*dt*k2[1], uzh+.5*dt*k2[2])
        k4 = self._nonlinear(uxh+dt*k3[0],    uyh+dt*k3[1],    uzh+dt*k3[2])
        uxh_n = uxh + dt/6*(k1[0]+2*k2[0]+2*k3[0]+k4[0])
        uyh_n = uyh + dt/6*(k1[1]+2*k2[1]+2*k3[1]+k4[1])
        uzh_n = uzh + dt/6*(k1[2]+2*k2[2]+2*k3[2]+k4[2])
        return self._project(uxh_n, uyh_n, uzh_n)

    def diagnostics(self, uxh, uyh, uzh, t: float, dt: float):
        ux = self._to_physical(uxh)
        uy = self._to_physical(uyh)
        uz = self._to_physical(uzh)
        E  = 0.5 * np.mean(ux**2 + uy**2 + uz**2)

        wx_h = 1j*self.ky*uzh - 1j*self.kz*uyh
        wy_h = 1j*self.kz*uxh - 1j*self.kx*uzh
        wz_h = 1j*self.kx*uyh - 1j*self.ky*uxh
        wx = self._to_physical(wx_h)
        wy = self._to_physical(wy_h)
        wz = self._to_physical(wz_h)
        om2 = wx**2 + wy**2 + wz**2

        W       = 0.5 * np.mean(om2)
        om_max  = float(np.sqrt(om2).max())
        H       = float(np.mean(ux*wx + uy*wy + uz*wz))
        Palin   = 0.5 * float(np.mean(
            self.k2 * np.abs(wx_h)**2 +
            self.k2 * np.abs(wy_h)**2 +
            self.k2 * np.abs(wz_h)**2) / self.N**6)

        self.t_hist.append(t);      self.E_hist.append(E)
        self.W_hist.append(W);      self.H_hist.append(H)
        self.omMax_hist.append(om_max)
        self.Palin_hist.append(Palin)
        self.dt_hist.append(dt)

        return E, W, H, om_max

    def energy_spectrum(self, uxh, uyh, uzh) -> Tuple[np.ndarray, np.ndarray]:
        k_int = np.round(self.k_mag).astype(int).ravel()
        E_tot = 0.5 * (np.abs(uxh)**2 + np.abs(uyh)**2 + np.abs(uzh)**2)
        k_max = self.N // 3
        k_bins = np.arange(0, k_max + 1)
        E_k   = np.zeros(k_max + 1)
        for k_val in range(k_max + 1):
            mask = k_int == k_val
            E_k[k_val] = E_tot.ravel()[mask].sum() / self.N**6
        return k_bins.astype(float), E_k

    def qr_invariants(self, uxh, uyh, uzh) -> dict:
        def gp(fh, k): return self._to_physical(1j * k * fh)
        A = np.zeros((3, 3, self.N, self.N, self.N))
        fields = [uxh, uyh, uzh]
        ks     = [self.kx, self.ky, self.kz]
        for i in range(3):
            for j in range(3):
                A[i, j] = gp(fields[i], ks[j])

        Q = -0.5 * np.einsum('ijxyz,jixyz->xyz', A, A)
        R = -(1.0/3.0) * np.einsum('ijxyz,jkxyz,kixyz->xyz', A, A, A)
        return dict(Q=Q, R=R,
                    Q_mean=float(Q.mean()), R_mean=float(R.mean()),
                    discriminant=(27*R**2 + 4*Q**3))

    def vorticity_slice(self, uxh, uyh, uzh) -> np.ndarray:
        wz_h = 1j*self.kx*uyh - 1j*self.ky*uxh
        return self._to_physical(wz_h)[:, :, self.N // 2]

    def run(self, n_steps: int = 600, diag_every: int = 10,
            adaptive_dt: bool = True) -> dict:
        ux, uy, uz = self.init_taylor_green()
        uxh = self._to_spectral(ux)
        uyh = self._to_spectral(uy)
        uzh = self._to_spectral(uz)
        t   = 0.0
        snapshots: dict = {}
        snap_steps = {0, n_steps//4, n_steps//2, 3*n_steps//4, n_steps-1}

        for i in range(n_steps):
            dt = self._cfl_dt(uxh, uyh, uzh) if adaptive_dt else self.dt_init
            if i % diag_every == 0:
                self.diagnostics(uxh, uyh, uzh, t, dt)
            if i in snap_steps:
                snapshots[round(t, 4)] = self.vorticity_slice(uxh, uyh, uzh)
            uxh, uyh, uzh = self.step_rk4(uxh, uyh, uzh, dt)
            t += dt

        return dict(t       = np.array(self.t_hist),
                    E       = np.array(self.E_hist),
                    W       = np.array(self.W_hist),
                    H       = np.array(self.H_hist),
                    omMax   = np.array(self.omMax_hist),
                    Palin   = np.array(self.Palin_hist),
                    dt_hist = np.array(self.dt_hist),
                    snapshots = snapshots)


# ══════════════════════════════════════════════════════════════════════════════
#  3-D NAVIER-STOKES  (ETD-RK4, Kolmogorov microscales)
# ══════════════════════════════════════════════════════════════════════════════

class NavierStokes3D(Euler3D):
    """
    Incompressible 3-D Navier-Stokes on [0,2π]³.
    Exponential time differencing RK4 (Cox & Matthews 2002).
    """

    def __init__(self, N: int = 32, dt: float = 1e-3,
                 viscosity: float = 1e-3):
        super().__init__(N=N, dt=dt, viscosity=viscosity,
                         cfl_safety=0.4)
        self._label = f"NS  N={N}  ν={viscosity:.1e}"
        self.eps_hist  : list = []
        self.Re_lam_hist: list = []

    @staticmethod
    def _phi1(z: np.ndarray) -> np.ndarray:
        safe = np.abs(z) > 1e-3
        out  = np.where(safe,
                        np.expm1(z) / np.where(safe, z, 1.0),
                        1 + z/2 + z**2/6 + z**3/24)
        return out

    @staticmethod
    def _phi2(z: np.ndarray) -> np.ndarray:
        safe = np.abs(z) > 1e-3
        out  = np.where(safe,
                        (np.expm1(z) - z) / np.where(safe, z**2, 1.0),
                        0.5 + z/6 + z**2/24 + z**3/120)
        return out

    def step_etdrk4(self, uxh, uyh, uzh, dt: float = None):
        dt  = dt or self.dt
        L   = -self.nu * self.k2
        E   = np.exp(L * dt)
        E2  = np.exp(L * dt * 0.5)

        z   = L * dt
        z2  = L * dt * 0.5
        ph1 = self._phi1(z)
        ph2 = self._phi2(z)
        ph1h= self._phi1(z2)

        def N_(a, b, c): return self._nonlinear(a, b, c)

        Na = N_(uxh, uyh, uzh)
        ax = E2*uxh + dt*0.5 * ph1h * Na[0]
        ay = E2*uyh + dt*0.5 * ph1h * Na[1]
        az = E2*uzh + dt*0.5 * ph1h * Na[2]
        ax, ay, az = self._project(ax, ay, az)

        Nb = N_(ax, ay, az)
        bx = E2*uxh + dt*0.5 * ph1h * Nb[0]
        by = E2*uyh + dt*0.5 * ph1h * Nb[1]
        bz = E2*uzh + dt*0.5 * ph1h * Nb[2]
        bx, by, bz = self._project(bx, by, bz)

        Nc = N_(bx, by, bz)
        cx = E*ax + dt * ph1h * (2*Nc[0] - Na[0])
        cy = E*ay + dt * ph1h * (2*Nc[1] - Na[1])
        cz = E*az + dt * ph1h * (2*Nc[2] - Na[2])
        cx, cy, cz = self._project(cx, cy, cz)

        Nd = N_(cx, cy, cz)

        uxh_n = (E*uxh + dt*(
            (ph1 - 3*ph2) * Na[0] +
            2*ph2          * (Nb[0] + Nc[0]) +
            (ph2 - ph1 + 1) * Nd[0]))
        uyh_n = (E*uyh + dt*(
            (ph1 - 3*ph2) * Na[1] +
            2*ph2          * (Nb[1] + Nc[1]) +
            (ph2 - ph1 + 1) * Nd[1]))
        uzh_n = (E*uzh + dt*(
            (ph1 - 3*ph2) * Na[2] +
            2*ph2          * (Nb[2] + Nc[2]) +
            (ph2 - ph1 + 1) * Nd[2]))
        return self._project(uxh_n, uyh_n, uzh_n)

    def diagnostics(self, uxh, uyh, uzh, t: float, dt: float):
        E, W, H, om_max = super().diagnostics(uxh, uyh, uzh, t, dt)

        eps = 2 * self.nu * W if self.nu > 0 else 0.0
        u_rms = np.sqrt(2 * E / 3)
        lam   = np.sqrt(5 * self.nu * E / (eps + 1e-300))
        Re_lam = u_rms * lam / (self.nu + 1e-300)

        self.eps_hist.append(eps)
        self.Re_lam_hist.append(Re_lam)
        return E, W, H, om_max

    def kolmogorov_scales(self) -> dict:
        if not self.eps_hist:
            return {}
        eps_mean = float(np.mean(self.eps_hist[-len(self.eps_hist)//4 or 1:]))
        eps_mean = max(eps_mean, 1e-30)
        eta   = (self.nu**3 / eps_mean)**0.25
        tau_e = (self.nu / eps_mean)**0.5
        v_e   = (self.nu * eps_mean)**0.25
        return dict(eta=eta, tau_eta=tau_e, v_eta=v_e,
                    eps_mean=eps_mean,
                    Re_lambda_mean=float(np.mean(self.Re_lam_hist) if self.Re_lam_hist else 0))

    def structure_functions(self, uxh, orders: list = None,
                             n_r: int = 20) -> dict:
        if orders is None:
            orders = [2, 4, 6]
        ux = self._to_physical(uxh)
        r_vals = np.linspace(1, self.N // 2, n_r).astype(int)
        Sp = {p: np.zeros(n_r) for p in orders}
        for i, r_shift in enumerate(r_vals):
            delta_u = np.roll(ux, int(r_shift), axis=0) - ux
            for p in orders:
                Sp[p][i] = float(np.mean(np.abs(delta_u)**p))
        r_phys = r_vals * self.dx
        return dict(r=r_phys, Sp=Sp)

    def run(self, n_steps: int = 600, diag_every: int = 10,
            adaptive_dt: bool = True) -> dict:
        ux, uy, uz = self.init_taylor_green()
        uxh = self._to_spectral(ux)
        uyh = self._to_spectral(uy)
        uzh = self._to_spectral(uz)
        t   = 0.0
        snapshots: dict = {}
        snap_steps = {0, n_steps//4, n_steps//2, 3*n_steps//4, n_steps-1}

        for i in range(n_steps):
            dt = self._cfl_dt(uxh, uyh, uzh) if adaptive_dt else self.dt_init
            if i % diag_every == 0:
                self.diagnostics(uxh, uyh, uzh, t, dt)
            if i in snap_steps:
                snapshots[round(t, 4)] = self.vorticity_slice(uxh, uyh, uzh)
            uxh, uyh, uzh = self.step_etdrk4(uxh, uyh, uzh, dt)
            t += dt

        kol = self.kolmogorov_scales()
        return dict(t       = np.array(self.t_hist),
                    E       = np.array(self.E_hist),
                    W       = np.array(self.W_hist),
                    H       = np.array(self.H_hist),
                    omMax   = np.array(self.omMax_hist),
                    Palin   = np.array(self.Palin_hist),
                    eps     = np.array(self.eps_hist),
                    Re_lam  = np.array(self.Re_lam_hist),
                    dt_hist = np.array(self.dt_hist),
                    snapshots = snapshots,
                    kolmogorov = kol)


# ══════════════════════════════════════════════════════════════════════════════
#  UNIFIED SINGULARITY SIMULATION
# ══════════════════════════════════════════════════════════════════════════════

class SingularitySimulation:
    """
    Unified singularity simulation combining:
      1. Kerr black hole          (GR, spin, frame dragging, GW chirp)
      2. 3-D inviscid Euler       (adaptive CFL, Q-R map, E(k) spectrum)
      3. 3-D viscous Navier-Stokes(ETD-RK4, Kolmogorov scales, Re_λ)
    """

    def __init__(
        self,
        bh_mass_solar : float = 10.0,
        bh_spin       : float = 0.0,
        fluid_N       : int   = 32,
        fluid_dt      : float = 1.5e-3,
        ns_viscosity  : float = 8e-4,
        n_steps       : int   = 900,
        adaptive_dt   : bool  = True,
    ):
        self.bh  = BlackHoleSingularity(mass_solar=bh_mass_solar, spin=bh_spin)
        self.eu  = Euler3D(N=fluid_N, dt=fluid_dt, viscosity=0.0)
        self.ns  = NavierStokes3D(N=fluid_N, dt=fluid_dt, viscosity=ns_viscosity)
        self.n_steps    = n_steps
        self.adaptive_dt= adaptive_dt

        self._eu_data  : Optional[dict] = None
        self._ns_data  : Optional[dict] = None
        self._geo_data : Optional[dict] = None
        self._pen_data : Optional[dict] = None
        self._gw_data  : Optional[dict] = None

    def run_all(self, gw_companion_solar: float = 1.4):
        print("► Running Kerr black-hole geodesic …")
        self._geo_data = self.bh.integrate_geodesic()
        self._pen_data = self.bh.penrose_diagram_data()

        print("► Computing gravitational wave chirp …")
        self._gw_data  = self.bh.gw_chirp_strain(
            m_companion_solar=gw_companion_solar)

        print("► Running 3-D Euler (adaptive CFL) …")
        self._eu_data  = self.eu.run(n_steps=self.n_steps,
                                     adaptive_dt=self.adaptive_dt)

        print("► Running 3-D Navier-Stokes (ETD-RK4) …")
        self._ns_data  = self.ns.run(n_steps=self.n_steps,
                                     adaptive_dt=self.adaptive_dt)
        print("✔  All simulations complete.")

    @property
    def blackhole(self) -> BlackHoleSingularity:   return self.bh
    @property
    def euler(self)     -> Euler3D:                return self.eu
    @property
    def navier_stokes(self) -> NavierStokes3D:     return self.ns
    @property
    def euler_data(self) -> dict:
        if self._eu_data is None: raise RuntimeError("Call run_all() first.")
        return self._eu_data
    @property
    def ns_data(self) -> dict:
        if self._ns_data is None: raise RuntimeError("Call run_all() first.")
        return self._ns_data
    @property
    def geodesic_data(self) -> dict:
        if self._geo_data is None: raise RuntimeError("Call run_all() first.")
        return self._geo_data
    @property
    def gw_data(self) -> dict:
        if self._gw_data is None: raise RuntimeError("Call run_all() first.")
        return self._gw_data

    def bkm_criterion(self) -> dict:
        def integrate_omMax(data):
            t   = data["t"]
            om  = data["omMax"]
            dt  = np.diff(t, prepend=t[0])
            return np.cumsum(om * dt)

        return dict(
            euler  = integrate_omMax(self._eu_data),
            ns     = integrate_omMax(self._ns_data),
            t_euler= self._eu_data["t"],
            t_ns   = self._ns_data["t"],
        )

    def compare_singularities(self) -> dict:
        if self._eu_data is None or self._ns_data is None:
            raise RuntimeError("Call run_all() first.")

        bkm   = self.bkm_criterion()
        r_arr = np.linspace(1.01*self.bh.r_plus, 20*self.bh.r_plus, 500)
        K     = self.bh.kretschner_scalar(r_arr)
        K_norm= K / K[0]

        eu_bkm_norm = bkm["euler"] / (bkm["euler"].max() + 1e-300)
        ns_bkm_norm = bkm["ns"]   / (bkm["ns"].max()    + 1e-300)

        return dict(
            r_over_rplus = r_arr / self.bh.r_plus,
            K_norm       = K_norm,
            t_euler      = bkm["t_euler"],
            euler_bkm    = eu_bkm_norm,
            t_ns         = bkm["t_ns"],
            ns_bkm       = ns_bkm_norm,
            bh_label     = self.bh._label,
            eu_label     = f"Euler N={self.eu.N}",
            ns_label     = f"NS N={self.ns.N} ν={self.ns.nu:.1e}",
        )

    def hawking_temperature(self)  -> float: return self.bh.T_H
    def bekenstein_entropy(self)   -> float: return self.bh.S_BH
    def penrose_process_efficiency(self) -> float: return self.bh.efficiency
    def isco_radius(self)          -> float: return self.bh.r_isco
    def spaghettification_radius(self)   -> float:
        return self.bh.spaghettification_radius()
    def kolmogorov_scales(self)    -> dict:  return self.ns.kolmogorov_scales()

    def singularity_strength(self, r: np.ndarray = None) -> np.ndarray:
        if r is None:
            r = np.linspace(1.01*self.bh.r_plus, 20*self.bh.r_plus, 500)
        return self.bh.kretschner_scalar(r)

    def simulate(self, mode: str = "all", n_steps: int = None,
                 return_raw: bool = False) -> dict:
        if n_steps is not None:
            self.n_steps = n_steps

        if mode in ("all", "blackhole") and self._geo_data is None:
            self._geo_data = self.bh.integrate_geodesic()
            self._pen_data = self.bh.penrose_diagram_data()
            self._gw_data  = self.bh.gw_chirp_strain()

        if mode in ("all", "euler") and self._eu_data is None:
            self._eu_data = self.eu.run(n_steps=self.n_steps,
                                        adaptive_dt=self.adaptive_dt)

        if mode in ("all", "ns") and self._ns_data is None:
            self._ns_data = self.ns.run(n_steps=self.n_steps,
                                        adaptive_dt=self.adaptive_dt)

        out = dict(
            bh_temperature   = self.bh.T_H,
            bh_entropy       = self.bh.S_BH,
            bh_r_s           = self.bh.r_s,
            bh_r_plus        = self.bh.r_plus,
            bh_r_isco        = self.bh.r_isco,
            bh_r_photo       = self.bh.r_photo,
            bh_spin          = self.bh.a_star,
            bh_efficiency    = self.bh.efficiency,
            bh_omega_H       = self.bh.Omega_H,
            bh_spaghetti_r   = self.bh.spaghettification_radius(),
        )
        if self._geo_data:
            out["bh_geodesic"]  = self._geo_data
        if self._pen_data:
            out["bh_penrose"]   = self._pen_data
        if self._gw_data:
            out["bh_gw"]        = self._gw_data
        if self._eu_data:
            out.update(euler_E      = self._eu_data["E"],
                       euler_W      = self._eu_data["W"],
                       euler_H      = self._eu_data["H"],
                       euler_omMax  = self._eu_data["omMax"],
                       euler_Palin  = self._eu_data["Palin"],
                       t_euler      = self._eu_data["t"])
        if self._ns_data:
            out.update(ns_E         = self._ns_data["E"],
                       ns_W         = self._ns_data["W"],
                       ns_H         = self._ns_data["H"],
                       ns_omMax     = self._ns_data["omMax"],
                       ns_eps       = self._ns_data["eps"],
                       ns_Re_lam    = self._ns_data["Re_lam"],
                       t_ns         = self._ns_data["t"],
                       kolmogorov   = self._ns_data.get("kolmogorov", {}))
        if self._eu_data and self._ns_data:
            out["bkm_integral"]      = self.bkm_criterion()
            out["singularity_cross"] = self.compare_singularities()
        return out


# ══════════════════════════════════════════════════════════════════════════════
#  visualization  (wraps SingularitySimulation)
# ══════════════════════════════════════════════════════════════════════════════

"""
visualization_robust.py  –  v2  (color-clarity + layout overhaul)
===================================================================
Complete rewrite of the 24-panel SingularitySimulation figure.

Key fixes vs v1
---------------
* Colorbars placed INSIDE the axes ([0.79, 0.02, 0.04, 0.96]) so their
  tick labels never bleed into neighbouring panels – the root cause of the
  text-chaos in rows 4-5.
* Every panel wrapped in try/except -> one bad panel never crashes the figure.
* Bolder line widths, brighter palette, higher-contrast grid.
* Snapshot panels: rms badge as a tight bbox; titles kept short.
* Section headers now span the full width as coloured fig.text banners.
* Effective-potential panel has a hard-coded Schwarzschild fallback so it
  never appears blank.
* All data access goes through _sget / _safe helpers (no raw dict indexing).
"""

from __future__ import annotations

import traceback
import warnings
from typing import Any, Dict, Optional, Tuple

import numpy as np
import matplotlib
import matplotlib.axes as maxes
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

C_L: float = 2.99792458e8          # speed of light [m/s]

# ══════════════════════════════════════════════════════════════════════════════
#  PALETTE
# ══════════════════════════════════════════════════════════════════════════════

BG_DEEP   = "#1a1f2e"
BG_PANEL  = "#0b1220"
BG_PANEL2 = "#0f1a2e"
GRID_COL  = "#1b2d44"
BORDER    = "#2a4870"

CLR_EU    = "#00e5ff"
CLR_NS    = "#ff4d6d"
CLR_BH    = "#ffb347"
CLR_KRET  = "#aaff00"
CLR_GW    = "#cc80ff"
CLR_HAWK  = "#ffa040"
CLR_EPS   = "#ff80ab"
CLR_RE    = "#64ffda"
CLR_PALIN = "#ea80fc"
CLR_HEL   = "#80d8ff"
CLR_ISCO  = "#76ff03"

TXT       = "#e8f4ff"
TXT_DIM   = "#6a8faf"
WHITE     = "#ffffff"

matplotlib.rcParams.update({
    "font.family"        : "monospace",
    "axes.unicode_minus" : False,
    "figure.dpi"         : 120,
})

# ══════════════════════════════════════════════════════════════════════════════
#  SAFE DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _safe(arr: Any, fallback: Any = None) -> Any:
    try:
        a = np.asarray(arr, dtype=float)
        return a if (a.size > 0 and np.any(np.isfinite(a))) else fallback
    except Exception:
        return fallback


def _sget(m: Any, key: str, fallback: Any = None) -> Any:
    try:
        return m[key]
    except (KeyError, TypeError):
        pass
    try:
        return getattr(m, key, fallback)
    except Exception:
        return fallback


def _safe_max(arr: Any, fallback: float = 1e-10) -> float:
    try:
        v = float(np.nanmax(np.abs(np.asarray(arr, dtype=float))))
        return v if (np.isfinite(v) and v > 0) else fallback
    except Exception:
        return fallback


def _clamp(arr: np.ndarray, lo: float = -1e30, hi: float = 1e30) -> np.ndarray:
    return np.clip(
        np.nan_to_num(np.asarray(arr, dtype=float),
                      nan=0.0, posinf=hi, neginf=lo), lo, hi)


def _bh_attr(bh: Any, attr: str, fallback: float = 0.0) -> float:
    try:
        v = float(getattr(bh, attr))
        return v if np.isfinite(v) else fallback
    except Exception:
        return fallback

# ══════════════════════════════════════════════════════════════════════════════
#  MATPLOTLIB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _ax_style(ax: plt.Axes, title: str,
              xlabel: str = "", ylabel: str = "",
              bg: str = BG_PANEL) -> None:
    ax.set_facecolor(bg)
    ax.tick_params(colors=TXT_DIM, labelsize=7, length=3, width=0.6)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(0.9)
    ax.xaxis.label.set_color(TXT_DIM)
    ax.yaxis.label.set_color(TXT_DIM)
    ax.set_title(title, color=TXT, fontsize=8, pad=4,
                 fontfamily="monospace", fontweight="bold")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=7, labelpad=2)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=7, labelpad=2)
    ax.grid(color=GRID_COL, linestyle="-", linewidth=0.4, alpha=0.9)
    ax.set_axisbelow(True)


def _vline(ax: plt.Axes, x: float, color: str = WHITE,
           lw: float = 0.8, ls: str = "--",
           label: Optional[str] = None) -> None:
    if not np.isfinite(x):
        return
    kw: Dict[str, Any] = dict(color=color, lw=lw, ls=ls, alpha=0.85)
    if label:
        kw["label"] = label
    ax.axvline(x, **kw)


def _legend(ax: plt.Axes, **kw: Any) -> None:
    handles, _ = ax.get_legend_handles_labels()
    if not handles:
        return
    leg = ax.legend(fontsize=6, facecolor=BG_PANEL2, labelcolor=TXT,
                    edgecolor=BORDER, framealpha=0.92, **kw)
    for ln in leg.get_lines():
        ln.set_linewidth(2.0)


def _colorbar(fig: plt.Figure, im: Any, ax: Any,
              label: str = "") -> Optional[Any]:
    """
    Inset colorbar placed INSIDE the axes (x = 79-83%) so tick labels
    never bleed into neighbouring panels.  Argument-order self-healing included.
    """
    try:
        if isinstance(im, maxes.Axes) and not isinstance(ax, maxes.Axes):
            im, ax = ax, im
        if not isinstance(ax, maxes.Axes) and hasattr(ax, "axes"):
            ax = ax.axes
        if not isinstance(ax, maxes.Axes):
            return None
        if isinstance(im, maxes.Axes):
            imgs = im.get_images()
            cols = getattr(im, "collections", [])
            im   = imgs[-1] if imgs else (cols[-1] if cols else None)
        if im is None or not isinstance(im, ScalarMappable):
            return None
        # *** INSIDE the axes -- no overflow into neighbouring panels ***
        cax = ax.inset_axes([0.79, 0.02, 0.04, 0.96])
        cax.set_facecolor(BG_PANEL)
        cb = fig.colorbar(im, cax=cax)
        cb.ax.tick_params(colors=TXT_DIM, labelsize=5, length=2, width=0.5)
        cb.outline.set_edgecolor(BORDER)
        cb.outline.set_linewidth(0.6)
        if label:
            cb.set_label(label, color=TXT_DIM, fontsize=5)
        return cb
    except Exception:
        traceback.print_exc()
        return None


def _panel_error(ax: plt.Axes, msg: str) -> None:
    ax.set_facecolor(BG_PANEL)
    short = msg.split("\n")[0][:60]
    ax.text(0.5, 0.5, f"panel error\n{short}",
            transform=ax.transAxes, ha="center", va="center",
            color=CLR_NS, fontsize=6.5, fontfamily="monospace")
    ax.set_xticks([])
    ax.set_yticks([])


def _section_bar(fig: plt.Figure, y: float, label: str, color: str) -> None:
    fig.text(0.03, y, label, ha="left", va="center",
             color=color, fontsize=6.8, fontweight="bold",
             fontfamily="monospace")
    fig.add_artist(matplotlib.lines.Line2D(
        [0.03, 0.97], [y - 0.002, y - 0.002],
        transform=fig.transFigure,
        color=color, linewidth=0.5, alpha=0.4))

# ══════════════════════════════════════════════════════════════════════════════
#  SPECTRAL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def _energy_spectrum_2d(snap: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    snap = np.asarray(snap, dtype=float)
    N = snap.shape[0]
    if N < 4:
        return np.array([1.0]), np.array([0.0])
    wh   = np.fft.fft2(snap)
    E2   = 0.5 * np.abs(wh) ** 2 / max(N ** 4, 1)
    kx   = np.fft.fftfreq(N, d=1.0 / N)
    ky   = np.fft.fftfreq(N, d=1.0 / N)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    Kmag = np.round(np.sqrt(KX**2 + KY**2)).astype(int).ravel()
    k_max  = max(N // 3, 1)
    k_bins = np.arange(1, k_max + 1, dtype=float)
    Ek     = np.zeros(k_max, dtype=float)
    E2_r   = E2.ravel()
    for ki in range(1, k_max + 1):
        m = Kmag == ki
        if m.any():
            Ek[ki - 1] = float(E2_r[m].sum())
    return k_bins, np.nan_to_num(Ek, nan=0.0, posinf=0.0, neginf=0.0)

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN  plot_all  --  24-panel figure
# ══════════════════════════════════════════════════════════════════════════════

def plot_all(sim: Any, save_path: Optional[str] = None) -> plt.Figure:

    eu  = _sget(sim, "_eu_data") or {}
    ns  = _sget(sim, "_ns_data") or {}
    geo = _sget(sim, "_geo_data") or {}
    pen = _sget(sim, "_pen_data") or {}
    bh  = _sget(sim, "bh")
    gw  = _sget(sim, "_gw_data") or {}

    def _bh(attr: str, fb: float = 0.0) -> float:
        return _bh_attr(bh, attr, fb)

    rp = max(_bh("r_plus", 1.0), 1e-30)
    th = np.linspace(0, 2 * np.pi, 400)

    # ── Figure ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(30, 40))
    fig.patch.set_facecolor(BG_DEEP)

    outer = gridspec.GridSpec(
        6, 4, figure=fig,
        hspace=0.56, wspace=0.42,
        left=0.055, right=0.975,
        top=0.960,  bottom=0.022,
    )

    # ── Master title ──────────────────────────────────────────────────────────
    fig.text(
        0.5, 0.978,
        "SINGULARITY SIMULATION  x  KERR BLACK HOLE"
        "  x  3-D EULER  x  3-D NAVIER-STOKES",
        ha="center", va="top", color=CLR_BH,
        fontsize=14, fontweight="bold", fontfamily="monospace")

    subtitle = (
        f"M = {_bh('M_solar'):.0f} Msun   "
        f"a* = {_bh('a_star'):.3f}   "
        f"r+ = {rp:.3e} m   "
        f"T_H = {_bh('T_H'):.3e} K   "
        f"S_BH = {_bh('S_BH'):.3e} nat   "
        f"eta_Penrose = {_bh('efficiency')*100:.2f}%"
    )
    fig.text(0.5, 0.970, subtitle, ha="center", va="top",
             color=TXT_DIM, fontsize=8, fontfamily="monospace")

    _section_bar(fig, 0.880,
                 "--- GR / KERR BLACK HOLE -----------------------------------------------------------",
                 CLR_BH)
    _section_bar(fig, 0.715,
                 "--- GRAVITATIONAL WAVES  .  HAWKING SPECTRUM --------------------------------------",
                 CLR_GW)
    _section_bar(fig, 0.548,
                 "--- FLUID DYNAMICS  .  GLOBAL DIAGNOSTICS -----------------------------------------",
                 CLR_EU)
    _section_bar(fig, 0.382,
                 "--- ENERGY SPECTRUM  .  DISSIPATION  .  Re_lambda ---------------------------------",
                 CLR_NS)
    _section_bar(fig, 0.218,
                 "--- VORTICITY SNAPSHOTS  .  EULER --------------------------------------------------",
                 CLR_EU)
    _section_bar(fig, 0.056,
                 "--- VORTICITY SNAPSHOTS  .  NAVIER-STOKES  .  STRUCTURE FUNCTIONS -----------------",
                 CLR_NS)

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 0 -- Kerr GR
    # ══════════════════════════════════════════════════════════════════════════

    # 0-0  Geodesic -----------------------------------------------------------
    ax00 = fig.add_subplot(outer[0, 0])
    try:
        _ax_style(ax00, "KERR GEODESIC", "x / r+", "y / r+")
        gx = _safe(_sget(geo, "x"))
        gy = _safe(_sget(geo, "y"))
        if gx is not None:
            ax00.plot(gx / rp, gy / rp,
                      color=CLR_BH, lw=1.4, label="Geodesic", zorder=3)
        ax00.fill(np.cos(th), np.sin(th), color="#00000f", zorder=4)
        ax00.plot(np.cos(th), np.sin(th), color=CLR_BH, lw=1.2, zorder=5,
                  label=f"r+ = {rp:.2e} m")
        for r_attr, col, ls, lbl in [
            ("r_isco",    CLR_ISCO, "--", f"ISCO ({_bh('r_isco')/rp:.1f}r+)"),
            ("r_photo",   CLR_NS,   ":",  f"photon orbit"),
            ("r_ergo_eq", CLR_GW,   "-.", "Ergosphere"),
        ]:
            rv = _bh(r_attr)
            if rv > 0:
                rn = rv / rp
                ax00.plot(rn * np.cos(th), rn * np.sin(th),
                          color=col, lw=0.9, ls=ls, label=lbl, alpha=0.9)
        if _sget(geo, "terminated") and gx is not None:
            ax00.plot(gx[-1] / rp, gy[-1] / rp,
                      "x", color=CLR_NS, ms=6, zorder=6, label="-> horizon")
        ax00.set_aspect("equal")
        ax00.set_xlim(-12, 12)
        ax00.set_ylim(-12, 12)
        _legend(ax00, loc="upper right")
    except Exception:
        _panel_error(ax00, traceback.format_exc(limit=2))

    # 0-1  Effective potential ------------------------------------------------
    ax01 = fig.add_subplot(outer[0, 1])
    try:
        _ax_style(ax01, "EFFECTIVE POTENTIAL  V_eff(r)", "r / r+", "V_eff (norm.)")
        r_arr   = np.linspace(1.3 * rp, 22 * rp, 800)
        plotted = False
        for L_fac, col in [(2.0, CLR_NS), (3.5, CLR_BH), (5.5, CLR_EU)]:
            try:
                L_ = L_fac * rp * C_L
                V  = np.asarray(
                    bh.effective_potential(r_arr, L=L_, E=0.95, mu=1.0),
                    dtype=float)
                V  = _clamp(V)
                mn = _safe_max(V, 1.0)
                ax01.plot(r_arr / rp, V / mn, color=col, lw=1.6,
                          label=f"L = {L_fac:.1f} r+c")
                plotted = True
            except Exception:
                pass
        if not plotted:
            # Schwarzschild fallback: V_eff = 1 - r_s/r + L^2/r^2 - L^2*r_s/r^3
            r_s = _bh("r_s", rp * 2)
            for L_fac, col in [(2.0, CLR_NS), (3.5, CLR_BH), (5.5, CLR_EU)]:
                L_ = L_fac * rp
                V  = (1 - r_s / r_arr
                      + L_**2 / r_arr**2
                      - L_**2 * r_s / r_arr**3)
                mn = _safe_max(V, 1.0)
                ax01.plot(r_arr / rp, V / mn, color=col, lw=1.6,
                          label=f"L = {L_fac:.1f} r+c (Schw.)")
        _vline(ax01, 1.0, color=CLR_BH, lw=1.2, label="horizon r+")
        ri = _bh("r_isco") / rp
        if 0 < ri < 25:
            _vline(ax01, ri, color=CLR_ISCO, ls=":", lw=1.0, label="ISCO")
        ax01.set_ylim(-2.2, 2.2)
        ax01.axhline(0, color=BORDER, lw=0.6)
        _legend(ax01, loc="upper right")
    except Exception:
        _panel_error(ax01, traceback.format_exc(limit=2))

    # 0-2  Kretschner curvature -----------------------------------------------
    ax02 = fig.add_subplot(outer[0, 2])
    try:
        _ax_style(ax02, "KRETSCHNER SCALAR  K(r)", "r / r+", "K  [m^-4]")
        r_k = np.linspace(1.02 * rp, 12 * rp, 700)
        K   = np.abs(_clamp(
            np.asarray(bh.kretschner_scalar(r_k), dtype=float),
            lo=0.0, hi=1e300))
        pos = K > 0
        if pos.any():
            ax02.semilogy(r_k[pos] / rp, K[pos], color=CLR_KRET, lw=1.8)
            ax02.fill_between(r_k[pos] / rp, K[pos], K[pos].min(),
                              alpha=0.18, color=CLR_KRET)
        _vline(ax02, 1.0, color=CLR_BH, lw=1.2, label="r+")
        ri = _bh("r_isco") / rp
        if 0 < ri < 25:
            _vline(ax02, ri, color=CLR_ISCO, ls=":", lw=1.0, label="ISCO")
        ax02.set_xlim(0.9, 12)
        K_lo = K[pos].min() if pos.any() else 1e-10
        K_hi = K[pos].max() if pos.any() else 1e-9
        ax02.fill_betweenx([K_lo, K_hi], 0, 1, alpha=0.10, color=CLR_BH)
        ax02.text(0.97, 0.93, "inf singularity\n  at r = 0",
                  transform=ax02.transAxes, ha="right", va="top",
                  color=CLR_KRET, fontsize=7, fontfamily="monospace")
        _legend(ax02)
    except Exception:
        _panel_error(ax02, traceback.format_exc(limit=2))

    # 0-3  Penrose / Kruskal-Szekeres -----------------------------------------
    ax03 = fig.add_subplot(outer[0, 3])
    try:
        _ax_style(ax03, "PENROSE  .  KRUSKAL-SZEKERES", "X", "T")
        r_c       = _safe(_sget(pen, "r_arr"), np.linspace(1, 10, 100))
        metric_tt = _safe(_sget(pen, "metric_tt"))
        if metric_tt is not None:
            n    = len(r_c)
            bg2d = _clamp(np.tile(metric_tt, (n, 1)), lo=-1.0, hi=0.0)
            im03 = ax03.imshow(bg2d, origin="lower",
                               extent=[-1.1, 1.1, -1.3, 1.3],
                               aspect="auto", cmap="RdYlBu_r",
                               alpha=0.6, vmin=-1.0, vmax=0.0)
            _colorbar(fig, im03, ax03, "g_tt")
        X_r = _safe(_sget(pen, "X_krus"))
        T_r = _safe(_sget(pen, "T_krus"))
        if X_r is not None and T_r is not None:
            fin = np.isfinite(X_r) & np.isfinite(T_r)
            if fin.any():
                sx = _safe_max(X_r[fin], 1.0)
                st = _safe_max(T_r[fin], 1.0)
                ax03.plot(X_r[fin] / sx, T_r[fin] / st * 1.2,
                          color=CLR_BH, lw=1.4, label="r = const.")
        xi = np.linspace(-1.0, 1.0, 120)
        ax03.plot( xi,  xi, color=WHITE, lw=0.8, ls="--", alpha=0.5,
                   label="null ray")
        ax03.plot( xi, -xi, color=WHITE, lw=0.8, ls="--", alpha=0.5)
        ax03.plot([-1.1, 0], [-1.1, 0], color=CLR_BH, lw=1.2, alpha=0.7)
        ax03.plot([0,  1.1], [0,  1.1], color=CLR_BH, lw=1.2, alpha=0.7)
        ax03.set_xlim(-1.1, 1.1)
        ax03.set_ylim(-1.3, 1.3)
        ax03.axhline(0, color=BORDER, lw=0.5)
        ax03.axvline(0, color=BORDER, lw=0.5)
        ax03.text(0.5, 0.74, "BH INTERIOR", color=CLR_NS, fontsize=6.5,
                  ha="center", transform=ax03.transAxes,
                  fontfamily="monospace", fontweight="bold")
        ax03.text(0.5, 0.22, "EXTERIOR", color=CLR_EU, fontsize=6.5,
                  ha="center", transform=ax03.transAxes,
                  fontfamily="monospace", fontweight="bold")
        _legend(ax03, loc="lower right")
    except Exception:
        _panel_error(ax03, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 1 -- Hawking + GW + parameter table
    # ══════════════════════════════════════════════════════════════════════════

    # 1-0  Hawking spectrum ---------------------------------------------------
    ax10 = fig.add_subplot(outer[1, 0])
    try:
        _ax_style(ax10, "HAWKING RADIATION SPECTRUM", "nu  [Hz]", "dP/dnu  [W/Hz]")
        hs = {}
        try:
            hs = bh.hawking_spectrum() or {}
        except Exception:
            pass
        nu    = _safe(_sget(hs, "nu"))
        dPdnu = _safe(_sget(hs, "dPdnu"))
        nu_pk = float(_sget(hs, "nu_peak", 0.0))
        if nu is not None and dPdnu is not None and nu_pk > 0:
            dp = np.maximum(dPdnu, 1e-400)
            ax10.semilogy(nu, dp, color=CLR_HAWK, lw=1.8, label="dP/dnu")
            ax10.fill_between(nu, dp, 1e-400, alpha=0.20, color=CLR_HAWK)
            _vline(ax10, nu_pk, color=WHITE, ls=":", lw=1.0, label="nu_peak")
            ax10.text(0.97, 0.93,
                      f"T_H = {_bh('T_H'):.2e} K\n"
                      f"P_tot = {float(_sget(hs,'total_power',0)):.2e} W",
                      transform=ax10.transAxes, ha="right", va="top",
                      color=CLR_HAWK, fontsize=7, fontfamily="monospace")
        else:
            ax10.text(0.5, 0.5,
                      "T_H ~ 0  (stellar BH)\nHawking flux negligible",
                      ha="center", va="center", transform=ax10.transAxes,
                      color=TXT_DIM, fontsize=8, fontfamily="monospace")
        _legend(ax10)
    except Exception:
        _panel_error(ax10, traceback.format_exc(limit=2))

    # 1-1  GW chirp strain ----------------------------------------------------
    ax11 = fig.add_subplot(outer[1, 1])
    try:
        _ax_style(ax11, "GW CHIRP STRAIN  h+(t)", "t  [s]", "h+")
        t_gw = _safe(_sget(gw, "t"))
        hp   = _safe(_sget(gw, "h_plus"))
        hx   = _safe(_sget(gw, "h_cross"))
        if t_gw is not None and hp is not None:
            fin = np.isfinite(hp)
            if fin.any():
                ax11.plot(t_gw[fin], hp[fin], color=CLR_GW, lw=1.4,
                          label="h+", alpha=0.95)
                if hx is not None:
                    fx = np.isfinite(hx)
                    if fx.any():
                        ax11.plot(t_gw[fx], hx[fx], color=CLR_EU, lw=0.9,
                                  ls="--", label="hx", alpha=0.75)
                ax11.set_xlim(t_gw[0], t_gw[fin].max())
        ax11.text(0.03, 0.93,
                  f"Mc = {float(_sget(gw,'Mc_solar',0)):.3f} Msun\n"
                  f"T_chirp = {float(_sget(gw,'T_chirp',0)):.2f} s",
                  transform=ax11.transAxes, va="top",
                  color=CLR_GW, fontsize=7, fontfamily="monospace")
        _legend(ax11)
    except Exception:
        _panel_error(ax11, traceback.format_exc(limit=2))

    # 1-2  GW chirp frequency -------------------------------------------------
    ax12 = fig.add_subplot(outer[1, 2])
    try:
        _ax_style(ax12, "GW CHIRP FREQUENCY  f(t)", "t  [s]", "f  [Hz]")
        t_gw = _safe(_sget(gw, "t"))
        f_t  = _safe(_sget(gw, "f_t"))
        if t_gw is not None and f_t is not None:
            fin = np.isfinite(f_t) & (f_t > 0)
            if fin.any():
                f_min = max(f_t[fin].min() * 0.9, 1e-10)
                ax12.semilogy(t_gw[fin], f_t[fin],
                              color=CLR_GW, lw=1.8, label="f(t)")
                ax12.fill_between(t_gw[fin], f_t[fin], f_min,
                                  alpha=0.20, color=CLR_GW)
                ax12.set_xlim(t_gw[0], t_gw[fin].max())
        ax12.axhline(100, color=TXT_DIM, lw=0.7, ls=":", label="100 Hz")
        _legend(ax12)
    except Exception:
        _panel_error(ax12, traceback.format_exc(limit=2))

    # 1-3  Parameter table ----------------------------------------------------
    ax13 = fig.add_subplot(outer[1, 3])
    try:
        ax13.set_facecolor(BG_PANEL2)
        for sp in ax13.spines.values():
            sp.set_edgecolor(CLR_BH)
            sp.set_linewidth(1.4)
        ax13.set_xticks([])
        ax13.set_yticks([])
        ax13.set_title("KERR BH  PARAMETER TABLE", color=CLR_BH,
                       fontsize=8, pad=5, fontfamily="monospace",
                       fontweight="bold")
        try:
            spag_s = f"{float(bh.spaghettification_radius()):.3e} m"
        except Exception:
            spag_s = "N/A"
        rows = [
            ("Mass",              f"{_bh('M_solar'):.1f} Msun"),
            ("Spin  a*",          f"{_bh('a_star'):.4f}"),
            ("Outer horizon r+",  f"{rp:.4e} m"),
            ("Inner horizon r-",  f"{_bh('r_minus'):.4e} m"),
            ("Schwarzschild r_s", f"{_bh('r_s'):.4e} m"),
            ("ISCO (prograde)",   f"{_bh('r_isco'):.4e} m"),
            ("Photon orbit",      f"{_bh('r_photo'):.4e} m"),
            ("Omega_H (horizon)", f"{_bh('Omega_H'):.4e} rad/s"),
            ("Hawking  T_H",      f"{_bh('T_H'):.4e} K"),
            ("Bekenstein S_BH",   f"{_bh('S_BH'):.4e} nat"),
            ("Penrose eta_max",   f"{_bh('efficiency')*100:.3f} %"),
            ("GW chirp mass Mc",  f"{float(_sget(gw,'Mc_solar',0)):.4f} Msun"),
            ("Spaghetti r",       spag_s),
        ]
        y0, dy = 0.93, 0.068
        for i, (k, v) in enumerate(rows):
            y = y0 - i * dy
            ax13.text(0.04, y, k, transform=ax13.transAxes, va="top",
                      color=TXT_DIM, fontsize=7, fontfamily="monospace")
            ax13.text(0.96, y, v, transform=ax13.transAxes, va="top",
                      ha="right", color=CLR_BH, fontsize=7,
                      fontfamily="monospace", fontweight="bold")
            if i < len(rows) - 1:
                ax13.axhline(y - dy * 0.42,
                             color=GRID_COL, lw=0.35, xmin=0.02, xmax=0.98)
    except Exception:
        _panel_error(ax13, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 2 -- Fluid global diagnostics
    # ══════════════════════════════════════════════════════════════════════════

    # 2-0  Kinetic energy -----------------------------------------------------
    ax20 = fig.add_subplot(outer[2, 0])
    try:
        _ax_style(ax20, "KINETIC ENERGY  E(t)", "tau", "E = 1/2 <|u|^2>")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            v = _safe(_sget(data, "E"))
            if t is not None and v is not None:
                ax20.plot(t, v, color=col, lw=1.6, ls=ls, label=lbl)
                ax20.fill_between(t, v, alpha=0.09, color=col)
        _legend(ax20)
    except Exception:
        _panel_error(ax20, traceback.format_exc(limit=2))

    # 2-1  Enstrophy ----------------------------------------------------------
    ax21 = fig.add_subplot(outer[2, 1])
    try:
        _ax_style(ax21, "ENSTROPHY  Omega(t)", "tau", "Omega")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            v = _safe(_sget(data, "W"))
            if t is not None and v is not None:
                ax21.semilogy(t, np.maximum(v, 1e-30),
                              color=col, lw=1.6, ls=ls, label=lbl)
        _legend(ax21)
    except Exception:
        _panel_error(ax21, traceback.format_exc(limit=2))

    # 2-2  Helicity -----------------------------------------------------------
    ax22 = fig.add_subplot(outer[2, 2])
    try:
        _ax_style(ax22, "HELICITY  H(t) = int u.omega dV", "tau", "H")
        for data, col, ls, lbl in [
            (eu, CLR_HEL, "-",  "Euler"),
            (ns, CLR_EPS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            H = _safe(_sget(data, "H"))
            if t is not None and H is not None:
                ax22.plot(t, H, color=col, lw=1.6, ls=ls, label=lbl)
        ax22.axhline(0, color=BORDER, lw=0.6)
        _legend(ax22)
    except Exception:
        _panel_error(ax22, traceback.format_exc(limit=2))

    # 2-3  BKM integral -------------------------------------------------------
    ax23 = fig.add_subplot(outer[2, 3])
    try:
        _ax_style(ax23, "BKM INTEGRAL  int ||omega||_inf dtau",
                  "tau", "int ||omega||_inf dtau")
        bkm = {}
        try:
            bkm = sim.bkm_criterion() or {}
        except Exception:
            pass
        for key_t, key_v, col, ls, lbl in [
            ("t_euler", "euler", CLR_EU, "-",  "Euler"),
            ("t_ns",    "ns",    CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(bkm, key_t))
            v = _safe(_sget(bkm, key_v))
            if t is not None and v is not None:
                ax23.plot(t, v, color=col, lw=1.6, ls=ls, label=lbl)
                ax23.fill_between(t, v, alpha=0.12, color=col)
        ax23.text(0.03, 0.93,
                  "Blowup only if integral -> inf\n(Beale-Kato-Majda 1984)",
                  transform=ax23.transAxes, va="top",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax23)
    except Exception:
        _panel_error(ax23, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 3 -- Spectrum, max vorticity, dissipation, Re_lambda
    # ══════════════════════════════════════════════════════════════════════════

    # 3-0  Energy spectrum ----------------------------------------------------
    ax30 = fig.add_subplot(outer[3, 0])
    try:
        _ax_style(ax30, "KINETIC ENERGY SPECTRUM  E(k)",
                  "k  [wavenumber]", "E(k)")
        eu_snaps = sorted((_sget(eu, "snapshots") or {}).items())
        ns_snaps = sorted((_sget(ns, "snapshots") or {}).items())
        ref_k = ref_E = None
        for lbl_, snaps_, col_ in [("Euler", eu_snaps, CLR_EU),
                                    ("NS",    ns_snaps, CLR_NS)]:
            if not snaps_:
                continue
            try:
                k_b, Ek = _energy_spectrum_2d(snaps_[-1][1])
                v = (Ek > 0) & np.isfinite(Ek)
                if v.any():
                    ax30.loglog(k_b[v], Ek[v], color=col_, lw=1.8, label=lbl_)
                    if lbl_ == "Euler":
                        ref_k, ref_E = k_b, Ek
            except Exception:
                pass
        if ref_k is not None and ref_E is not None:
            vv = ref_E > 0
            if vv.sum() > 3:
                idx = vv.nonzero()[0][len(vv.nonzero()[0]) // 3]
                E0 = ref_E[idx]
                k0 = ref_k[idx]
                kr = np.logspace(np.log10(ref_k[vv][0]),
                                 np.log10(ref_k[vv][-1]), 50)
                ax30.loglog(kr, E0 * (kr / k0) ** (-5/3),
                            color=TXT_DIM, lw=1.0, ls="--",
                            label="k^(-5/3) K41")
        _legend(ax30)
    except Exception:
        _panel_error(ax30, traceback.format_exc(limit=2))

    # 3-1  Max vorticity ------------------------------------------------------
    ax31 = fig.add_subplot(outer[3, 1])
    try:
        _ax_style(ax31, "MAX VORTICITY  ||omega||_inf(t)",
                  "tau", "||omega||_inf")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t  = _safe(_sget(data, "t"))
            om = _safe(_sget(data, "omMax"))
            if t is not None and om is not None:
                ax31.plot(t, om, color=col, lw=1.6, ls=ls, label=lbl)
        _legend(ax31)
    except Exception:
        _panel_error(ax31, traceback.format_exc(limit=2))

    # 3-2  Dissipation & palinstrophy -----------------------------------------
    ax32 = fig.add_subplot(outer[3, 2])
    try:
        _ax_style(ax32, "DISSIPATION eps(t)  .  PALINSTROPHY P(t)",
                  "tau", "eps")
        ax32r = ax32.twinx()
        ax32r.tick_params(colors=TXT_DIM, labelsize=7)
        ax32r.yaxis.label.set_color(CLR_PALIN)
        for sp in ax32r.spines.values():
            sp.set_edgecolor(BORDER)
        t_ns = _safe(_sget(ns, "t"))
        eps  = _safe(_sget(ns, "eps"))
        if t_ns is not None and eps is not None:
            ax32.plot(t_ns, np.maximum(eps, 1e-30),
                      color=CLR_EPS, lw=1.6, label="eps (NS)")
        for data, ls, alpha in [(eu, "--", 0.85), (ns, "-", 0.55)]:
            t = _safe(_sget(data, "t"))
            p = _safe(_sget(data, "Palin"))
            if t is not None and p is not None:
                ax32r.semilogy(t, np.maximum(p, 1e-30),
                               color=CLR_PALIN, lw=1.2, ls=ls, alpha=alpha)
        ax32r.set_ylabel("P(t)  [log]", fontsize=7, color=CLR_PALIN)
        ax32.text(0.04, 0.93, "solid = eps   dashed = P(t)",
                  transform=ax32.transAxes, va="top",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax32)
    except Exception:
        _panel_error(ax32, traceback.format_exc(limit=2))

    # 3-3  Taylor Reynolds ----------------------------------------------------
    ax33 = fig.add_subplot(outer[3, 3])
    try:
        _ax_style(ax33, "TAYLOR REYNOLDS  Re_lambda(t)  .  NS",
                  "tau", "Re_lambda")
        t_ns  = _safe(_sget(ns, "t"))
        Re_la = _safe(_sget(ns, "Re_lam"))
        if t_ns is not None and Re_la is not None:
            ax33.plot(t_ns, Re_la, color=CLR_RE, lw=1.8, label="Re_lambda")
            ax33.fill_between(t_ns, Re_la, alpha=0.14, color=CLR_RE)
        kol = _sget(ns, "kolmogorov") or {}
        if not kol:
            try:
                kol = sim.ns.kolmogorov_scales() or {}
            except Exception:
                kol = {}
        if kol:
            ax33.text(0.97, 0.95,
                      f"eta   = {kol.get('eta',0):.4f}\n"
                      f"tau_eta = {kol.get('tau_eta',0):.4f}\n"
                      f"v_eta = {kol.get('v_eta',0):.4f}\n"
                      f"<Re_lam> = {kol.get('Re_lambda_mean',0):.1f}",
                      transform=ax33.transAxes, va="top", ha="right",
                      color=CLR_RE, fontsize=6.8, fontfamily="monospace")
        _legend(ax33)
    except Exception:
        _panel_error(ax33, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 4 -- Euler vorticity snapshots + cross-comparison
    # ══════════════════════════════════════════════════════════════════════════

    eu_snaps_d = _sget(eu, "snapshots") or {}
    eu_times   = sorted(eu_snaps_d.keys())[:3]

    for ci, t_snap in enumerate(eu_times):
        ax4 = fig.add_subplot(outer[4, ci])
        try:
            snap = np.asarray(eu_snaps_d[t_snap], dtype=float)
            vmax = _safe_max(snap, 1e-10)
            im4  = ax4.imshow(snap.T, origin="lower", cmap="seismic",
                              vmin=-vmax, vmax=vmax, aspect="auto",
                              extent=[0, 2*np.pi, 0, 2*np.pi])
            _ax_style(ax4, f"EULER wz  tau={t_snap:.3f}", "x", "y")
            _colorbar(fig, im4, ax4)
            rms = float(np.sqrt(np.mean(snap**2)))
            ax4.text(0.02, 0.96, f"rms={rms:.3f}",
                     transform=ax4.transAxes, va="top",
                     color=CLR_EU, fontsize=6.5, fontfamily="monospace",
                     bbox=dict(facecolor=BG_PANEL, edgecolor="none",
                               alpha=0.75, pad=1.5))
        except Exception:
            _panel_error(ax4, traceback.format_exc(limit=2))

    for ci in range(len(eu_times), 3):
        _ax_style(fig.add_subplot(outer[4, ci]),
                  "EULER wz  (no snapshot)", "x", "y")

    # 4-3  Cross-comparison ---------------------------------------------------
    ax43 = fig.add_subplot(outer[4, 3])
    try:
        _ax_style(ax43, "SINGULARITY STRENGTH (norm.)", "", "value (norm.)")
        cross = {}
        try:
            cross = sim.compare_singularities() or {}
        except Exception:
            pass
        ax43_top = ax43.twiny()
        ax43_top.tick_params(colors=CLR_BH, labelsize=7)
        ax43_top.set_xlabel("r / r+  (GR)", fontsize=7,
                            color=CLR_BH, labelpad=2)
        r_n = _safe(_sget(cross, "r_over_rplus"))
        K_n = _safe(_sget(cross, "K_norm"))
        if r_n is not None and K_n is not None and (K_n > 0).any():
            ax43_top.semilogy(r_n, np.maximum(K_n, 1e-30),
                              color=CLR_BH, lw=1.8, label="K(r)/K_ref  [GR]")
            ax43_top.set_xlim(r_n[0], r_n[-1])
        for kt, kv, col, ls, lbl in [
            ("t_euler", "euler_bkm", CLR_EU, "-",  "BKM Euler"),
            ("t_ns",    "ns_bkm",   CLR_NS, "--", "BKM NS"),
        ]:
            t = _safe(_sget(cross, kt))
            v = _safe(_sget(cross, kv))
            if t is not None and v is not None:
                tm = float(t.max()) if t.max() > 0 else 1.0
                ax43.plot(t / tm, v, color=col, lw=1.8, ls=ls, label=lbl)
        ax43.set_xlabel("tau / tau_max  (fluid)",
                        fontsize=7, color=TXT_DIM, labelpad=2)
        ax43.text(0.03, 0.05,
                  "K(r)->inf as r->0  [GR]\nint||omega||->inf => blowup [BKM]",
                  transform=ax43.transAxes, va="bottom",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax43, loc="upper right")
    except Exception:
        _panel_error(ax43, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 5 -- NS vorticity snapshots + structure functions
    # ══════════════════════════════════════════════════════════════════════════

    ns_snaps_d = _sget(ns, "snapshots") or {}
    ns_times   = sorted(ns_snaps_d.keys())[:3]

    for ci, t_snap in enumerate(ns_times):
        ax5 = fig.add_subplot(outer[5, ci])
        try:
            snap = np.asarray(ns_snaps_d[t_snap], dtype=float)
            vmax = _safe_max(np.abs(snap), 1e-10)
            im5  = ax5.imshow(np.abs(snap.T), origin="lower", cmap="inferno",
                              vmin=0, vmax=vmax, aspect="auto",
                              extent=[0, 2*np.pi, 0, 2*np.pi])
            _ax_style(ax5, f"NS |wz|  tau={t_snap:.3f}", "x", "y")
            _colorbar(fig, im5, ax5)
            rms = float(np.sqrt(np.mean(snap**2)))
            ax5.text(0.02, 0.96, f"rms={rms:.3f}",
                     transform=ax5.transAxes, va="top",
                     color=CLR_NS, fontsize=6.5, fontfamily="monospace",
                     bbox=dict(facecolor=BG_PANEL, edgecolor="none",
                               alpha=0.75, pad=1.5))
        except Exception:
            _panel_error(ax5, traceback.format_exc(limit=2))

    for ci in range(len(ns_times), 3):
        _ax_style(fig.add_subplot(outer[5, ci]),
                  "NS |wz|  (no snapshot)", "x", "y")

    # 5-3  Structure functions ------------------------------------------------
    ax53 = fig.add_subplot(outer[5, 3])
    try:
        _ax_style(ax53, "STRUCTURE FUNCTIONS  S_p(r)",
                  "r  [code units]", "S_p(r)")
        if ns_times:
            ns_last = np.asarray(ns_snaps_d[ns_times[-1]], dtype=float)
            N_s     = ns_last.shape[0]
            r_sh    = np.linspace(1, max(N_s // 2, 2), 15).astype(int)
            r_ph    = r_sh * (2 * np.pi / max(N_s, 1))
            for p, col in zip([2, 4, 6], [CLR_EU, CLR_BH, CLR_NS]):
                Sp = np.array([
                    float(np.mean(
                        np.abs(np.roll(ns_last, int(r), axis=0) - ns_last) ** p))
                    for r in r_sh])
                v = (Sp > 0) & np.isfinite(Sp)
                if v.any():
                    ax53.loglog(r_ph[v], Sp[v], color=col, lw=1.6,
                                marker=".", ms=4, label=f"S_{p}(r)")
            if len(r_ph) >= 6:
                rr = np.logspace(np.log10(r_ph[1]),
                                 np.log10(r_ph[-1]), 30)
                ax53.loglog(rr, 0.3 * (rr / r_ph[5]) ** (2/3),
                            color=TXT_DIM, lw=1.0, ls=":",
                            label="r^(2/3)  K41")
        kol_f = _sget(ns, "kolmogorov") or {}
        if not kol_f:
            try:
                kol_f = sim.ns.kolmogorov_scales() or {}
            except Exception:
                kol_f = {}
        if kol_f:
            ax53.text(0.97, 0.05,

                     "Kolmogorov scales\n"
                     r"$\eta$ = "        + f"{kol_f.get(r'$\eta$', 0):.4f}\n"
                     r"$\tau_{\eta}$ = " + f"{kol_f.get(r'$\tau_{\eta}$', 0):.4f}\n"
                     r"$v_{\eta}$ = "    + f"{kol_f.get('v_eta', 0):.4f}\n"
                     r"$\langle \varepsilon \rangle$ = " + f"{kol_f.get('eps_mean', 0):.3e}\n"
                     r"$\langle Re_{\lambda} \rangle$ = " + f"{kol_f.get('Re_lambda_mean', 0):.1f}",
                     transform=ax53.transAxes, va="bottom", ha="right",
                     color=CLR_RE, fontsize=6.5, fontfamily="monospace",
                     bbox=dict(facecolor=BG_PANEL2, edgecolor=BORDER,
                                boxstyle="round,pad=0.4", alpha=0.92))
        _legend(ax53, loc="upper left")
    except Exception:
        _panel_error(ax53, traceback.format_exc(limit=2))

    # ── Footer ────────────────────────────────────────────────────────────────
    try:
        eu_obj = getattr(sim, "eu", None)
        ns_obj = getattr(sim, "ns", None)
        footer = (
            f"BH: {getattr(bh,'_label','Kerr')}   "
            f"Euler: N={getattr(eu_obj,'N','?')}   "
            f"NS: N={getattr(ns_obj,'N','?')}  "
            f"nu={getattr(ns_obj,'nu',0):.1e}   "
            f"steps={getattr(sim,'n_steps','?')}   "
            "[Kerr GR  .  BKM  .  Kolmogorov  .  ETD-RK4  .  adaptive CFL]"
        )
    except Exception:
        footer = "[SingularitySimulation]"
    fig.text(0.5, 0.010, footer, ha="center", va="bottom",
             color=TXT_DIM, fontsize=6.5, fontfamily="monospace")

    # ── Save -----------------------------------------------------------------
    if save_path:
        try:
            plt.savefig(save_path, dpi=150, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            print(f"[OK] Saved -> {save_path}")
        except Exception as exc:
            print(f"[WARN] Save failed: {exc}")

    return fig


# ══════════════════════════════════════════════════════════════════════════════
#  WRAPPER CLASS
# ══════════════════════════════════════════════════════════════════════════════

class visualization:
    """
    Instantiates and runs SingularitySimulation, then renders the
    24-panel figure via plot_all().

    Parameters
    ----------
    training_dataset : unused (kept for API compatibility)
    bh_mass_solar    : BH mass [solar masses]
    fluid_N          : grid points per side
    fluid_dt         : fluid time-step (code units)
    ns_viscosity     : kinematic viscosity (NS solver)
    bh_spin          : dimensionless spin  a* in [0, 1)
    n_steps          : integration steps
    """

    def __init__(
        self,
        training_dataset: Any = None,
        bh_mass_solar: float = 30.0,
        fluid_N:       int   = 32,
        fluid_dt:      float = 1.5e-3,
        ns_viscosity:  float = 8e-4,
        bh_spin:       float = 0.68,
        n_steps:       int   = 900,
    ) -> None:
        self.training_dataset = training_dataset
        try:
            from singularity_simulation import SingularitySimulation  # type: ignore
        except ImportError as exc:
            raise ImportError(
                "SingularitySimulation not found. "
                "Ensure singularity_simulation.py is on sys.path."
            ) from exc

        self.sim = SingularitySimulation(
            bh_mass_solar=bh_mass_solar,
            bh_spin=bh_spin,
            fluid_N=fluid_N,
            fluid_dt=fluid_dt,
            ns_viscosity=ns_viscosity,
            n_steps=n_steps,
        )

    def run_singularity_plot(
        self, save_path: str = "singularity_full.png"
    ) -> plt.Figure:
        """Run simulation and produce the 24-panel figure."""
        self.sim.run_all()
        return plot_all(self.sim, save_path=save_path)


In [ ]:
# @title


"""
=============================================================================
           Fluid Singularity Research Framework
=============================================================================
  Integrates three solvers from the 'Discovery of Unstable Singularities'

    1. IPMSolver         — Incompressible Porous Media
    2. BoussinesqSolver  — 2D Boussinesq (atmospheric/buoyancy flows)
    3. Euler3DSolver     — 3D Euler with wall boundary
=============================================================================
"""

from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.fft import fft2, ifft2, fftfreq
from typing import Optional
import warnings
warnings.filterwarnings("ignore")


#from phi.jax.flow import *
# from phi.flow import *  # If JAX is not installed. You can use phi.torch or phi.tf as well.
#from phi.field._point_cloud import distribute_points
from tqdm.notebook import trange
from scipy import special

from typing import Tuple, Dict
import torch as pt
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from scipy import ndimage

from typing import Callable, Tuple, List, Optional, Union
from functools import partial


from scipy.optimize import NonlinearConstraint
import pymc as pm
import arviz as az
from scipy.integrate import solve_ivp

from scipy.stats import norm

import csv
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd


from scipy.fft import fft, ifft
from scipy.special import expit as sigmoid
from scipy.stats import norm

from typing import Callable, Optional, Union,Tuple,Any
from pymc import Model, Normal, sample
from scipy.stats import norm


from scipy.optimize import minimize
from bayes_opt import BayesianOptimization
from matplotlib import gridspec


import numpy as np
import pandas as pd
from datetime import datetime, timedelta
import random

from scipy.stats import norm
from scipy.interpolate import interp1d
from scipy.stats import norm
from scipy import ndimage
from scipy.sparse.linalg import cg


import scipy.sparse.linalg as splinalg
from scipy import interpolate
from scipy.sparse import diags

from scipy.sparse.linalg import LinearOperator, cg
from scipy.interpolate import RegularGridInterpolator
from scipy.ndimage import laplace
from scipy.optimize import minimize
from scipy.interpolate import griddata

import arviz as az
#%load_ext autoreload
#%autoreload 2

seed = 42
#az.style.use(['arviz-white', 'arviz-plasmish'])

from scipy.special import expit as sigmoid
from scipy.linalg.blas import saxpy
from scipy.stats import norm

from matplotlib import pyplot, cm
from mpl_toolkits.mplot3d import Axes3D
from scipy.integrate import solve_ivp


import torch
import torch.nn as nn
import numpy as np
import scipy.io
from matplotlib import pyplot as plt
import matplotlib.animation as animation


# Define grid parameters
Lx = Ly = Lz = 1.0
Nx = Ny = Nz = 32
dx = dy = dz = Lx / Nx

# Create 3D grid
x = np.linspace(0, Lx, Nx)
y = np.linspace(0, Ly, Ny)
z = np.linspace(0, Lz, Nz)
X, Y, Z = np.meshgrid(x, y, z, indexing='ij')

# Initialize arrays
u = np.zeros((Ny, Nx, Nz))
v = np.zeros((Ny, Nx, Nz))
w = np.zeros((Ny, Nx, Nz))
p = np.zeros((Ny, Nx, Nz))
f = np.zeros((Ny, Nx, Nz))


# Set time parameters
dt = 0.001
t_end = 10.0
t = 0.0


S = 100  # Initial stock price
K = 125  # Strike price
T = 3  # Time to expiration in years
r = 0.05  # Risk-free interest rate
sigma = 0.9  # Volatility
dt = 2  # Time step for simulation
initial_portfolio_value = 150000  # Initial portfolio value
Ts = [40, 50,60,80]  # Time horizons for the rough volatility model

from scipy.stats import expon
U = 1000  # Initial capital
c = 500   # Premium rate
claim_frequency = 0.05  # Claim frequency
F = expon(scale=100)  # Exponential distribution for claim sizes
T_max = 1000000     # Maximum simulation time
jump_max = 50       # Maximum number of jumps


spot_prices = [100, 105,130]
strike_prices = [102, 103,200]
times_to_maturity = [0.5, 1.0, 1.5]
interest_rates = [0.02, 0.03, 0.04]
risk_free_rates = [0.02, 0.03,0.4]
volatilities = [0.20, 0.25,0.7]

num_particles=10000


P = 100  # Initial price
k = 100000  # Constant parameter
alpha = 0.001  # Alpha parameter
mu = 0.02  # Market impact coefficient
theta = 0.005  # Limit order fill rate
beta = 0.002  # Beta parameter
gamma = 0.1  # Gamma parameter
sigma = 0.03  # Volatility
v=0.04


a = 0.02  # Rate of mean reversion
b = 0.05  # Long-term mean
sigma = 0.01  # Volatility
r0 = 0.03  # Initial interest rate
T = 3     # Time horizon
N = 10000  # Number of steps





"""
NS3DSOLved — Unified Fluid Singularity Solver Suite
=========================================================
Integrates the three singularity solvers:
  • _IPMSolver          (2-D Incompressible Porous Media)
  • _BoussinesqSolver   (2-D Inviscid Boussinesq)
  • _Euler3DSolver      (3-D Incompressible Euler)

All following the Wang et al. (2025) / Hou–Luo / Chen–Hou line of work on
stable and unstable finite-time singularities.

Usage
-----
    suite = NS3DSOLved(
        family="S0",
        ipm_config        = dict(N=128, dt=5e-4),
        boussinesq_config = dict(N=128, dt=2e-4, ic_type="hou_luo"),
        euler3d_config    = dict(Nx=32, Ny=32, Nz=32, dt=1e-3, ic_type="hou_luo"),
    )
    suite.run_all()
    suite.plot_all(save_path="singularity_dashboard.png")
    print(suite.full_summary())
"""

from __future__ import annotations

import textwrap
import warnings
from typing import Optional

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as ticker
from matplotlib.colors import LogNorm, SymLogNorm, Normalize
from matplotlib.patches import FancyArrowPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable



matplotlib.rcParams.update({
    "font.family":        "monospace",
    "axes.titlepad":      6,
    "axes.labelpad":      3,
    "figure.dpi":         150,
    "savefig.dpi":        200,
    "axes.grid":          False,
})




"""
NS3DSOLved — Unified Fluid Singularity Solver Suite
=========================================================
Integrates three singularity solvers:
  • _IPMSolver          (2-D Incompressible Porous Media)
  • _BoussinesqSolver   (2-D Inviscid Boussinesq)
  • _Euler3DSolver      (3-D Incompressible Euler)

All following the Wang et al. (2025) / Hou–Luo / Chen–Hou line of work on
stable and unstable finite-time singularities.

Usage
-----
    suite = NS3DSOLved(
        family="S0",
        ipm_config        = dict(N=128, dt=5e-4),
        boussinesq_config = dict(N=128, dt=2e-4, ic_type="hou_luo"),
        euler3d_config    = dict(Nx=32, Ny=32, Nz=32, dt=1e-3, ic_type="hou_luo"),
    )
    suite.run_all()
    suite.plot_all(save_path="singularity_dashboard.png")
    print(suite.full_summary())

@property accessors
-------------------
    suite.ipm          →  _IPMSolver instance
    suite.boussinesq   →  _BoussinesqSolver instance
    suite.euler3d      →  _Euler3DSolver instance
    suite.family       →  active singularity family label (str)
    suite.ipm_results  →  cached result dict (None until run_ipm called)
    suite.bsq_results  →  cached result dict (None until run_boussinesq called)
    suite.e3d_results  →  cached result dict (None until run_euler3d called)
"""

from __future__ import annotations

import warnings
from dataclasses import dataclass, field
from typing import Callable, Literal, NamedTuple, Optional

import numpy as np
from numpy.fft import fft, ifft, fft2, ifft2, fftfreq
from scipy.fft import dct, idct

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.offsetbox import AnchoredText
from mpl_toolkits.axes_grid1 import make_axes_locatable

matplotlib.rcParams.update({
    "font.family":   "monospace",
    "axes.titlepad": 6,
    "axes.labelpad": 3,
    "figure.dpi":    150,
    "savefig.dpi":   200,
    "axes.grid":     False,
})

PI = np.pi

# ══════════════════════════════════════════════════════════════════════════════
# Dormand–Prince RK4(5) Butcher tableau  (shared by all solvers)
# ══════════════════════════════════════════════════════════════════════════════

_DP_A = np.array([
    [0,           0,           0,           0,           0,           0,     0],
    [1/5,         0,           0,           0,           0,           0,     0],
    [3/40,        9/40,        0,           0,           0,           0,     0],
    [44/45,      -56/15,       32/9,        0,           0,           0,     0],
    [19372/6561, -25360/2187,  64448/6561, -212/729,     0,           0,     0],
    [9017/3168,  -355/33,      46732/5247,  49/176,     -5103/18656,  0,     0],
    [35/384,      0,           500/1113,    125/192,    -2187/6784,   11/84, 0],
], dtype=np.float64)
_DP_B5 = _DP_A[6]
_DP_B4 = np.array(
    [5179/57600, 0, 7571/16695, 393/640, -92097/339200, 187/2100, 1/40],
    dtype=np.float64,
)
_DP_E = _DP_B5 - _DP_B4


# ══════════════════════════════════════════════════════════════════════════════
# 4th-order z FD stencils  (shared by Boussinesq and Euler3D)
# ══════════════════════════════════════════════════════════════════════════════

def _fd4_dz(f: np.ndarray, dz: float) -> np.ndarray:
    c   = np.array([-1, 8, 0, -8, 1], dtype=np.float64) / (12.0 * dz)
    out = np.zeros_like(f)
    out[..., 2:-2] = (c[0]*f[..., :-4] + c[1]*f[..., 1:-3]
                    + c[3]*f[..., 3:-1] + c[4]*f[..., 4:])
    dz2 = 2.0 * dz
    out[..., 1]  = (f[..., 2]  - f[..., 0])  / dz2
    out[..., -2] = (f[..., -1] - f[..., -3]) / dz2
    out[..., 0]  = (-3*f[..., 0] + 4*f[..., 1] - f[..., 2]) / (2*dz)
    out[..., -1] = ( 3*f[..., -1] - 4*f[..., -2] + f[..., -3]) / (2*dz)
    return out


def _fd4_dz2(f: np.ndarray, dz: float) -> np.ndarray:
    c   = np.array([-1, 16, -30, 16, -1], dtype=np.float64) / (12.0 * dz**2)
    out = np.zeros_like(f)
    out[..., 2:-2] = (c[0]*f[..., :-4] + c[1]*f[..., 1:-3]
                    + c[2]*f[..., 2:-2] + c[3]*f[..., 3:-1] + c[4]*f[..., 4:])
    inv_dz2 = 1.0 / dz**2
    out[..., 0]  = (2*f[..., 0] - 5*f[..., 1] + 4*f[..., 2] - f[..., 3]) * inv_dz2
    out[..., -1] = (2*f[..., -1] - 5*f[..., -2] + 4*f[..., -3] - f[..., -4]) * inv_dz2
    out[..., 1]  = (f[..., 0] - 2*f[..., 1] + f[..., 2]) * inv_dz2
    out[..., -2] = (f[..., -3] - 2*f[..., -2] + f[..., -1]) * inv_dz2
    return out



from __future__ import annotations
from typing import Literal

from typing import Literal,TypeAlias
from dataclasses import dataclass

import textwrap
import warnings
# ══════════════════════════════════════════════════════════════════════════════
# ── SECTION 1 :  2-D IPM SOLVER ──────────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

#_SingularityKind = Literal["stable", "unstable"]


@dataclass(frozen=True)
class SingularityFamily:
    """IPM self-similar blow-up family (Wang et al. 2025)."""
    label:       str
    kind:        Literal["stable", "unstable"] #_SingularityKind
    order:       int
    alpha:       float
    beta:        float
    gamma:       float
    codim:       int
    description: str

    @property
    def blowup_scale(self) -> tuple[float, float]:
        return self.alpha, self.beta

    @classmethod
    def empirical(cls, n: int) -> "SingularityFamily":
        alpha_0, delta_alpha = 1/3, 0.5
        alpha = alpha_0 + n * delta_alpha
        beta  = 1.0 - alpha
        gamma = alpha
        kind: _SingularityKind = "stable" if n == 0 else "unstable"
        return cls(
            label=f"S{n}" if n == 0 else f"U{n}",
            kind=kind, order=n, alpha=alpha, beta=beta, gamma=gamma, codim=n,
            description=(
                f"{'Stable' if n == 0 else f'Unstable (codim {n})'} IPM singularity, "
                f"α={alpha:.4f}, β={beta:.4f}, γ={gamma:.4f}"
            ),
        )


IPM_SINGULARITY_CATALOGUE: dict[str, SingularityFamily] = {
    ("S0" if n == 0 else f"U{n}"): SingularityFamily.empirical(n)
    for n in range(4)
}



class IPMBlowupEstimate(NamedTuple):
    T_star:       float
    alpha_fit:    float
    residual:     float
    family_match: Optional[str]
    confidence:   float


def _fit_blowup_ipm(times, max_grad, tol=0.05) -> Optional[IPMBlowupEstimate]:
    if len(times) < 8:
        return None
    t  = np.asarray(times[-16:], dtype=np.float64)
    mg = np.asarray(max_grad[-16:], dtype=np.float64)
    if np.any(mg <= 0) or (np.log(mg[-1]) - np.log(mg[0])) < 0.5:
        return None
    log_mg = np.log(mg)
    T_cands = t[-1] + np.logspace(-4, 1, 200)
    best_res, best_T, best_alpha = np.inf, T_cands[-1], 0.0
    for T in T_cands:
        log_dt = np.log(T - t)
        A      = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            cf, _, _, _ = np.linalg.lstsq(A, log_mg, rcond=None)
        except np.linalg.LinAlgError:
            continue
        alpha_cand = -cf[1]
        if alpha_cand < 0.05:
            continue
        res = float(np.sqrt(np.mean((log_mg - A @ cf)**2)))
        if res < best_res:
            best_res, best_T, best_alpha = res, T, alpha_cand
    if best_alpha < 0.1:
        return None
    fmatch = min(IPM_SINGULARITY_CATALOGUE,
                 key=lambda nm: abs(IPM_SINGULARITY_CATALOGUE[nm].alpha - best_alpha))
    min_diff   = abs(IPM_SINGULARITY_CATALOGUE[fmatch].alpha - best_alpha)
    confidence = max(0.0, 1.0 - best_res / 0.5) * (1.0 if min_diff < tol else 0.5)
    return IPMBlowupEstimate(
        T_star=best_T, alpha_fit=best_alpha, residual=best_res,
        family_match=fmatch if min_diff < 0.15 else None,
        confidence=confidence,
    )


@dataclass
class IPMSelfSimilarProfile:
    t: float; xi: np.ndarray; zeta: np.ndarray; R: np.ndarray
    alpha: float; beta: float; gamma: float; T_star: float; family: Optional[str]



class _IPMSpectralOps:
    def __init__(self, N, Lx, Lz):
        self.N = N; self.Lx = Lx; self.Lz = Lz
        kx = fftfreq(N, d=Lx / (2*PI*N))
        kz = (PI/Lz) * np.arange(N, dtype=np.float64)
        self.KX, self.KZ = np.meshgrid(kx, kz, indexing='ij')
        self.K2 = self.KX**2 + self.KZ**2
        self.K2[0, 0] = 1.0
        kmax_x = (N//3)*(2*PI/Lx); kmax_z = (N//3)*(PI/Lz)
        self.DA = (np.abs(self.KX) < kmax_x) & (self.KZ < kmax_z)
        K2r = 1.0 + self.K2
        self.W_H1 = np.sqrt(K2r); self.W_H2 = K2r

    def fwd(self, f):
        tmp = fft(f, axis=0)
        return (dct(tmp.real, type=2, axis=1, norm='ortho')
              + 1j*dct(tmp.imag, type=2, axis=1, norm='ortho'))

    def inv(self, fh):
        tmp_r = idct(fh.real, type=2, axis=1, norm='ortho')
        tmp_i = idct(fh.imag, type=2, axis=1, norm='ortho')
        return ifft(tmp_r + 1j*tmp_i, axis=0).real

    def dx(self, fh):         return 1j*self.KX*fh
    def dz_hat(self, fh):     return -self.KZ*fh          # cosine → negative sine coeff
    def solve_pressure(self, rh):
        ph = (self.KZ*rh)/self.K2; ph[0,0] = 0.0; return ph
    def sobolev_norm(self, fh, s=1):
        W = {1: self.W_H1, 2: self.W_H2}.get(s, self.W_H1)
        return float(np.sqrt(np.sum((W*np.abs(fh))**2)))



class _IPMSolver:
    """
    2-D Incompressible Porous-Media pseudo-spectral solver.

    Physics:
        ∂ρ/∂t + u·∇ρ = 0
        u = −∇p + ρ ê_z,   Δp = ∂ρ/∂z
    """

    def __init__(self, N=256, Lx=2*PI, Lz=2*PI, dt=5e-4,
                 rtol=1e-6, atol=1e-9, family=None,
                 nu_sponge=0.0, blowup_threshold=1e3):
        if N % 2 != 0:
            raise ValueError("N must be even.")
        self.N = N; self.Lx = Lx; self.Lz = Lz
        self.dt = dt; self.rtol = rtol; self.atol = atol
        self.nu_sponge = nu_sponge; self.blowup_threshold = blowup_threshold
        self.t = 0.0; self._step = 0; self._near_blowup = False

        x = np.linspace(0, Lx, N, endpoint=False)
        z = np.linspace(0, Lz, N, endpoint=False)
        self.X, self.Z = np.meshgrid(x, z, indexing='ij')
        self._sp = _IPMSpectralOps(N, Lx, Lz)

        if family is not None:
            if family not in IPM_SINGULARITY_CATALOGUE:
                raise ValueError(f"Unknown IPM family '{family}'.")
            self._family = IPM_SINGULARITY_CATALOGUE[family]
            self.rho = self._make_family_ic(self._family)
        else:
            self._family = None
            amp = 0.05
            self.rho = -np.tanh((self.Z - Lz/2)/0.1)
            self.rho += amp*np.cos(2*PI*self.X/Lx)*np.exp(-((self.Z - Lz/2)**2)/0.05)

        self.times:       list[float] = []
        self.max_grad:    list[float] = []
        self.enstrophy:   list[float] = []
        self.energy:      list[float] = []
        self.h1_norm:     list[float] = []
        self.h2_norm:     list[float] = []
        self.blowup_rate: list[float] = []
        self.dt_history:  list[float] = []
        self.blowup_estimate: Optional[IPMBlowupEstimate] = None
        self.self_similar_snapshots: list[IPMSelfSimilarProfile] = []
        self._k_prev: Optional[np.ndarray] = None


    def _make_family_ic(self, fam):
        n, alpha, beta = fam.order, fam.alpha, fam.beta
        sigma_z = self.Lz*0.05*(1.0 + 0.5*beta)
        z_c     = self.Lz*0.25
        rho = -np.tanh((self.Z - z_c)/sigma_z)
        k_pert = max(1, n)
        x_wave = np.cos(2*PI*k_pert*self.X/self.Lx)
        amp_b  = 0.05*np.exp(-((self.Z - z_c)**2)/(2*sigma_z**2))
        rho += amp_b*x_wave
        for m in range(1, n+1):
            phase_z  = PI*m/self.Lz*self.Z
            mode_amp = 0.02/(m**1.5)*np.exp(-((self.Z - z_c*m/(n+1))**2)/(sigma_z**2))
            rho += mode_amp*np.cos(2*PI*(k_pert+m)*self.X/self.Lx)*np.cos(phase_z)
        rho -= rho.mean()
        return rho



    def _rhs(self, rho):
        sp = self._sp
        rh = sp.fwd(rho)*sp.DA
        ph = sp.solve_pressure(rh)
        ux_h = -sp.dx(ph); uz_h = sp.KZ*ph + rh
        ux_h *= sp.DA; uz_h *= sp.DA
        ux = sp.inv(ux_h); uz = sp.inv(uz_h)
        drho_dx = sp.inv(sp.dx(rh))
        drho_dz = sp.inv(sp.dz_hat(rh))
        adv = ux*drho_dx + uz*drho_dz
        drho_dt_h = -sp.fwd(adv)*sp.DA
        if self.nu_sponge > 0.0:
            drho_dt_h -= self.nu_sponge*(sp.K2**4)*rh
        return sp.inv(drho_dt_h)


    def _dp45_step(self, rho, dt):
        A, B4, B5, E = _DP_A, _DP_B4, _DP_B5, _DP_E
        k = np.zeros((7, *rho.shape))
        k[0] = self._rhs(rho) if self._k_prev is None else self._k_prev
        for i in range(1, 7):
            y_i = rho + dt*sum(A[i,j]*k[j] for j in range(i))
            k[i] = self._rhs(y_i)
        rho5 = rho + dt*sum(B5[i]*k[i] for i in range(7))
        rho4 = rho + dt*sum(B4[i]*k[i] for i in range(7))
        ev   = dt*sum(E[i]*k[i] for i in range(7))
        self._k_prev = k[6]
        scale = self.atol + self.rtol*np.maximum(np.abs(rho), np.abs(rho5))
        err   = float(np.sqrt(np.mean((ev/scale)**2)))
        return rho5, rho4, err

    def _adapt_dt(self, err, dt):
        factor = np.clip(0.9*err**(-1/5) if err > 0 else 5.0, 0.1, 10.0)
        new_dt = dt*factor
        if self.blowup_estimate is not None:
            margin = max(1e-7, 0.02*(self.blowup_estimate.T_star - self.t))
            new_dt = min(new_dt, margin)
        return float(np.clip(new_dt, 1e-9, 0.1))

    def _record_diagnostics(self, rho):
        sp  = self._sp
        rh  = sp.fwd(rho)*sp.DA
        gx  = sp.inv(sp.dx(rh))
        gz  = sp.inv(sp.dz_hat(rh))
        mg  = float(np.sqrt(gx**2 + gz**2).max())
        ph  = sp.solve_pressure(rh)
        ux_h = -sp.dx(ph); uz_h = sp.KZ*ph + rh
        om_h  = sp.dx(uz_h) - sp.dz_hat(ux_h)   # ∂uz/∂x − ∂ux/∂z
        omega = sp.inv(om_h)
        cell  = (self.Lx/self.N)*(self.Lz/self.N)
        ux = sp.inv(ux_h); uz = sp.inv(uz_h)
        self.times.append(self.t)
        self.max_grad.append(mg)
        self.enstrophy.append(float(0.5*np.sum(omega**2)*cell))
        self.energy.append(float(0.5*np.sum(ux**2 + uz**2)*cell))
        self.h1_norm.append(sp.sobolev_norm(rh, s=1))
        self.h2_norm.append(sp.sobolev_norm(rh, s=2))
        if len(self.times) % 20 == 0:
            est = _fit_blowup_ipm(self.times, self.max_grad)
            if est is not None:
                self.blowup_estimate = est
                self.blowup_rate.append(est.alpha_fit)
        if mg > self.blowup_threshold:
            self._near_blowup = True


    def _extract_self_similar_snapshot(self, fam):
        if self.blowup_estimate is None:
            return None
        T   = self.blowup_estimate.T_star
        tau = T - self.t
        if tau <= 0:
            return None
        sp = self._sp
        rh = sp.fwd(self.rho)*sp.DA
        gz = sp.inv(sp.dz_hat(rh))
        bot = gz[:, :self.N//8]
        ix_star = int(np.argmax(np.abs(bot).max(axis=1)))
        x_star  = self.X[ix_star, 0]
        Nss = 64; L_xi = 6.0; L_zeta = 6.0
        xi   = np.linspace(-L_xi,   L_xi,   Nss)
        zeta = np.linspace(0,       L_zeta, Nss)
        XI, ZETA = np.meshgrid(xi, zeta, indexing='ij')
        X_p = np.mod(x_star + XI*tau**fam.alpha, self.Lx)
        Z_p = np.clip(ZETA*tau**fam.beta, 0, self.Lz)
        dx  = self.Lx/self.N; dz = self.Lz/self.N
        ix  = (X_p/dx).astype(int) % self.N
        iz  = np.clip((Z_p/dz).astype(int), 0, self.N-1)
        R   = self.rho[ix, iz]*tau**(-fam.gamma)
        return IPMSelfSimilarProfile(
            t=self.t, xi=xi, zeta=zeta, R=R,
            alpha=fam.alpha, beta=fam.beta, gamma=fam.gamma,
            T_star=T, family=fam.label,
        )

    def step(self, n_steps=1, record_every=4):
        for _ in range(n_steps):
            for _ in range(15):
                rho5, rho4, err = self._dp45_step(self.rho, self.dt)
                if err <= 1.0 or self.dt < 1e-9:
                    self.rho = rho5; self.t += self.dt
                    self._step += 1; self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    self.dt *= max(0.1, 0.9*err**(-0.2))
                    self._k_prev = None
            else:
                warnings.warn(f"IPM step at t={self.t:.4f} failed.", RuntimeWarning)
            if self._step % record_every == 0:
                self._record_diagnostics(self.rho)
            if self._near_blowup and self.blowup_estimate is not None:
                last_t = (self.self_similar_snapshots[-1].t
                          if self.self_similar_snapshots else -np.inf)
                if self.t - last_t > 1e-5:
                    fl  = (self.blowup_estimate.family_match
                           or (self._family.label if self._family else "S0"))
                    fam = IPM_SINGULARITY_CATALOGUE.get(fl, SingularityFamily.empirical(0))
                    sn  = self._extract_self_similar_snapshot(fam)
                    if sn is not None:
                        self.self_similar_snapshots.append(sn)

    def run_until(self, T_end, record_every=10, max_steps=500_000):
        steps = 0
        while self.t < T_end and steps < max_steps:
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(1, record_every); steps += 1
            if self._near_blowup and self.blowup_estimate is not None:
                est = self.blowup_estimate
                if self.t >= est.T_star - 1e-6:
                    print(f"[IPMSolver] Blow-up t≈{self.t:.6f}  T*={est.T_star:.6f}  "
                          f"α={est.alpha_fit:.4f}  family={est.family_match or '?'}")
                    break


    def spectral_energy_budget(self):
        sp = self._sp
        rh = sp.fwd(self.rho)*sp.DA
        ph = sp.solve_pressure(rh)
        ux_h = -sp.dx(ph); uz_h = sp.KZ*ph + rh
        E_h  = 0.5*(np.abs(ux_h)**2 + np.abs(uz_h)**2)
        K    = np.sqrt(sp.KX**2 + sp.KZ**2)
        kmax = K.max()
        bins = np.linspace(0, kmax, 64)
        k_sh = 0.5*(bins[:-1] + bins[1:])
        E_k  = np.array([E_h[(K >= bins[i]) & (K < bins[i+1])].sum()
                         for i in range(len(bins)-1)])
        mask = (k_sh > 1) & (E_k > 0)
        slope = (float(np.polyfit(np.log(k_sh[mask]), np.log(E_k[mask]), 1)[0])
                 if mask.sum() > 4 else float("nan"))
        return {"k_shells": k_sh, "E_k": E_k, "slope": slope}

    def linearised_growth_rate(self, epsilon=1e-5):
        rng   = np.random.default_rng(0)
        delta = rng.standard_normal(self.rho.shape)
        delta = delta/np.linalg.norm(delta)*epsilon
        f0    = self._rhs(self.rho)
        fp    = self._rhs(self.rho + delta)
        return float(np.linalg.norm(fp - f0) / np.linalg.norm(delta))

    def summary(self):
        lines = [
            "="*62, " _IPMSolver — 2-D Incompressible Porous Media", "="*62,
            f"  Grid       : {self.N}×{self.N}  (domain {self.Lx:.2f}×{self.Lz:.2f})",
            f"  Time       : t = {self.t:.6f}",
            f"  Steps      : {self._step}",
            f"  Current dt : {self.dt:.2e}",
        ]
        if self.max_grad:
            lines += [
                f"  ‖∇ρ‖_∞    : {self.max_grad[-1]:.4e}",
                f"  Enstrophy  : {self.enstrophy[-1]:.4e}",
                f"  Kinetic E  : {self.energy[-1]:.4e}",
                f"  H¹ norm ρ  : {self.h1_norm[-1]:.4e}",
                f"  H² norm ρ  : {self.h2_norm[-1]:.4e}",
            ]
        if self.blowup_estimate:
            e = self.blowup_estimate
            lines += [
                "", "  ── Blow-up estimate ──",
                f"  T*         : {e.T_star:.6f}",
                f"  α_fit      : {e.alpha_fit:.4f}",
                f"  Family     : {e.family_match or 'unknown'}",
                f"  Confidence : {e.confidence:.2f}",
            ]
        if self._family:
            f = self._family
            lines += [
                "", "  ── Singularity family ──",
                f"  {f.description}",
            ]
        lines.append("="*62)
        return "\n".join(lines)

    def __repr__(self):
        return (f"_IPMSolver(N={self.N}, t={self.t:.4f}, dt={self.dt:.2e}, "
                f"family={self._family.label if self._family else None})")


# ══════════════════════════════════════════════════════════════════════════════
# ── SECTION 2 :  2-D BOUSSINESQ SOLVER ───────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

_BKind = Literal["stable", "unstable"]


@dataclass(frozen=True)
class BoussinesqSingularityFamily:
    """
    2-D Boussinesq blow-up family (Wang et al. 2025 §3 analogous to IPM/Euler).

    Self-similar ansatz:
        ω(x,z,t) ≈ (T−t)^{γ_ω} Ω̃(x/(T−t)^c, z/(T−t)^c)
        θ(x,z,t) ≈ (T−t)^{γ_θ} Θ̃(x/(T−t)^c, z/(T−t)^c)

    Empirical law (analogous to Euler):
        c(n) = c₀ + n·0.5,  c₀ = 0.5
        γ_ω = -(1 - c)
        γ_θ = -(2 - c)   (θ drives the vorticity source ∂θ/∂x)
    """
    label:    str
    kind:     _BKind
    order:    int
    c:        float
    gamma_om: float
    gamma_th: float

    @classmethod
    def empirical(cls, n: int) -> "BoussinesqSingularityFamily":
        c0, dc = 0.5, 0.5
        c = c0 + n*dc
        kind: _BKind = "stable" if n == 0 else "unstable"
        return cls(
            label    = "S0" if n == 0 else f"U{n}",
            kind     = kind, order=n, c=c,
            gamma_om = -(1.0 - c),
            gamma_th = -(2.0 - c),
        )

    @property
    def bkm_integrable(self) -> bool:
        return self.c > 1.0

    def __str__(self):
        return (f"{self.label} ({self.kind}, n={self.order}): "
                f"c={self.c:.4f}  γ_ω={self.gamma_om:+.4f}  γ_θ={self.gamma_th:+.4f}")


BSQ_CATALOGUE: dict[str, BoussinesqSingularityFamily] = {
    ("S0" if n == 0 else f"U{n}"): BoussinesqSingularityFamily.empirical(n)
    for n in range(4)
}


class BSQBlowupEstimate(NamedTuple):
    T_star:       float
    c_fit:        float
    gamma_om_fit: float
    bkm_integral: float
    family_match: Optional[str]
    residual:     float
    confidence:   float


def _fit_blowup_bsq(times, max_om, bkm_int, tol=0.05) -> Optional[BSQBlowupEstimate]:
    if len(times) < 10:
        return None
    t  = np.array(times[-20:], dtype=np.float64)
    mo = np.array(max_om[-20:], dtype=np.float64)
    bk = np.array(bkm_int[-20:], dtype=np.float64)
    if np.any(mo <= 0) or (mo[-1]/(mo[0]+1e-14)) < 2.0:
        return None
    log_mo = np.log(mo)
    T_cands = t[-1] + np.logspace(-5, 1, 300)
    best_res, best_T, best_c = np.inf, T_cands[0], 0.0
    for T in T_cands:
        log_dt = np.log(T - t)
        A = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            cf, _, _, _ = np.linalg.lstsq(A, log_mo, rcond=None)
        except np.linalg.LinAlgError:
            continue
        c_cand = 1.0 + cf[1]          # γ_ω = -(1-c) → slope = -(1-c) → c = 1+slope
        if not (0.05 < c_cand < 3.0):
            continue
        res = float(np.sqrt(np.mean((log_mo - A @ cf)**2)))
        if res < best_res:
            best_res, best_T, best_c = res, T, c_cand
    if best_c < 0.05:
        return None
    fm = min(BSQ_CATALOGUE, key=lambda nm: abs(BSQ_CATALOGUE[nm].c - best_c))
    md = abs(BSQ_CATALOGUE[fm].c - best_c)
    cf = max(0.0, 1.0 - best_res/0.5)*(1.0 if md < tol else 0.5)
    return BSQBlowupEstimate(
        T_star=best_T, c_fit=best_c, gamma_om_fit=best_c - 1.0,
        bkm_integral=float(bk[-1]),
        family_match=fm if md < 0.20 else None,
        residual=best_res, confidence=cf,
    )


@dataclass
class BSQSelfSimilarSnapshot:
    t: float; xi: np.ndarray; zeta: np.ndarray
    Omega: np.ndarray; Theta: np.ndarray
    c: float; T_star: float; family: Optional[str]


class _BoussinesqSpectralOps:
    """Pseudo-spectral ops for 2-D periodic Boussinesq on [0,L]²."""

    def __init__(self, N, L):
        self.N = N; self.L = L
        k    = fftfreq(N, d=L/(2*PI*N))
        KX2d, KZ2d = np.meshgrid(k, k, indexing='ij')
        self.KX = KX2d; self.KZ = KZ2d
        self.K2 = KX2d**2 + KZ2d**2
        self.K2[0, 0] = 1.0
        kmax = (N//3)*(2*PI/L)
        self.DA = (np.abs(KX2d) < kmax) & (np.abs(KZ2d) < kmax)
        K2r = 1.0 + self.K2
        self.W_H1 = np.sqrt(K2r); self.W_H2 = K2r

    def fwd(self, f):  return fft2(f, axes=(0,1))
    def inv(self, fh): return ifft2(fh, axes=(0,1)).real

    def dx(self, fh):           return self.inv(1j*self.KX*fh)
    def dz(self, fh):           return self.inv(1j*self.KZ*fh)
    def solve_stream(self, oh): return oh / (-self.K2)  # Δψ = -ω  →  ψ̂ = ω̂/K²

    def sobolev_h1(self, f):
        fh = self.fwd(f)
        return float(np.sqrt(np.sum((self.W_H1*np.abs(fh))**2)))
    def sobolev_h2(self, f):
        fh = self.fwd(f)
        return float(np.sqrt(np.sum((self.W_H2*np.abs(fh))**2)))


class _BoussinesqSolver:
    """
    2-D Inviscid Boussinesq pseudo-spectral solver.

    Governing equations (vorticity-streamfunction form):
        ∂ω/∂t + u·∇ω = ∂θ/∂x        (buoyancy source)
        ∂θ/∂t + u·∇θ = 0             (temperature advection)
        Δψ = −ω                       (stream function)
        u = ∂ψ/∂z,  w = −∂ψ/∂x      (velocity from ψ)

    Domain: [0, L]² periodic.
    Spectral method: 2-D DFT with 2/3-rule dealiasing.
    Time integrator: adaptive Dormand–Prince RK4(5).
    """

    def __init__(self, N=128, L=2*PI, dt=2e-4, rtol=1e-6, atol=1e-9,
                 family=None, nu_hyp=0.0, blowup_threshold=1e2, ic_type="default"):
        if N % 2 != 0:
            raise ValueError("N must be even.")
        self.N = N; self.L = L
        self.dt = dt; self.rtol = rtol; self.atol = atol
        self.nu_hyp = nu_hyp; self.blowup_threshold = blowup_threshold
        self.t = 0.0; self._step = 0; self._near_blowup = False

        x = np.linspace(0, L, N, endpoint=False)
        self.X, self.Z = np.meshgrid(x, x, indexing='ij')
        self._sp = _BoussinesqSpectralOps(N, L)

        if family is not None:
            if family not in BSQ_CATALOGUE:
                raise ValueError(f"Unknown Boussinesq family '{family}'.")
            self._family = BSQ_CATALOGUE[family]
        else:
            self._family = BSQ_CATALOGUE.get("S0")

        self.omega, self.theta = self._make_ic(ic_type, self._family)

        self.times:      list[float] = []
        self.max_omega:  list[float] = []
        self.bkm:        list[float] = []
        self.energy:     list[float] = []
        self.enstrophy:  list[float] = []
        self.h1_omega:   list[float] = []
        self.h2_omega:   list[float] = []
        self.h1_theta:   list[float] = []
        self.dt_history: list[float] = []
        self._bkm_acc:   float       = 0.0
        self._energy0:   float       = 0.0
        self._enst0:     float       = 0.0

        self.blowup_estimate:        Optional[BSQBlowupEstimate]     = None
        self.self_similar_snapshots: list[BSQSelfSimilarSnapshot]    = []
        self._k_prev: Optional[tuple] = None

    # ── Initial conditions ───────────────────────────────────────────────────

    def _make_ic(self, ic_type, fam):
        X, Z, L = self.X, self.Z, self.L
        n = fam.order if fam else 0
        c = fam.c     if fam else 0.5

        if ic_type == "hou_luo" or fam is not None:
            # Hou–Luo-inspired: concentrated shear layer with buoyancy gradient
            z_c   = L/2; sigma = L*0.08*(1.0 + 0.2*c)
            g     = np.exp(-((Z - z_c)**2)/(2*sigma**2))
            omega = np.sin(2*PI*X/L)*g
            theta = np.cos(PI*Z/L)*(1.0 + 0.1*np.cos(2*PI*X/L))
            for m in range(1, n+1):
                eps = 0.03/m**1.5
                omega += eps*np.sin(2*PI*(m+1)*X/L)*g
                theta += 0.02*np.cos(2*PI*(m+1)*X/L)*g
        else:
            omega = (np.sin(2*PI*X/L)*np.cos(2*PI*Z/L)
                   - 0.5*np.cos(4*PI*X/L)*np.sin(2*PI*Z/L))
            theta = np.cos(PI*Z/L) + 0.05*np.cos(2*PI*X/L)

        omega -= omega.mean(); theta -= theta.mean()
        return omega, theta

    # ── RHS ──────────────────────────────────────────────────────────────────

    def _rhs(self, omega, theta):
        sp = self._sp
        oh = sp.fwd(omega)*sp.DA
        th = sp.fwd(theta)*sp.DA

        # Stream function ψ and velocity
        # Δψ = −ω  →  ψ̂ = ω̂/K²   (solve_stream)
        # u = ∂ψ/∂z  →  û = iKZ ψ̂
        # w = −∂ψ/∂x →  ŵ = −iKX ψ̂
        psi_h = sp.solve_stream(oh)
        psi_h[0, 0] = 0.0
        u = sp.inv(1j*sp.KZ*psi_h)
        w = sp.inv(-1j*sp.KX*psi_h)

        # Advection of ω
        dom_dx = sp.dx(oh); dom_dz = sp.dz(oh)
        adv_om = u*dom_dx + w*dom_dz
        # Buoyancy source ∂θ/∂x
        dth_dx = sp.inv(1j*sp.KX*th)
        rhs_om_h = sp.fwd(-adv_om + dth_dx)*sp.DA

        # Advection of θ
        dth_dx_p = sp.inv(1j*sp.KX*th)
        dth_dz_p = sp.inv(1j*sp.KZ*th)
        adv_th   = u*dth_dx_p + w*dth_dz_p
        rhs_th_h = sp.fwd(-adv_th)*sp.DA

        if self.nu_hyp > 0.0:
            K2h = sp.K2
            rhs_om_h -= self.nu_hyp*(K2h**4)*oh
            rhs_th_h -= self.nu_hyp*(K2h**4)*th

        return sp.inv(rhs_om_h), sp.inv(rhs_th_h)

    # ── DP45 step ────────────────────────────────────────────────────────────

    def _dp45_step(self, omega, theta, dt):
        A, B4, B5, E = _DP_A, _DP_B4, _DP_B5, _DP_E
        ko = np.zeros((7, *omega.shape))
        kt = np.zeros_like(ko)
        if self._k_prev is not None:
            ko[0], kt[0] = self._k_prev
        else:
            ko[0], kt[0] = self._rhs(omega, theta)
        for i in range(1, 7):
            o_i = omega + dt*sum(A[i,j]*ko[j] for j in range(i))
            t_i = theta + dt*sum(A[i,j]*kt[j] for j in range(i))
            ko[i], kt[i] = self._rhs(o_i, t_i)
        o5 = omega + dt*sum(B5[i]*ko[i] for i in range(7))
        t5 = theta + dt*sum(B5[i]*kt[i] for i in range(7))
        o4 = omega + dt*sum(B4[i]*ko[i] for i in range(7))
        t4 = theta + dt*sum(B4[i]*kt[i] for i in range(7))
        eo = dt*sum(E[i]*ko[i] for i in range(7))
        et = dt*sum(E[i]*kt[i] for i in range(7))
        self._k_prev = (ko[6], kt[6])
        sc_o = self.atol + self.rtol*np.maximum(np.abs(omega), np.abs(o5))
        sc_t = self.atol + self.rtol*np.maximum(np.abs(theta), np.abs(t5))
        err  = float(np.sqrt((np.mean((eo/sc_o)**2) + np.mean((et/sc_t)**2))/2))
        return o5, t5, o4, t4, err

    def _adapt_dt(self, err, dt):
        factor = min(5.0, max(0.1, 0.9*err**(-0.2) if err > 0 else 5.0))
        new_dt = dt*factor
        if self.blowup_estimate is not None:
            margin = max(1e-8, 0.02*(self.blowup_estimate.T_star - self.t))
            new_dt = min(new_dt, margin)
        return float(np.clip(new_dt, 1e-10, 0.05))

    # ── Diagnostics ──────────────────────────────────────────────────────────

    def _record_diagnostics(self, omega, theta):
        sp   = self._sp
        mo   = float(np.abs(omega).max())
        if self.times:
            dt_rec = self.t - self.times[-1]
            self._bkm_acc += 0.5*(self.max_omega[-1] + mo)*dt_rec
        cell = (self.L/self.N)**2
        enst = float(0.5*np.sum(omega**2)*cell)
        oh   = sp.fwd(omega)*sp.DA
        psi_h = sp.solve_stream(oh); psi_h[0,0] = 0.0
        u  = sp.inv(1j*sp.KZ*psi_h)
        w  = sp.inv(-1j*sp.KX*psi_h)
        ek = float(0.5*np.sum(u**2 + w**2)*cell)
        self.times.append(self.t)
        self.max_omega.append(mo)
        self.bkm.append(self._bkm_acc)
        self.enstrophy.append(enst)
        self.energy.append(ek)
        self.h1_omega.append(sp.sobolev_h1(omega))
        self.h2_omega.append(sp.sobolev_h2(omega))
        self.h1_theta.append(sp.sobolev_h1(theta))
        if len(self.times) == 1:
            self._energy0 = ek; self._enst0 = enst
        if len(self.times) % 20 == 0:
            est = _fit_blowup_bsq(self.times, self.max_omega, self.bkm)
            if est is not None:
                self.blowup_estimate = est
        if mo > self.blowup_threshold:
            self._near_blowup = True

    def _extract_self_similar_snapshot(self, fam):
        if self.blowup_estimate is None:
            return None
        T   = self.blowup_estimate.T_star
        tau = T - self.t
        if tau < 1e-10:
            return None
        c  = fam.c
        ix_star = int(np.argmax(np.abs(self.omega).max(axis=1)))
        iz_star = int(np.argmax(np.abs(self.omega).max(axis=0)))
        x_star  = self.X[ix_star, 0]; z_star = self.Z[0, iz_star]
        Nss = 48; R = 4.0
        xi   = np.linspace(-R, R, Nss); zeta = np.linspace(-R, R, Nss)
        XI, ZE = np.meshgrid(xi, zeta, indexing='ij')
        X_p = np.mod(x_star + XI*tau**c, self.L)
        Z_p = np.mod(z_star + ZE*tau**c, self.L)
        dx  = self.L/self.N
        ix  = (X_p/dx).astype(int) % self.N
        iz  = (Z_p/dx).astype(int) % self.N
        Om  = self.omega[ix, iz]*tau**(-fam.gamma_om)
        Th  = self.theta[ix, iz]*tau**(-fam.gamma_th)
        return BSQSelfSimilarSnapshot(
            t=self.t, xi=xi, zeta=zeta, Omega=Om, Theta=Th,
            c=c, T_star=T, family=fam.label,
        )

    # ── Integration ──────────────────────────────────────────────────────────

    def step(self, n_steps=1, record_every=1):
        for _ in range(n_steps):
            for _ in range(20):
                o5, t5, o4, t4, err = self._dp45_step(self.omega, self.theta, self.dt)
                if err <= 1.0 or self.dt < 1e-10:
                    self.omega = o5; self.theta = t5
                    self.t += self.dt; self._step += 1
                    self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    self.dt *= max(0.1, 0.9*err**(-0.2))
                    self._k_prev = None
            else:
                warnings.warn(f"BSQ step at t={self.t:.5f} failed.", RuntimeWarning)
            if self._step % record_every == 0:
                self._record_diagnostics(self.omega, self.theta)
            if self._near_blowup and self.blowup_estimate is not None:
                last_t = (self.self_similar_snapshots[-1].t
                          if self.self_similar_snapshots else -np.inf)
                if self.t - last_t > 1e-5:
                    fl  = (self.blowup_estimate.family_match
                           or (self._family.label if self._family else "S0"))
                    fam = BSQ_CATALOGUE.get(fl, BoussinesqSingularityFamily.empirical(0))
                    sn  = self._extract_self_similar_snapshot(fam)
                    if sn is not None:
                        self.self_similar_snapshots.append(sn)

    def run_until(self, T_end, record_every=10, max_steps=1_000_000):
        steps = 0
        while self.t < T_end and steps < max_steps:
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(1, record_every); steps += 1
            if self._near_blowup and self.blowup_estimate is not None:
                est = self.blowup_estimate
                if self.t >= est.T_star - 1e-6:
                    print(f"[BSQSolver] Blow-up t≈{self.t:.6f}  T*={est.T_star:.6f}  "
                          f"c={est.c_fit:.4f}  family={est.family_match or '?'}")
                    break

    def spectral_energy_budget(self):
        sp   = self._sp
        oh   = sp.fwd(self.omega)*sp.DA
        ph   = sp.solve_stream(oh); ph[0,0] = 0.0
        u    = sp.inv(1j*sp.KZ*ph); w = sp.inv(-1j*sp.KX*ph)
        E_h  = 0.5*(np.abs(sp.fwd(u))**2 + np.abs(sp.fwd(w))**2)
        K    = np.sqrt(sp.KX**2 + sp.KZ**2)
        kmax = K.max()
        bins = np.linspace(0, kmax, 64)
        k_sh = 0.5*(bins[:-1] + bins[1:])
        E_k  = np.array([E_h[(K >= bins[i]) & (K < bins[i+1])].sum()
                         for i in range(len(bins)-1)])
        mask  = (k_sh > 1) & (E_k > 0)
        slope = (float(np.polyfit(np.log(k_sh[mask]), np.log(E_k[mask]), 1)[0])
                 if mask.sum() > 4 else float("nan"))
        return {"k_shells": k_sh, "E_k": E_k, "slope": slope}

    def conservation_errors(self):
        if not self.energy:
            return {"delta_energy_rel": 0.0, "delta_enstrophy_rel": 0.0}
        e0  = self._energy0  + 1e-14
        es0 = self._enst0    + 1e-14
        return {
            "delta_energy_rel":    abs(self.energy[-1]   - e0)  / abs(e0),
            "delta_enstrophy_rel": abs(self.enstrophy[-1] - es0) / abs(es0),
        }

    def bkm_criterion_met(self, threshold=30.0):
        if len(self.max_omega) < 3:
            return False
        return bool(self._bkm_acc > threshold
                    and self.max_omega[-1] > self.max_omega[-2] > self.max_omega[-3])

    def linearised_growth_rate(self, epsilon=1e-5):
        rng  = np.random.default_rng(0)
        do   = rng.standard_normal(self.omega.shape)
        dt_  = rng.standard_normal(self.theta.shape)
        nrm  = np.sqrt(np.linalg.norm(do)**2 + np.linalg.norm(dt_)**2)
        do  *= epsilon/nrm; dt_ *= epsilon/nrm
        f0   = self._rhs(self.omega, self.theta)
        fp   = self._rhs(self.omega+do, self.theta+dt_)
        Ln   = np.sqrt(np.linalg.norm(fp[0]-f0[0])**2 + np.linalg.norm(fp[1]-f0[1])**2)
        dn   = np.sqrt(np.linalg.norm(do)**2 + np.linalg.norm(dt_)**2)
        return float(Ln/dn)

    def instability_order_estimate(self):
        """Estimate codimension of the unstable manifold by sign changes in ω."""
        if not self.max_omega:
            return -1
        mid = self.omega[:, self.N//2]
        sign_changes = int(np.sum(np.diff(np.sign(mid)) != 0))
        return sign_changes

    def summary(self):
        lines = [
            "="*62, " _BoussinesqSolver — 2-D Inviscid Boussinesq", "="*62,
            f"  Grid       : {self.N}×{self.N}  (domain {self.L:.2f}²)",
            f"  Time       : t = {self.t:.6f}",
            f"  Steps      : {self._step}",
            f"  Current dt : {self.dt:.2e}",
        ]
        if self.max_omega:
            lines += [
                f"  ‖ω‖_∞      : {self.max_omega[-1]:.4e}",
                f"  BKM ∫‖ω‖   : {self._bkm_acc:.4e}",
                f"  Enstrophy  : {self.enstrophy[-1]:.4e}",
                f"  Kinetic E  : {self.energy[-1]:.4e}",
                f"  H¹ norm ω  : {self.h1_omega[-1]:.4e}",
                f"  H² norm ω  : {self.h2_omega[-1]:.4e}",
                f"  BKM met?   : {self.bkm_criterion_met()}",
            ]
        if self.blowup_estimate:
            e = self.blowup_estimate
            lines += [
                "", "  ── Blow-up estimate ──",
                f"  T*         : {e.T_star:.6f}",
                f"  c_fit      : {e.c_fit:.4f}",
                f"  γ_ω fit    : {e.gamma_om_fit:.4f}",
                f"  Family     : {e.family_match or 'unknown'}",
                f"  Confidence : {e.confidence:.2f}",
            ]
        if self._family:
            lines += ["", f"  ── Family: {self._family}"]
        lines.append("="*62)
        return "\n".join(lines)

    def __repr__(self):
        return (f"_BoussinesqSolver(N={self.N}, t={self.t:.4f}, dt={self.dt:.2e}, "
                f"family={self._family.label if self._family else None})")


# ══════════════════════════════════════════════════════════════════════════════
# ── SECTION 3 :  3-D EULER SOLVER  (fixed from doc 3) ────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

_EKind = Literal["stable", "unstable"]


@dataclass(frozen=True)
class EulerSingularityFamily:
    """3-D Euler boundary blow-up family (Wang et al. 2025)."""
    label:    str
    kind:     _EKind
    order:    int
    c:        float
    gamma_u:  float
    gamma_om: float
    gamma_S:  float
    gamma_p:  float

    @classmethod
    def empirical(cls, n: int) -> "EulerSingularityFamily":
        c0, dc = 0.44, 0.50; c = c0 + n*dc
        kind: _EKind = "stable" if n == 0 else "unstable"
        return cls(
            label="S0" if n == 0 else f"U{n}",
            kind=kind, order=n, c=c,
            gamma_u  = -(1.0 - c),
            gamma_om = -(2.0 - c),
            gamma_S  = -(3.0 - c),
            gamma_p  = -(2.0 - 2.0*c),
        )

    @property
    def bkm_integrable(self) -> bool:
        return self.c > 1.0

    def __str__(self):
        return (f"{self.label} ({self.kind}, n={self.order}): "
                f"c={self.c:.4f}  γ_u={self.gamma_u:+.4f}  "
                f"γ_ω={self.gamma_om:+.4f}  γ_S={self.gamma_S:+.4f}  "
                f"BKM-integrable={self.bkm_integrable}")


EULER3D_CATALOGUE: dict[str, EulerSingularityFamily] = {
    ("S0" if n == 0 else f"U{n}"): EulerSingularityFamily.empirical(n)
    for n in range(4)
}


class E3DBlowupEstimate(NamedTuple):
    T_star:       float
    c_fit:        float
    gamma_om_fit: float
    bkm_integral: float
    vortex_stretch: float
    family_match: Optional[str]
    residual:     float
    confidence:   float


def _fit_blowup_euler3d(times, max_om, bkm_int, tol=0.05) -> Optional[E3DBlowupEstimate]:
    if len(times) < 10:
        return None
    t  = np.array(times[-20:], dtype=np.float64)
    mo = np.array(max_om[-20:], dtype=np.float64)
    bk = np.array(bkm_int[-20:], dtype=np.float64)
    if np.any(mo <= 0) or (mo[-1]/(mo[0]+1e-14)) < 2.0:
        return None
    log_mo = np.log(mo)
    T_cands = t[-1] + np.logspace(-5, 1, 400)
    best_res, best_T, best_c = np.inf, T_cands[0], 0.0
    for T in T_cands:
        log_dt = np.log(T - t)
        A = np.column_stack([np.ones_like(log_dt), log_dt])
        try:
            cf, _, _, _ = np.linalg.lstsq(A, log_mo, rcond=None)
        except np.linalg.LinAlgError:
            continue
        c_cand = 2.0 + cf[1]
        if not (0.05 < c_cand < 3.0):
            continue
        res = float(np.sqrt(np.mean((log_mo - A @ cf)**2)))
        if res < best_res:
            best_res, best_T, best_c = res, T, c_cand
    if best_c < 0.05:
        return None
    fm = min(EULER3D_CATALOGUE, key=lambda nm: abs(EULER3D_CATALOGUE[nm].c - best_c))
    md = abs(EULER3D_CATALOGUE[fm].c - best_c)
    cf = max(0.0, 1.0 - best_res/0.5)*(1.0 if md < tol else 0.5)
    return E3DBlowupEstimate(
        T_star=best_T, c_fit=best_c, gamma_om_fit=best_c - 2.0,
        bkm_integral=float(bk[-1]), vortex_stretch=float(mo[-1]),
        family_match=fm if md < 0.20 else None,
        residual=best_res, confidence=cf,
    )


@dataclass
class E3DSelfSimilarSnapshot:
    t: float; xi: np.ndarray; zeta: np.ndarray
    U_tilde: np.ndarray; Om_tilde: np.ndarray
    c: float; T_star: float; family: Optional[str]


class _E3DSpectralOps:
    def __init__(self, Nx, Ny, Nz, Lxy, Lz):
        self.Nx = Nx; self.Ny = Ny; self.Nz = Nz
        self.Lxy = Lxy; self.Lz = Lz; self.dz = Lz/Nz
        kx = fftfreq(Nx, d=Lxy/(2*PI*Nx))
        ky = fftfreq(Ny, d=Lxy/(2*PI*Ny))
        KX2d, KY2d = np.meshgrid(kx, ky, indexing='ij')
        self.KX = KX2d[:,:,np.newaxis]; self.KY = KY2d[:,:,np.newaxis]
        self.KXY2 = self.KX**2 + self.KY**2; self.KXY2[0,0,0] = 1.0
        kmax_x = (Nx//3)*(2*PI/Lxy); kmax_y = (Ny//3)*(2*PI/Lxy)
        da2d = (np.abs(KX2d) < kmax_x) & (np.abs(KY2d) < kmax_y)
        self.DA = da2d[:,:,np.newaxis]
        K2r = 1.0 + KX2d**2 + KY2d**2
        self.W_H1_2d = np.sqrt(K2r); self.W_H2_2d = K2r

    def fwd_xy(self, f):  return fft2(f, axes=(0,1))
    def inv_xy(self, fh): return ifft2(fh, axes=(0,1)).real
    def dx(self, fh):     return self.inv_xy(1j*self.KX*fh)
    def dy(self, fh):     return self.inv_xy(1j*self.KY*fh)
    def dz(self, f):      return _fd4_dz(f, self.dz)
    def dz2(self, f):     return _fd4_dz2(f, self.dz)

    def sobolev_h1(self, f):
        fh = self.fwd_xy(f)
        return float(np.sqrt(((self.W_H1_2d[:,:,np.newaxis]*np.abs(fh))**2).sum()*self.dz))
    def sobolev_h2(self, f):
        fh = self.fwd_xy(f)
        return float(np.sqrt(((self.W_H2_2d[:,:,np.newaxis]*np.abs(fh))**2).sum()*self.dz))

    def energy_spectrum(self, u, v, w_vel, nb=64):
        uh = self.fwd_xy(u); vh = self.fwd_xy(v); wh = self.fwd_xy(w_vel)
        E_h = 0.5*(np.abs(uh)**2 + np.abs(vh)**2 + np.abs(wh)**2)
        K2d = np.sqrt(self.KX[:,:,0]**2 + self.KY[:,:,0]**2)
        E_2d = E_h.mean(axis=2)
        bins = np.linspace(0, K2d.max(), nb+1)
        k_sh = 0.5*(bins[:-1] + bins[1:])
        E_k  = np.array([E_2d[(K2d >= bins[i]) & (K2d < bins[i+1])].sum()
                         for i in range(nb)])
        mask  = (k_sh > 1.0) & (E_k > 0)
        slope = (float(np.polyfit(np.log(k_sh[mask]), np.log(E_k[mask]), 1)[0])
                 if mask.sum() > 4 else float("nan"))
        return k_sh, E_k, slope


class _Euler3DSolver:
    """
    3-D Incompressible Euler — spectral (x,y) + 4th-order FD (z).
    All four bugs from the original code are fixed here (see FIX-1 … FIX-4).
    """

    def __init__(self, Nx=64, Ny=64, Nz=64, Lxy=2*PI, Lz=PI,
                 dt=1e-3, rtol=1e-6, atol=1e-9, family=None,
                 nu_hyp=0.0, blowup_threshold=1e2, ic_type="default"):
        for N, name in [(Nx,"Nx"),(Ny,"Ny"),(Nz,"Nz")]:
            if N % 2 != 0:
                raise ValueError(f"{name} must be even.")
        self.Nx = Nx; self.Ny = Ny; self.Nz = Nz
        self.Lxy = Lxy; self.Lz = Lz; self.dz = Lz/Nz
        self.dt = dt; self.rtol = rtol; self.atol = atol
        self.nu_hyp = nu_hyp; self.blowup_threshold = blowup_threshold
        self.t = 0.0; self._step = 0
        self._blown_up = False; self._blowup_time: Optional[float] = None
        self._near_blowup = False

        x  = np.linspace(0, Lxy, Nx, endpoint=False)
        y  = np.linspace(0, Lxy, Ny, endpoint=False)
        z  = np.linspace(0, Lz,  Nz, endpoint=False)
        self.X, self.Y, self.Z = np.meshgrid(x, y, z, indexing='ij')
        self._sp = _E3DSpectralOps(Nx, Ny, Nz, Lxy, Lz)

        if family is not None and family not in EULER3D_CATALOGUE:
            raise ValueError(f"Unknown Euler3D family '{family}'.")
        self._family = EULER3D_CATALOGUE[family] if family else None

        self.u, self.v, self.w = self._make_ic(ic_type, self._family)
        self._project()

        self.times:          list[float] = []
        self.max_omega:      list[float] = []
        self.bkm:            list[float] = []
        self.vortex_stretch: list[float] = []
        self.energy:         list[float] = []
        self.helicity:       list[float] = []
        self.h1_u:           list[float] = []
        self.h2_u:           list[float] = []
        self.palinstrophy:   list[float] = []
        self.dt_history:     list[float] = []
        self._bkm_acc:       float       = 0.0

        self.blowup_estimate:        Optional[E3DBlowupEstimate]    = None
        self.self_similar_snapshots: list[E3DSelfSimilarSnapshot]   = []
        self._k_prev: Optional[tuple] = None

    def _make_ic(self, ic_type, fam):
        X, Y, Z = self.X, self.Y, self.Z
        Lxy, Lz = self.Lxy, self.Lz
        A = 1.0; n = fam.order if fam else 0; c = fam.c if fam else 0.44
        if ic_type == "tgv":
            u = A*np.sin(X)*np.cos(Y)*np.cos(Z); v = -A*np.cos(X)*np.sin(Y)*np.cos(Z)
            w = np.zeros_like(u)
            u += 0.05*np.sin(2*X)*np.cos(Y)*np.cos(2*Z)
            v += 0.05*np.cos(X)*np.sin(2*Y)*np.cos(2*Z)
        elif ic_type == "hou_luo" or (fam is not None and ic_type == "default"):
            z_c = Lz/2; sigma = Lz*0.08*(1.0 + 0.25*c)
            g   = np.exp(-((Z - z_c)**2)/(2*sigma**2))
            u   = A*np.sin(X)*np.cos(Y)*g; v = -A*np.cos(X)*np.sin(Y)*g
            w   = (2*A/3)*np.cos(X)*np.cos(Y)*np.sin(Z)*g*0.2
            for m in range(1, n+1):
                eps_m = 0.03/m**1.5; phi = 2*PI*m/(n+1)
                gm    = np.exp(-((Z - z_c)**2)/sigma**2)
                u += eps_m*np.sin((m+1)*X + phi)*np.cos(Y)*gm
                v += eps_m*np.cos(X)*np.sin((m+1)*Y + phi)*gm
        else:
            u = A*np.sin(X)*np.cos(Y)*np.cos(Z); v = -A*np.cos(X)*np.sin(Y)*np.cos(Z)
            w = (2*A/3)*np.cos(X)*np.cos(Y)*np.sin(Z)
            u += 0.05*np.sin(2*X)*np.cos(Y)*np.cos(2*Z)
            v += 0.05*np.cos(X)*np.sin(2*Y)*np.cos(2*Z)
        w[:,:,0] = 0.0; w[:,:,-1] = 0.0
        return u, v, w


    def _project(self):
        """
        Leray–Helmholtz projection onto ∇·u=0 subspace.
        FIX-3: 4th-order z-Laplacian in the tridiagonal solve.
        """
        sp = self._sp
        uh = sp.fwd_xy(self.u)*sp.DA
        vh = sp.fwd_xy(self.v)*sp.DA
        div_u = sp.inv_xy(1j*sp.KX*uh)
        div_v = sp.inv_xy(1j*sp.KY*vh)
        div_w = _fd4_dz(self.w, self.dz)
        div_  = div_u + div_v + div_w
        div_h = sp.fwd_xy(div_)

        dz = self.dz; Nz = self.Nz; c4 = 1.0/(12.0*dz**2)
        Lz_mat = np.zeros((Nz, Nz), dtype=np.complex128)
        for r in range(2, Nz-2):
            Lz_mat[r, r-2] = -c4;   Lz_mat[r, r-1] =  16.0*c4
            Lz_mat[r, r  ] = -30.0*c4; Lz_mat[r, r+1] = 16.0*c4; Lz_mat[r, r+2] = -c4
        inv_dz2 = 1.0/dz**2
        Lz_mat[1, 0] =  inv_dz2; Lz_mat[1, 1] = -2.0*inv_dz2; Lz_mat[1, 2] =  inv_dz2
        Lz_mat[-2,-3] = inv_dz2; Lz_mat[-2,-2] = -2.0*inv_dz2; Lz_mat[-2,-1] = inv_dz2
        Lz_mat[0,:]  = 0.0; Lz_mat[0,  0 ] = 1.0
        Lz_mat[-1,:] = 0.0; Lz_mat[-1,-1]  = 1.0

        phi_h   = np.zeros_like(div_h)
        kxy2_fl = (sp.KX[:,:,0]**2 + sp.KY[:,:,0]**2)
        for i in range(self.Nx):
            for j in range(self.Ny):
                if i == 0 and j == 0:
                    continue
                k2 = float(kxy2_fl[i,j]); L = Lz_mat.copy()
                for r in range(1, Nz-1):
                    L[r, r] += k2
                rhs = div_h[i,j,:].copy(); rhs[0] = 0.0; rhs[-1] = 0.0
                phi_h[i,j,:] = np.linalg.solve(L, rhs)

        phi = sp.inv_xy(phi_h)
        self.u -= sp.dx(sp.fwd_xy(phi)*sp.DA)
        self.v -= sp.dy(sp.fwd_xy(phi)*sp.DA)
        self.w -= _fd4_dz(phi, self.dz)
        self.w[:,:,0] = 0.0; self.w[:,:,-1] = 0.0

    def _vorticity(self, u, v, w):
        sp = self._sp
        uh = sp.fwd_xy(u)*sp.DA; vh = sp.fwd_xy(v)*sp.DA; wh = sp.fwd_xy(w)*sp.DA
        om_x = sp.inv_xy(1j*sp.KY*wh) - _fd4_dz(v, self.dz)
        om_y = _fd4_dz(u, self.dz)    - sp.inv_xy(1j*sp.KX*wh)
        om_z = sp.inv_xy(1j*sp.KX*vh) - sp.inv_xy(1j*sp.KY*uh)
        return om_x, om_y, om_z


    def _rhs(self, u, v, w):
        """
        FIX-2: Leray projector now includes ∂(adv_w)/∂z in div_a_h.
        """
        sp = self._sp
        uh = sp.fwd_xy(u)*sp.DA; vh = sp.fwd_xy(v)*sp.DA; wh = sp.fwd_xy(w)*sp.DA
        du_dx = sp.inv_xy(1j*sp.KX*uh); du_dy = sp.inv_xy(1j*sp.KY*uh); du_dz = _fd4_dz(u, self.dz)
        dv_dx = sp.inv_xy(1j*sp.KX*vh); dv_dy = sp.inv_xy(1j*sp.KY*vh); dv_dz = _fd4_dz(v, self.dz)
        dw_dx = sp.inv_xy(1j*sp.KX*wh); dw_dy = sp.inv_xy(1j*sp.KY*wh); dw_dz = _fd4_dz(w, self.dz)
        adv_u = u*du_dx + v*du_dy + w*du_dz
        adv_v = u*dv_dx + v*dv_dy + w*dv_dz
        adv_w = u*dw_dx + v*dw_dy + w*dw_dz
        au_h = sp.fwd_xy(adv_u)*sp.DA
        av_h = sp.fwd_xy(adv_v)*sp.DA
        aw_h = sp.fwd_xy(adv_w)*sp.DA
        # FIX-2: include z-divergence of adv_w
        daw_dz_h = sp.fwd_xy(_fd4_dz(sp.inv_xy(aw_h), self.dz))*sp.DA
        div_a_h  = 1j*sp.KX*au_h + 1j*sp.KY*av_h + daw_dz_h
        au_h -= 1j*sp.KX*div_a_h/sp.KXY2
        av_h -= 1j*sp.KY*div_a_h/sp.KXY2
        rhs_u = -sp.inv_xy(au_h); rhs_v = -sp.inv_xy(av_h); rhs_w = -sp.inv_xy(aw_h)
        rhs_w[:,:,0] = 0.0; rhs_w[:,:,-1] = 0.0
        if self.nu_hyp > 0.0:
            K2h = sp.KXY2
            rhs_u -= self.nu_hyp*sp.inv_xy((K2h**4)*sp.fwd_xy(u)*sp.DA)
            rhs_v -= self.nu_hyp*sp.inv_xy((K2h**4)*sp.fwd_xy(v)*sp.DA)
            rhs_w -= self.nu_hyp*sp.inv_xy((K2h**4)*sp.fwd_xy(w)*sp.DA)
        return rhs_u, rhs_v, rhs_w


    def _dp45_step(self, u, v, w, dt):
        A, B4, B5, E = _DP_A, _DP_B4, _DP_B5, _DP_E
        ku = np.zeros((7,*u.shape)); kv = np.zeros_like(ku); kw = np.zeros_like(ku)
        if self._k_prev is not None:
            ku[0], kv[0], kw[0] = self._k_prev
        else:
            ku[0], kv[0], kw[0] = self._rhs(u, v, w)
        for i in range(1, 7):
            u_i = u + dt*sum(A[i,j]*ku[j] for j in range(i))
            v_i = v + dt*sum(A[i,j]*kv[j] for j in range(i))
            w_i = w + dt*sum(A[i,j]*kw[j] for j in range(i))
            ku[i], kv[i], kw[i] = self._rhs(u_i, v_i, w_i)
        u5 = u + dt*sum(B5[i]*ku[i] for i in range(7))
        v5 = v + dt*sum(B5[i]*kv[i] for i in range(7))
        w5 = w + dt*sum(B5[i]*kw[i] for i in range(7))
        u4 = u + dt*sum(B4[i]*ku[i] for i in range(7))
        v4 = v + dt*sum(B4[i]*kv[i] for i in range(7))
        w4 = w + dt*sum(B4[i]*kw[i] for i in range(7))
        self._k_prev = (ku[6], kv[6], kw[6])
        eu = dt*sum(E[i]*ku[i] for i in range(7))
        ev = dt*sum(E[i]*kv[i] for i in range(7))
        ew = dt*sum(E[i]*kw[i] for i in range(7))
        sc_u = self.atol + self.rtol*np.maximum(np.abs(u), np.abs(u5))
        sc_v = self.atol + self.rtol*np.maximum(np.abs(v), np.abs(v5))
        sc_w = self.atol + self.rtol*np.maximum(np.abs(w), np.abs(w5))
        err  = float(np.sqrt((np.mean((eu/sc_u)**2) + np.mean((ev/sc_v)**2)
                               + np.mean((ew/sc_w)**2))/3.0))
        return u5, v5, w5, u4, v4, w4, err

    def _adapt_dt(self, err, dt):
        factor = min(5.0, max(0.1, 0.9*err**(-0.2) if err > 0 else 5.0))
        new_dt = dt*factor
        if self.blowup_estimate is not None:
            margin = max(1e-8, 0.02*(self.blowup_estimate.T_star - self.t))
            new_dt = min(new_dt, margin)
        return float(np.clip(new_dt, 1e-10, 0.02))

    def _record_diagnostics(self, u, v, w):
        """FIX-1: h2_u is now computed and appended."""
        sp = self._sp
        om_x, om_y, om_z = self._vorticity(u, v, w)
        om_mag = np.sqrt(om_x**2 + om_y**2 + om_z**2)
        max_om = float(om_mag.max())
        if self.times:
            dt_rec = self.t - self.times[-1]
            self._bkm_acc += 0.5*(self.max_omega[-1] + max_om)*dt_rec
        uh = sp.fwd_xy(u)*sp.DA; vh = sp.fwd_xy(v)*sp.DA
        du_dx = sp.inv_xy(1j*sp.KX*uh); du_dy = sp.inv_xy(1j*sp.KY*uh); du_dz = _fd4_dz(u, self.dz)
        dv_dx = sp.inv_xy(1j*sp.KX*vh); dv_dy = sp.inv_xy(1j*sp.KY*vh); dv_dz = _fd4_dz(v, self.dz)
        dw_dx = sp.inv_xy(1j*sp.KX*sp.fwd_xy(w)*sp.DA)
        dw_dy = sp.inv_xy(1j*sp.KY*sp.fwd_xy(w)*sp.DA)
        dw_dz = _fd4_dz(w, self.dz)
        Sx = om_x*du_dx + om_y*du_dy + om_z*du_dz
        Sy = om_x*dv_dx + om_y*dv_dy + om_z*dv_dz
        Sz = om_x*dw_dx + om_y*dw_dy + om_z*dw_dz
        vs = float(np.sqrt(Sx**2 + Sy**2 + Sz**2).max())
        dV = (self.Lxy/self.Nx)*(self.Lxy/self.Ny)*self.dz
        energy   = float(0.5*(u**2 + v**2 + w**2).sum()*dV)
        helicity = float((u*om_x + v*om_y + w*om_z).sum()*dV)
        h1_u = sp.sobolev_h1(u)
        h2_u = sp.sobolev_h2(u)   # FIX-1: was missing
        om_xh = sp.fwd_xy(om_x)*sp.DA; om_yh = sp.fwd_xy(om_y)*sp.DA
        gox = sp.inv_xy(1j*sp.KX*om_xh); goy = sp.inv_xy(1j*sp.KY*om_yh)
        pali = float(0.5*(gox**2 + goy**2).sum()*dV)
        self.times.append(self.t); self.max_omega.append(max_om)
        self.bkm.append(self._bkm_acc); self.vortex_stretch.append(vs)
        self.energy.append(energy); self.helicity.append(helicity)
        self.h1_u.append(h1_u); self.h2_u.append(h2_u)   # FIX-1
        self.palinstrophy.append(pali)
        if len(self.times) % 20 == 0:
            est = _fit_blowup_euler3d(self.times, self.max_omega, self.bkm)
            if est is not None:
                self.blowup_estimate = est._replace(vortex_stretch=vs)
        if max_om > self.blowup_threshold:
            self._near_blowup = True


    def _extract_self_similar_snapshot(self, fam):
        if self.blowup_estimate is None:
            return None
        T = self.blowup_estimate.T_star; tau = T - self.t
        if tau < 1e-10:
            return None
        c = fam.c; sp = self._sp
        om_x, om_y, om_z = self._vorticity(self.u, self.v, self.w)
        om_mag = np.sqrt(om_x**2 + om_y**2 + om_z**2)
        iz_strip = slice(self.Nz//3, 2*self.Nz//3)
        sub  = om_mag[:,:,iz_strip]; flat = np.argmax(sub)
        ix_s = flat//(sub.shape[1]*sub.shape[2])
        iz_s = flat % sub.shape[2] + self.Nz//3
        x_star = self.X[ix_s,0,0]; z_star = self.Z[0,0,iz_s]
        Nss = 32; R = 4.0
        xi = np.linspace(-R,R,Nss); zeta = np.linspace(-R,R,Nss)
        XI, ZE = np.meshgrid(xi, zeta, indexing='ij')
        X_p = np.mod(x_star + XI*tau**c, self.Lxy)
        Z_p = np.clip(z_star + ZE*tau**c, 0, self.Lz)
        dx = self.Lxy/self.Nx; dz_g = self.Lz/self.Nz
        ix = (X_p/dx).astype(int) % self.Nx
        iz = np.clip((Z_p/dz_g).astype(int), 0, self.Nz-1)
        iy_m = int((self.Lxy/2)/(self.Lxy/self.Ny)) % self.Ny
        def ex(f): return f[ix, iy_m, iz]
        U_t  = np.stack([ex(self.u), ex(self.v), ex(self.w)], axis=-1)*tau**(-fam.gamma_u)
        Om_t = np.stack([ex(om_x),   ex(om_y),   ex(om_z)],  axis=-1)*tau**(-fam.gamma_om)
        return E3DSelfSimilarSnapshot(t=self.t, xi=xi, zeta=zeta,
            U_tilde=U_t, Om_tilde=Om_t, c=c, T_star=T, family=fam.label)


    def step(self, n_steps=1, record_every=4):
        """FIX-4: _project() called after every accepted RK step."""
        for _ in range(n_steps):
            for _ in range(20):
                u5, v5, w5, u4, v4, w4, err = self._dp45_step(self.u, self.v, self.w, self.dt)
                if err <= 1.0 or self.dt < 1e-10:
                    self.u = u5; self.v = v5; self.w = w5
                    self.w[:,:,0] = 0.0; self.w[:,:,-1] = 0.0
                    self._project()               # FIX-4
                    self._k_prev = None           # invalidate FSAL after projection
                    self.t += self.dt; self._step += 1
                    self.dt_history.append(self.dt)
                    self.dt = self._adapt_dt(err, self.dt)
                    break
                else:
                    self.dt *= max(0.1, 0.9*err**(-0.2)); self._k_prev = None
            else:
                warnings.warn(f"Euler3D step at t={self.t:.5f} failed.", RuntimeWarning)
            if self._step % record_every == 0:
                self._record_diagnostics(self.u, self.v, self.w)
            if self._near_blowup and self.blowup_estimate is not None:
                last_t = (self.self_similar_snapshots[-1].t
                          if self.self_similar_snapshots else -np.inf)
                if self.t - last_t > 1e-5:
                    fl  = (self.blowup_estimate.family_match
                           or (self._family.label if self._family else "S0"))
                    fam = EULER3D_CATALOGUE.get(fl, EulerSingularityFamily.empirical(0))
                    sn  = self._extract_self_similar_snapshot(fam)
                    if sn is not None:
                        self.self_similar_snapshots.append(sn)


    def run_until(self, T_end, record_every=10, max_steps=1_000_000):
        steps = 0
        while self.t < T_end and steps < max_steps:
            if self.t + self.dt > T_end:
                self.dt = T_end - self.t
            self.step(1, record_every); steps += 1
            if self._near_blowup and self.blowup_estimate is not None:
                est = self.blowup_estimate
                if self.t >= est.T_star - 1e-6:
                    self._blown_up = True; self._blowup_time = self.t
                    print(f"[Euler3DSolver] Blow-up t≈{self.t:.6f}  T*={est.T_star:.6f}  "
                          f"c={est.c_fit:.4f}  family={est.family_match or '?'}  "
                          f"‖ω‖={self.max_omega[-1]:.3e}")
                    break

    def spectral_energy_budget(self):
        k_sh, E_k, slope = self._sp.energy_spectrum(self.u, self.v, self.w)
        return {"k_shells": k_sh, "E_k": E_k, "slope": slope}

    def bkm_criterion_met(self, threshold=50.0):
        if len(self.max_omega) < 3:
            return False
        return bool(self._bkm_acc > threshold
                    and self.max_omega[-1] > self.max_omega[-2] > self.max_omega[-3])

    def energy_conservation_error(self):
        if len(self.energy) < 2:
            return 0.0
        return abs(self.energy[-1] - self.energy[0])/(abs(self.energy[0]) + 1e-14)

    def linearised_growth_rate(self, epsilon=1e-5):
        rng = np.random.default_rng(0)
        du  = rng.standard_normal(self.u.shape)
        dv  = rng.standard_normal(self.v.shape)
        dw  = rng.standard_normal(self.w.shape)
        nrm = np.sqrt(np.linalg.norm(du)**2 + np.linalg.norm(dv)**2 + np.linalg.norm(dw)**2)
        du *= epsilon/nrm; dv *= epsilon/nrm; dw *= epsilon/nrm
        f0  = self._rhs(self.u, self.v, self.w)
        fp  = self._rhs(self.u+du, self.v+dv, self.w+dw)
        Lf  = tuple((fp[i]-f0[i])/epsilon for i in range(3))
        Ln  = np.sqrt(sum(np.linalg.norm(Lf[i])**2 for i in range(3)))
        dn  = np.sqrt(np.linalg.norm(du)**2 + np.linalg.norm(dv)**2 + np.linalg.norm(dw)**2)
        return float(Ln/dn)

    def summary(self):
        lines = [
            "="*70, " _Euler3DSolver — 3-D Incompressible Euler", "="*70,
            f"  Grid       : {self.Nx}×{self.Ny}×{self.Nz}  Lxy={self.Lxy:.4f}  Lz={self.Lz:.4f}",
            f"  Time       : t = {self.t:.6f}",
            f"  Steps      : {self._step}",
            f"  Current dt : {self.dt:.2e}",
        ]
        if self.max_omega:
            lines += [
                f"  ‖ω‖_∞      : {self.max_omega[-1]:.4e}",
                f"  BKM ∫‖ω‖   : {self._bkm_acc:.4e}",
                f"  Vortex str : {self.vortex_stretch[-1]:.4e}",
                f"  Kinetic E  : {self.energy[-1]:.4e}",
                f"  Helicity   : {self.helicity[-1]:.4e}",
                f"  H¹ ‖u‖     : {self.h1_u[-1]:.4e}",
                f"  H² ‖u‖     : {self.h2_u[-1]:.4e}",
                f"  Palistr.   : {self.palinstrophy[-1]:.4e}",
                f"  ΔE_rel     : {self.energy_conservation_error():.4e}",
                f"  BKM met?   : {self.bkm_criterion_met()}",
                f"  Blown up?  : {self._blown_up}"
                + (f"  (t*≈{self._blowup_time:.6f})" if self._blown_up else ""),
            ]
        if self.blowup_estimate:
            e = self.blowup_estimate
            lines += ["", "  ── Blow-up estimate ──",
                      f"  T*={e.T_star:.6f}  c={e.c_fit:.4f}  family={e.family_match or '?'}"]
        if self._family:
            lines += ["", f"  ── Family: {self._family}"]
        lines.append("="*70)
        return "\n".join(lines)

    def __repr__(self):
        return (f"_Euler3DSolver(N={self.Nx}×{self.Ny}×{self.Nz}, "
                f"t={self.t:.4f}, dt={self.dt:.2e}, "
                f"family={self._family.label if self._family else None})")


# ══════════════════════════════════════════════════════════════════════════════
# ── SECTION 4 :  RESULT EXTRACTION HELPERS ────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

def _extract_ipm_results(s: _IPMSolver) -> dict:
    N = s.N
    x = np.linspace(0, s.Lx, N, endpoint=False)
    z = np.linspace(0, s.Lz, N, endpoint=False)
    X, Z = np.meshgrid(x, z, indexing="ij")
    eb   = s.spectral_energy_budget()
    return {
        "rho": s.rho.copy(), "X": X, "Z": Z,
        "times": list(s.times), "max_grad": list(s.max_grad),
        "enstrophy": list(s.enstrophy), "energy": list(s.energy),
        "h1_norm": list(s.h1_norm), "h2_norm": list(s.h2_norm),
        "dt_history": list(s.dt_history), "blowup_rate": list(s.blowup_rate),
        "k_shells": eb["k_shells"], "E_k": eb["E_k"], "spec_slope": eb["slope"],
        "blowup_estimate": s.blowup_estimate,
        "self_similar_snapshots": s.self_similar_snapshots,
        "family": s._family,
        "growth_rate": s.linearised_growth_rate() if s.times else float("nan"),
    }


def _extract_boussinesq_results(s: _BoussinesqSolver) -> dict:
    N = s.N
    x = np.linspace(0, s.L, N, endpoint=False)
    X, Z = np.meshgrid(x, x, indexing="ij")
    eb   = s.spectral_energy_budget()
    ce   = s.conservation_errors()
    return {
        "omega": s.omega.copy(), "theta": s.theta.copy(), "X": X, "Z": Z,
        "times": list(s.times), "max_omega": list(s.max_omega),
        "bkm": list(s.bkm), "energy": list(s.energy),
        "enstrophy": list(s.enstrophy), "h1_omega": list(s.h1_omega),
        "h2_omega": list(s.h2_omega), "h1_theta": list(s.h1_theta),
        "dt_history": list(s.dt_history),
        "k_shells": eb["k_shells"], "E_k": eb["E_k"], "spec_slope": eb["slope"],
        "blowup_estimate": s.blowup_estimate,
        "self_similar_snapshots": s.self_similar_snapshots,
        "family": s._family,
        "bkm_met": s.bkm_criterion_met(), "conservation": ce,
        "growth_rate": s.linearised_growth_rate() if s.times else float("nan"),
        "instability_order": s.instability_order_estimate() if s.times else -1,
    }


def _extract_euler3d_results(s: _Euler3DSolver) -> dict:
    eb  = s.spectral_energy_budget()
    iz  = s.Nz // 2
    return {
        "u": s.u.copy(), "v": s.v.copy(), "w": s.w.copy(),
        "X": s.X, "Y": s.Y, "Z": s.Z, "iz_mid": iz,
        "times": list(s.times), "max_omega": list(s.max_omega),
        "bkm": list(s.bkm), "vortex_stretch": list(s.vortex_stretch),
        "energy": list(s.energy), "helicity": list(s.helicity),
        "h1_u": list(s.h1_u), "palinstrophy": list(s.palinstrophy),
        "dt_history": list(s.dt_history),
        "k_shells": eb["k_shells"], "E_k": eb["E_k"], "spec_slope": eb["slope"],
        "blowup_estimate": s.blowup_estimate,
        "self_similar_snapshots": s.self_similar_snapshots,
        "family": s._family,
        "blown_up": s._blown_up, "blowup_time": s._blowup_time,
        "bkm_met": s.bkm_criterion_met(),
        "energy_err": s.energy_conservation_error(),
        "growth_rate": s.linearised_growth_rate() if s.times else float("nan"),
    }



"""
NS3DSOLved — Diagnostics & Convergence Plotting Extension
==========================================================
Drop-in addition to the NS3DSOLved suite.

Exports
-------
    plot_solver_specs(ipm, bsq, e3d, save_path=None)
        Configuration panels: domain, BCs, resolution, dealiasing,
        CFL / time-step control, viscosity.

    ConvergenceStudy(solver_cls, base_config, N_levels, T_end)
        Runs the same solver at multiple resolutions and builds
        convergence / T* stabilisation data.

    plot_convergence(study, save_path=None)
        Two-panel figure:  ‖ω‖_∞  vs t at every level  +  T* vs N.

    plot_diagnostics_dashboard(ipm_res, bsq_res, e3d_res, save_path=None)
        Quantitative dashboard: vorticity growth rates, BKM integrals,
        energy spectra, dt history, and self-similar profile (if available).
"""

from __future__ import annotations

import textwrap, warnings
from typing import Optional, Dict, List
from dataclasses import dataclass, field

import numpy as np
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import LogNorm, SymLogNorm, Normalize
from matplotlib.patches import FancyArrowPatch, FancyBboxPatch
from mpl_toolkits.axes_grid1 import make_axes_locatable

# ── palette ──────────────────────────────────────────────────────────────────
_BG   = "#808080"
_FG   = "#008080"
_GRID = "#DC143C"
_C = {          # one accent per solver + neutral
    "ipm":  "#4fc3f7",   # cyan
    "bsq":  "#81c784",   # green
    "e3d":  "#ff8a65",   # amber-orange
    "grey": "#90a0b0",
    "gold": "#ffd54f",
}

_MONO = {"family": "monospace"}
_SANS = {"family": "DejaVu Sans"}




# ══════════════════════════════════════════════════════════════════════════════
# helpers
# ══════════════════════════════════════════════════════════════════════════════

def _ax_spine_color(ax, color=_GRID):
    for sp in ax.spines.values():
        sp.set_edgecolor(color)

def _ax_off(ax):
    ax.axis("off")

def _title_bar(ax, label, color):
    ax.set_title(label, fontsize=9, fontweight="bold",
                 color=color, pad=4, **_SANS)
    ax.title.set_position([0.5, 1.0])

def _col_line(ax, xs, ys, color, label=None, lw=1.6, alpha=1.0, ls="-"):
    return ax.plot(xs, ys, color=color, lw=lw, alpha=alpha,
                   ls=ls, label=label)[0]

def _safe_log_ax(ax):
    ax.set_yscale("log")


def _stamp(ax, txt, color, x=0.03, y=0.97, fs=7):
    ax.text(x, y, txt, transform=ax.transAxes,
            fontsize=fs, color=color, va="top", ha="left",
            fontfamily="monospace",
            bbox=dict(boxstyle="round,pad=0.25", fc="#0d0f1488", ec="none"))


def _fit_powerlaw_window(times, vals, window=12):
    """Fit ‖ω‖ ~ (T*-t)^{-γ} over last `window` points; return T*, γ, residual."""
    if len(times) < window + 2 or min(vals[-window:]) <= 0:
        return None, None, None
    t  = np.array(times[-window:], dtype=np.float64)
    v  = np.array(vals[-window:],  dtype=np.float64)
    lv = np.log(v)
    T_cands = t[-1] + np.logspace(-5, 0, 200)
    best = (np.inf, t[-1]+0.01, 0.0)
    for T in T_cands:
        ldt = np.log(T - t)
        A   = np.column_stack([np.ones_like(ldt), ldt])
        cf, *_ = np.linalg.lstsq(A, lv, rcond=None)
        res = float(np.sqrt(np.mean((lv - A @ cf)**2)))
        if res < best[0]:
            best = (res, T, -cf[1])
    return best[1], best[2], best[0]          # T*, gamma, residual


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 1 — Solver Configuration Panels
# ══════════════════════════════════════════════════════════════════════════════

def _make_spec_table(solver_kind: str, solver) -> List[str]:
    """Return list of two-column rows [(label, value), ...]."""
    PI = np.pi
    rows: list[tuple[str, str]] = []

    if solver_kind == "ipm":
        N  = solver.N;  Lx = solver.Lx; Lz = solver.Lz
        sp = solver._sp
        kmax_x = int((N//3) * (2*PI/Lx) / (2*PI/Lx))   # in grid units
        rows = [
            ("Equations",       "2-D IPM  (∂ρ/∂t + u·∇ρ = 0)"),
            ("Domain",          f"[0, {Lx/PI:.2g}π] × [0, {Lz/PI:.2g}π]"),
            ("Boundary (x)",    "Periodic  (Fourier)"),
            ("Boundary (z)",    "Half-range cosine (DCT-II)"),
            ("Resolution",      f"{N} × {N}  ({N*N:,} DoF)"),
            ("Dealiasing",      f"2/3-rule  |kx|<{N//3}, kz<{N//3}"),
            ("Time scheme",     "Dormand–Prince RK4(5)"),
            ("CFL control",     "Adaptive — err-based factor ∈ [0.1, 10]"),
            ("dt₀",             f"{solver.dt:.2e}"),
            ("rtol / atol",     f"{solver.rtol:.0e} / {solver.atol:.0e}"),
            ("Hyp. viscosity ν","None" if solver.nu_sponge == 0 else
                                f"{solver.nu_sponge:.2e}  (sponge layer)"),
            ("Blow-up thresh",  f"{solver.blowup_threshold:.0e}"),
            ("IC family",       solver._family.label if solver._family else "default"),
        ]

    elif solver_kind == "bsq":
        N  = solver.N; L = solver.L; sp = solver._sp
        rows = [
            ("Equations",       "2-D Boussinesq  (ω-ψ-θ form)"),
            ("Domain",          f"[0, {L/PI:.2g}π]²  fully periodic"),
            ("Boundary (all)",  "Periodic  (2-D FFT)"),
            ("Resolution",      f"{N} × {N}  ({N*N:,} DoF)"),
            ("Dealiasing",      f"2/3-rule  |kx|, |kz| < {N//3}"),
            ("Time scheme",     "Dormand–Prince RK4(5)  (FSAL)"),
            ("CFL control",     "Adaptive — err-based, margin near T*"),
            ("dt₀",             f"{solver.dt:.2e}"),
            ("rtol / atol",     f"{solver.rtol:.0e} / {solver.atol:.0e}"),
            ("Hyp. viscosity ν","None" if solver.nu_hyp == 0 else
                                f"{solver.nu_hyp:.2e}  (∇⁸ hyper-visc.)"),
            ("Blow-up thresh",  f"{solver.blowup_threshold:.0e}"),
            ("IC type",         getattr(solver, '_ic_type', 'hou_luo')),
            ("BKM ∫‖ω‖",       f"{solver._bkm_acc:.4g}  "
                                f"(met: {solver.bkm_criterion_met()})"),
        ]

    else:   # e3d
        sp = solver._sp
        rows = [
            ("Equations",       "3-D Incompressible Euler"),
            ("Domain",          f"[0, {solver.Lxy/PI:.2g}π]² × [0, {solver.Lz/PI:.2g}π]"),
            ("Boundary (x,y)",  "Periodic  (2-D FFT)"),
            ("Boundary (z)",    "Stress-free / no-penetration  (w=0)"),
            ("Resolution",      f"{solver.Nx}×{solver.Ny}×{solver.Nz}  "
                                f"({solver.Nx*solver.Ny*solver.Nz:,} DoF)"),
            ("Dealiasing",      f"2/3-rule (xy); 4th-order FD stencil (z)"),
            ("Projection",      "Leray–Helmholtz  (per RK step, FIX-4)"),
            ("Time scheme",     "Dormand–Prince RK4(5)  (FSAL)"),
            ("CFL control",     "Adaptive — err-based, margin near T*"),
            ("dt₀",             f"{solver.dt:.2e}"),
            ("rtol / atol",     f"{solver.rtol:.0e} / {solver.atol:.0e}"),
            ("Hyp. viscosity ν","None" if solver.nu_hyp == 0 else
                                f"{solver.nu_hyp:.2e}  (∇⁸ hyper-visc.)"),
            ("Blow-up thresh",  f"{solver.blowup_threshold:.0e}"),
            ("IC family",       solver._family.label if solver._family else "default"),
            ("ΔE_rel",          f"{solver.energy_conservation_error():.3e}"),
        ]

    return rows


def _draw_spec_panel(ax, rows, accent, title):
    """Render a neat two-column spec table inside an Axes."""
    _ax_off(ax)
    # Header bar
    ax.add_patch(FancyBboxPatch(
        (0, 0.94), 1, 0.06, transform=ax.transAxes, clip_on=False,
        boxstyle="square,pad=0", fc=accent+"44", ec=accent, lw=1.2))
    ax.text(0.5, 0.97, title, transform=ax.transAxes,
            ha="center", va="center", fontsize=9, fontweight="bold",
            color=accent, **_SANS)

    y      = 0.90
    dy     = 0.90 / max(len(rows), 1)
    dy     = min(dy, 0.062)
    col_x  = [0.03, 0.44]
    for i, (lbl, val) in enumerate(rows):
        bg_alpha = "#1a1f2e" if i % 2 == 0 else _BG
        ax.add_patch(FancyBboxPatch(
            (0, y - dy*0.85), 1, dy*0.88,
            transform=ax.transAxes, clip_on=True,
            boxstyle="square,pad=0", fc=bg_alpha, ec="none"))
        ax.text(col_x[0], y - dy*0.3, lbl + ":", transform=ax.transAxes,
                fontsize=6.4, color=_C["grey"], va="center", ha="left",
                **_MONO)
        ax.text(col_x[1], y - dy*0.3, val, transform=ax.transAxes,
                fontsize=6.4, color=_FG, va="center", ha="left",
                **_MONO)
        y -= dy


def plot_solver_specs(ipm, bsq, e3d, save_path: Optional[str] = None):
    """
    Render a three-column configuration panel (one per solver).
    Shows domain, BCs, resolution, dealiasing, CFL/dt control, viscosity.
    """
    fig = plt.figure(figsize=(18, 9), facecolor=_BG)
    fig.suptitle(
        "NS3DSOLved — Solver Configuration Reference",
        fontsize=13, fontweight="bold", color=_FG, y=0.99, **_SANS)

    gs = gridspec.GridSpec(1, 3, figure=fig, wspace=0.06,
                           left=0.01, right=0.99, top=0.93, bottom=0.02)

    specs = [
        ("ipm", ipm, _C["ipm"],  "_IPMSolver — 2-D Incompressible Porous Media"),
        ("bsq", bsq, _C["bsq"],  "_BoussinesqSolver — 2-D Inviscid Boussinesq"),
        ("e3d", e3d, _C["e3d"],  "_Euler3DSolver — 3-D Incompressible Euler"),
    ]
    for col, (kind, solver, accent, title) in enumerate(specs):
        ax = fig.add_subplot(gs[0, col])
        rows = _make_spec_table(kind, solver)
        _draw_spec_panel(ax, rows, accent, title)

    # Shared footer: dealiasing diagram
    _draw_dealiasing_schematic(fig)

    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"[plot_solver_specs] saved → {save_path}")
    return fig


def _draw_dealiasing_schematic(fig):
    """Small inline k-space dealiasing diagram at the bottom."""
    ax = fig.add_axes([0.03, 0.00, 0.94, 0.05])
    _ax_off(ax)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    msg = ("Dealiasing:  all three solvers use the 2/3-rule  —  "
           "modes with |k| > N/3 · (2π/L) are zeroed in spectral space  "
           "to prevent aliasing errors in quadratic nonlinearities.")
    ax.text(0.5, 0.5, msg, ha="center", va="center",
            fontsize=6.5, color=_C["grey"], **_MONO,
            bbox=dict(boxstyle="round,pad=0.3", fc=_GRID+"88", ec=_GRID))


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 2 — Convergence Study
# ══════════════════════════════════════════════════════════════════════════════

@dataclass
class ConvergenceLevel:
    N:           int
    T_end:       float
    times:       List[float]
    max_vals:    List[float]     # ‖ω‖_∞ or ‖∇ρ‖_∞
    T_star_est:  Optional[float]
    gamma_est:   Optional[float]
    residual:    Optional[float]
    energy_err:  float


@dataclass
class ConvergenceStudy:
    """
    Runs one solver at multiple resolutions and collects
    ‖ω‖_∞(t), estimated T*, and γ for each level.

    Parameters
    ----------
    solver_cls   :  _IPMSolver | _BoussinesqSolver | _Euler3DSolver
    base_config  :  dict of kwargs (N will be overridden at each level)
    N_levels     :  list of grid sizes, e.g. [32, 64, 128]
    T_end        :  integration horizon for every level
    label        :  short display label
    val_key      :  attribute name of the growth diagnostic list
                    ('max_grad' for IPM, 'max_omega' for BSQ/Euler3D)
    """
    solver_cls:  type
    base_config: dict
    N_levels:    List[int]
    T_end:       float
    label:       str        = "solver"
    val_key:     str        = "max_omega"
    levels:      List[ConvergenceLevel] = field(default_factory=list)

    def run(self, record_every: int = 5, verbose: bool = True):
        """Execute all resolution levels."""
        self.levels.clear()
        for N in self.N_levels:
            cfg = dict(self.base_config)
            cfg["N"] = N
            if verbose:
                print(f"[ConvergenceStudy/{self.label}]  N={N} …", flush=True)
            try:
                s = self.solver_cls(**cfg)
                s.run_until(self.T_end, record_every=record_every)
            except Exception as exc:
                warnings.warn(f"N={N} failed: {exc}")
                continue
            vals  = list(getattr(s, self.val_key, []))
            times = list(s.times)
            T_star, gamma, res = _fit_powerlaw_window(times, vals, window=16)
            # energy error
            if hasattr(s, "energy_conservation_error"):
                ee = s.energy_conservation_error()
            elif hasattr(s, "conservation_errors"):
                ce = s.conservation_errors()
                ee = ce.get("delta_energy_rel", 0.0)
            else:
                ee = 0.0
            self.levels.append(ConvergenceLevel(
                N=N, T_end=self.T_end,
                times=times, max_vals=vals,
                T_star_est=T_star, gamma_est=gamma, residual=res,
                energy_err=ee,
            ))
            if verbose:
                tag = f"T*≈{T_star:.5f}  γ≈{gamma:.3f}" if T_star else "no estimate"
                print(f"          done — {tag}")
        return self


def plot_convergence(study: ConvergenceStudy, save_path: Optional[str] = None):
    """
    Two-row figure:
      Row 1 — ‖ω‖_∞(t) on log scale at every resolution (colour-coded by N)
      Row 2 — T* stabilisation and γ convergence vs 1/N
    """
    if not study.levels:
        raise RuntimeError("ConvergenceStudy.run() has not been called yet.")

    fig = plt.figure(figsize=(14, 9), facecolor=_BG)
    fig.suptitle(
        f"Convergence Study  ·  {study.label}  ·  N = "
        + ", ".join(str(lv.N) for lv in study.levels),
        fontsize=11, fontweight="bold", color=_FG, y=1.01, **_SANS)

    gs  = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.35,
                            left=0.07, right=0.97, top=0.95, bottom=0.09)

    ax_growth  = fig.add_subplot(gs[0, :2])   # ‖ω‖_∞ traces
    ax_spec    = fig.add_subplot(gs[0, 2])    # dt history
    ax_Tstar   = fig.add_subplot(gs[1, 0])    # T* vs N
    ax_gamma   = fig.add_subplot(gs[1, 1])    # γ vs N
    ax_energy  = fig.add_subplot(gs[1, 2])    # energy error vs N

    Ns    = [lv.N for lv in study.levels]
    cmap  = mpl.colormaps["plasma"]
    cols  = [cmap(i / max(len(study.levels)-1, 1)) for i in range(len(study.levels))]

    # ── Row 0 left: ‖ω‖_∞(t) ────────────────────────────────────────────────
    ax = ax_growth
    for lv, col in zip(study.levels, cols):
        if not lv.times:
            continue
        ax.semilogy(lv.times, np.maximum(lv.max_vals, 1e-14),
                    color=col, lw=1.6, label=f"N={lv.N}")
        if lv.T_star_est:
            ax.axvline(lv.T_star_est, color=col, lw=0.8, ls="--", alpha=0.5)
    ax.set_xlabel("t", fontsize=8); ax.set_ylabel("‖ω‖_∞  (log)", fontsize=8)
    _title_bar(ax, "Vorticity / Gradient Growth  ‖ω‖_∞(t)", _C["ipm"])
    ax.legend(fontsize=7, ncol=2, loc="upper left")
    _ax_spine_color(ax)
    _stamp(ax, "dashed verticals = estimated T*", _C["grey"], x=0.6, y=0.12)

    # ── Row 0 right: growth-rate trajectory ──────────────────────────────────
    ax = ax_spec
    for lv, col in zip(study.levels, cols):
        if len(lv.times) < 4:
            continue
        # local logarithmic slope  d/dt log‖ω‖
        t  = np.array(lv.times)
        mv = np.maximum(lv.max_vals, 1e-14)
        dldt = np.gradient(np.log(mv), t)
        ax.plot(t, dldt, color=col, lw=1.3, label=f"N={lv.N}")
    ax.set_xlabel("t", fontsize=8); ax.set_ylabel("d/dt  log ‖ω‖_∞", fontsize=8)
    _title_bar(ax, "Instantaneous Growth Rate", _C["bsq"])
    ax.legend(fontsize=7, loc="upper left")
    _ax_spine_color(ax)
    ax.axhline(0, color=_C["grey"], lw=0.6, ls=":")

    # ── Row 1: T* stabilisation ───────────────────────────────────────────────
    ax = ax_Tstar
    Tvals = [lv.T_star_est for lv in study.levels]
    valid = [(N, T) for N, T in zip(Ns, Tvals) if T is not None]
    if valid:
        Nv, Tv = zip(*valid)
        ax.semilogx(Nv, Tv, "o-", color=_C["gold"], lw=2,
                    ms=6, mec=_BG, mew=1)
        ax.axhline(np.mean(Tv[-3:]) if len(Tv) >= 3 else Tv[-1],
                   color=_C["gold"], lw=0.8, ls="--", alpha=0.5)
    ax.set_xlabel("N  (grid size)", fontsize=8); ax.set_ylabel("T*", fontsize=8)
    _title_bar(ax, "T* Stabilisation vs N", _C["gold"])
    _ax_spine_color(ax)

    # ── Row 1 mid: γ convergence ──────────────────────────────────────────────
    ax = ax_gamma
    gvals = [lv.gamma_est for lv in study.levels]
    valid_g = [(N, g) for N, g in zip(Ns, gvals) if g is not None]
    if valid_g:
        Ng, Gv = zip(*valid_g)
        ax.semilogx(Ng, Gv, "s-", color=_C["bsq"], lw=2,
                    ms=6, mec=_BG, mew=1)
        # dashed reference from finest level
        ax.axhline(Gv[-1], color=_C["bsq"], lw=0.7, ls="--", alpha=0.45)
        ax.set_ylim(max(0, min(Gv)*0.7), max(Gv)*1.3)
    ax.set_xlabel("N  (grid size)", fontsize=8); ax.set_ylabel("γ  (blow-up exponent)", fontsize=8)
    _title_bar(ax, "Blow-up Exponent γ vs N", _C["bsq"])
    _ax_spine_color(ax)

    # ── Row 1 right: energy error ─────────────────────────────────────────────
    ax = ax_energy
    evals = [lv.energy_err for lv in study.levels]
    ax.loglog(Ns, np.maximum(evals, 1e-16), "D-",
              color=_C["e3d"], lw=2, ms=6, mec=_BG, mew=1)
    # spectral convergence reference line
    if len(Ns) >= 2:
        xs = np.array(Ns, dtype=float)
        ax.loglog(xs, evals[-1]*(xs[-1]/xs)**4,
                  "--", color=_C["grey"], lw=0.8, label="~N⁻⁴ ref")
        ax.legend(fontsize=7)
    ax.set_xlabel("N  (grid size)", fontsize=8); ax.set_ylabel("ΔE_rel", fontsize=8)
    _title_bar(ax, "Energy Conservation Error vs N", _C["e3d"])
    _ax_spine_color(ax)

    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"[plot_convergence] saved → {save_path}")
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 3 — Full Quantitative Diagnostics Dashboard
# ══════════════════════════════════════════════════════════════════════════════

def plot_diagnostics_dashboard(
        ipm_res:  Optional[dict],
        bsq_res:  Optional[dict],
        e3d_res:  Optional[dict],
        save_path: Optional[str] = None,
):
    """
    Comprehensive multi-panel diagnostics figure.

    Each result dict is the return value of the corresponding
    _extract_*_results() helper (or None to skip that solver).

    Panels
    ------
    Row 0 — Growth curves:  ‖∇ρ‖_∞ (IPM) | ‖ω‖_∞ (BSQ) | ‖ω‖_∞ (Euler3D)
    Row 1 — BKM / Enstrophy | Energy spectrum | dt history
    Row 2 — Self-similar profile (latest snapshot, each solver)
    """
    fig = plt.figure(figsize=(18, 14), facecolor=_BG)
    fig.suptitle(
        "NS3DSOLved — Quantitative Diagnostics Dashboard",
        fontsize=13, fontweight="bold", color=_FG, y=1.005, **_SANS)

    gs_outer = gridspec.GridSpec(3, 1, figure=fig, hspace=0.40,
                                 left=0.06, right=0.98, top=0.97, bottom=0.05)
    gs0 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_outer[0], wspace=0.30)
    gs1 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_outer[1], wspace=0.30)
    gs2 = gridspec.GridSpecFromSubplotSpec(1, 3, subplot_spec=gs_outer[2], wspace=0.30)

    results  = [ipm_res, bsq_res, e3d_res]
    accents  = [_C["ipm"], _C["bsq"], _C["e3d"]]
    names    = ["IPM 2-D", "Boussinesq 2-D", "Euler 3-D"]
    val_keys = ["max_grad", "max_omega", "max_omega"]
    enst_keys= ["enstrophy", "enstrophy", None]
    bkm_keys = [None, "bkm", "bkm"]

    # ── ROW 0:  Growth curves ─────────────────────────────────────────────────
    for col, (res, acc, name, vk) in enumerate(zip(results, accents, names, val_keys)):
        ax = fig.add_subplot(gs0[col])
        if res is None or not res.get("times"):
            _ax_off(ax); ax.text(0.5, 0.5, "no data", ha="center",
                                 va="center", color=_C["grey"], fontsize=9,
                                 transform=ax.transAxes); continue
        t  = np.array(res["times"])
        mv = np.maximum(res[vk], 1e-14)

        _col_line(ax, t, mv, acc, label="‖ω / ∇ρ‖_∞")
        _safe_log_ax(ax)

        # Annotate blow-up estimate
        be = res.get("blowup_estimate")
        if be is not None:
            T_star = getattr(be, "T_star", None) or getattr(be, "T_star", None)
            if T_star and T_star > t[0]:
                ax.axvline(T_star, color=_C["gold"], lw=1.0, ls="--")
                ax.text(T_star, mv.max()*0.5, f"T*={T_star:.4f}",
                        color=_C["gold"], fontsize=6.5, rotation=90,
                        va="top", ha="right", **_MONO)

        # Growth-rate annotation: fit power law over tail
        T_s, gam, res_fit = _fit_powerlaw_window(list(t), list(mv))
        if gam is not None:
            _stamp(ax, f"γ ≈ {gam:.3f}  (T*≈{T_s:.4f})", acc)

        ax.set_xlabel("t", fontsize=8)
        ax.set_ylabel("‖ω‖_∞  (log)", fontsize=8)
        _title_bar(ax, f"{name}  ·  Growth Curve", acc)
        _ax_spine_color(ax)
        ax.legend(fontsize=7, loc="upper left")

    # ── ROW 1a:  BKM / Enstrophy ──────────────────────────────────────────────
    ax_bkm = fig.add_subplot(gs1[0])
    for res, acc, name, bk, ek in zip(results, accents, names, bkm_keys, enst_keys):
        if res is None or not res.get("times"):
            continue
        t = np.array(res["times"])
        if bk and bk in res and res[bk]:
            ax_bkm.plot(t, res[bk], color=acc, lw=1.4,
                        ls="-", label=f"{name} BKM ∫‖ω‖")
        if ek and ek in res and res[ek]:
            ax_bkm.plot(t, res[ek], color=acc, lw=1.0,
                        ls="--", alpha=0.6, label=f"{name} Enstrophy")
    _safe_log_ax(ax_bkm)
    ax_bkm.set_xlabel("t", fontsize=8); ax_bkm.set_ylabel("value  (log)", fontsize=8)
    _title_bar(ax_bkm, "BKM Integral & Enstrophy", _C["grey"])
    ax_bkm.legend(fontsize=6, ncol=1, loc="upper left")
    _ax_spine_color(ax_bkm)

    # ── ROW 1b:  Energy spectra ───────────────────────────────────────────────
    ax_spec = fig.add_subplot(gs1[1])
    _last_Ek = None  # track last valid Ek for the k^-5/3 reference

    for res, acc, name in zip(results, accents, names):
        if res is None:
            continue
        ks = res.get("k_shells"); Ek = res.get("E_k")
        if ks is None or Ek is None:
            continue

        mask = (ks > 0) & (Ek > 0)
        ax_spec.loglog(ks[mask], Ek[mask], color=acc, lw=1.4,
                       label=f"{name}  ({res.get('spec_slope', float('nan')):.2f})")
        _last_Ek = Ek  # keep a reference for the guide line below

    # Kolmogorov -5/3 reference — only draw if at least one spectrum was plotted
    if _last_Ek is not None:
        k_ref = np.logspace(0, 2, 80)
        _mask_ref = _last_Ek > 0
        _norm = _last_Ek[_mask_ref].max() if _mask_ref.any() else 1.0
        E_ref = k_ref ** (-5 / 3) * _norm
        ax_spec.loglog(k_ref, E_ref, ":", color=_C["grey"], lw=0.9, label="k⁻⁵/³")

    ax_spec.set_xlabel("k", fontsize=8); ax_spec.set_ylabel("E(k)  (log)", fontsize=8)
    _title_bar(ax_spec, "Energy Spectra  (slope in legend)", _C["grey"])
    ax_spec.legend(fontsize=6.5, loc="upper right")
    _ax_spine_color(ax_spec)

    # ── ROW 1c:  dt history ───────────────────────────────────────────────────
    ax_dt = fig.add_subplot(gs1[2])
    for res, acc, name in zip(results, accents, names):
        if res is None or not res.get("dt_history"):
            continue
        dts = res["dt_history"]
        ax_dt.semilogy(range(len(dts)), dts, color=acc, lw=0.9, alpha=0.85,
                       label=name)
    ax_dt.set_xlabel("Step index", fontsize=8); ax_dt.set_ylabel("Δt  (log)", fontsize=8)
    _title_bar(ax_dt, "Adaptive Time-Step History", _C["grey"])
    ax_dt.legend(fontsize=7, loc="upper right")
    _ax_spine_color(ax_dt)
    _stamp(ax_dt, "shrinks near T*", _C["gold"])

    # ── ROW 2:  Self-similar snapshots ────────────────────────────────────────
    snap_labels = ["IPM  R̃(ξ,ζ)", "BSQ  Ω̃(ξ,ζ)", "Euler3D  |Ω̃|(ξ,ζ)"]
    for col, (res, acc, name, slbl) in enumerate(
            zip(results, accents, names, snap_labels)):
        ax = fig.add_subplot(gs2[col])
        if res is None:
            _ax_off(ax); continue
        snaps = res.get("self_similar_snapshots", [])
        if not snaps:
            _ax_off(ax)
            ax.text(0.5, 0.5,
                    f"{name}\n(no self-similar\nsnapshot yet)",
                    ha="center", va="center", color=_C["grey"],
                    fontsize=8, transform=ax.transAxes)
            continue
        sn   = snaps[-1]
        # choose field
        if col == 0:
            field = getattr(sn, "R", None)
        elif col == 1:
            field = getattr(sn, "Omega", None)
        else:
            Om   = getattr(sn, "Om_tilde", None)
            field = (np.sqrt((Om**2).sum(axis=-1))
                     if Om is not None and Om.ndim == 3
                     else Om)
        if field is None:
            _ax_off(ax); continue

        xi   = sn.xi; zeta = sn.zeta
        ext  = [xi[0], xi[-1], zeta[0], zeta[-1]]
        vmax = np.abs(field).max()
        if vmax < 1e-14:
            vmax = 1.0
        im = ax.imshow(field.T, origin="lower", extent=ext,
                       cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
        div  = make_axes_locatable(ax)
        cax  = div.append_axes("right", size="4%", pad=0.06)
        cb   = fig.colorbar(im, cax=cax)
        cb.ax.yaxis.set_tick_params(labelcolor=_FG, labelsize=6)
        cb.outline.set_edgecolor(_GRID)
        ax.set_xlabel("ξ", fontsize=8); ax.set_ylabel("ζ", fontsize=8)
        c_val = getattr(sn, "c", getattr(sn, "alpha", "?"))
        T_s   = getattr(sn, "T_star", "?")
        _title_bar(ax, f"{slbl}  ·  t={sn.t:.4f}  c={c_val:.3f}  T*={T_s:.4f}", acc)
        _ax_spine_color(ax)
        ax.contour(xi, zeta, field.T, levels=8,
                   colors="white", linewidths=0.35, alpha=0.5)

    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"[plot_diagnostics_dashboard] saved → {save_path}")
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 4 — Vorticity Growth Rate Detail Panel
# ══════════════════════════════════════════════════════════════════════════════

def plot_growth_rate_detail(
        ipm_res:  Optional[dict],
        bsq_res:  Optional[dict],
        e3d_res:  Optional[dict],
        window:   int = 16,
        save_path: Optional[str] = None,
):
    """
    Detailed two-row panel for vorticity / gradient growth-rate analysis.

    Row 0 — (T*-t)·‖ω‖_∞ compensated plot  (should flatten near blow-up)
    Row 1 — Local exponent γ(t) = -d log‖ω‖/d log(T*-t) and its convergence
    """
    fig, axes = plt.subplots(2, 3, figsize=(16, 8), facecolor=_BG)
    fig.subplots_adjust(hspace=0.38, wspace=0.30,
                        left=0.07, right=0.97, top=0.93, bottom=0.09)
    fig.suptitle("Vorticity Growth Rate Analysis",
                 fontsize=12, fontweight="bold", color=_FG, y=0.98, **_SANS)

    triples = [
        (ipm_res, _C["ipm"], "IPM 2-D",    "max_grad",  "max_grad"),
        (bsq_res, _C["bsq"], "BSQ 2-D",    "max_omega", "max_omega"),
        (e3d_res, _C["e3d"], "Euler3D",    "max_omega", "max_omega"),
    ]
    for col, (res, acc, name, vk, _) in enumerate(triples):
        ax_comp = axes[0, col]
        ax_exp  = axes[1, col]

        if res is None or not res.get("times"):
            for ax in (ax_comp, ax_exp):
                _ax_off(ax)
                ax.text(0.5, 0.5, "no data", ha="center", va="center",
                        color=_C["grey"], fontsize=9, transform=ax.transAxes)
            continue

        t  = np.array(res["times"],  dtype=np.float64)
        mv = np.maximum(res[vk],     1e-14).astype(np.float64)
        be = res.get("blowup_estimate")
        T_star, gamma_fit, res_fit = _fit_powerlaw_window(list(t), list(mv), window)

        # ── compensated plot ────────────────────────────────────────────────
        ax = ax_comp
        if T_star and T_star > t[-1]:
            tau = np.maximum(T_star - t, 1e-14)
            compensated = tau**gamma_fit * mv
            ax.plot(t, compensated, color=acc, lw=1.6)
            ax.axhline(compensated[-min(window, len(compensated)):].mean(),
                       color=_C["gold"], lw=0.9, ls="--",
                       label="plateau = C (prefactor)")
            ax.set_xlabel("t", fontsize=8)
            ax.set_ylabel(f"(T*−t)^γ · ‖ω‖_∞  (γ≈{gamma_fit:.3f})", fontsize=7.5)
            _stamp(ax, f"T*={T_star:.5f}   γ={gamma_fit:.4f}   res={res_fit:.4f}", acc)
            ax.legend(fontsize=7)
        else:
            ax.semilogy(t, mv, color=acc, lw=1.5)
            _stamp(ax, "T* not yet resolved", _C["grey"])
            ax.set_xlabel("t", fontsize=8); ax.set_ylabel("‖ω‖_∞", fontsize=8)
        _title_bar(ax, f"{name}  Compensated  (T*-t)^γ·‖ω‖", acc)
        _ax_spine_color(ax)

        # ── local exponent ──────────────────────────────────────────────────
        ax = ax_exp
        if T_star and T_star > t[-1] and len(t) > 6:
            tau   = np.maximum(T_star - t, 1e-14)
            # γ_local = -d log(mv) / d log(tau)
            log_tau = np.log(tau)
            log_mv  = np.log(mv)
            # central differences
            dlm  = np.gradient(log_mv,  t)
            dlt  = np.gradient(log_tau, t)
            with np.errstate(divide="ignore", invalid="ignore"):
                gamma_local = -np.where(np.abs(dlt) > 1e-12, dlm/dlt, np.nan)
            ax.plot(t, gamma_local, color=acc, lw=1.4)
            ax.axhline(gamma_fit, color=_C["gold"], lw=1.0, ls="--",
                       label=f"global γ={gamma_fit:.3f}")
            ax.set_ylim(max(-1, np.nanmin(gamma_local)-0.5),
                        min(10, np.nanmax(gamma_local)+0.5))
            ax.set_xlabel("t", fontsize=8)
            ax.set_ylabel("γ(t)  local exponent", fontsize=8)
            ax.legend(fontsize=7)
        else:
            _ax_off(ax)
            ax.text(0.5, 0.5, "insufficient data\nfor local exponent",
                    ha="center", va="center",
                    color=_C["grey"], fontsize=8, transform=ax.transAxes)
        _title_bar(ax, f"{name}  Local Blow-up Exponent γ(t)", acc)
        _ax_spine_color(ax)

    if save_path:
        fig.savefig(save_path, bbox_inches="tight")
        print(f"[plot_growth_rate_detail] saved → {save_path}")
    return fig


# ══════════════════════════════════════════════════════════════════════════════
# SECTION 5 — All-in-one convenience wrapper  (to be called from NS3DSolved)
# ══════════════════════════════════════════════════════════════════════════════

def plot_all_diagnostics(
        ipm_solver,
        bsq_solver,
        e3d_solver,
        ipm_results:  Optional[dict] = None,
        bsq_results:  Optional[dict] = None,
        e3d_results:  Optional[dict] = None,
        save_prefix:  str            = "ns3d",
):
    """
    Convenience wrapper: generates all four figures and saves them.

    Returns (fig_specs, fig_convergence_placeholder, fig_dashboard, fig_growth)
    """
    from pathlib import Path

    print("[plot_all_diagnostics]  Generating specs panel …")
    f1 = plot_solver_specs(ipm_solver, bsq_solver, e3d_solver,
                           save_path=f"{save_prefix}_specs.png")

    print("[plot_all_diagnostics]  Generating dashboard …")
    f3 = plot_diagnostics_dashboard(ipm_results, bsq_results, e3d_results,
                                    save_path=f"{save_prefix}_dashboard.png")

    print("[plot_all_diagnostics]  Generating growth-rate panel …")
    f4 = plot_growth_rate_detail(ipm_results, bsq_results, e3d_results,
                                 save_path=f"{save_prefix}_growth_rates.png")

    print("[plot_all_diagnostics]  Done.")
    return f1, None, f3, f4


import numpy as np


# ══════════════════════════════════════════════════════════════════════════════
# ── SECTION 5 :  PLOT STYLE HELPERS ──────────────────────────────────────────
# ══════════════════════════════════════════════════════════════════════════════

# ACADEMIC & LEGIBLE COLOR PALETTE
# Replaced dark backgrounds with white and assigned distinct, professional colors.

# --- Core Layout Colors ---
_DARK  = "#000000"       # Pure Black (Best for axes, labels, and text)
_PANEL = "#FFFFFF"       # White background (The only choice for LaTeX PDFs)
_EDGE  = "#BCBCBC"       # Light gray for subtle grid lines (won't distract)
_DIM   = "#7F7F7F"       # True Medium Gray for auxiliary data

# --- Data Series (High Contrast & Professional) ---
_RED   = "#D62728"       # Sharp Academic Red
_AMBER = "#FF7F0E"       # High-visibility Orange (Colorblind safe)
_GREEN = "#2CA02C"       # Forest Green (Readable on white)
_CYAN  = "#17BECF"       # Deep Cyan
_WHITE = "#FFFFFF"       # Actual Pure White (Use for plot backgrounds)

# --- Colormaps (Perceptually Uniform) ---
_CMAP_FIELD = "RdBu_r"    # Diverging: Blue to Red (Great for heatmaps)
_CMAP_SPEED = "viridis"   # Sequential: Purple to Yellow (Best for magnitude)
_CMAP_OMEGA = "seismic"   # Diverging: Centered on zero
_CMAP_SPEC  = "magma"     # High-contrast alternative to inferno



def _ax_style(ax, title="", xlabel="", ylabel="", col=_RED, fs=8):
    ax.set_facecolor(_PANEL)
    for sp in ax.spines.values():
        sp.set_edgecolor(_EDGE); sp.set_linewidth(0.8)
    ax.tick_params(axis="both", labelsize=6, colors=_DIM, length=3, width=0.6)
    ax.xaxis.label.set_color(_DIM); ax.yaxis.label.set_color(_DIM)
    if title:   ax.set_title(title, color=col, fontsize=fs, fontweight="bold", pad=4)
    if xlabel:  ax.set_xlabel(xlabel, fontsize=6, color=_DIM)
    if ylabel:  ax.set_ylabel(ylabel, fontsize=6, color=_DIM)


def _colorbar(fig, ax, im, label="", col=_DIM):
    div = make_axes_locatable(ax); cax = div.append_axes("right", size="4%", pad=0.04)
    cb  = fig.colorbar(im, cax=cax)
    cb.ax.tick_params(labelsize=5, colors=col, length=2)
    cb.outline.set_edgecolor(_EDGE)
    if label: cb.set_label(label, fontsize=5, color=col)
    return cb


def _annotate_blowup(ax, T_star, col=_RED):
    ax.axvline(T_star, color=col, lw=0.9, ls="--", alpha=0.85,
               label=f"T*≈{T_star:.4f}")
    ax.legend(fontsize=5, facecolor=_PANEL, edgecolor=_EDGE,
              labelcolor=col, framealpha=0.9)


def _label_box(ax, text, col=_RED, loc="upper left"):
    at = AnchoredText(text, loc=loc,
                      prop=dict(size=5, color=col, family="monospace"),
                      frameon=True, pad=0.3, borderpad=0.4)
    at.patch.set(facecolor=_PANEL, edgecolor=_EDGE, alpha=0.9)
    ax.add_artist(at)


def _draw_family_card(ax, family, blowup_est, col, solver):
    ax.set_facecolor(_PANEL); ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values(): sp.set_edgecolor(_EDGE)
    lines = [f"[ {solver} — Family Card ]", ""]
    if family is not None:
        lines += [f"Label : {family.label}", f"Kind  : {family.kind}",
                  f"Order : n = {family.order}"]
        if hasattr(family, "alpha"):
            lines += [f"α     : {family.alpha:.4f}", f"β     : {family.beta:.4f}",
                      f"γ     : {family.gamma:.4f}"]
        elif hasattr(family, "gamma_th"):
            lines += [f"c     : {family.c:.4f}", f"γ_ω   : {family.gamma_om:.4f}",
                      f"γ_θ   : {family.gamma_th:.4f}"]
        elif hasattr(family, "gamma_u"):
            lines += [f"c     : {family.c:.4f}", f"γ_u   : {family.gamma_u:.4f}",
                      f"γ_ω   : {family.gamma_om:.4f}"]
    lines.append("")
    if blowup_est is not None:
        lines += ["── Blow-up estimate ──", f"T*    : {blowup_est.T_star:.5f}"]
        if hasattr(blowup_est, "alpha_fit"):
            lines += [f"α_fit : {blowup_est.alpha_fit:.4f}",
                      f"Match : {blowup_est.family_match or '?'}",
                      f"Conf  : {blowup_est.confidence:.2f}"]
        elif hasattr(blowup_est, "c_fit"):
            lines += [f"c_fit : {blowup_est.c_fit:.4f}",
                      f"Match : {blowup_est.family_match or '?'}",
                      f"Conf  : {blowup_est.confidence:.2f}"]
    else:
        lines.append("Blow-up : not yet detected")
    ax.text(0.5, 0.5, "\n".join(lines), ha="center", va="center",
            transform=ax.transAxes, color=col, fontsize=6, fontfamily="monospace",
            linespacing=1.6,
            bbox=dict(facecolor=_DARK, edgecolor=_EDGE,
                      boxstyle="round,pad=0.5", linewidth=0.8))
    ax.set_title(f"{solver} — Singularity Card", color=col,
                 fontsize=8, fontweight="bold", pad=4)


import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.ticker import LogLocator, LogFormatterSciNotation
import warnings
warnings.filterwarnings("ignore")



class PoisoningAttackCleanLabelBackdoor:
    """
    This class implements a poisoning attack with a clean label backdoor.
    """

    def __init__(
        self,
        trigger_func: Callable,
        target_label: Union[int, str, np.ndarray],
        dirty_label: Union[int, str, np.ndarray],
        flip_prob: float = 0.5,
        trigger_alpha: float = 0.5,
        poison_rate: float = 0.1,
        backdoor_trigger: Optional[Union[int, str, np.ndarray]] = None,
        backdoor_target: Optional[Union[int, str, np.ndarray]] = None,
        training_dataset: Optional[np.ndarray] = None,
        training_params: Optional[dict] = None,
        prior_mean: float = 0,
        prior_std: float = 1,

        bh_mass_solar : float = 30.0,
        fluid_N       : int   = 32,
        fluid_dt      : float = 1.5e-3,
        ns_viscosity  : float = 8e-4,
        bh_spin       : float = 0.68,
        n_steps       : int   = 900,

        family:            str            = "S0",
        ipm_config:        Optional[dict] = None,
        boussinesq_config: Optional[dict] = None,
        euler3d_config:    Optional[dict] = None,
    ) -> None:
        """
        Initialize the PoisoningAttackCleanLabelBackdoor instance.
        """
        self.trigger_func = trigger_func
        self.target_label = target_label
        self.dirty_label = dirty_label
        self.flip_prob = flip_prob
        self.trigger_alpha = trigger_alpha
        self.poison_rate = poison_rate
        self.backdoor_trigger = backdoor_trigger if backdoor_trigger is not None else 0
        self.backdoor_target = backdoor_target
        self.training_dataset = training_dataset
        self.training_params = training_params
        self.prior_mean = prior_mean
        self.prior_std = prior_std
        self.prior = np.random.normal(prior_mean, prior_std)
        self._family_label = family

        # ── Build solver instances ─────────────────────────────────────────
        ipm_kw = dict(N=128, Lx=2*PI, Lz=2*PI, dt=5e-4)
        ipm_kw.update(ipm_config or {})
        ipm_kw["family"] = family
        self._ipm_solver = _IPMSolver(**ipm_kw)

        bsq_kw = dict(N=128, L=2*PI, dt=2e-4, ic_type="hou_luo")
        bsq_kw.update(boussinesq_config or {})
        bsq_kw["family"] = family
        self._bsq_solver = _BoussinesqSolver(**bsq_kw)

        e3d_kw = dict(Nx=32, Ny=32, Nz=32, Lxy=2*PI, Lz=PI, dt=1e-3, ic_type="hou_luo")
        e3d_kw.update(euler3d_config or {})
        e3d_kw["family"] = family
        self._e3d_solver = _Euler3DSolver(**e3d_kw)

        # ── Results caches ─────────────────────────────────────────────────
        self._ipm_results: Optional[dict] = None
        self._bsq_results: Optional[dict] = None
        self._e3d_results: Optional[dict] = None

        # ── Plot figure caches (None = not yet rendered) ───────────────────
        self._fig_specs:       Optional["plt.Figure"] = None
        self._fig_dashboard:   Optional["plt.Figure"] = None
        self._fig_growth:      Optional["plt.Figure"] = None
        self._fig_convergence: Optional["plt.Figure"] = None

        # ── Optional convergence study (set via attach_convergence_study) ──
        self._convergence_study: Optional["ConvergenceStudy"] = None

        self._singularity_sim = SingularitySimulation(
        bh_mass_solar = bh_mass_solar,
        fluid_N       = fluid_N,
        fluid_dt      = fluid_dt,
        ns_viscosity  = ns_viscosity,
        n_steps       = n_steps )

        self._sim_result : dict | None = None

        # Instantiate the singularity simulation with the provided parameters
        self.sim = SingularitySimulation(
        bh_mass_solar=30.0,   # Schwarzschild radius ~88 km (LIGO-like)
        bh_spin=0.68,         # Spinning Kerr BH
        fluid_N=32,
        fluid_dt=1.5e-3,
        ns_viscosity=8e-4,
        n_steps=900,
         )
        self.sim.run_all(gw_companion_solar=1.4)

        self.sim = SingularitySimulation(
            bh_mass_solar=bh_mass_solar,
            bh_spin=bh_spin,
            fluid_N=fluid_N,
            fluid_dt=fluid_dt,
            ns_viscosity=ns_viscosity,
            n_steps=n_steps,
        )

    def run_singularity_plot(
        self, save_path: str = "singularity_full.png"
       ) -> plt.Figure:
        """Run simulation and produce the 24-panel figure."""
        self.sim.run_all()
        return plot_all(self.sim, save_path=save_path)


    # ──   Core lazy property ────────────────────────────────────────────────
    @property
    def singularity(self) -> dict:
        """Run everything once, then cache forever."""
        if self._sim_result is None:
            self._sim_result = self._singularity_sim.simulate(mode="all")
        return self._sim_result

    # ──   Convenience scalar properties ────────────────────────────────────
    @property
    def hawking_temperature(self) -> float:
        """Hawking temperature T_H = ħc³ / (8πGMk_B)  [K]."""
        return self.singularity["bh_temperature"]

    @property
    def bekenstein_entropy(self) -> float:
        """Bekenstein-Hawking entropy S = 4πGM² / (ħc)  [dimensionless]."""
        return self.singularity["bh_entropy"]

    @property
    def schwarzschild_radius(self) -> float:
        """Event-horizon radius r_s = 2GM/c²  [m]."""
        return self.singularity["bh_r_s"]

    @property
    def euler_energy(self) -> np.ndarray:
        """Kinetic energy time-series E(t) for the inviscid Euler run."""
        return self.singularity["euler_E"]

    @property
    def euler_enstrophy(self) -> np.ndarray:
        """Enstrophy Ω(t) = ½∫|ω|² for the inviscid Euler run."""
        return self.singularity["euler_W"]

    @property
    def euler_max_vorticity(self) -> np.ndarray:
        """‖ω‖∞(t) — BKM blowup indicator for Euler."""
        return self.singularity["euler_omMax"]

    @property
    def ns_energy(self) -> np.ndarray:
        """Kinetic energy time-series E(t) for the Navier-Stokes run."""
        return self.singularity["ns_E"]

    @property
    def ns_enstrophy(self) -> np.ndarray:
        """Enstrophy Ω(t) for the Navier-Stokes run."""
        return self.singularity["ns_W"]

    @property
    def ns_max_vorticity(self) -> np.ndarray:
        """‖ω‖∞(t) — BKM blowup indicator for NS."""
        return self.singularity["ns_omMax"]

    @property
    def bkm_integral(self) -> dict:
        """
        Beale-Kato-Majda integral ∫₀ᵗ ‖ω‖∞ dτ.
        Keys: 'euler', 'ns', 't_euler', 't_ns'.
        If this stays bounded  →  no singularity.
        If it diverges         →  finite-time blowup.
        """
        return self.singularity["bkm_integral"]

    @property
    def geodesic(self) -> dict:
        """Full geodesic trajectory dict (r, phi, x, y, tau, r_s, label)."""
        return self.singularity["bh_geodesic"]

    # ──   Re-run with fresh parameters ─────────────────────────────────────
    def resimulate(self, mode: str = "all", n_steps: int = 900) -> dict:
        """
        Force a fresh run and refresh the cache.

        Parameters
        ----------
        mode    : 'all' | 'blackhole' | 'euler' | 'ns'
        n_steps : how many time steps to integrate
        """
        self._sim_result = None               # invalidate cache
        self._singularity_sim._eu_data  = None
        self._singularity_sim._ns_data  = None
        self._singularity_sim._geo_data = None
        self._singularity_sim.eu.t_hist = []; self._singularity_sim.eu.E_hist = []
        self._singularity_sim.eu.W_hist = []; self._singularity_sim.eu.omMax  = []
        self._singularity_sim.ns.t_hist = []; self._singularity_sim.ns.E_hist = []
        self._singularity_sim.ns.W_hist = []; self._singularity_sim.ns.omMax  = []
        self._sim_result = self._singularity_sim.simulate(mode=mode,
                                                          n_steps=n_steps)
        return self._sim_result

    # ──   Direct model access ───────────────────────────────────────────────
    @property
    def blackhole_model(self) -> BlackHoleSingularity:
        return self._singularity_sim.blackhole

    @property
    def euler_model(self):
        return self._singularity_sim.euler

    @property
    def ns_model(self):
        return self._singularity_sim.navier_stokes


    def summarise(self):
        print(f"\n{'='*64}")
        print(f"  {self.name}")
        print(f"{'='*64}")

        print("\n── Kerr Black Hole ──────────────────────────────────────")
        print(f"  Mass             : {self.bh_mass_solar:.1f} M☉")
        print(f"  Spin  a*         : {self.bh_spin:.4f}")
        print(f"  Outer horizon r+ : {self.bh_outer_horizon:.4e} m")
        print(f"  Inner horizon r− : {self.bh_inner_horizon:.4e} m")
        print(f"  ISCO (prograde)  : {self.bh_isco_radius:.4e} m")
        print(f"  Photon orbit     : {self.bh_photon_orbit_radius:.4e} m")
        print(f"  Hawking T_H      : {self.bh_hawking_temperature:.4e} K")
        print(f"  Bekenstein S_BH  : {self.bh_bekenstein_entropy:.4e} nat")
        print(f"  Penrose η_max    : {self.bh_penrose_efficiency*100:.3f} %")
        print(f"  Spaghetti radius : {self.bh_spaghettification_radius:.3e} m")
        print(f"  Ω_H              : {self.bh_horizon_angular_velocity:.4e} rad/s")

        print("\n── Gravitational Waves ──────────────────────────────────")
        print(f"  Chirp mass Mc    : {self.gw_chirp_mass_solar:.4f} M☉")
        print(f"  Chirp time       : {self.gw_chirp_time:.3f} s")
        print(f"  Peak |h+|        : {self.gw_peak_strain:.3e}")

        print("\n── Euler (final step) ───────────────────────────────────")
        print(f"  E                : {self.euler_final_kinetic_energy:.6f}")
        print(f"  Ω (enstrophy)    : {self.euler_final_enstrophy:.6f}")
        print(f"  ‖ω‖∞             : {self.euler_final_max_vorticity:.4f}")
        print(f"  BKM integral     : {self.bkm_euler_final:.4f}")

        print("\n── Navier-Stokes (final step) ───────────────────────────")
        print(f"  E                : {self.ns_final_kinetic_energy:.6f}")
        print(f"  Ω (enstrophy)    : {self.ns_final_enstrophy:.6f}")
        print(f"  ε (dissipation)  : {self.ns_final_dissipation_rate:.4e}")
        print(f"  BKM integral     : {self.bkm_ns_final:.4f}")

        kol = self.kolmogorov_scales
        print("\n── Kolmogorov Microscales ───────────────────────────────")
        print(f"  η   (length)     : {self.kolmogorov_eta:.6f}")
        print(f"  τ_η (time)       : {self.kolmogorov_tau:.6f}")
        print(f"  v_η (velocity)   : {self.kolmogorov_velocity:.6f}")
        print(f"  ⟨ε⟩              : {self.kolmogorov_mean_dissipation:.3e}")
        print(f"  ⟨Re_λ⟩           : {self.kolmogorov_mean_taylor_re:.2f}")
        print(f"\n  repr: {self!r}\n")

    def run_and_plot(self, save_path: str = "singularity_simulation.png"):
        self.run()
        self.summarise()
        print(f"\n► Rendering 24-panel figure → {save_path}")
        self.show_plot(save_path=save_path)
        print("✔  Complete.")




    # =========================================================================
    # ❻  plot_singularity()  — COMPLETE 12-PANEL FIGURE
    # =========================================================================
    def plot_singularity(self, save_path: str = "singularity_plot.png") -> None:
        """
        Render and save the full 12-panel singularity figure.

        Panels
        ──────
        Row 0 │ BH geodesic orbit │ GR effective potential │ Kretschner scalar
        Row 1 │ Kinetic energy E(t) │ Enstrophy Ω(t) │ Max vorticity ‖ω‖∞
        Row 2 │ BKM integral (both)│ Euler ωz snapshots (×2)
        Row 3 │ NS |ωz| snapshots (×3)

        Parameters
        ----------
        save_path : file path for the output PNG (default 'singularity_plot.png')
        """

        # ── Ensure data is ready ─────────────────────────────────────────────
        _ = self.singularity                         # triggers lazy run if needed
        sim  = self._singularity_sim
        eu   = sim._eu_data
        ns   = sim._ns_data
        geo  = sim._geo_data
        bh   = sim.bh
        rs   = bh.r_s

        # ── Style constants ──────────────────────────────────────────────────
        BG        = "#FFFFFF"
        PANEL_BG  = "#000000"
        GRID_C    = "#0072B2"
        CLR_EU    = "#56B4E9"    # cyan  — Euler
        CLR_NS    = "#E69F00"    # coral — Navier-Stokes
        CLR_BH    = "#F0E442"    # gold  — black hole
        CLR_KRET  = "#CC79A7"    # violet — curvature
        CLR_BKM_E = "#009E73"    # lime  — BKM Euler
        CLR_BKM_N = "#D55E00"    # orange — BKM NS

        def _ax(ax, title, xlabel="", ylabel=""):
            """Uniform dark-theme axis styling."""
            ax.set_facecolor(PANEL_BG)
            ax.tick_params(colors="white", labelsize=7, which="both")
            for spine in ax.spines.values():
                spine.set_edgecolor(GRID_C)
            ax.xaxis.label.set_color("white")
            ax.yaxis.label.set_color("white")
            ax.set_title(title, color="white", fontsize=8.5,
                         fontweight="bold", pad=5)
            ax.set_xlabel(xlabel, fontsize=7)
            ax.set_ylabel(ylabel, fontsize=7)
            ax.grid(color=GRID_C, linestyle="--", linewidth=0.4, alpha=0.7)

        def _legend(ax):
            leg = ax.legend(fontsize=6.5, facecolor=PANEL_BG,
                            labelcolor="white", edgecolor=GRID_C,
                            framealpha=0.85)
            return leg

        def _cbar(fig, im, ax, label=""):
            cb = fig.colorbar(im, ax=ax, pad=0.02, fraction=0.046)
            cb.ax.yaxis.set_tick_params(colors="white", labelsize=5)
            cb.set_label(label, color="white", fontsize=6)
            cb.outline.set_edgecolor(GRID_C)

        # ── Figure & grid ────────────────────────────────────────────────────
        fig = plt.figure(figsize=(26, 22))
        fig.patch.set_facecolor(BG)

        gs = gridspec.GridSpec(
            4, 3, figure=fig,
            hspace=0.46, wspace=0.38,
            left=0.055, right=0.97,
            top=0.955, bottom=0.04,
        )

        fig.suptitle(
            "SINGULARITY SIMULATION  ·  "
            "Schwarzschild Black Hole  ×  3-D Euler  ×  3-D Navier–Stokes",
            color="white", fontsize=14, fontweight="bold",
        )

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (0,0)  Geodesic orbit in the equatorial plane
        # ─────────────────────────────────────────────────────────────────────
        ax00 = fig.add_subplot(gs[0, 0])
        _ax(ax00, "BH Geodesic — Equatorial Plane", "x / r_s", "y / r_s")

        x_n  = geo["x"] / rs
        y_n  = geo["y"] / rs
        # colour the track by radial distance (proximity to singularity)
        r_n  = np.hypot(x_n, y_n)
        from matplotlib.collections import LineCollection
        pts  = np.array([x_n, y_n]).T.reshape(-1, 1, 2)
        segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
        lc   = LineCollection(segs, cmap="plasma",
                              norm=plt.Normalize(r_n.min(), r_n.max()),
                              linewidth=0.9, zorder=3)
        lc.set_array(r_n[:-1])
        ax00.add_collection(lc)
        _cbar(fig, lc, ax00, "r / r_s")

        # Event horizon circle
        th   = np.linspace(0, 2*np.pi, 360)
        ax00.fill(np.cos(th), np.sin(th), color="#000000", zorder=4)
        ax00.plot(np.cos(th), np.sin(th), color="white", lw=0.7, zorder=5,
                  label="Event horizon")
        ax00.text(0, 0, "✦", color=CLR_BH, ha="center", va="center",
                  fontsize=11, zorder=6)

        # Photon sphere at r = 1.5 r_s
        ax00.plot(1.5*np.cos(th), 1.5*np.sin(th),
                  color="#aaaaff", lw=0.5, ls=":", label="Photon sphere")
        ax00.set_aspect("equal")
        _legend(ax00)
        ax00.set_xlim(-max(abs(x_n).max(), 2)-0.5, max(abs(x_n).max(), 2)+0.5)
        ax00.set_ylim(-max(abs(y_n).max(), 2)-0.5, max(abs(y_n).max(), 2)+0.5)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (0,1)  GR Effective Potential
        # ─────────────────────────────────────────────────────────────────────
        ax01 = fig.add_subplot(gs[0, 1])
        _ax(ax01, "GR Effective Potential  V_eff(r)", "r / r_s", "V_eff  (norm.)")

        r_arr = np.linspace(1.3*rs, 30*rs, 900)
        L_vals = [2.0*rs*C_L, 3.5*rs*C_L, 5.0*rs*C_L, 7.0*rs*C_L]
        colors_ = ["#ff9f1c", CLR_BH, "#2ec4b6", "#c77dff"]
        for L_, c_ in zip(L_vals, colors_):
            V = bh.effective_potential(r_arr, L=L_, mu=1.0)
            Vn = V / np.nanmax(np.abs(V))
            ax01.plot(r_arr / rs, Vn, lw=0.9, color=c_,
                      label=f"L={L_/(rs*C_L):.1f} r_s c")
        ax01.axvline(1.0, color="white", lw=0.6, ls="--", label="Horizon")
        ax01.axvline(1.5, color="#aaaaff", lw=0.5, ls=":", label="Photon sphere")
        ax01.axhline(0.0, color="#444466", lw=0.4)
        ax01.set_ylim(-1.5, 1.5)
        _legend(ax01)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (0,2)  Kretschner curvature scalar  K(r) → ∞ as r→0
        # ─────────────────────────────────────────────────────────────────────
        ax02 = fig.add_subplot(gs[0, 2])
        _ax(ax02, "Kretschner Curvature  K = 48G²M²/r⁶c⁴",
            "r / r_s", "K  [m⁻⁴]")

        r_k  = np.linspace(1.05*rs, 12*rs, 800)
        K_   = bh.kretschner_scalar(r_k)
        ax02.semilogy(r_k/rs, K_, color=CLR_KRET, lw=1.1)
        ax02.axvline(1.0, color="white", lw=0.6, ls="--", label="Horizon r_s")

        # Shade the region inside the horizon
        y_lo, y_hi = K_.min(), K_.max()
        ax02.axvspan(0, 1.0, alpha=0.12, color="white")
        ax02.text(0.55, np.sqrt(y_lo*y_hi), "Singularity\nregion",
                  color="white", fontsize=6, ha="center")
        ax02.yaxis.set_major_locator(LogLocator(numticks=6))
        ax02.yaxis.set_major_formatter(LogFormatterSciNotation())
        _legend(ax02)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (1,0)  Kinetic energy E(t)
        # ─────────────────────────────────────────────────────────────────────
        ax10 = fig.add_subplot(gs[1, 0])
        _ax(ax10, "Kinetic Energy  E(t)", "time  [arb.]", "E")

        ax10.plot(eu["t"], eu["E"], color=CLR_EU, lw=1.1, label="Euler (ν=0)")
        ax10.plot(ns["t"], ns["E"], color=CLR_NS, lw=1.1, ls="--",
                  label=f"NS (ν={sim.ns.nu:.0e})")

        # Energy decay slope annotation
        if len(eu["t"]) > 10:
            t_mid = eu["t"][len(eu["t"])//2]
            E_mid = eu["E"][len(eu["E"])//2]
            ax10.annotate("Conservation\n(Euler)", xy=(t_mid, E_mid),
                          xytext=(t_mid*0.1, E_mid*1.0),  #xytext=(t_mid*0.4, E_mid*1.1),
                          color=CLR_EU, fontsize=6,
                          arrowprops=dict(arrowstyle="->",
                                          color=CLR_EU, lw=0.6))
        _legend(ax10)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (1,1)  Enstrophy Ω(t) — blowup signal
        # ─────────────────────────────────────────────────────────────────────
        ax11 = fig.add_subplot(gs[1, 1])
        _ax(ax11, "Enstrophy  Ω(t) = ½∫|ω|²  — Singularity Signal",
            "time  [arb.]", "Ω  (log scale)")

        ax11.semilogy(eu["t"], np.maximum(eu["W"], 1e-30),
                      color=CLR_EU, lw=1.1, label="Euler")
        ax11.semilogy(ns["t"], np.maximum(ns["W"], 1e-30),
                      color=CLR_NS, lw=1.1, ls="--", label="NS")
        _legend(ax11)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (1,2)  Max vorticity  ‖ω‖∞  (BKM criterion)
        # ─────────────────────────────────────────────────────────────────────
        ax12 = fig.add_subplot(gs[1, 2])
        _ax(ax12, "Max Vorticity  ‖ω‖∞  — BKM Blowup Criterion",
            "time  [arb.]", "‖ω‖∞")

        ax12.plot(eu["t"], eu["omMax"], color=CLR_EU, lw=1.1, label="Euler")
        ax12.plot(ns["t"], ns["omMax"], color=CLR_NS, lw=1.1, ls="--",
                  label="NS")
        ax12.fill_between(eu["t"], eu["omMax"],
                          alpha=0.12, color=CLR_EU)
        ax12.fill_between(ns["t"], ns["omMax"],
                          alpha=0.12, color=CLR_NS)

        # BKM theorem label
        ax12.text(0.97, 0.97,
                  "BKM: if  ∫‖ω‖∞ dt < ∞\nthen no blowup",
                  transform=ax12.transAxes,
                  color="#cccccc", fontsize=6, va="top", ha="right",
                  bbox=dict(facecolor=PANEL_BG, edgecolor=GRID_C, pad=3))
        _legend(ax12)

        # ─────────────────────────────────────────────────────────────────────
        # PANEL (2,0)  BKM integral  ∫‖ω‖∞ dτ  (both models)
        # ─────────────────────────────────────────────────────────────────────
        ax20 = fig.add_subplot(gs[2, 0])
        _ax(ax20, "BKM Integral  ∫₀ᵗ ‖ω‖∞ dτ  (Blowup Test)",
            "time  [arb.]", "Cumulative ∫‖ω‖∞")

        bkm = self.bkm_integral
        ax20.plot(bkm["t_euler"], bkm["euler"],
                  color=CLR_BKM_E, lw=1.1, label="Euler")
        ax20.plot(bkm["t_ns"],    bkm["ns"],
                  color=CLR_BKM_N, lw=1.1, ls="--", label="NS")
        ax20.fill_between(bkm["t_euler"], bkm["euler"],
                          alpha=0.15, color=CLR_BKM_E)
        ax20.fill_between(bkm["t_ns"],    bkm["ns"],
                          alpha=0.15, color=CLR_BKM_N)
        ax20.text(0.05, 0.90,
                  "Bounded integral\n→ regularity (no blowup)",
                  transform=ax20.transAxes, color="#cccccc",
                  fontsize=6, va="top",
                  bbox=dict(facecolor=PANEL_BG, edgecolor=GRID_C, pad=3))
        _legend(ax20)

        # ─────────────────────────────────────────────────────────────────────
        # PANELS (2,1) & (2,2)  Euler vorticity snapshots
        # ─────────────────────────────────────────────────────────────────────
        eu_snaps = list(eu["snapshots"].items())
        for col_off, idx in enumerate([1, 2]):            # 2nd and 3rd snapshot
            if idx >= len(eu_snaps):
                continue
            t_s, snap = eu_snaps[idx]
            ax_ = fig.add_subplot(gs[2, col_off + 1])
            vmax = max(float(np.abs(snap).max()), 1e-10)
            im   = ax_.imshow(snap.T, origin="lower", cmap="seismic",
                              vmin=-vmax, vmax=vmax, aspect="auto",
                              interpolation="bilinear")
            _ax(ax_, f"Euler  ωz  slice  t = {t_s:.4f}")
            _cbar(fig, im, ax_, "ωz")
            ax_.set_xlabel("x  [grid]", fontsize=7)
            ax_.set_ylabel("y  [grid]", fontsize=7)

        # ─────────────────────────────────────────────────────────────────────
        # PANELS (3,0–2)  Navier-Stokes vorticity snapshots
        # ─────────────────────────────────────────────────────────────────────
        ns_snaps = list(ns["snapshots"].items())
        ns_cmaps = ["inferno", "magma", "plasma"]
        for col, (t_s, snap) in enumerate(ns_snaps[:3]):
            ax_ = fig.add_subplot(gs[3, col])
            mag  = np.abs(snap)
            vmax = max(float(mag.max()), 1e-10)
            im   = ax_.imshow(mag.T, origin="lower", cmap=ns_cmaps[col],
                              vmin=0, vmax=vmax, aspect="auto",
                              interpolation="bilinear")
            _ax(ax_, f"NS  |ωz|  slice  t = {t_s:.4f}")
            _cbar(fig, im, ax_, "|ωz|")
            ax_.set_xlabel("x  [grid]", fontsize=7)
            ax_.set_ylabel("y  [grid]", fontsize=7)

        # ─────────────────────────────────────────────────────────────────────
        # Info footer
        # ─────────────────────────────────────────────────────────────────────
        footer = (
            f"BH mass = {bh.M/M_SUN:.0f} M☉   "
            f"r_s = {rs:.3e} m   "
            f"T_Hawking = {bh.T_H:.3e} K   "
            f"S_BH = {bh.S_BH:.3e}   "
            f"Euler N = {sim.eu.N}³   "
            f"NS ν = {sim.ns.nu:.0e}   "
            f"Δt = {sim.eu.dt:.2e}"
        )
        fig.text(0.5, 0.005, footer, ha="center", va="bottom",
                 color="#888888", fontsize=6.5)

        # ─────────────────────────────────────────────────────────────────────
        # Save
        # ─────────────────────────────────────────────────────────────────────
        plt.savefig(
            save_path,
            dpi=160,
            bbox_inches="tight",
            facecolor=fig.get_facecolor(),
        )
        plt.close(fig)
        print(f"✔  plot_singularity() → saved to: {save_path}")



    def _invalidate_plot_cache(self) -> None:
        """
        Discard all cached figures.

        Called automatically at the end of every run_* method so that
        the next property access re-renders with the latest solver data.
        """
        self._fig_specs       = None
        self._fig_dashboard   = None
        self._fig_growth      = None
        self._fig_convergence = None


    def attach_convergence_study(self, study: "ConvergenceStudy") -> None:
        """
        Attach a completed ConvergenceStudy so convergence_figure works.

        Parameters
        ----------
        study : ConvergenceStudy
            A study on which ``.run()`` has already been called.
        """
        if not study.levels:
            raise ValueError("ConvergenceStudy has no levels — call .run() first.")
        self._convergence_study = study
        self._fig_convergence   = None          # force re-render

# ── Plot figure properties ─────────────────────────────────────────────────

    @property
    def specs_figure(self) -> "plt.Figure":
        """
        Configuration reference panel for all three solvers.

        Returns a cached matplotlib Figure showing domain, boundary conditions,
        resolution, dealiasing, CFL/dt control, and viscosity for each solver.
        Re-renders only when the underlying solvers have changed (cache is
        invalidated by run_ipm / run_boussinesq / run_euler3d).

        Returns
        -------
        matplotlib.figure.Figure
        """
        if self._fig_specs is None:
            self._fig_specs = plot_solver_specs(
                self._ipm_solver,
                self._bsq_solver,
                self._e3d_solver,
            )
        return self._fig_specs

    @property
    def dashboard_figure(self) -> "plt.Figure":
        """
        Full quantitative diagnostics dashboard.

        Nine-panel figure combining:
          • Growth curves  ‖ω‖_∞(t) / ‖∇ρ‖_∞(t) with annotated T* and γ
          • BKM integral and enstrophy traces
          • Energy spectra with spectral slope
          • Adaptive time-step history
          • Self-similar coordinate rescaled snapshots (when available)

        Requires at least one of run_ipm / run_boussinesq / run_euler3d to
        have been called first; solvers with no result data are shown as
        placeholder panels rather than raising.

        Returns
        -------
        matplotlib.figure.Figure
        """
        if self._fig_dashboard is None:
            self._fig_dashboard = plot_diagnostics_dashboard(
                self._ipm_results,
                self._bsq_results,
                self._e3d_results,
            )
        return self._fig_dashboard

    @property
    def growth_rate_figure(self) -> "plt.Figure":
        """
        Detailed vorticity / gradient growth-rate analysis panel.

        Six-panel figure (two rows × three solvers):
          Row 0 — Compensated plot  (T*−t)^γ · ‖ω‖_∞  (should plateau near T*)
          Row 1 — Local blow-up exponent  γ(t) = −d log‖ω‖ / d log(T*−t)
                   alongside the global power-law fit as a reference line.

        Returns
        -------
        matplotlib.figure.Figure
        """
        if self._fig_growth is None:
            self._fig_growth = plot_growth_rate_detail(
                self._ipm_results,
                self._bsq_results,
                self._e3d_results,
            )
        return self._fig_growth

    @property
    def convergence_figure(self) -> Optional["plt.Figure"]:
        """
        Resolution convergence panel.

        Shows T* stabilisation and blow-up exponent γ vs grid size N across
        all ConvergenceLevel objects stored in self._convergence_study.
        Returns None (and emits a warning) if no convergence study has been
        attached yet; populate it via::

            suite.attach_convergence_study(study)

        where ``study`` is a completed ``ConvergenceStudy`` instance.

        Returns
        -------
        matplotlib.figure.Figure or None
        """
        if self._convergence_study is None:
            import warnings
            warnings.warn(
                "No ConvergenceStudy attached. "
                "Call suite.attach_convergence_study(study) first.",
                UserWarning,
                stacklevel=2,
            )
            return None
        if self._fig_convergence is None:
            self._fig_convergence = plot_convergence(self._convergence_study)
        return self._fig_convergence

    @property
    def all_figures(self) -> dict[str, "plt.Figure"]:
        """
        Mapping of every available plot figure, keyed by name.

        Accessing this property generates all figures that have data behind
        them and returns them in a single dict.  Figures whose prerequisites
        are not yet met (e.g. convergence_figure before a study is attached)
        are silently omitted rather than raising.

        Returns
        -------
        dict[str, matplotlib.figure.Figure]
          Keys: ``"specs"``, ``"dashboard"``, ``"growth_rates"``,
                ``"convergence"``  (only when study is attached).

        Example
        -------
        >>> figs = suite.all_figures
        >>> figs["dashboard"].savefig("dashboard.png", bbox_inches="tight")
        """
        out: dict = {
            "specs":       self.specs_figure,
            "dashboard":   self.dashboard_figure,
            "growth_rates": self.growth_rate_figure,
        }
        cv = self.convergence_figure          # warns + returns None if missing
        if cv is not None:
            out["convergence"] = cv
        return out



    @property
    def ipm(self) -> _IPMSolver:
        """Live _IPMSolver instance."""
        return self._ipm_solver

    @property
    def boussinesq(self) -> _BoussinesqSolver:
        """Live _BoussinesqSolver instance."""
        return self._bsq_solver

    @property
    def euler3d(self) -> _Euler3DSolver:
        """Live _Euler3DSolver instance."""
        return self._e3d_solver

    @property
    def family(self) -> str:
        """Active singularity family label (e.g. 'S0', 'U1')."""
        return self._family_label

    @property
    def ipm_results(self) -> Optional[dict]:
        """Cached result dict from the most recent run_ipm() call."""
        return self._ipm_results

    @property
    def bsq_results(self) -> Optional[dict]:
        """Cached result dict from the most recent run_boussinesq() call."""
        return self._bsq_results

    @property
    def e3d_results(self) -> Optional[dict]:
        """Cached result dict from the most recent run_euler3d() call."""
        return self._e3d_results

    # Convenience aliases matching the original dashboard variable names
    @property
    def _ipm_results_alias(self) -> Optional[dict]:
        return self._ipm_results



    def run_ipm(
        self,
        steps: int = 400,
        every: int = 5,
        T_end: Optional[float] = None,
    ) -> dict:
        """Run the 2-D IPM solver and return a result dict."""
        s = self.ipm
        print(f"▶  IPM solver  [{self.family}]  N={s.N}×{s.N}")
        if T_end is not None:
            s.run_until(T_end, record_every=every)
        else:
            s.step(n_steps=steps, record_every=every)
        self._ipm_results = _extract_ipm_results(s)
        r  = self._ipm_results
        mg = r["max_grad"]
        growth = mg[-1]/(mg[0]+1e-14) if mg else float("nan")
        be     = r["blowup_estimate"]
        be_str = (f"  T*≈{be.T_star:.4f}  α={be.alpha_fit:.3f}  [{be.family_match}]"
                  if be else "  no blow-up detected")
        print(f"   ✓  t={s.t:.5f}  |∇ρ| growth={growth:.2f}×{be_str}")
        return self._ipm_results

    def run_boussinesq(
        self,
        steps: int = 330,
        every: int = 5,
        T_end: Optional[float] = None,
    ) -> dict:
        """Run the 2-D Boussinesq solver and return a result dict."""
        s = self.boussinesq
        print(f"▶  Boussinesq solver  [{self.family}]  N={s.N}×{s.N}")
        if T_end is not None:
            s.run_until(T_end, record_every=every)
        else:
            s.step(n_steps=steps, record_every=every)
        self._bsq_results = _extract_boussinesq_results(s)
        r  = self._bsq_results
        mw = r["max_omega"]
        growth = mw[-1]/(mw[0]+1e-14) if mw else float("nan")
        bkm    = r["bkm"][-1] if r["bkm"] else 0.0
        be     = r["blowup_estimate"]
        be_str = (f"  T*≈{be.T_star:.4f}  c={be.c_fit:.3f}  [{be.family_match}]"
                  if be else "  no blow-up detected")
        print(f"   ✓  t={s.t:.5f}  ‖ω‖ growth={growth:.2f}×"
              f"  BKM={bkm:.3f}  BKM-met={r['bkm_met']}{be_str}")
        return self._bsq_results

    def run_euler3d(
        self,
        steps: int = 110,
        every: int = 5,
        T_end: Optional[float] = None,
    ) -> dict:
        """Run the 3-D Euler solver and return a result dict."""
        s = self.euler3d
        print(f"▶  3-D Euler solver  [{self.family}]  N={s.Nx}×{s.Ny}×{s.Nz}")
        if T_end is not None:
            s.run_until(T_end, record_every=every)
        else:
            s.step(n_steps=steps, record_every=every)
        self._e3d_results = _extract_euler3d_results(s)
        r  = self._e3d_results
        mw = r["max_omega"]
        if mw:
            growth = mw[-1]/(mw[0]+1e-14)
            be     = r["blowup_estimate"]
            be_str = (f"  T*≈{be.T_star:.4f}  c={be.c_fit:.3f}  [{be.family_match}]"
                      if be else "  no blow-up detected")
            status = "⚠ BLOW-UP" if r["blown_up"] else f"‖ω‖ growth={growth:.2f}×"
            bt_str = f"  T*={r['blowup_time']:.5f}" if r["blown_up"] else ""
            print(f"   ✓  t={s.t:.5f}  {status}{bt_str}{be_str}")
        return self._e3d_results

    def run_all(
        self,
        ipm_steps=1700, ipm_every=5,
        bsq_steps=1600, bsq_every=5,
        e3d_steps=1900, e3d_every=4,
        ipm_T_end=None, bsq_T_end=None, e3d_T_end=None,
    ) -> None:
        """Run all three solvers sequentially."""
        sep = "═"*2000
        print(sep)
        print(f"  Unified Fluid Singularity Suite — family [{self.family}]")
        print(sep)
        self.run_ipm(steps=ipm_steps, every=ipm_every, T_end=ipm_T_end)
        self.run_boussinesq(steps=bsq_steps, every=bsq_every, T_end=bsq_T_end)
        self.run_euler3d(steps=e3d_steps, every=e3d_every, T_end=e3d_T_end)
        print(sep); print(" All solvers complete."); print(sep)


    def full_summary(self) -> str:
        parts = []
        if self._ipm_results  is not None: parts.append(self.ipm.summary())
        if self._bsq_results  is not None: parts.append(self.boussinesq.summary())
        if self._e3d_results  is not None: parts.append(self.euler3d.summary())
        return "\n\n".join(parts) if parts else "(no solvers have been run yet)"



    def plot_all(self, save_path: Optional[str] = "singularity_dashboard.png") -> None:  # noqa: C901
        N_ROWS, N_COLS = 3, 6
        fig = plt.figure(figsize=(26, 14), facecolor=_DARK)
        fig.patch.set_facecolor(_DARK)
        gs  = gridspec.GridSpec(N_ROWS, N_COLS, figure=fig,
                                hspace=0.52, wspace=0.42,
                                left=0.04, right=0.97, top=0.92, bottom=0.05)
        fig.text(0.5, 0.965,
                 f"SINGULARITY [{self.family}]  ·  "
                 f"IPM  |  Boussinesq  |  3D Euler",
                 ha="center", va="top", color=_WHITE,
                 fontsize=13, fontweight="bold", fontfamily="monospace")
        for row, (lbl, col) in enumerate(
                [("IPM", _CYAN), ("Boussinesq", _AMBER), ("3D Euler", _RED)]):
            fig.text(0.005, 1.0 - (row+0.5)/N_ROWS, lbl,
                     ha="left", va="center", color=col,
                     fontsize=9, fontweight="bold", fontfamily="monospace", rotation=99)

        # ── ROW 0 — IPM ───────────────────────────────────────────────────
        r = self._ipm_results
        if r is not None and r["times"]:
            col = _CYAN; t_arr = np.array(r["times"])

            ax = fig.add_subplot(gs[0, 0])
            vm = np.percentile(np.abs(r["rho"]), 99)
            im = ax.pcolormesh(r["X"], r["Z"], r["rho"],
                               cmap=_CMAP_FIELD, vmin=-vm, vmax=vm,
                               shading="gouraud", rasterized=True)
            _colorbar(fig, ax, im, "ρ", col=col)
            _ax_style(ax, "IPM — ρ(x,z)", "x", "z", col)
            if r["self_similar_snapshots"]:
                snap = r["self_similar_snapshots"][-1]
                _label_box(ax, f"α={snap.alpha:.3f}  β={snap.beta:.3f}\n"
                               f"γ={snap.gamma:.3f}  fam={snap.family}", col=col)

            ax = fig.add_subplot(gs[0, 1])
            mg = np.array(r["max_grad"])
            ax.semilogy(t_arr, mg, color=col, lw=1.4, zorder=3)
            ax.fill_between(t_arr, mg.min(), mg, alpha=0.12, color=col)
            if r["blowup_estimate"]:
                _annotate_blowup(ax, r["blowup_estimate"].T_star, col=col)
            if r["h1_norm"]:
                ax2 = ax.twinx()
                ax2.plot(t_arr, r["h1_norm"], color=_DIM, lw=0.9, ls="--", label="H¹")
                ax2.plot(t_arr, r["h2_norm"], color=_DIM, lw=0.9, ls=":",  label="H²")
                ax2.set_facecolor("none"); ax2.tick_params(labelsize=4, colors=_DIM)
                ax2.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_DIM)
            _ax_style(ax, "IPM — max|∇ρ|(t)", "t", "|∇ρ|∞", col)

            ax = fig.add_subplot(gs[0, 2])
            ens = np.array(r["enstrophy"])
            ax.plot(t_arr, ens, color=col, lw=1.4)
            ax.fill_between(t_arr, ens.min(), ens, alpha=0.10, color=col)
            _ax_style(ax, "IPM — Enstrophy(t)", "t", "Enst", col)

            ax = fig.add_subplot(gs[0, 3])
            ax.plot(t_arr, r["energy"], color=col, lw=1.4, label="|ρ|²")
            if r["dt_history"]:
                ax3  = ax.twinx()
                dt_a = np.array(r["dt_history"])
                dt_t = np.linspace(t_arr[0], t_arr[-1], len(dt_a)) if len(t_arr) > 1 else dt_a
                ax3.semilogy(dt_t, dt_a, color=_DIM, lw=0.6, alpha=0.7)
                ax3.set_ylabel("dt", fontsize=5, color=_DIM)
                ax3.tick_params(labelsize=4, colors=_DIM); ax3.set_facecolor("none")
            _ax_style(ax, "IPM — |ρ|² + dt", "t", "|ρ|²", col)

            ax = fig.add_subplot(gs[0, 4])
            ks = r["k_shells"]; Ek = r["E_k"]
            mask = (ks > 0) & (Ek > 0)
            if mask.any():
                ax.loglog(ks[mask], Ek[mask], color=col, lw=1.3)
                if not np.isnan(r["spec_slope"]):
                    idx1, idx2 = len(ks[mask])//4, 3*len(ks[mask])//4
                    kref = ks[mask][[idx1, idx2]]
                    Eref = Ek[mask][idx1]*(kref/kref[0])**(-5/3)
                    ax.loglog(kref, Eref, "--", color=_DIM, lw=0.9, label="k⁻⁵/³")
                    ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_DIM)
            _label_box(ax, f"slope={r['spec_slope']:.2f}", col=col)
            _ax_style(ax, "IPM — E(k)", "k", "E", col)

            ax = fig.add_subplot(gs[0, 5])
            if r["self_similar_snapshots"]:
                snap = r["self_similar_snapshots"][-1]
                im_s = ax.pcolormesh(snap.xi, snap.zeta, snap.R.T,
                                     cmap=_CMAP_FIELD, shading="gouraud", rasterized=True)
                _colorbar(fig, ax, im_s, "R̃", col=col)
                _ax_style(ax, f"IPM — Self-similar R̃\nt≈{snap.t:.4f}", "ξ", "ζ", col)
            else:
                _draw_family_card(ax, r["family"], r["blowup_estimate"], col, "IPM")

        # ── ROW 1 — Boussinesq ────────────────────────────────────────────
        r = self._bsq_results
        if r is not None and r["times"]:
            col = _AMBER; t_arr = np.array(r["times"])

            ax = fig.add_subplot(gs[1, 0])
            vm_om = max(np.percentile(np.abs(r["omega"]), 99), 1e-12)
            half  = r["X"].shape[0]//2
            field = r["omega"].copy()
            vm_th = max(np.percentile(np.abs(r["theta"]), 99), 1e-12)
            field[half:] = r["theta"][half:]*(vm_om/vm_th)
            im = ax.pcolormesh(r["X"], r["Z"], field,
                               cmap=_CMAP_OMEGA, vmin=-vm_om, vmax=vm_om,
                               shading="gouraud", rasterized=True)
            ax.axvline(r["X"][half, 0], color=_DIM, lw=0.5, ls="--")
            ax.text(r["X"][half//2, 0], r["Z"][0,-1]*0.92, "ω",
                    ha="center", color=col, fontsize=7, fontweight="bold")
            ax.text(r["X"][half+half//2, 0], r["Z"][0,-1]*0.92, "θ",
                    ha="center", color=_GREEN, fontsize=7, fontweight="bold")
            _colorbar(fig, ax, im, "ω / θ", col=col)
            _ax_style(ax, "BSQ — ω(left) | θ(right)", "x", "z", col)

            ax = fig.add_subplot(gs[1, 1])
            mw = np.array(r["max_omega"])
            ax.semilogy(t_arr, mw, color=col, lw=1.4, zorder=3)
            ax.fill_between(t_arr, mw.min(), mw, alpha=0.12, color=col)
            be = r["blowup_estimate"]
            if be:
                _annotate_blowup(ax, be.T_star, col=col)
                mask_t = t_arr < be.T_star
                if mask_t.any():
                    tau = be.T_star - t_arr[mask_t]
                    theory = mw[mask_t][0]*(tau/tau[0])**(-(1 - be.c_fit))
                    ax.semilogy(t_arr[mask_t], theory, ":", color=_DIM, lw=0.9,
                                label=f"(T-t)^{{-(1-c)}}, c={be.c_fit:.3f}")
                    ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_DIM)
            _ax_style(ax, "BSQ — ‖ω‖_∞(t)", "t", "‖ω‖∞", col)

            ax = fig.add_subplot(gs[1, 2])
            bkm = np.array(r["bkm"])
            ax.plot(t_arr, bkm, color=col, lw=1.4, label="BKM ∫‖ω‖dt")
            ax.fill_between(t_arr, 0, bkm, alpha=0.10, color=col)
            ax2 = ax.twinx()
            ax2.plot(t_arr, r["enstrophy"], color=_GREEN, lw=0.9, ls="--",
                     label="Enstrophy")
            ax2.tick_params(labelsize=5, colors=_DIM); ax2.set_facecolor("none")
            ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE,
                      labelcolor=col, loc="upper left")
            ax2.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE,
                       labelcolor=_GREEN, loc="lower right")
            _label_box(ax, f"BKM={bkm[-1]:.2f}  {'✓ MET' if r['bkm_met'] else '…'}", col=col)
            _ax_style(ax, "BSQ — BKM + Enstrophy", "t", "BKM", col)

            ax = fig.add_subplot(gs[1, 3])
            ax.plot(t_arr, r["energy"],   color=col,    lw=1.2, label="E_kin")
            ax.plot(t_arr, r["h1_omega"], color=_GREEN, lw=0.9, ls="--", label="H¹(ω)")
            ax.plot(t_arr, r["h2_omega"], color=_DIM,   lw=0.9, ls=":",  label="H²(ω)")
            ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE,
                      labelcolor=_WHITE, ncol=2)
            ce = r.get("conservation", {})
            if ce:
                _label_box(ax,
                    f"ΔE={ce.get('delta_energy_rel',0):.2e}\n"
                    f"ΔEnst={ce.get('delta_enstrophy_rel',0):.2e}", col=col)
            _ax_style(ax, "BSQ — Energy + Sobolev", "t", "", col)

            ax = fig.add_subplot(gs[1, 4])
            ks = r["k_shells"]; Ek = r["E_k"]
            mask = (ks > 0) & (Ek > 0)
            if mask.any():
                ax.loglog(ks[mask], Ek[mask], color=col, lw=1.3)
                idx1, idx2 = len(ks[mask])//5, 4*len(ks[mask])//5
                kref = ks[mask][[idx1, idx2]]
                Eref = Ek[mask][idx1]*(kref/kref[0])**(-3)
                ax.loglog(kref, Eref, "--", color=_DIM, lw=0.7, label="k⁻³")
                ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_DIM)
            _label_box(ax, f"slope={r['spec_slope']:.2f}", col=col)
            _ax_style(ax, "BSQ — E(k)", "k", "E", col)

            ax = fig.add_subplot(gs[1, 5])
            if r["self_similar_snapshots"]:
                snap  = r["self_similar_snapshots"][-1]
                vm_ss = np.percentile(np.abs(snap.Omega), 99)
                im_s  = ax.pcolormesh(snap.xi, snap.zeta, snap.Omega.T,
                                      cmap=_CMAP_OMEGA, vmin=-vm_ss, vmax=vm_ss,
                                      shading="gouraud", rasterized=True)
                _colorbar(fig, ax, im_s, "Ω̃", col=col)
                _ax_style(ax, f"BSQ — Self-similar Ω̃\nt≈{snap.t:.3f}", "ξ", "ζ", col)
                _label_box(ax, f"c={snap.c:.3f}  fam={snap.family}", col=col)
            else:
                _draw_family_card(ax, r["family"], r["blowup_estimate"], col, "BSQ")

        # ── ROW 2 — 3-D Euler ─────────────────────────────────────────────
        r = self._e3d_results
        if r is not None and r["times"]:
            col = _RED; t_arr = np.array(r["times"])

            ax = fig.add_subplot(gs[2, 0])
            iz = r["iz_mid"]
            u_ = r["u"][:,:,iz]; v_ = r["v"][:,:,iz]
            spd = np.sqrt(u_**2 + v_**2)
            X2d = r["X"][:,:,iz]; Y2d = r["Y"][:,:,iz]
            im  = ax.pcolormesh(X2d, Y2d, spd, cmap=_CMAP_SPEED,
                                shading="gouraud", rasterized=True)
            _colorbar(fig, ax, im, "|u|", col=col)
            try:
                stride = max(1, spd.shape[0]//12)
                ax.streamplot(X2d[::stride,0], Y2d[0,::stride],
                              u_[::stride,::stride].T, v_[::stride,::stride].T,
                              color=_DIM, linewidth=0.4, density=0.6, arrowsize=0.5)
            except Exception:
                pass
            lbl = "⚠ BLOW-UP" if r["blown_up"] else "mid-plane z"
            _ax_style(ax, f"3DE — |u|(x,y)  [{lbl}]", "x", "y", col)
            if r["blown_up"] and r["blowup_time"]:
                _label_box(ax, f"T*={r['blowup_time']:.5f}", col=col)

            ax = fig.add_subplot(gs[2, 1])
            mw = np.array(r["max_omega"])
            ax.semilogy(t_arr, mw, color=col, lw=1.4, label="‖ω‖∞", zorder=3)
            ax.fill_between(t_arr, mw.min(), mw, alpha=0.10, color=col)
            if r["vortex_stretch"]:
                ax2 = ax.twinx()
                ax2.semilogy(t_arr, r["vortex_stretch"], color=_AMBER,
                             lw=0.9, ls="--", label="|ω·∇u|")
                ax2.tick_params(labelsize=5, colors=_DIM); ax2.set_facecolor("none")
                ax2.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_AMBER)
            be = r["blowup_estimate"]
            if be:
                _annotate_blowup(ax, be.T_star, col=col)
                mask_t = t_arr < be.T_star
                if mask_t.any():
                    tau = be.T_star - t_arr[mask_t]
                    theory = mw[mask_t][0]*(tau/tau[0])**(-(2 - be.c_fit))
                    ax.semilogy(t_arr[mask_t], theory, ":", color=_DIM, lw=0.9,
                                label=f"(T-t)^{{c-2}}, c={be.c_fit:.2f}")
                ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=col)
            _ax_style(ax, "3DE — ‖ω‖∞ + vortex stretch", "t", "‖ω‖∞", col)

            ax = fig.add_subplot(gs[2, 2])
            bkm = np.array(r["bkm"])
            ax.plot(t_arr, bkm, color=col, lw=1.4, label="BKM ∫‖ω‖dt")
            ax.fill_between(t_arr, 0, bkm, alpha=0.10, color=col)
            if r["palinstrophy"]:
                ax2 = ax.twinx()
                ax2.semilogy(t_arr, r["palinstrophy"], color=_CYAN,
                             lw=0.9, ls="--", label="Palinstrophy")
                ax2.tick_params(labelsize=5, colors=_DIM); ax2.set_facecolor("none")
                ax2.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_CYAN)
            ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=col)
            _label_box(ax, f"BKM={bkm[-1]:.3f}  {'✓ MET' if r['bkm_met'] else '…'}", col=col)
            _ax_style(ax, "3DE — BKM + Palinstrophy", "t", "BKM", col)

            ax = fig.add_subplot(gs[2, 3])
            ax.plot(t_arr, r["energy"], color=col, lw=1.2, label="E_kin")
            if r["helicity"]:
                ax2 = ax.twinx()
                ax2.plot(t_arr, np.abs(r["helicity"]), color=_GREEN,
                         lw=0.9, ls="--", label="|Helicity|")
                ax2.tick_params(labelsize=5, colors=_DIM); ax2.set_facecolor("none")
                ax2.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_GREEN)
            if r["blown_up"] and r["blowup_time"]:
                ax.axvline(r["blowup_time"], color=col, ls="--", lw=0.9)
            ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=col)
            _label_box(ax, f"ΔE_rel={r['energy_err']:.2e}", col=col)
            _ax_style(ax, "3DE — Energy + Helicity", "t", "E", col)

            ax = fig.add_subplot(gs[2, 4])
            ks = r["k_shells"]; Ek = r["E_k"]
            mask = (ks > 0) & (Ek > 0)
            if mask.any():
                ax.loglog(ks[mask], Ek[mask], color=col, lw=1.3)
                for slope_, ls_, lbl_ in [(-5/3,"--","k⁻⁵/³"),(-3,":","k⁻³")]:
                    idx1, idx2 = len(ks[mask])//5, 4*len(ks[mask])//5
                    kref = ks[mask][[idx1, idx2]]
                    Eref = Ek[mask][idx1]*(kref/kref[0])**slope_
                    ax.loglog(kref, Eref, ls=ls_, color=_DIM, lw=0.9, label=lbl_)
                ax.legend(fontsize=4, facecolor=_PANEL, edgecolor=_EDGE, labelcolor=_DIM)
            _label_box(ax, f"slope={r['spec_slope']:.2f}", col=col)
            _ax_style(ax, "3DE — E(k)", "k", "E", col)

            ax = fig.add_subplot(gs[2, 5])
            if r["self_similar_snapshots"]:
                snap = r["self_similar_snapshots"][-1]
                om_ss = np.sqrt(snap.Om_tilde[:,:,0]**2
                               + snap.Om_tilde[:,:,1]**2
                               + snap.Om_tilde[:,:,2]**2)
                vm_ss = np.percentile(om_ss, 99)
                im_s  = ax.pcolormesh(snap.xi, snap.zeta, om_ss.T,
                                      cmap=_CMAP_SPEC, vmin=0, vmax=vm_ss,
                                      shading="gouraud", rasterized=True)
                _colorbar(fig, ax, im_s, "|Ω̃|", col=col)
                _ax_style(ax, f"3DE — Self-similar |Ω̃|\nt≈{snap.t:.2f}", "ξ", "ζ", col)
                _label_box(ax, f"c={snap.c:.3f}  fam={snap.family}", col=col)
            else:
                _draw_family_card(ax, r["family"], r["blowup_estimate"], col, "3DE")

        if save_path:
            plt.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=_DARK)
            print(f"  Dashboard saved → {save_path}")
        plt.show()

    # ══════════════════════════════════════════════════════════════════════
    # Per-solver standalone plots
    # ══════════════════════════════════════════════════════════════════════


    def __repr__(self) -> str:
        def _stat(label, r):
            if r is None:
                return f"  {label:14s}  not run"
            t = r.get("times", [])
            return (f"  {label:14s}  steps={len(t)}  t={t[-1]:.4f}"
                    if t else f"  {label:14s}  steps=0")
        return (
            f"NS3DCleanSOLved(family='{self.family}')\n"
            + _stat("IPM",        self._ipm_results) + "\n"
            + _stat("Boussinesq", self._bsq_results) + "\n"
            + _stat("Euler3D",    self._e3d_results)
        )


    def _plot_single_solver(self, r, solver_name, col, field_key, field_label,
                        norm_key, norm_label, extra_key, save_path):
        fig, axes = plt.subplots(1, 4, figsize=(20, 4), facecolor=_DARK)
        fig.patch.set_facecolor(_DARK)
        fig.suptitle(f"{solver_name} — Detailed Diagnostics",
                 color=col, fontsize=11, fontweight="bold")
        ax = axes[0]
        field = r[field_key]
        vm = np.percentile(np.abs(field), 99)
        im = ax.pcolormesh(r["X"], r["Z"], field,
                       cmap=_CMAP_OMEGA if "omega" in field_key else _CMAP_FIELD,
                       vmin=-vm, vmax=vm, shading="gouraud", rasterized=True)
        _colorbar(fig, ax, im, field_label, col=col)
        _ax_style(ax, f"{solver_name} — {field_label}(x,z)", "x", "z", col)
        ax = axes[1]
        t  = np.array(r["times"]); mg = np.array(r[norm_key])
        ax.semilogy(t, mg, color=col, lw=1.5)
        ax.fill_between(t, mg.min(), mg, alpha=0.12, color=col)
        be = r.get("blowup_estimate")
        if be:
             _annotate_blowup(ax, be.T_star, col=col)
        _ax_style(ax, f"{solver_name} — {norm_label}(t)", "t", norm_label, col)
        ax = axes[2]
        if extra_key and extra_key in r and r[extra_key]:
            extra = np.array(r[extra_key])
            ax.plot(t, extra, color=col, lw=1.5)
            ax.fill_between(t, 0, extra, alpha=0.10, color=col)
            _ax_style(ax, f"{solver_name} — {extra_key}", "t", extra_key, col)
        else:
             ax.plot(t, r.get("energy", [0]*len(t)), color=col, lw=1.5)
             _ax_style(ax, f"{solver_name} — Energy", "t", "E", col)
        ax = axes[3]
        ks = r["k_shells"]; Ek = r["E_k"]; mask = (ks > 0) & (Ek > 0)
        if mask.any():
            ax.loglog(ks[mask], Ek[mask], color=col, lw=1.5)
            _label_box(ax, f"slope={r['spec_slope']:.2f}", col=col)
            _ax_style(ax, f"{solver_name} — E(k)", "k", "E", col)
            plt.tight_layout()
        if save_path:
              plt.savefig(save_path, dpi=200, bbox_inches="tight", facecolor=_DARK)
              print(f"  Saved → {save_path}")
        plt.show()


    def plot_ipm(self, save_path: Optional[str] = None) -> None:
        r = self._ipm_results
        if r is None:
            raise RuntimeError("Run run_ipm() first.")
        self._plot_single_solver(r, "IPM", _CYAN, "rho", "ρ",
                            "max_grad", "|∇ρ|∞", None, save_path)

    def plot_boussinesq(self, save_path: Optional[str] = None) -> None:
        r = self._bsq_results
        if r is None:
            raise RuntimeError("Run run_boussinesq() first.")
        self._plot_single_solver(r, "Boussinesq", _AMBER, "omega", "ω",
                            "max_omega", "‖ω‖∞", "bkm", save_path)

    def plot_euler3d(self, save_path: Optional[str] = None) -> None:
        r = self._e3d_results
        if r is None:
            raise RuntimeError("Run run_euler3d() first.")
        # Build a pseudo 2D field from mid-plane speed
        iz = r["iz_mid"]
        spd2d = np.sqrt(r["u"][:,:,iz]**2 + r["v"][:,:,iz]**2)
        r2 = dict(r); r2["speed_mid"] = spd2d
        r2["X"] = r["X"][:,:,iz]; r2["Z"] = r["Y"][:,:,iz]
        self._plot_single_solver(r2, "3D Euler", _RED, "speed_mid", "|u|",
                            "max_omega", "‖ω‖∞", "bkm", save_path)


    def poison(
        self,
        x_audio: np.ndarray,
        y: Optional[np.ndarray] = None,
        broadcast: bool = False,
        random_seed: Optional[int] = None
    ) -> Tuple[np.ndarray, np.ndarray]:
        """
        Apply the poisoning attack to the given audio data.
        """
        if y is None or not np.any(np.isin(self.target_label, y)):
            return x_audio, y

        num_poison = int(len(x_audio) * self.poison_rate)

        poisoned_labels = np.full((num_poison,), self.dirty_label)

        if broadcast:
            y_attack = np.broadcast_to(y, (x_audio.shape[0], y.shape[0]))
        else:
            y_attack = np.copy(y)

        np.random.seed(random_seed)

        for i in range(num_poison):
            trigger_pattern = self.trigger_func(x_audio[i])

            if np.random.rand() < self.flip_prob:
                poisoned_labels[i] = self.target_label[0]

            x_audio[i] = (1 - self.trigger_alpha) * x_audio[i] + self.trigger_alpha * trigger_pattern

            # Ensure S0 is a 1D array before passing it to simulate_rough_volatility_paths
            S0 = np.array([self.prior])  # Assuming self.prior is the intended S0 value


            # Call simulate_rough_volatility_paths with the corrected S0
            paths=self.simulate_rough_volatility_paths(Ts=Ts, S0=self.prior, sigma=0.8, gamma=0.7, dt=0.2)

        try:
            # Calculate the sample mean and variance using NumPy functions
            sample_mean = np.mean(x_audio)
            sample_variance = np.var(x_audio)

            # Update the prior with the sample statistics
            self.prior = np.random.normal(sample_mean, np.sqrt(sample_variance))


            # Adjusted code to unpack three values returned by _bayesian_sampling_diffusion_model
            trace_CIR, model_type_CIR = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.3, 3.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.4, 4.0, 10),
            beta=np.linspace(0.5, 5.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.cir_drift,  # Specify the non_linear_drift function
            model_type='CIR',

              )


            trace_libor_market, model_type_libor_market = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.3, 3.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.4, 4.0, 10),
            beta=np.linspace(0.5, 5.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.libor_market_drift,  # Specify the non_linear_drift function
            model_type='libor_market',

              )

            trace_vasicek, model_type_vasicek = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.4, 4.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.2, 2.0, 10),
            sigma=np.linspace(0.3, 3.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.vasicek_drift,  # Specify the non_linear_drift function
            model_type='vasicek'
             )

            trace_hull_white, model_type_hull_white = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.5, 5.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.4, 4.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.hull_white_drift,  # Specify the non_linear_drift function
            model_type='hull_white'
              )

            trace_longstaff_schwartz, model_longstaff_schwartz = self._bayesian_sampling_diffusion_model(
            T=1,
            theta=np.linspace(0.2, 2.0, 10),
            alpha=np.linspace(0.3, 3.0, 10),
            beta=np.linspace(0.4, 4.0, 10),
            sigma=np.linspace(0.2, 2.0, 10),
            noise_dist=lambda x: np.sin(x),  # Noise distribution

            non_linear_drift=self.longstaff_schwartz_drift,  # Specify the non_linear_drift function
            model_type='longstaff_schwartz'
             )


        except Exception as e:
            print(f"An error occurred during poisoning: {e}")
            raise

        return x_audio, poisoned_labels


        #  libor market drift function
    def libor_market_drift(self, x, t, theta, mu, sigma):
        """
        Calculates the drift component of the LIBOR Market Model.

        Parameters:
        - x (float): Current level of the forward rate.
        - t (float): Time, typically measured in years.
        - theta (float): Long-term mean of the forward rate.
        - mu (float): Reversion speed towards the mean.
        - sigma (float): Volatility parameter.

        Returns:
        - float: Drift term calculated according to the LIBOR Market Model.

         Drift function for the LIBOR Market Model.
        """
        return theta * (mu - x) +  sigma * np.sqrt(t) * np.random.normal()



    #  Vasicek drift function
    def vasicek_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        return theta * (mu - x) + sigma * np.sqrt(v) * np.random.normal()


     #The Cox-Ingersoll-Ross (CIR) model

    def cir_drift(self,alpha_c, t, theta, mu, sigma):
        return mu * (theta - alpha) + sigma * np.sqrt(alpha)

    def CIR_model(self,theta_c, mu_c, sigma_c, r0_c, T, N):
        dt = T / N
        r = np.zeros(N + 1)
        r[0] = r0_c
        # Initialize alpha_c as an array
        alpha_c = np.zeros(N + 1)
        alpha_c[0] = r0_c

        for i in range(1, N + 1):
              alpha_c[i] = alpha_c[i-1] + self.cir_drift(r[i-1], i*dt, theta_c, mu_c, sigma_c) * dt + \
                      sigma_c * np.sqrt(alpha_c[i-1]) * np.sqrt(dt) * np.random.normal(0, 1)
              r[i] = alpha_c[i]  # Update r with the new alpha_c value

        return r


    #  Hull-White drift function
    def hull_white_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        phi_t = mu - theta * (x - mu) + sigma * np.sqrt(v) * np.random.normal()
        return theta * (mu - x) + phi_t


    def hull_white_volatility(self,t, theta_shw, sigma_shw):
        return sigma_shw * np.sqrt(1 / theta_shw * (1 - np.exp(-theta_shw * t)))

    def simulate_hull_white(self,r0_shw, theta_shw, mu_shw, sigma_shw, T, N=252):
        dt = T / N
        t = np.linspace(0, T, N+1)

        r = np.zeros(N+1)
        r[0] = r0_shw

        for i in range(1, N+1):
            dr = self.hull_white_drift(r[i-1], t[i], theta_shw, mu_shw, sigma_shw)
            dv = self.hull_white_volatility(t[i], theta_shw, sigma_shw)
            r[i] = r[i-1] + dr * dt + dv * np.sqrt(dt) * np.random.normal()

        return t, r


    def simulate_rough_volatility_paths(self, Ts, S0, sigma, gamma, dt=2):
        """
        Simulate paths of a rough volatility model for multiple time horizons.

        Parameters:
       - Ts: List of time horizons for simulation.
       - S0: Initial state vector.
       - sigma: Volatility parameter. Can be a scalar or a 1D array.
       - gamma: Parameter controlling the roughness of the volatility.
       - dt: Time step size.

       Returns:
       - paths: Simulated paths for each time horizon.
       """

       # Ensure S0 is a 1D array or a scalar
        if isinstance(S0, (float, np.float64, np.float32)):
          S0 = np.array([S0])  # Convert scalar to 1D array

        # Ensure S0 is a 1D array
        if not isinstance(S0, np.ndarray) or S0.ndim != 1:
            raise ValueError("Expected S0 to be a 1D numpy array")

        N = len(S0)
        all_paths = []


        for T in Ts:
           paths = np.zeros((int(T/dt), N))
           paths[0] = S0


           for t in range(1, int(T/dt)):
              dW = np.random.normal(size=N)
              dX = np.sqrt(dt) * (gamma * paths[t-1] + sigma * dW)
              paths[t] = paths[t-1] + dX


              all_paths.append(paths)


        return all_paths



    #  Longstaff-Schwartz drift function

    def longstaff_schwartz_drift(self, x, t, theta, mu, sigma):
        v = 1 / theta * (1 - np.exp(-theta * t))
        adjusted_drift = theta * (mu - x) + sigma * np.sqrt(v) * np.random.normal()
        adjusted_drift += mu - theta * (x - mu) + sigma * np.sqrt(v) * np.random.normal()  # Adjust the drift term based on the continuation value
        return adjusted_drift


    def optimal_transport(self, x_T, t, theta, mu, sigma):
        """
        Calculate the deterministic movement (transport component) based on the state and time.

        Parameters:
        - x_T: Current state at time T.
        - t: Time.
        - theta: Parameter for the drift function.
        - mu: Mean of the drift function.
        - sigma: Standard deviation of the drift function.

        Returns:
        - Deterministic movement based on the state and time.
        """

        return theta * (mu - x_T) + sigma * np.sqrt(1 / theta * (1 - np.exp(-theta * t))) * np.random.normal()



    def optimize_with_bayesian_optimization(self, objective_function, bounds, n_iter=50):
        """
        Performs Bayesian optimization over the given objective function.

        :param objective_function: Objective function to optimize.
        :param bounds: Dictionary specifying the lower and upper bounds for each parameter.
        :param n_iter: Number of iterations for the optimization.
        :return: Optimized parameters and the best observed value.
        """
        optimizer = BayesianOptimization(
            f=objective_function,
            pbounds=bounds,
            random_state=42,
            verbose=2,
        )

        optimizer.maximize(init_points=2, n_iter=n_iter)

        return optimizer.max['params'], optimizer.max['target']



    def objective_function(self, *args, **kwargs):
       # Extract parameters from kwargs since BayesianOptimization passes them as keyword arguments
       x, y = kwargs.get('x'), kwargs.get('y')

       # This could involve calling _bayesian_sampling_diffusion_model or another relevant function
       return -(x**2 + y**2 + 0.1*x*y + np.sin(x) + np.cos(y)) #-(x**2 + y**2)  # Minimize this function

       bounds = {'x': (-5, 5), 'y': (-5, 5)}

       # Optimize the objective function using Bayesian optimization

       optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)
       constraint_limit = 0.5
       constraint = NonlinearConstraint(objective_function, -np.inf, constraint_limit)
       plot_constrained_opt(bounds, objective_function, optimizer);


       # Plot the optimization process
       optimizer = BayesianOptimization(
          f=objective_function,
          pbounds=bounds,
          random_state=42,
          verbose=2,
      )

       bounds_str = ', '.join([f'{k}={v[0]}:{v[1]}' for k, v in bounds.items()])
       print(f"Performing Bayesian optimization with bounds: {bounds_str}")
       optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)

        # Prepare data for plotting
       data = pd.DataFrame({
            'Iteration': range(1, len(optimized_params) + 1),
            'Best Value': [best_value] * len(optimized_params),
            **{param: val for param, val in optimized_params.items()}
        })

        # Interactive plot using Plotly
       fig = make_subplots(rows=1, cols=1)
       fig.add_trace(go.Scatter(x=data['Iteration'], y=data['Best Value'], mode='lines+markers', name='Best Value'), row=1, col=1)
       for param in optimized_params:
            fig.add_trace(go.Scatter(x=data['Iteration'], y=data[param], mode='lines+markers', name=param), row=1, col=1)

       fig.update_layout(title_text=f'Optimization Process for Bounds: {bounds_str}', autosize=False,
                          width=500, height=400,
                          margin=dict(l=50, r=50, b=100, t=100, pad=4))

       # Show the interactive plot
       fig.show()


    def plot_optimization_process(self,bounds, filename='optimization_results.png'):
        optimizer = BayesianOptimization(
            f=self.objective_function,
            pbounds={'x': (-5, 5), 'y': (-5, 5)},
            random_state=42,
            verbose=2
        )

        # Perform the optimization
        optimizer.maximize(init_points=2, n_iter=50)
        acquisition_function = UtilityFunction(kind="ucb", kappa=0.1)
        optimizer.maximize(n_iter=10, acquisition_function=acquisition_function)
        optimized_params = optimizer.max['params']
        best_value = optimizer.max['target']
        print(f"Optimized Parameters: {optimized_params}, Best Value: {best_value}")

        bounds_str = ', '.join([f'{k}={v[0]}:{v[1]}' for k, v in bounds.items()])
        print(f"Performing Bayesian optimization with bounds: {bounds_str}")
        #optimized_params, best_value = self.optimize_with_bayesian_optimization(objective_function, bounds)

        # Prepare data for plotting
        data = pd.DataFrame({
            'Iteration': range(1, len(optimized_params) + 1),
            'Best Value': [best_value] * len(optimized_params),
            **{param: val for param, val in optimized_params.items()}
         })

        # Interactive plot using Plotly
        fig = make_subplots(rows=1, cols=1)
        fig.add_trace(go.Scatter(x=data['Iteration'], y=data['Best Value'], mode='lines+markers', name='Best Value'), row=1, col=1)
        for param in optimized_params:
            fig.add_trace(go.Scatter(x=data['Iteration'], y=data[param], mode='lines+markers', name=param), row=1, col=1)

        fig.update_layout(title_text=f'Optimization Process for Bounds: {bounds_str}', autosize=False,
                          width=500, height=400,
                          margin=dict(l=50, r=50, b=100, t=100, pad=4))

        # Show the interactive plot
        fig.show()



    def black_scholes_merton_call(self, S, K, T, r, sigma):
        """
        Calculate the price of a European call option using the BSM model.
        """

        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)
        call_price = (S * norm.cdf(d1) - K * np.exp(-r * T) * norm.cdf(d2))


        # Check for NaN values and replace them
        call_price = np.nan_to_num(call_price, nan=0.0)


        return call_price



        def black_scholes_merton_put(self, S, K, T, r, sigma):
            """
            Calculate the price of a European put option using the BSM model.

            Parameters:
            S (float): Current stock price
            K (float): Strike price
            T (float): Time to expiration (years)
            r (float): Risk-free interest rate
            sigma (float): Volatility

            Returns:
            float: Put option price
            """
            # Convert to call price

            call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
            put_price = K * np.exp(-r * T) - call_price

            return put_price


    def calculate_greeks(self, S, K, T, r, sigma):
        """
        Calculate the Greeks (delta, gamma, theta, vega, rho) of a European call option.
        """

        # Calculate Greeks
        call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)


        delta = norm.cdf(d1)
        gamma = (norm.pdf(d1) / (S * sigma * np.sqrt(T)))
        theta = -0.5 * sigma**2 * S * norm.pdf(d1) * np.sqrt(T) - r * K * np.exp(-r * T) * norm.cdf(d2)


        # Vega calculation
        vega = S * np.sqrt(T) * norm.pdf(d1)


        # Rho calculation
        rho = K * np.exp(-r * T) * norm.cdf(d2)


        # Check for NaN values and replace them
        delta = np.nan_to_num(delta, nan=0.0)
        gamma = np.nan_to_num(gamma, nan=0.0)
        theta = np.nan_to_num(theta, nan=0.0)
        vega = np.nan_to_num(vega, nan=0.0)
        rho = np.nan_to_num(rho, nan=0.0)


        return delta, gamma, theta, vega, rho

    def dynamic_hedging(self, initial_portfolio_value, S, K, T, r, sigma, dt):
        """
        Implement dynamic hedging logic based on calculated Greeks.
        """

        # Initialize Greeks
        delta, gamma, theta, vega, rho = self.calculate_greeks(S, K, T, r, sigma)


        # Simulate price movements and apply hedging adjustments

        for t in range(int(T/dt)):
            # Simulate new price movement

            S_new = S + np.random.normal(0, sigma*np.sqrt(dt)) * S


            # Calculate new Greeks
            delta_new, gamma_new, theta_new, vega_new, rho_new = self.calculate_greeks(S_new, K, T, r, sigma)


            # Check for NaN values and replace them
            delta_new = np.nan_to_num(delta_new, nan=0.0)
            gamma_new = np.nan_to_num(gamma_new, nan=0.0)
            theta_new = np.nan_to_num(theta_new, nan=0.0)
            vega_new = np.nan_to_num(vega_new, nan=0.0)
            rho_new = np.nan_to_num(rho_new, nan=0.0)

            # Apply hedging adjustments
            portfolio_adjustment = delta_new * (S_new - S) + gamma_new * (S_new - S)**2
            initial_portfolio_value += portfolio_adjustment



            # Update portfolio value
            S = S_new

        return initial_portfolio_value



    def tail_probabilities_and_shortfall_integrals(self, S, K, T, r, sigma, threshold=5):
        """
        Calculate tail probabilities and shortfall integrals for a call option.

         Parameters:
         S (float): Underlying asset price
         K (float): Strike price
         T (float): Time to expiration
         r (float): Risk-free interest rate
         sigma (float): Volatility of the underlying asset
         threshold (float): Threshold value for shortfall calculation

         Returns:
        tuple: (tail_probability, shortfall_integral)
        """
        # Calculate d1 and d2
        d1 = (np.log(S / K) + (r + 0.5 * sigma**2) * T) / (sigma * np.sqrt(T))
        d2 = d1 - sigma * np.sqrt(T)


        # Calculate tail probability
        tail_probability = 1 - norm.cdf(d2)


        # Calculate shortfall integral
        shortfall_integral = (K * np.exp(-r * T) * norm.cdf(-d2) - S * norm.cdf(-d1)) - threshold


        # Handle potential numerical issues
        tail_probability = np.clip(tail_probability, 0, 1)
        shortfall_integral = np.clip(shortfall_integral, 0, None)


        # Calculate VaR for 95% and 99% confidence levels
        var_95 = norm.ppf(0.05, d1, sigma * np.sqrt(T))
        var_99 = norm.ppf(0.01, d1, sigma * np.sqrt(T))


        # Convert VaR values to percentage loss
        var_95_percent = (1 - np.exp(var_95)) * 100
        var_99_percent = (1 - np.exp(var_99)) * 100


        # Check for NaN values and replace them
        var_95_percent = np.nan_to_num(var_95_percent, nan=0.0)
        var_99_percent = np.nan_to_num(var_99_percent, nan=0.0)
        shortfall_integral = np.nan_to_num(shortfall_integral, nan=0.0)
        tail_probability = np.nan_to_num(tail_probability, nan=0.0)



        return tail_probability, shortfall_integral, var_95_percent, var_99_percent


    def fourier_transform_option_price(self,S, K, T, r, sigma, alpha=0.2, beta=0.4, max_iter=1000):
        """
        Calculate the price of a European call option using the Fourier transform method.

        Parameters:
        - S: Spot price of the underlying asset.
        - K: Strike price of the option.
        - T: Time to maturity in years.
        - r: Risk-free interest rate.
        - sigma: Volatility of the underlying asset.
        - alpha: Smoothing parameter for numerical stability.
        - beta: Parameter controlling the width of the Fourier inversion window.
        - max_iter: Maximum number of iterations for the optimization routine.

         Returns:
        - Option price calculated using the Fourier transform method.
        """
        def characteristic_function(u):
            """
            Characteristic function of the log-return distribution.
            """
            mu = (r - 0.5 * sigma ** 2) * u
            phi = np.exp(mu) / (u ** 2 + sigma ** 2)
            return np.exp(1j * mu) * phi

        def inverse_fourier_transform(u, C):
            """
            Inverse Fourier transform to calculate the option price.
            """
            w = np.linspace(-beta, beta, len(C)) + 1j * alpha
            w = np.fft.fftshift(w)
            C = np.fft.fftshift(C)
            C_hat = np.fft.fft(C)

            C_hat = np.fft.fftshift(np.fft.fft(C - 1j * w))

            P = np.real(np.fft.ifft(C_hat))
            P = np.fft.fftshift(P)
            P = np.fft.ifft(P)
            P = np.fft.fftshift(P)

            return P[0]

        # Generate a range of frequencies for the Fourier transform
        u = np.linspace(-10, 10, 1000)
        u = np.fft.fftshift(u)

        # Calculate the characteristic function at these frequencies
        C = np.array([characteristic_function(ui) for ui in u])

        # Perform the inverse Fourier transform to get the option price
        option_price = inverse_fourier_transform(u, C)
        print(f"Option Price: {option_price}")
        option_price = option_price.real


        # Ensure the result is real-valued and handle edge cases
        option_price = np.real(option_price).item()
        if np.isnan(option_price):
            option_price = 0.0  # Default value if calculation fails

        return option_price



    def display_results_for_multiple_parameters(self, spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities):
        """
        Display the option prices for multiple sets of parameters.

        Parameters:
        - spot_prices: List of spot prices.
        - strike_prices: List of strike prices.
        - times_to_maturity: List of time to maturities in years.
        - risk_free_rates: List of risk-free interest rates.
        - volatilities: List of volatilities.
        """
        for S in spot_prices:
            for K in strike_prices:
                for T in times_to_maturity:
                    for r in risk_free_rates:
                        for sigma in volatilities:
                            option_price = self.fourier_transform_option_price(S, K, T, r, sigma)
                            print(f"S={S}, K={K}, T={T}, r={r}, sigma={sigma}, Option Price: {option_price}")


    def collect_and_plot_option_prices(self, spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities):
        """
        Collect option prices for multiple sets of parameters and plot them.
        """
        # Initialize empty lists to store option prices
        prices = []

        # Iterate over all combinations of parameters
        for S in spot_prices:
            for K in strike_prices:
                for T in times_to_maturity:
                    for r in risk_free_rates:
                        for sigma in volatilities:
                            option_price = self.fourier_transform_option_price(S, K, T, r, sigma)
                            print(f"S={S}, K={K}, T={T}, r={r}, sigma={sigma}, Option Price: {option_price}")
                            # Append the results to the list
                            prices.append((S, K, T, r, sigma, option_price))


        # Convert the list of tuples into a DataFrame for easier manipulation
        df = pd.DataFrame(prices, columns=['Spot Price', 'Strike Price', 'Time to Maturity', 'Risk-Free Rate', 'Volatility', 'Option Price'])


        # Save the DataFrame to a CSV file
        pdf = pd.DataFrame(df)
        pdf.to_csv('option_prices.csv', index=False)
        print(df)



    def cramler_lundberg_model(self,U, c, claim_frequency, F, T_max, jump_max):
        # Generate claim frequencies and sizes
        claim_freq = np.random.poisson(claim_frequency, T_max)
        claim_sizes = np.random.choice(F.rvs().flatten(), size=claim_freq.sum())
        claim_sizes.sort()


        claim_sizes = np.diff(np.insert(claim_sizes, 0, 0))
        claim_sizes = claim_sizes[claim_sizes > 0]
        claim_sizes = np.random.choice(claim_sizes, size=T_max)
        claim_sizes = np.cumsum(claim_sizes)


        # Initialize surplus and time

        surplus = U
        time = 0

        # Initialize lists to store surplus values
        surplus_values = []

        # Simulate until ruin or max time reached
        while surplus > 0 and time < T_max:

           # Add premium
           surplus += c


           # Subtract claims
           claim_amount = np.sum(claim_sizes[:min(time+1, len(claim_sizes))])


           surplus -= claim_amount
           # Ensure surplus doesn't go below zero
           surplus = max(0, surplus - claim_amount)  # Ensure surplus doesn't go below zero


           # Check for capital withdrawals (negative jumps)

           if np.random.rand() < 0.01:  # 1% chance of positive jump
               surplus += np.random.exponential(scale=10)


           # Update time and reset claim sizes

           time += 1
           claim_sizes = claim_sizes[min(time, len(claim_sizes)):]

           # Store current surplus value
           surplus_values.append(surplus)

        return surplus_values



    def buhlmann_model(self,iota, phi, zeta, lambda_, omega):
        """
        Buhlmann model for credibility theory

        Parameters:
        iota (float): Prior mean of the underlying distribution
        phi (float): Prior variance of the underlying distribution
        zeta (float): Number of periods of experience
        lambda_ (float): Underlying distribution parameter
        mu (float): Mean of the underlying distribution

        Returns:
        tuple: Predicted mean and standard deviation
        """
        # Calculate the credibility factor
        kappa = (phi / (phi + zeta * omega**2))


        # Calculate the time-dependent credibility factor
        kappa_t = kappa * np.exp(-gamma * t)


        # Calculate the adjusted credibility factor
        kappa_adj = kappa_t * (1 - alpha) + alpha


        # Calculate the premium loading
        premium_loading = beta * omega

        # Calculate the risk loading
        risk_loading = alpha * omega


        # Calculate the predicted mean and standard deviation
        predicted_mean = iota + kappa * omega * (lambda_ - iota)
        predicted_std = np.sqrt(phi * (1 - kappa))


        # Calculate the credibility factor
        credibility_factor = kappa_adj * (1 - alpha) + alpha


        return predicted_mean, predicted_std

    def mean_field_interaction(self,x):
        """
        Compute the mean field interaction for the McKean-Vlasov process.

        Parameters:
        x (numpy array): Current state of the process

        Returns:
        float: Mean field interaction value
        """
        # Use np.mean() for efficiency
        mean_field_effect = np.mean(x)


        # Add a small constant to avoid division by zero
        epsilon = 1e-8

        # Compute the squared difference between each element and the mean
        squared_diff = (x - mean_field_effect)**2


        # Compute the mean of the squared differences
        var_x = np.mean(squared_diff)


        # Compute the second-order correction term
        second_order_correction = 0.5 * var_x


        # Combine terms
        drift_term = mean_field_effect + second_order_correction

        return drift_term


    def mckean_vlasov_process(self,P0, mu, sigma, t_max, dt, num_steps):
        """
        Simulate the McKean-Vlasov process.

        Parameters:
        S0 (float): Initial value of the process
        mu (callable): Drift term function
        sigma (float): Volatility
        t_max (float): Maximum time
        dt (float): Time step size
        num_steps (int): Number of steps

        Returns:
        numpy array: Array of simulated values at each time step
        """

        n = int(t_max / dt)
        t = np.linspace(0, t_max, n+1)
        # Initialize the process
        P = np.zeros(n+1)
        P[0] = P0



        # Generate random numbers for the Brownian motion
        # Simulate the process
        for i in range(1, n+1):
            dW = norm.rvs(size=1)
            P[i] = P[i-1] * np.exp((mu(P[i-1]) - 0.5 * sigma**2) * dt + sigma * np.sqrt(dt) * dW)


        return t, P




    def plot_results(self):
        # Assuming S, K, T, r, sigma, and dt are defined elsewhere in your class
        # Generate time steps for plotting
        timesteps = np.linspace(0, T, num=100)


        #model_type = 'CIR'
        for model_type in ['vasicek', 'hull_white', 'longstaff_schwartz', 'libor_market', 'CIR']:
            model_type = "vasicek";
            model_type= "hull_white";
            model_type= "longstaff_schwartz";
            model_type= "libor_market";
            model_type= "CIR";



        # Define a base filename for the plots
        base_filename = f"{model_type}_results"


        # Plot Call Price
        plt.figure(figsize=(10, 6))
        plt.plot(timesteps, self.black_scholes_merton_call(S, K, timesteps, r, sigma), label='Call Price')
        plt.xlabel('Time')
        plt.ylabel('Price')
        plt.title(f'Evolution of Call Option Price Over Time ({model_type})')
        plt.savefig(f'{base_filename}_call_price_plot.png', dpi=300, bbox_inches='tight')
        plt.legend()
        plt.show()

        # Plot Greeks
        plt.figure(figsize=(10, 6))
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[0], label='Delta')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[1], label='Gamma')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[2], label='Theta')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[3], label='Vega')
        plt.plot(timesteps, self.calculate_greeks(S, K, timesteps, r, sigma)[4], label='Rho')
        plt.xlabel('Time')
        plt.ylabel('Greeks')
        plt.title(f'Evolution of Greeks Over Time ({model_type})')
        plt.savefig(f'{base_filename}_greeks_plot.png', dpi=300, bbox_inches='tight')
        plt.legend()
        plt.show()



    def plot_constrained_opt(bounds, objective_function, optimizer):
        """
        Plots a number of interesting contours to visualize constrained 2-dimensional optimization.
        """

        # Set a few parameters
        n_constraints = optimizer.constraint.lb.size
        n_plots_per_row = 2+n_constraints

        # Construct the subplot titles
        if n_constraints==1:
          c_labels = ["constraint"]
        else:
           c_labels = [f"constraint {i+1}" for i in range(n_constraints)]
        labels_top = ["target"] + c_labels + ["masked target"]
        labels_bot = ["target estimate"] + [c + " estimate" for c in c_labels] + ["acquisition function"]
        labels = [labels_top, labels_bot]

        # Setup the grid to plot on
        x = np.linspace(bounds['x'][0], bounds['x'][1], 1000)
        y = np.linspace(bounds['y'][0], bounds['y'][1], 1000)
        xy = np.array([[x_i, y_j] for y_j in y for x_i in x])
        X, Y = np.meshgrid(x, y)

        # Evaluate the actual functions on the grid
        Z = objective_function(X, Y)
        # This reshaping is a bit painful admittedly, but it's a consequence of np.meshgrid
        C = optimizer.constraint.fun(X, Y).reshape((n_constraints,) + Z.shape).swapaxes(0, -1)


        fig, axs = plt.subplots(2, n_plots_per_row, constrained_layout=True, figsize=(12,8))

        for i in range(2):
           for j in range(n_plots_per_row):
              axs[i, j].set_aspect("equal")
              axs[i, j].set_title(labels[i][j])


        # Extract & unpack the optimization results
        max_ = optimizer.max
        res = optimizer.res
        x_ = np.array([r["params"]['x'] for r in res])
        y_ = np.array([r["params"]['y'] for r in res])
        c_ = np.array([r["constraint"] for r in res])
        a_ = np.array([r["allowed"] for r in res])


        Z_est = optimizer._gp.predict(xy).reshape(Z.shape)
        C_est = optimizer.constraint.approx(xy).reshape(Z.shape + (n_constraints,))
        P_allowed = optimizer.constraint.predict(xy).reshape(Z.shape)

        Acq = np.where(Z_est >0, Z_est * P_allowed, Z_est / (0.5 + P_allowed))


        target_vbounds = np.min([Z, Z_est]), np.max([Z, Z_est])
        constraint_vbounds = np.min([C, C_est]), np.max([C, C_est])


        axs[0,0].contourf(X, Y, Z, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])
        for i in range(n_constraints):
          axs[0,1+i].contourf(X, Y, C[:,:,i], cmap=plt.cm.coolwarm, vmin=constraint_vbounds[0], vmax=constraint_vbounds[1])
        Z_mask = Z

        Z_mask[~np.squeeze(optimizer.constraint.allowed(C))] = np.nan
        axs[0,n_plots_per_row-1].contourf(X, Y, Z_mask, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])

        axs[1,0].contourf(X, Y, Z_est, cmap=plt.cm.coolwarm, vmin=target_vbounds[0], vmax=target_vbounds[1])
        for i in range(n_constraints):
           axs[1,1+i].contourf(X, Y, C_est[:, :, i], cmap=plt.cm.coolwarm, vmin=constraint_vbounds[0], vmax=constraint_vbounds[1])
        axs[1,n_plots_per_row-1].contourf(X, Y, Acq, cmap=plt.cm.coolwarm, vmin=0, vmax=1)

        for i in range(2):
           for j in range(n_plots_per_row):
              axs[i,j].scatter(x_[a_], y_[a_], c='white', s=80, edgecolors='black')
              axs[i,j].scatter(x_[~a_], y_[~a_], c='red', s=80, edgecolors='black')
              axs[i,j].scatter(max_["params"]['x'], max_["params"]['y'], s=80, c='green', edgecolors='black')

        return fig, axs



    def constraint_function_2_dim(x, y):
        return np.array([- np.cos(x) * np.cos(y) + np.sin(x) * np.sin(y), - np.cos(x) * np.cos(-y) + np.sin(x) * np.sin(-y)])

    constraint_lower = np.array([-np.inf, -np.inf])
    constraint_upper = np.array([0.6, 0.6])



    def simulate_high_frequency_trading(self,prices, bid_ask_spreads, liquidity_factor):
        """
        Simulate a high-frequency trading strategy with the given market prices and bid-ask spreads.

        prices: Array of simulated prices
        bid_ask_spreads: Array of simulated bid-ask spreads
        liquidity_factor: Factor affecting market liquidity (lower values indicate less liquidity)

        Returns: Cumulative profit, cumulative slippage, and plot of results
        """
        n_trades = len(prices)
        slippage = []
        profits = []

        # Simulate trades at every price point
        for i in range(1, n_trades):
            # Simulated trade with price slippage due to bid-ask spread
            trade_price = prices[i] + np.random.uniform(-bid_ask_spreads[i], bid_ask_spreads[i])
            trade_price *= liquidity_factor  # Apply liquidity factor
            trade_price = np.clip(trade_price, prices[i] - bid_ask_spreads[i], prices[i] + bid_ask_spreads[i])


            # Calculate slippage (difference between expected and actual price)
            trade_slippage = abs(trade_price - prices[i])
            slippage.append(trade_slippage)


            # Profit from this trade (difference between buy and sell price)
            profit = prices[i] - prices[i - 1] - trade_slippage
            profits.append(profit)


            cumulative_profit = np.cumsum(profits)
            cumulative_slippage = np.cumsum(slippage)


            return cumulative_profit, cumulative_slippage


    def simulate_market_liquidity(self,prices, bid_ask_spreads, liquidity_factor):
        np.random.seed(42)
        n_trades = len(prices)
        prices = np.cumsum(np.random.randn(n_trades) * 0.5 + 100)
        bid_ask_spreads = np.random.uniform(0.001, 0.02 / liquidity_factor, size=n_trades)
        return prices, bid_ask_spreads

    def simulate_trade_execution(self,prices, bid_ask_spreads, liquidity_factor):
        n_trades = len(prices)
        slippage = []
        profits = []

        for i in range(1, n_trades):
            # Simulate trade with price slippage due to bid-ask spread
            trade_price = prices[i] + np.random.uniform(-bid_ask_spreads[i], bid_ask_spreads[i]) * liquidity_factor

            # Apply more sophisticated slippage calculation
            slippage_term = np.abs(trade_price - prices[i]) * np.exp(-(i/n_trades))  # Exponential decay of slippage
            trade_slippage = slippage_term

            # Calculate profit from this trade
            profit = prices[i] - prices[i-1] - trade_slippage

            # Apply stop-loss/take-profit logic
            if profit > 0 and np.random.rand() < 0.1:  # 10% chance of taking profit
                   profit = max(profit, profit * 0.9)  # Take 90% of profit
            elif profit < 0 and np.random.rand() < 0.1:  # 10% chance of cutting loss
                   profit = min(profit, profit * 0.9)  # Cut 90% of loss

                   slippage.append(trade_slippage)
                   profits.append(profit)

            return np.cumsum(profits), np.cumsum(slippage)



    def run_simulation(self,prices, bid_ask_spreads, liquidity_factor):
        try:
            # Simulate market liquidity
            prices_sim, bid_ask_spreads_sim = self.simulate_market_liquidity(prices, bid_ask_spreads, liquidity_factor)
            print(f"Simulated prices: {prices_sim}")
            print(f"Simulated bid-ask spreads: {bid_ask_spreads_sim}")

            # Simulate trade execution
            cumulative_profit, cumulative_slippage = self.simulate_trade_execution(prices_sim, bid_ask_spreads_sim, liquidity_factor)


            return cumulative_profit, cumulative_slippage
        except Exception as e:
            print(f"An error occurred: {str(e)}")
            return None, None, None



    def logistic_map(self,r_le, x):
          return r_le * x * (1 - x)

    def lyapunov_exponent(self,r_le, num_iterations=1000, num_discard=100):
          x = 0.5
          lyap = 0

          for _ in range(num_discard):
              x = self.logistic_map(r_le, x)

          for _ in range(num_iterations):
                 x = self.logistic_map(r_le, x)
                 lyap += np.log(abs(r_le * (1 - 2*x)))

          return lyap / num_iterations



    def bifurcation_diagram(self,r_min, r_max, num_r, num_iterations, num_discard):
        r_range = np.linspace(r_min, r_max, num_r)
        x = np.ones(num_r) * 0.5

        results = []
        for r in r_range:
            for _ in range(num_discard):
                    x = self.logistic_map(r, x)
            for _ in range(num_iterations):
                    x = self.logistic_map(r, x)
                    results.append((r, x))

        return np.array(results)



    def calculate_optimal_speed(self,S, t):
        return P / (T + k / alpha)

    def simulate_execution(self,num_steps=10000):
        dt = T / num_steps
        S = np.zeros(num_steps)
        Q_mkt = np.zeros(num_steps)
        Q_lim = np.zeros(num_steps)

        S[0] = 100  # Initial stock price
        Q_mkt[0] = P * mu / (mu + theta)
        Q_lim[0] = P * mu / (mu + beta)

        for i in range(1, num_steps):
            nu_star_mkt = self.calculate_optimal_speed(S[i-1], i*dt)
            nu_star_lim = nu_star_mkt * beta / mu

            # Market order execution
            dS_mkt = sigma * np.sqrt(dt) * np.random.normal() + theta * nu_star_mkt * dt
            dQ_mkt = -nu_star_mkt * dt

            S[i] += dS_mkt
            Q_mkt[i] = max(Q_mkt[i-1] + dQ_mkt, 0)

            # Limit order execution
            fill_rate = norm.cdf(nu_star_lim * gamma * np.sqrt(dt))
            dQ_lim = -nu_star_lim * dt * fill_rate

            Q_lim[i] = max(Q_lim[i-1] + dQ_lim, 0)

            # Update total liquidated shares
            Q_total = Q_mkt[i] + Q_lim[i]

        return S, Q_mkt, Q_lim, Q_total




        def calculate_optimal_speed(self,S, t):
          return P / (T + k / alpha)

        def simulate_execution(self,num_steps=10000):
            dt = T / num_steps
            S = np.zeros(num_steps)
            Q_mkt = np.zeros(num_steps)
            Q_lim = np.zeros(num_steps)

            S[0] = 100  # Initial stock price
            Q_mkt[0] = P * mu / (mu + theta)
            Q_lim[0] = P * mu / (mu + beta)

            for i in range(1, num_steps):
                nu_star_mkt = self.calculate_optimal_speed(S[i-1], i*dt)
                nu_star_lim = nu_star_mkt * beta / mu

                # Market order execution
                dS_mkt = sigma * np.sqrt(dt) * np.random.normal() + theta * nu_star_mkt * dt
                dQ_mkt = -nu_star_mkt * dt

                S[i] += dS_mkt
                Q_mkt[i] = max(Q_mkt[i-1] + dQ_mkt, 0)

                # Limit order execution
                fill_rate = norm.cdf(nu_star_lim * gamma * np.sqrt(dt))
                dQ_lim = -nu_star_lim * dt * fill_rate

                Q_lim[i] = max(Q_lim[i-1] + dQ_lim, 0)

                # Update total liquidated shares
                Q_total = Q_mkt[i] + Q_lim[i]

            return S, Q_mkt, Q_lim, Q_total

    def run_multiple_scenarios(self):
        num_steps_list = [5000, 10000, 20000]
        initial_price_list = [100, 150, 200]
        volatility_list = [0.2, 0.3, 0.4]
        drift_list = [0.05, 0.1, 0.15]
        theta_list = [0.01, 0.02, 0.03]
        beta_list = [0.005, 0.0075, 0.01]
        gamma_list = [0.5, 1.0, 1.5]
        alpha_list = [0.1, 0.2, 0.3]

        results = []
        for num_steps in num_steps_list:
            print(f"Running scenario with num_steps = {num_steps}")
            S, Q_mkt, Q_lim, Q_total = self.simulate_execution(num_steps)
            results.append((S, Q_mkt, Q_lim, Q_total))

        return results




    def initialize_system(self,Lx, Ly, Lz, Nx, Ny, Nz):
        dx = Lx / (Nx - 1)
        dy = Ly / (Ny - 1)
        dz = Lz / (Nz - 1)
        t = 0.0
        t_end = 1.0
        dt = 0.001


        u = np.random.rand(Nx, Ny, Nz)
        v = np.random.rand(Nx, Ny, Nz)
        w = np.random.rand(Nx, Ny, Nz)


        # Smooth initial velocity profile
        x, y, z = np.meshgrid(
             np.linspace(0, Lx, Nx),
             np.linspace(0, Ly, Ny),
             np.linspace(0, Lz, Nz)


               )

        r = np.sqrt(x**2 + y**2 + z**2)
        u[:] = np.sin(r) * np.exp(-r)
        v[:] = np.cos(r) * np.exp(-r)
        w[:] = np.sin(r) * np.exp(-r)


        # Set initial velocity profiles
        u[:, :, :] = 1.0 * (np.sin(np.pi * np.arange(Nx) / Nx) * np.cos(np.pi * np.arange(Ny) / Ny) *
                       np.exp(-(np.arange(Nz) / Nz)**2))
        v[:, :, :] = 0.5 * (np.sin(np.pi * np.arange(Ny) / Ny) * np.cos(np.pi * np.arange(Nx) / Nx) *
                       np.exp(-(np.arange(Nz) / Nz)**2))
        w[:, :, :] = 0.25 * (np.sin(np.pi * np.arange(Nz) / Nz) * np.cos(np.pi * np.arange(Nx) / Nx) *
                         np.exp(-(np.arange(Ny) / Ny)**2))


        return u, v, w, dx, dy, dz

    def navier_stokes(self,u, v, w, dx, dy, dz, dt, rho, mu, k, T, Pr=0.7, alpha=0.01, epsilon=0.1,epsilon_turb=0.1,Cepsilon1=1.44, Cepsilon2=1.92, Ckmu=0.09, Cp=1000):

        """
        Solve the Navier-Stokes equations using the stable fluids algorithm.

        Parameters:
        u, v, w (ndarray): Initial velocity components
        dx, dy, dz (float): Grid spacings
        dt (float): Time step
        rho (float): Fluid density
        mu (float): Dynamic viscosity
        alpha (float): Smoothing parameter (default: 0.01)
        beta (float): Projection parameter (default: 0.5)

        Returns:
        u_new, v_new, w_new (ndarray): Updated velocity components
        """

        Re = rho * dx * np.sqrt(u**2 + v**2 + w**2) / mu


        # Compute velocity gradients
        du_dx = (np.roll(u, -1, axis=1) - u) / dx
        dv_dy = (np.roll(v, -1, axis=0) - v) / dy
        dw_dz = (np.roll(w, -1, axis=2) - w) / dz



        # Compute strain rates
        e11 = du_dx
        e22 = dv_dy
        e33 = dw_dz
        e12 = (du_dx + np.roll(u, -1, axis=1) - 2*u) / (2*dx)
        e23 = (dv_dy + np.roll(v, -1, axis=0) - 2*v) / (2*dy)
        e31 = (dw_dz + np.roll(w, -1, axis=2) - 2*w) / (2*dz)


        # Compute viscous stresses
        tau11 = 2 * mu * e11
        tau22 = 2 * mu * e22
        tau33 = 2 * mu * e33
        tau12 = mu * e12
        tau23 = mu * e23
        tau31 = mu * e31



        # Compute pressure gradient
        grad_p = -(rho * (u * du_dx + v * dv_dy + w * dw_dz +
                       tau11 + tau22 + tau33 + tau12 + tau23 + tau31))



        # Compute divergence of velocity
        div_v = (np.sum(np.array([du_dx, dv_dy, dw_dz]), axis=0))


        # Compute nonlinear term
        nonlinear_term = div_v * np.array([u, v, w])
        nonlinear_term = nonlinear_term.reshape(3, Nx, Ny, Nz)
        nonlinear_term = np.sum(nonlinear_term, axis=0)

        nonlinear_term = nonlinear_term.sum(axis=0)
        nonlinear_term = -1.0 * nonlinear_term
        nonlinear_term = nonlinear_term * Re
        nonlinear_term = nonlinear_term * dt
        nonlinear_term = nonlinear_term / 2.0


        # Apply Laplacian smoothing
        laplacian_u = ndimage.laplace(u)
        laplacian_v = ndimage.laplace(v)
        laplacian_w = ndimage.laplace(w)



        # Update velocities with smoothing
        u_smooth = u - alpha * dt * laplacian_u
        v_smooth = v - alpha * dt * laplacian_v
        w_smooth = w - alpha * dt * laplacian_w


        # Update velocities
        u_new = u_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        v_new = v_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        w_new = w_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)


        # Update velocities
        u_new = u_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[0])
        v_new = v_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[1])
        w_new = w_smooth - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31 + nonlinear_term[2])


        # Periodic boundary conditions
        u[:, :, :] = np.roll(u, 1, axis=1)
        v[:, :, :] = np.roll(v, 1, axis=0)
        w[:, :, :] = np.roll(w, 1, axis=2)


        # Update velocities
        u_new = u - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        v_new = v - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)
        w_new = w - dt * (grad_p + tau11 + tau22 + tau33 + tau12 + tau23 + tau31)



        # Pressure correction
        pressure_correction = splinalg.cg(
            A=splinalg.LinearOperator(
               shape=(np.prod(u.shape), np.prod(u.shape)),
               matvec=lambda x: ndimage.laplace(x.reshape(u.shape)).flatten(),
                      ),
             b=np.zeros(np.prod(u.shape)),
             maxiter=None,
                )[0].reshape(u.shape)



        # Correct velocities

        u_new -= pressure_correction
        v_new -= pressure_correction
        w_new -= pressure_correction


        # Update velocities with smoothing
        u_smooth = u - alpha * dt * laplacian_u
        v_smooth = v - alpha * dt * laplacian_v
        w_smooth = w - alpha * dt * laplacian_w


        # Turbulence model (k-ε)

        T = np.random.rand(Nx, Ny, Nz) * 100
        k = np.random.rand(Nx, Ny, Nz) * 1e-6
        mu_turb = np.random.rand(Nx, Ny, Nz) * 1e-5

        epsilon = np.random.rand(Nx, Ny, Nz) * 1e-6
        rho = np.random.rand(Nx, Ny, Nz) * 1.0 + 1.0
        mu = np.random.rand(Nx, Ny, Nz) * 1e-5 + 1e-5
        Pr = 0.7



        k_new = k - dt * (mu_turb / rho * epsilon)
        epsilon_new = epsilon - dt * (
        Cepsilon1 * epsilon**2 / k +
        Cepsilon2 * epsilon / k * mu_turb / rho * epsilon
             )

        # Heat transfer equation
        T_new = T - dt * (
        Pr * mu / (rho * Cp) * ndimage.laplace(T)
                   )


        # Apply wall functions for boundary layers ; https://www.simscale.com/forum/t/what-is-y-yplus/82394
        u_wall = np.sqrt(mu_turb / rho * epsilon)
        v_wall = 0.0
        w_wall = 0.0
        T_wall = T_new[0]

        # Apply free surface condition
        # This is a simplification; actual implementation would involve solving for free surface height
        rho_free_surface = 1.225  # Air density https://en.wikipedia.org/wiki/Density_of_air
        mu_free_surface = 1.81e-5  # Air viscosity. https://en.wikipedia.org/wiki/Viscosity
        if np.any(rho < rho_free_surface):

            # Mix fluid properties at interface
            mix_rho = np.where(rho < rho_free_surface, rho, rho_free_surface)
            mix_mu = np.where(rho < rho_free_surface, mu, mu_free_surface)

        laplacian_k = ndimage.laplace(k)
        laplacian_epsilon = ndimage.laplace(epsilon)
        k_smooth = k - alpha * dt * laplacian_k
        epsilon_smooth = epsilon - alpha * dt * laplacian_epsilon


        # Update turbulence variables. https://en.wikipedia.org/wiki/Turbulence_modeling
        k_new = k_smooth + dt * (
        mu_turb / rho * epsilon +
        Ckmu * mu_turb / rho * epsilon**2 / k
            )
        epsilon_new = epsilon_smooth + dt * (
        Cepsilon1 * epsilon**2 / k * mu_turb / rho +
        Cepsilon2 * mu_turb / rho * epsilon**2 / k
             )


        # Pressure correction
        def pressure_correction():
            A = ndimage.laplace(np.stack([u_smooth, v_smooth, w_smooth], axis=2))
            b = div_v
            p, _ = cg(A, b)
            return p.reshape(u_smooth.shape)


        # Correct velocities
        u_new = u_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v
        v_new = v_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v
        w_new = w_smooth - dt * grad_p - dt * (tau11 + tau22 + tau33 + tau12 + tau23 + tau31) + epsilon * dt * div_v

        # Periodic boundary conditions
        u_new[:, :, :] = np.roll(u_new, 1, axis=1)
        v_new[:, :, :] = np.roll(v_new, 1, axis=0)
        w_new[:, :, :] = np.roll(w_new, 1, axis=2)


        # Compute convective acceleration terms
        convective_u = u * du_dx + v * dv_dy + w * dw_dz
        convective_v = u * du_dx + v * dv_dy + w * dw_dz
        convective_w = u * du_dx + v * dv_dy + w * dw_dz


        # Compute pressure gradient
        grad_p = -(rho * (convective_u + convective_v + convective_w +
                   tau11 + tau22 + tau33 + tau12 + tau23 + tau31))


        # Add terms representing the proof of existence and regularity
        laplacian_u = (du_dx + dv_dy + dw_dz)
        laplacian_v = (du_dx + dv_dy + dw_dz)
        laplacian_w = (du_dx + dv_dy + dw_dz)

        # Add regularization terms
        regularization_u = alpha * laplacian_u
        regularization_v = alpha * laplacian_v
        regularization_w = alpha * laplacian_w

        # Update velocity components
        u_new = u + dt * (-convective_u - tau11 + mu * laplacian_u + regularization_u)
        v_new = v + dt * (-convective_v - tau22 + mu * laplacian_v + regularization_v)
        w_new = w + dt * (-convective_w - tau33 + mu * laplacian_w + regularization_w)


        # Update temperature
        #T_new = T_new + pressure_correction


        # Compute turbulent dissipation rate
        epsilon_turb_new = epsilon_turb + alpha * (epsilon_turb / k - Cepsilon1 * epsilon_turb**2 / (Cepsilon2 * k + epsilon))


        # Update velocities using semi-Lagrangian advection https://physics.stackexchange.com/questions/356407/validity-of-the-navier-stokes-equations-for-turbulent-flows
        u_new = u + dt * (nonlinear_term[0] + mu * du_dx + epsilon_turb_new * u)
        v_new = v + dt * (nonlinear_term[1] + mu * dv_dy + epsilon_turb_new * v)
        w_new = w + dt * (nonlinear_term[2] + mu * dw_dz + epsilon_turb_new * w)


        # Update kinetic energy
        k_new = k + dt * (0.5 * (nonlinear_term[0]**2 + nonlinear_term[1]**2 + nonlinear_term[2]**2) / 3 - mu * (du_dx**2 + dv_dy**2 + dw_dz**2) - epsilon_turb)


        # Update temperature
        T_new = T + dt * (Cp * (nonlinear_term[0] * du_dx + nonlinear_term[1] * dv_dy + nonlinear_term[2] * dw_dz) / rho)


        # Update velocities
        u_new = u - dt * (nonlinear_term[0] + alpha * (grad_p[0] / dx))
        v_new = v - dt * (nonlinear_term[1] + alpha * (grad_p[1] / dy))
        w_new = w - dt * (nonlinear_term[2] + alpha * (grad_p[2] / dz))


        # Add artificial diffusion
        u_new += epsilon * dt * (u_new - u)
        v_new += epsilon * dt * (v_new - v)
        w_new += epsilon * dt * (w_new - w)


        # Additional checks for smoothness (e.g., second derivatives)
        d2u_dx2 = (np.roll(du_dx, -1, axis=1) - du_dx) / dx
        d2v_dy2 = (np.roll(dv_dy, -1, axis=0) - dv_dy) / dy
        d2w_dz2 = (np.roll(dw_dz, -1, axis=2) - dw_dz) / dz
        u_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)
        v_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)
        w_new += alpha * (d2u_dx2 + d2v_dy2 + d2w_dz2)


        # Add turbulent dissipation term
        dissipation_term = epsilon_turb * (u**2 + v**2 + w**2)

        # Compute turbulent dissipation rate
        epsilon_turb = Cepsilon1 * k**1.5 / (Cepsilon2 * k + mu / rho)
        # Compute total dissipation rate
        epsilon_total = epsilon + epsilon_turb

        # Update velocity components
        u_new = u + dt * (-u * du_dx - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)

        v_new = v + dt * (-u * dv_dy - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)

        w_new = w + dt * (-u * dw_dz - v * dv_dy - w * dw_dz +
                    tau11 + tau22 + tau33 + tau12 + tau23 + tau31 -
                    alpha * epsilon_total)


        # Update velocities
        u_new = u - dt * (grad_p + dissipation_term)
        v_new = v - dt * (grad_p + dissipation_term)
        w_new = w - dt * (grad_p + dissipation_term)


        # Compute turbulent kinetic energy  https://en.wikipedia.org/wiki/Turbulence_kinetic_energy
        K = 0.5 * (u**2 + v**2 + w**2)

        # Compute dissipation rate https://physics.stackexchange.com/questions/441561/energy-dissipation-for-force-free-incompressible-navier-stokes-equation-with-ta
        epsilon = 2 * mu * (e11**2 + e22**2 + e33**2 + 2*e12**2 + 2*e23**2 + 2*e31**2)**0.5 / 3


        # Compute eddy viscosity
        nu_turb = Ckmu * K**1.5 / epsilon

        # Update velocities using the modified Navier-Stokes equations
        u_new = u - dt * (grad_p + mu * (du_dx + epsilon_turb * du_dx))
        v_new = v - dt * (grad_p + mu * (dv_dy + epsilon_turb * dv_dy))
        w_new = w - dt * (grad_p + mu * (dw_dz + epsilon_turb * dw_dz))



        # Add new variables for potential singularity handling
        lambda_min = 1e-6  # Minimum value for lambda
        lambda_max = 10   # Maximum value for lambda
        lambda_step = 0.01  # Step size for lambda


        # Add new calculations for potential singularity handling
        lambda_values = np.arange(lambda_min, lambda_max, lambda_step)

        # Calculate lambda values for each point in the grid
        lambda_grid = np.zeros_like(u)
        for i in range(len(lambda_values)):
            lambda_grid += lambda_values[i] / (lambda_values[i]**2 + div_v**2)

        # Normalize lambda values
        lambda_normalized = lambda_grid / np.max(lambda_grid)

        # Apply numerical method for handling singularities
        u_new = u - dt * (u * du_dx + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)
        v_new = v - dt * (u * dv_dy + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)
        w_new = w - dt * (u * dw_dz + v * dv_dy + w * dw_dz) / (1 + lambda_normalized)


        # Regularization function
        def regularize(field):
            epsilon_reg = alpha * mu / dx
            return np.maximum(field - epsilon_reg, 0)

        # Apply regularization to velocity components
        u_regularized = regularize(u)
        v_regularized = regularize(v)
        w_regularized = regularize(w)

        Re = rho * dx * np.sqrt(u_regularized**2 + v_regularized**2 + w_regularized**2) / mu

        # Compute velocity gradients
        du_dx = (np.roll(u_regularized, -1, axis=1) - u_regularized) / dx
        dv_dy = (np.roll(v_regularized, -1, axis=0) - v_regularized) / dy
        dw_dz = (np.roll(w_regularized, -1, axis=2) - w_regularized) / dz


        return u_new, v_new, w_new




    import numpy as np

    def generate_chaotic_initial_conditions(self,N=32, L=1.0, seed=42):
        """
        Generates a divergence-free, chaotic 3D velocity field
        using a Passot-Pouquet energy spectrum.
        """
        np.random.seed(seed)
        dx = L / (N - 1)
        k0 = 2 * np.pi / L

        # 1. Create wave numbers in Fourier Space
        kx = np.fft.fftfreq(N, d=dx/(2*np.pi))
        kz, ky, kx = np.meshgrid(kx, kx, kx, indexing='ij')
        k_sq = kx**2 + ky**2 + kz**2
        k = np.sqrt(k_sq)
        k[0, 0, 0] = 1.0  # Avoid division by zero at the origin

        # 2. Define the Energy Spectrum E(k)
        # Peak wave number kp (determines the scale of initial chaos)
        kp = 4.0 * k0
        # Passot-Pouquet Spectrum: E(k) ~ k^4 * exp(-2(k/kp)^2)
        energy_spectrum = (k**4) * np.exp(-2.0 * (k / kp)**2)

        # 3. Generate random phases in Fourier space
        # Magnitude is derived from energy spectrum: |u_hat|^2 = E(k) / (4 * pi * k^2)
        amplitude = np.sqrt(energy_spectrum / (4.0 * np.pi * k_sq))

        u_hat = amplitude * (np.random.randn(N, N, N) + 1j * np.random.randn(N, N, N))
        v_hat = amplitude * (np.random.randn(N, N, N) + 1j * np.random.randn(N, N, N))
        w_hat = amplitude * (np.random.randn(N, N, N) + 1j * np.random.randn(N, N, N))

        # 4. Project onto Divergence-Free Space (Leray Projection)
        # Ensures k_i * u_hat_i = 0
        dot_product = (kx * u_hat + ky * v_hat + kz * w_hat) / k_sq
        u_hat -= kx * dot_product
        v_hat -= ky * dot_product
        w_hat -= kz * dot_product

        # 5. Transform back to Physical Space
        U = np.real(np.fft.ifftn(u_hat))
        V = np.real(np.fft.ifftn(v_hat))
        W = np.real(np.fft.ifftn(w_hat))

        # Normalize to a specific peak velocity (e.g., 1.0)
        max_vel = np.max(np.sqrt(U**2 + V**2 + W**2))
        return U/max_vel, V/max_vel, W/max_vel



    def compute_drag_coefficient(self,u, v, w, dx, dy, dz, dt, rho, mu):
        # Compute drag coefficient using the force balance method.  https://www1.grc.nasa.gov/beginners-guide-to-aeronautics/drag-balance-operation/ , https://www.simscale.com/docs/simwiki/lift-drag-pitch/what-is-drag-coefficient/
        def force_balance(self,x, y, z):
            u_x, u_y, u_z = u[x, y, z], v[x, y, z], w[x, y, z]
            f_x = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_y = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_z = 2 * mu * (u_x * dudx(x, y, z) + u_y * dudy(y, z) + u_z * dwdz(z))
            f_x = -f_x
            f_y = -f_y
            f_z = -f_z
            return f_x, f_y, f_z

        def dudx(self,x, y, z):
            return (np.roll(u, -1, axis=1)[x, y, z] - u[x, y, z]) / dx

        def dudy(self,y, z):
            return (np.roll(v, -1, axis=0)[x, y, z] - v[x, y, z]) / dy

        def dwdz(self,z):
            return (np.roll(w, -1, axis=2)[x, y, z] - w[x, y, z]) / dz

            total_force_x = sum(force_balance(x, y, z)[0] for x in range(Nx) for y in range(Ny) for z in range(Nz))
            total_force_y = sum(force_balance(x, y, z)[1] for x in range(Nx) for y in range(Ny) for z in range(Nz))
            total_force_z = sum(force_balance(x, y, z)[2] for x in range(Nx) for y in range(Ny) for z in range(Nz))


            drag_force = np.sqrt(total_force_x**2 + total_force_y**2 + total_force_z**2)
            drag_area = 1.0  # Assuming unit area
            drag_coefficient = drag_force / (0.5 * rho * velocity**2 * drag_area)

            density = rho
            velocity = np.sqrt(u**2 + v**2 + w**2).mean()
            cd = drag_force / (0.5 * density * velocity**2 * drag_area)
            return cd



    def lorenz_system(self,x, y, z, s_a=10, r_a=28, b_a=2.667):
        dx_dt = s_a * (y - x)
        dy_dt = r_a * x - y - x * z
        dz_dt = x * y - b_a * z
        return dx_dt, dy_dt, dz_dt

    def simulate_lorenz(self,initial_state, num_steps, dt=0.01):
        # Ensure num_steps is an integer before using it in np.zeros
        if not isinstance(num_steps, (int, list)):
           raise TypeError(f"num_steps must be an integer or a list of integers, got {type(num_steps)}")

        if isinstance(num_steps, list):
          if not all(isinstance(n, int) for n in num_steps):
            raise TypeError("All elements in num_steps must be integers")

        trajectory = np.zeros((num_steps, 3))
        state = initial_state

        for i in range(num_steps):
            dx, dy, dz = self.lorenz_system(*state)
            state += np.array([dx, dy, dz]) * dt
            trajectory[i] = state

        return trajectory


    def system(self,X, t):
        x, y = X
        dx = y
        dy = -np.sin(x)
        return [dx, dy]

    def plot_phase_portrait(self,ax, x_range, y_range):
        x = np.linspace(x_range[0], x_range[1], 20)
        y = np.linspace(y_range[0], y_range[1], 20)
        X, Y = np.meshgrid(x, y)
        U = Y
        V = -np.sin(X)
        ax.streamplot(X, Y, U, V, density=1)
        ax.set_xlabel('x')
        ax.set_ylabel('y')
        ax.set_title('Phase Portrait')



    def multi_factor_bates(self,S0_b, v0_b, r_b, kappa_b, theta_b, sigma_b, rho_b, lambda_, mu_j, sigma_j, T, N, M, num_factors=2):
         dt = T/N
         S_b = np.zeros((M, N+1))
         v_b = np.zeros((M, N+1, num_factors))
         S_b[:, 0] = S0_b
         v_b[:, 0, :] = v0_b

         for i in range(1, N+1):
             dW = np.random.normal(0, np.sqrt(dt), (M, num_factors+1))

             # Correlated Brownian motions
             for j in range(1, num_factors+1):
                 dW[:, j] = rho_b[j-1] * dW[:, 0] + np.sqrt(1 - rho_b[j-1]**2) * dW[:, j]

                 # Update volatilities
                 for j in range(num_factors):
                     v_b[:, i, j] = v_b[:, i-1, j] + kappa_b[j] * (theta_b[j] - v_b[:, i-1, j]) * dt + sigma_b[j] * np.sqrt(v_b[:, i-1, j]) * dW[:, j+1]
                     v_b[:, i, j] = np.maximum(v_b[:, i, j], 0)

                     # Jump process
                     dN = np.random.poisson(lambda_ * dt, M)
                     J = np.random.normal(mu_j, sigma_j, M) * dN

                     # Update asset price
                     total_var = np.sum(v_b[:, i], axis=1)
                     S_b[:, i] = S_b[:, i-1] * np.exp((r_b - 0.5*total_var - lambda_*(np.exp(mu_j + 0.5*sigma_j**2) - 1))*dt +
                                     np.sqrt(total_var)*dW[:, 0] + J)

         return S_b, v_b



    def incompressible_navier_stokes(self,u, v, dx, dy, dt, viscosity):
        # Compute spatial derivatives
        du_dx = np.gradient(u, dx, axis=1)
        du_dy = np.gradient(u, dy, axis=0)
        dv_dx = np.gradient(v, dx, axis=1)
        dv_dy = np.gradient(v, dy, axis=0)


        # Compute Laplacians
        laplacian_u = np.gradient(du_dx, dx, axis=1) + np.gradient(du_dy, dy, axis=0)
        laplacian_v = np.gradient(dv_dx, dx, axis=1) + np.gradient(dv_dy, dy, axis=0)


         # Update velocities
        u_new = u - dt * (u * du_dx + v * du_dy) + dt * viscosity * laplacian_u
        v_new = v - dt * (u * dv_dx + v * dv_dy) + dt * viscosity * laplacian_v

        # Periodic boundary conditions
        u_new[:, :] = np.roll(u_new, 1, axis=1)
        v_new[:, :] = np.roll(v_new, 1, axis=0)


        return u_new, v_new


    def simulate_sde(self, t_span, x0, theta, alpha, beta, sigma):
        """
        Simulates geometric random walk SDE
        dx_t = θx_t dt + αx_t dW_t + βx_t dt + σx_t dW_t
        """
        def sde_func(t, x):
            drift = (theta + beta) * x
            diffusion = (alpha + sigma) * x
            return drift + np.random.normal(0, diffusion)

        t_eval = np.linspace(t_span[0], t_span[1], 100)
        solution = solve_ivp(
           lambda t, x: sde_func(t, x),
           t_span,
           x0,
           t_eval=t_eval,
           method='RK45'
                     )
        return solution.y.T



    def LV(self,x, y):
        return np.array([x - x*y, x*y - y])

    def rk4(self,f, x0, y0, h, n):

        v = [0]*(n+1)
        v[0] = np.array([x0, y0])
        x = x0
        y = y0
        for i in range(1, n + 1):
            k1 = h*f(x, y)
            k2 = h*f(x + 0.5*k1[0], y + 0.5*k1[1])
            k3 = h*f(x + 0.5*k2[0], y + 0.5*k2[1])
            k4 = h*f(x + k3[0], y + k3[1])
            v[i] =  v[i-1] + (k1 + k2 + k2 + k3 + k3 + k4)/6
            x = v[i][0]
            y = v[i][1]

        t = np.array([i*h for i in range(0, n+1)])
        return t, np.array(v)

    def euler(self,f, x0, y0, h, n):

         v = [0]*(n+1)
         v[0] = np.array([x0, y0])
         x = x0
         y = y0

         for i in range(1, n + 1):
             v[i] =  v[i-1] + h*f(x, y)
             x = v[i][0]
             y = v[i][1]

         t = np.array([i*h for i in range(0, n+1)])
         return t, np.array(v)

    def plot_integrator(self,v_euler, v_rk4, t_euler, t_rk4, v_true, t_true, h):

         fig = plt.figure(figsize=(18,8))
         ax0 = fig.add_subplot(121)
         ax1 = fig.add_subplot(122)

         ax0.plot(t_euler, v_euler, marker = 'x')
         ax1.plot(t_rk4, v_rk4, marker = 'x')

         ax0.plot(t_true, v_true)
         ax1.plot(t_true, v_true)

         ax0.set_ylim(0, 3.5)
         ax1.set_ylim(0, 3.5)

         ax0.set_xlabel(r"t", fontsize=25)
         ax0.set_title("Euler, $h=$"+h, fontsize=25)
         ax0.legend(["x Euler", "y Euler", "x True", "y True"])
         ax1.set_xlabel(r"$t$", fontsize=25)
         ax1.set_title("RK4, $h=$"+h, fontsize=25)
         ax1.legend(["x RK4", "y RK4", "x True", "y True"])



    def create_causal_graph(self,model):
           g = pgv.AGraph(directed=True)

           # Add nodes
           g.add_node("theta", label="θ", shape="box")
           g.add_node("alpha", label="α", shape="box")
           g.add_node("beta", label="β", shape="box")
           g.add_node("sigma", label="σ", shape="box")
           g.add_node("x_T", label="x_T", shape="ellipse")
           g.add_node("x_t_minus_1", label="x_t-1", shape="ellipse")
           g.add_node("z", label="z", shape="ellipse")
           g.add_node("noise_dist", label="noise_dist", shape="diamond")
           g.add_node("backdoor_trigger", label="backdoor_trigger", shape="diamond")
           g.add_node("x_t", label="x_t", shape="ellipse")
           g.add_node("y", label="y", shape="ellipse")
           g.add_node("x_t_plus_1", label="x_t+1", shape="ellipse")


           # Add edges
           g.add_edge("theta", "x_T")
           g.add_edge("alpha", "x_T")
           g.add_edge("beta", "x_T")
           g.add_edge("sigma", "x_T")
           g.add_edge("x_T", "x_t_minus_1")
           g.add_edge("z", "x_t_minus_1")
           g.add_edge("backdoor_trigger", "x_T")
           g.add_edge("x_t_minus_1", "x_t")
           g.add_edge("noise_dist", "x_t")
           g.add_edge("x_t", "y")
           g.add_edge("y", "x_t_plus_1")
           g.add_edge("x_t_plus_1", "z")

           return g


    def function(self, x, y, t):

        res = self.net(torch.hstack((x, y, t)))
        psi, p = res[:, 0:1], res[:, 1:2]

        u = torch.autograd.grad(psi, y, grad_outputs=torch.ones_like(psi), create_graph=True)[0] #retain_graph=True,
        v = -1.*torch.autograd.grad(psi, x, grad_outputs=torch.ones_like(psi), create_graph=True)[0]

        u_x = torch.autograd.grad(u, x, grad_outputs=torch.ones_like(u), create_graph=True)[0]
        u_xx = torch.autograd.grad(u_x, x, grad_outputs=torch.ones_like(u_x), create_graph=True)[0]
        u_y = torch.autograd.grad(u, y, grad_outputs=torch.ones_like(u), create_graph=True)[0]
        u_yy = torch.autograd.grad(u_y, y, grad_outputs=torch.ones_like(u_y), create_graph=True)[0]
        u_t = torch.autograd.grad(u, t, grad_outputs=torch.ones_like(u), create_graph=True)[0]

        v_x = torch.autograd.grad(v, x, grad_outputs=torch.ones_like(v), create_graph=True)[0]
        v_xx = torch.autograd.grad(v_x, x, grad_outputs=torch.ones_like(v_x), create_graph=True)[0]
        v_y = torch.autograd.grad(v, y, grad_outputs=torch.ones_like(v), create_graph=True)[0]
        v_yy = torch.autograd.grad(v_y, y, grad_outputs=torch.ones_like(v_y), create_graph=True)[0]
        v_t = torch.autograd.grad(v, t, grad_outputs=torch.ones_like(v), create_graph=True)[0]

        p_x = torch.autograd.grad(p, x, grad_outputs=torch.ones_like(p), create_graph=True)[0]
        p_y = torch.autograd.grad(p, y, grad_outputs=torch.ones_like(p), create_graph=True)[0]

        f = u_t + u * u_x + v * u_y + p_x - nu * (u_xx + u_yy)
        g = v_t + u * v_x + v * v_y + p_y - nu * (v_xx + v_yy)

        return u, v, p, f, g


    def chorin_projection_method(self,nx, ny, nt, dt, dx, dy, rho, nu):
        # Initialize velocity fields (u, v) and pressure (p)
        u = np.zeros((ny, nx))
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))

        # Intermediate velocities
        u_star = np.zeros_like(u)
        v_star = np.zeros_like(v)

        for n in range(nt):
           # ---------------------------------------------------------
           # STEP 1: Predictor Step (Solve momentum without pressure)
           # Calculate intermediate velocity (u_star, v_star)
           # using diffusion and advection terms
           # ---------------------------------------------------------
           for j in range(1, ny-1):
               for i in range(1, nx-1):
                   # Central differencing for advection and diffusion
                   u_star[j, i] = u[j, i] - dt * (
                      u[j, i] * (u[j, i] - u[j, i-1]) / dx +
                      v[j, i] * (u[j, i] - u[j-1, i]) / dy
                   ) + nu * dt * (
                       (u[j, i+1] - 2*u[j, i] + u[j, i-1]) / dx**2 +
                       (u[j+1, i] - 2*u[j, i] + u[j-1, i]) / dy**2
                   )

                   v_star[j, i] = v[j, i] - dt * (
                       u[j, i] * (v[j, i] - v[j, i-1]) / dx +
                       v[j, i] * (v[j, i] - v[j-1, i]) / dy
                   ) + nu * dt * (
                      (v[j, i+1] - 2*v[j, i] + v[j, i-1]) / dx**2 +
                      (v[j+1, i] - 2*v[j, i] + v[j-1, i]) / dy**2
                   )

           # ---------------------------------------------------------
           # STEP 2: Solve the Pressure Poisson Equation
           # Enforce the incompressibility constraint (divergence-free)
           # ---------------------------------------------------------
           # Compute the divergence of the intermediate velocity field
           div_star = np.zeros_like(p)
           for j in range(1, ny-1):
                for i in range(1, nx-1):
                    div_star[j, i] = (u_star[j, i+1] - u_star[j, i-1]) / (2 * dx) + \
                                 (v_star[j+1, i] - v_star[j-1, i]) / (2 * dy)

           # Iteratively solve for pressure
           p_new = np.copy(p)
           for _ in range(50): # 50 iterations for Poisson solver
                for j in range(1, ny-1):
                    for i in range(1, nx-1):
                        p_new[j, i] = ((p[j, i+1] + p[j, i-1]) * dy**2 +
                                   (p[j+1, i] + p[j-1, i]) * dx**2 -
                                   (rho * dx**2 * dy**2 / dt) * div_star[j, i]) / (2 * (dx**2 + dy**2))
                p = np.copy(p_new)

           # ---------------------------------------------------------
           # STEP 3: Corrector Step
           # Update the final velocity field using the new pressure gradient
           # ---------------------------------------------------------
           for j in range(1, ny-1):
                for i in range(1, nx-1):
                   u[j, i] = u_star[j, i] - (dt / rho) * (p[j, i+1] - p[j, i-1]) / (2 * dx)
                   v[j, i] = v_star[j, i] - (dt / rho) * (p[j+1, i] - p[j-1, i]) / (2 * dy)



        return u, v, p

    def chorin_projection_method(self,nx, ny, nt, dt, dx, dy, rho, nu):
        """
        Solve incompressible Navier-Stokes via Chorin's projection method.

        Parameters
        ----------
        nx, ny : int   – grid points in x and y
        nt     : int   – number of time steps
        dt     : float – time step
        dx, dy : float – grid spacing
        rho    : float – density
        nu     : float – kinematic viscosity

        Returns
        -------
        u, v, p : (ny, nx) arrays
        """
        u = np.zeros((ny, nx))
        v = np.zeros((ny, nx))
        p = np.zeros((ny, nx))

        # Lid-driven cavity: top wall moves at u=1
        def apply_bc(self,u, v):
            u[:, 0]  = 0;  u[:, -1] = 0   # left / right walls
            u[0, :]  = 0                    # bottom wall
            u[-1, :] = 1.0                  # top lid  ← driven boundary
            v[:, 0]  = 0;  v[:, -1] = 0
            v[0, :]  = 0;  v[-1, :] = 0
            return u, v

        def pressure_poisson(self,p, u, v, dx, dy, dt, rho, niter=50):
            pn = p.copy()
            b  = (rho / dt) * (
                (u[1:-1, 2:] - u[1:-1, :-2]) / (2*dx) +
                (v[2:, 1:-1] - v[:-2, 1:-1]) / (2*dy))
            for _ in range(niter):
                pn = p.copy()
                p[1:-1, 1:-1] = (
                    ((pn[1:-1, 2:] + pn[1:-1, :-2]) * dy**2 +
                    (pn[2:, 1:-1] + pn[:-2, 1:-1]) * dx**2) /
                    (2*(dx**2 + dy**2)) -
                    dx**2 * dy**2 / (2*(dx**2 + dy**2)) * b)
                # Neumann BCs for pressure
                p[:, -1] = p[:, -2]
                p[:, 0]  = p[:, 1]
                p[-1, :] = p[-2, :]
                p[0, :]  = 0.0
            return p

            u, v = self.apply_bc(u, v)

            for _ in range(nt):
             un, vn = u.copy(), v.copy()

             # Intermediate velocity (advection + diffusion, no pressure)
             u[1:-1, 1:-1] = (
                  un[1:-1, 1:-1]
                  - dt * (un[1:-1, 1:-1] * (un[1:-1, 1:-1] - un[1:-1, :-2]) / dx
                      + vn[1:-1, 1:-1] * (un[1:-1, 1:-1] - un[:-2, 1:-1]) / dy)
                  + nu * dt * (
                    (un[1:-1, 2:] - 2*un[1:-1, 1:-1] + un[1:-1, :-2]) / dx**2 +
                    (un[2:, 1:-1] - 2*un[1:-1, 1:-1] + un[:-2, 1:-1]) / dy**2))

             v[1:-1, 1:-1] = (
                  vn[1:-1, 1:-1]
                 - dt * (un[1:-1, 1:-1] * (vn[1:-1, 1:-1] - vn[1:-1, :-2]) / dx
                      + vn[1:-1, 1:-1] * (vn[1:-1, 1:-1] - vn[:-2, 1:-1]) / dy)
                  + nu * dt * (
                   (vn[1:-1, 2:] - 2*vn[1:-1, 1:-1] + vn[1:-1, :-2]) / dx**2 +
                   (vn[2:, 1:-1] - 2*vn[1:-1, 1:-1] + vn[:-2, 1:-1]) / dy**2))

            u, v = self.apply_bc(u, v)

            # Pressure solve
            p = self.pressure_poisson(p, u, v, dx, dy, dt, rho)

            # Projection: correct velocity
            u[1:-1, 1:-1] -= dt/rho * (p[1:-1, 2:] - p[1:-1, :-2]) / (2*dx)
            v[1:-1, 1:-1] -= dt/rho * (p[2:, 1:-1] - p[:-2, 1:-1]) / (2*dy)

            u, v = self.apply_bc(u, v)

        return u, v, p



    def _bayesian_sampling_diffusion_model(
        self,
        T: int,
        theta: float,
        alpha: np.ndarray,
        beta: np.ndarray,
        sigma: np.ndarray,
        noise_dist: Callable[[Any], np.ndarray],
        non_linear_drift: Callable[[float, int], float],
        model_type: str = 'vasicek',
        trace_vasicek = None,
        trace_hull_white = None,
        trace_CIR = None,
        bounds=None,
        objective_function=None,
        cramler_lundberg_model=True,
        buhlmann_model=None,
        mckean_vlasov_process=None,


        trace_longstaff_schwartz=None  # Add this line
    ) -> pm.backends.base.MultiTrace: # Modified return type to include model_type

        assert isinstance(T, int), "Expected T to be an integer"
        assert isinstance(alpha, np.ndarray) and alpha.ndim == 1, "Expected alpha to be a 1D numpy array"
        assert isinstance(beta, np.ndarray) and beta.ndim == 1, "Expected beta to be a 1D numpy array"
        assert isinstance(sigma, np.ndarray) and sigma.ndim == 1, "Expected sigma to be a 1D numpy array"
        assert callable(noise_dist), "Expected noise_dist to be a callable function"
        assert callable(non_linear_drift), "Expected non_linear_drift to be a callable function"

        assert model_type in ['vasicek', 'hull_white', 'longstaff_schwartz', 'libor_market', 'CIR'], "Invalid model type"

        # Define the drift function based on the model type
        drift_function_map = {
            'CIR': self.cir_drift,
            'vasicek': self.vasicek_drift,
            'hull_white': self.hull_white_drift,
            'longstaff_schwartz': self.longstaff_schwartz_drift,
            'libor_market': self.libor_market_drift
         }
        drift_function = drift_function_map.get(model_type)



        r_range = np.linspace(2.5, 6.0, 1000)
        lyap_exponents = [self.lyapunov_exponent(r_le) for r_le in r_range]

        plt.figure(figsize=(12, 6))
        plt.plot(r_range, lyap_exponents)
        plt.title('Lyapunov Exponent of the Logistic Map')
        plt.xlabel('r')
        plt.ylabel('Lyapunov Exponent')
        plt.axhline(y=0, color='r', linestyle='--')
        plt.grid(True)
        plt.savefig('lyapunov_exponent_logistic_map.png', dpi=300, bbox_inches='tight')
        plt.show()



        h = 1.2
        LV = lambda x, y: np.array([x - x*y, x*y - y])

        t_euler, v_euler = self.euler(LV, 1., 2., h, 60)
        t_rk4, v_rk4 = self.rk4(LV, 1., 2., h, 60)
        t_true, v_true = self.rk4(LV, 1., 2., 0.003, 4000)

        self.plot_integrator(v_euler, v_rk4, t_euler, t_rk4, v_true, t_true, str(h))
        plt.savefig('integrator.png', dpi=300, bbox_inches='tight')
        plt.show()



        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

        # Stable equilibrium point
        self.plot_phase_portrait(ax1, [-np.pi, np.pi], [-2, 2])
        ax1.plot(0, 0, 'ro', markersize=10)
        ax1.set_title('Stable Equilibrium FinanceLLMsBackRL')


        # Unstable equilibrium point
        self.plot_phase_portrait(ax2, [0, 2*np.pi], [-2, 2])
        ax2.plot(np.pi, 0, 'ro', markersize=10)
        ax2.set_title('Unstable Equilibrium FinanceLLMsBackRL')

        plt.tight_layout()
        plt.savefig('phase_portraits.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage
        prices = np.array([100, 105, 110, 115, 120, 125, 130, 135, 140, 145])
        bid_ask_spreads = np.array([0.005, 0.004, 0.003, 0.002, 0.001, 0.0005, 0.0003, 0.0002, 0.0001, 0.00005])
        liquidity_factor = 1.5

        results = self.run_simulation(prices, bid_ask_spreads, liquidity_factor)
        if results:
              cumulative_profit, cumulative_slippage = results
        else:
              print("Simulation failed.")



        # Parameters
        r0_shw = 0.02  # Initial short rate
        theta_shw = 0.04  # Mean reversion speed
        mu_shw = 0.06  # Long-term mean
        sigma_shw = 0.1  # Volatility
        T = 3  # Total time horizon

        # Simulation
        t, r = self.simulate_hull_white(r0_shw, theta_shw, mu_shw, sigma_shw, T)

        # Plot results
        plt.figure(figsize=(10, 6))
        plt.plot(t, r)
        plt.xlabel('Time')
        plt.ylabel('Short Rate')
        plt.title('Hull-White Model Simulation')
        plt.grid(True)
        plt.savefig('hull_white_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage
        S0_b, r_b = 100, 0.05
        S0_b, r_b = 100, 0.05
        v0_b = [0.1, 0.05]
        kappa_b = [2, 1]
        theta_b = [0.1, 0.05]
        sigma_b = [0.3, 0.2]
        rho_b = [-0.5, -0.3]
        lambda_, mu_j, sigma_j = 0.1, -0.05, 0.1
        T, N, M = 3, 252, 1000


        S, v = self.multi_factor_bates(S0_b, v0_b, r_b, kappa_b, theta_b, sigma_b, rho_b, lambda_, mu_j, sigma_j, T, N, M)

        plt.figure(figsize=(12, 8))
        plt.plot(S[:5].T)
        plt.title('FinanceLLMsBackRL: Multi-Factor Bates Model: Sample Price Paths')
        plt.xlabel('Time Steps')
        plt.ylabel('Asset Price')
        plt.savefig('multi_factor_bates_model_sample_price_paths.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Example usage

        initial_state = [1.0, 1.0, 1.0]
        num_steps = 10000
        trajectory = self.simulate_lorenz(initial_state, num_steps)

        fig = plt.figure(figsize=(10, 8))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(trajectory[:, 0], trajectory[:, 1], trajectory[:, 2])
        ax.set_title("Lorenz Attractor: FinanceLLMsBackRL")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
        plt.savefig('lorenz_attractor.png', dpi=300, bbox_inches='tight')
        plt.show()


        r0_c = 0.05
        theta_c = 0.02
        mu_c = 0.01
        sigma_c = 0.05
        N = 1000
        r=self.CIR_model(theta_c, mu_c, sigma_c, r0_c, T, N)


        plt.figure(figsize=(10, 6))
        plt.plot(np.linspace(0, T, N+1), r)
        plt.title('Cox-Ingersoll-Ross Model Simulation')
        plt.xlabel('Time')
        plt.ylabel('Interest Rate')
        plt.grid(True)
        plt.savefig('cox_ingersoll_ross_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()


        # Example usage
        nx, ny = 50, 50
        dx, dy = 1.0 / nx, 1.0 / ny
        x, y = np.meshgrid(np.linspace(0, 1, nx), np.linspace(0, 1, ny))

        u = np.sin(2 * np.pi * x) * np.cos(2 * np.pi * y)
        v = -np.cos(2 * np.pi * x) * np.sin(2 * np.pi * y)

        dt = 0.001
        viscosity = 0.1

        for _ in range(100):
            u, v = self.incompressible_navier_stokes(u, v, dx, dy, dt, viscosity)

        plt.streamplot(x, y, u, v)
        plt.title("FinanceLLMsBackRL: Incompressible Flow Field")
        plt.xlabel("X")
        plt.ylabel("Y")
        plt.savefig('incompressible_flow_field.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Simulate rough volatility paths
        Ts = [40, 50,60,80]  # Example time horizons
        rough_volatility_paths = self.simulate_rough_volatility_paths(Ts=Ts, S0=self.prior, sigma=0.8, gamma=0.7, dt=0.2)
        print(f"Rough Volatility Paths: {rough_volatility_paths}")


        # Calculate the initial portfolio value
        S = rough_volatility_paths[-1]
        K = 125
        T = 1
        r = 0.09
        dt = 2
        S0 = self.prior
        initial_portfolio_value = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Initial Portfolio Value: {initial_portfolio_value}")
        print(f"S: {S}")


        call_price_ft = self.fourier_transform_option_price(100, 105, 1, 0.05, 0.2)
        print(f"Fourier Transform Call Price: {call_price_ft}")

        call_price_ft_results_for_multiple_parameters = self.display_results_for_multiple_parameters(spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities)
        print(f"Fourier Transform Call Price for Multiple Parameters: {call_price_ft_results_for_multiple_parameters}")

        # Collect and plot option prices
        self.collect_and_plot_option_prices(spot_prices, strike_prices, times_to_maturity, risk_free_rates, volatilities)
        print(f"Collected and plotted option prices.")



        tail_prob, shortfall_int,var_95_percent, var_99_percent = self.tail_probabilities_and_shortfall_integrals(S, K, T, r, sigma, threshold=5)
        print(f"Tail Probability: {tail_prob}")
        print(f"Shortfall Integral: {shortfall_int}")
        print(f"95% VaR: {var_95_percent}")
        print(f"99% VaR: {var_99_percent}")


        # Example 1: Cramer-Lundberg model

        eta = 0.8  # Scale parameter
        rho = 0.5  # Shape parameter
        initial_capital = 500000
        premium_rate = 0.05
        claim_frequency = 0.02


        surplus_history = self.cramler_lundberg_model(U, c, claim_frequency, F, T_max, jump_max)
        print(f"Initial Capital: {initial_capital}")
        print(f"Premium Rate: {premium_rate}")
        print(f"Claim Frequency: {claim_frequency}")
        print(f"Surplus History: {surplus_history}")

        plt.figure(figsize=(12, 6))
        plt.plot(surplus_history)
        plt.xlabel('Time')
        plt.ylabel('Surplus')
        plt.title('Cramer-Lundberg Model Simulation')
        plt.grid(True)
        plt.savefig('cramler_lundberg_model_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()


        # Example 2: Buhlmann model

        iota = 1000
        phi = 4000000
        zeta = 5
        lambda_ = 0.001
        omega = 30000


        predicted_mean, predicted_std = self.buhlmann_model(100000 ,phi, zeta, lambda_,omega)
        print(f"Credibility prediction: Mean={predicted_mean}, Standard Deviation={predicted_std}")


        bsm_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Black-Scholes-Merton Call Price: {bsm_price}")


        t, P = self.mckean_vlasov_process(P0=1.0, mu=self.mean_field_interaction, sigma=0.5, t_max=1.0, dt=0.01, num_steps=10000)
        print(f"Mckean-Vlasov Process: Time={t}, Value={S}")


        plt.figure(figsize=(12, 6))
        plt.plot(t, P)
        plt.title("McKean-Vlasov Process Simulation")
        plt.xlabel("Time")
        plt.ylabel("Value")
        plt.grid(True)
        plt.savefig('mckean_vlasov_process_simulation.png', dpi=300, bbox_inches='tight')
        plt.show()

        # Plotting

        plt.figure(figsize=(10, 6))
        for i, path in enumerate(rough_volatility_paths):
           plt.plot(path)

        plt.title('Rough Volatility Paths')
        plt.xlabel('Time Steps')
        plt.ylabel('Volatility Path Value')
        plt.legend()
        plt.savefig('rough_volatility_paths.png', dpi=300, bbox_inches='tight')
        plt.show()



        # Dynamic Hedging
        call_price = self.black_scholes_merton_call(S, K, T, r, sigma)
        print(f"Call Price: {call_price}")
        initial_portfolio_value = call_price
        print(f"Initial Portfolio Value: {initial_portfolio_value}")

        greeks = self.calculate_greeks(S, K, T, r, sigma)
        print(f"Greeks: Delta={greeks[0]}, Gamma={greeks[1]}, Theta={greeks[2]}, vega={greeks[3]}, rho={greeks[4]}")


        final_portfolio_value = self.dynamic_hedging(initial_portfolio_value, S, K, T, r, sigma, dt)
        print(f"Final Portfolio Value after Hedging: {final_portfolio_value}")


       # self.plot_results()


        u, v, w, dx, dy, dz = self.initialize_system(Lx, Ly, Lz, Nx, Ny, Nz)


        Pr = 0.7
        k = np.random.rand(Nx, Ny, Nz) * 1e-6
        #T = np.random.rand(Nx, Ny, Nz) * 100


        u_new, v_new, w_new = self.navier_stokes(u, v, w, dx, dy, dz, dt,rho,mu, Pr ,k ,T)

        print(f"Updated velocities: u_new={u_new}, v_new={v_new}, w_new={w_new}")
        print(f"Updated dx={dx}, dy={dy}, dz={dz}")

        print("System initialized.")



        def plot_navier_stokes_results(u, v, w):
            fig, axes = plt.subplots(3, 3, figsize=(15, 12))

            fields = [("U Velocity", u), ("V Velocity", v), ("W Velocity", w)]
            slice_labels = ["XY Plane (mid Z)", "XZ Plane (mid Y)", "YZ Plane (mid X)"]

            for row, (title, data) in enumerate(fields):
                mid_z = data[:, :, data.shape[2] // 2]  # XY slice
                mid_y = data[:, data.shape[1] // 2, :]  # XZ slice
                mid_x = data[data.shape[0] // 2, :, :]  # YZ slice

                slices = [mid_z, mid_y, mid_x]

                for col, (slc, slice_label) in enumerate(zip(slices, slice_labels)):
                    ax = axes[row, col]
                    im = ax.imshow(slc, cmap="viridis", origin="lower", aspect="auto")
                    ax.set_title(f"{title}\n{slice_label}", fontsize=10, fontweight="bold")
                    ax.set_xlabel("Axis 1")
                    ax.set_ylabel("Axis 2")
                    fig.colorbar(im, ax=ax, shrink=0.8)

            plt.suptitle("Navier-Stokes — Final Velocity Fields (Mid-Plane Slices)", fontsize=14, y=1.01)
            fig.subplots_adjust(top=0.93, hspace=0.5, wspace=0.4)
            plt.savefig("Navier-Stokes — Final Velocity Fields (Mid-Plane Slices)", dpi=300, bbox_inches='tight')
            plt.show()

        # Usage
        plot_navier_stokes_results(u_new, v_new, w_new)


        for _ in range(50):  # Run simulation for 500 steps
              u, v, w = self.navier_stokes(u, v, w, dx, dy, dz, dt, rho, mu,Pr ,k ,T)
              u_new, v_new, w_new = self.navier_stokes(u_new, v_new, w_new, dx, dy, dz, dt, rho, mu, Pr ,k ,T)


              print("Simulation step completed.")


        # Optional: Compute drag coefficient every 50 iterations
        if _ % 50 == 0:
              cd = self.compute_drag_coefficient(u, v, w, dx, dy, dz, dt, rho, mu
                                                 )
              print(f"Iteration {_}: Drag Coefficient = {cd:.4f}")

              print("Simulation completed.")


              print(f"Initial kinetic energy: {np.mean(0.5*(u**2 + v**2 + w**2))}")
              print(f"Final kinetic energy: {np.mean(0.5*(u_new**2 + v_new**2 + w_new**2))}")



        # Run simulation
        S, Q_mkt, Q_lim, Q_total = self.simulate_execution()


        print(f"S: {S}")
        print(f"Q_mkt: {Q_mkt}")
        print(f"Q_lim: {Q_lim}")
        print(f"Q_total: {Q_total}")



        results = self.run_multiple_scenarios()
        print(f"Results: {results}")

        results = self.run_multiple_scenarios()
        print(f"Results: {results}")

        # Store and analyze results
        for i, result in enumerate(results):
            S, Q_mkt, Q_lim, Q_total = result
            # Add your analysis code here
            print(f"Scenario {i+1} analysis:")
            # Example: Print statistics
            print(f"S mean: {np.mean(S)}")
            print(f"S std: {np.std(S)}")
            print(f"Q_mkt mean: {np.mean(Q_mkt)}")
            print(f"Q_lim mean: {np.mean(Q_lim)}")
            print(f"Q_total mean: {np.mean(Q_total)}")
            print("---")


        # Perform Bayesian optimization
        bounds = {'x': (-5, 5), 'y': (-5, 5)}


        prices = np.arange(10000) + np.random.normal(0, 1, 10000)
        bid_ask_spreads = np.abs(np.random.normal(0, 0.01, 9999))  # Exclude last price
        liquidity_factor = 1.0  # Higher liquidity factor means tighter bid-ask spreads

        cumulative_profit, cumulative_slippage = self.simulate_high_frequency_trading(prices, bid_ask_spreads, liquidity_factor)
        print(f"Cumulative Profit: {cumulative_profit[-1]}")
        print(f"Cumulative Slippage: {cumulative_slippage[-1]}")


        # Adjust liquidity factor and re-run simulation
        for liquidity_factor in [0.5, 1.0, 2.0]:
            print(f"\nLiquidity Factor: {liquidity_factor:.1f}")
            cumulative_profit, cumulative_slippage = self.simulate_high_frequency_trading(prices, bid_ask_spreads, liquidity_factor)
            print(f"Cumulative Profit: {cumulative_profit[-1]}")
            print(f"Cumulative Slippage: {cumulative_slippage[-1]}")


        # Create an instance of UtilityFunction for the acquisition function
        optimizer = BayesianOptimization(

            f=self.objective_function,
            pbounds=bounds,
            random_state=42,
            verbose=2,


        )
        # Set Gaussian Process parameters
        # Perform the optimization
        optimizer.set_gp_params(alpha=1e-3, n_restarts_optimizer=5)
        optimizer.maximize(
        init_points=2,
        n_iter=5

                  )

        # Prepare data for CSV

        import matplotlib.ticker as ticker
        from mpl_toolkits.mplot3d import Axes3D

        def plot_iv_surface(data, x="Log Moneyness", y='Time to Maturity (years)', z='iv'):
            """ Plots the IV surface
            """
            fig = plt.figure(figsize=(12, 6))
            ax = fig.gca(projection='3d')
            ax.azim = 120
            ax.elev = 13

            ax.set_xlabel(x)
            ax.set_ylabel(y)
            ax.set_zlabel(z)

            ax.invert_xaxis()
            ax.xaxis.set_major_formatter(ticker.FormatStrFormatter('%0.2f'))

            surf = ax.plot_trisurf(data[x], data[y], data[z], antialiased=True, cmap = plt.cm.Spectral)
            fig.colorbar(surf, shrink=0.7, aspect=10)

            plt.tight_layout()

            plt.show()

            plot_iv_surface(rough_volatility_paths, z='iv')


        # Visualization using Plotly for interactivity
        fig = go.Figure()


        # Extracting the optimized parameters and corresponding values

        optimized_params = optimizer.max['params']
        optimized_value = optimizer.max['target']

        # Adding scatter plot for optimized parameter with enhanced styling
        fig.add_trace(go.Scatter(x=[optimized_params["x"]], y=[optimized_value], mode='markers',
                         marker=dict(size=10, color='red'), name='Optimized Max'))

        # Generating a grid for plotting the objective function
        x_grid = np.linspace(-5, 5, 100)
        y_grid = np.linspace(-5, 5, 100)
        X, Y = np.meshgrid(x_grid, y_grid)
        Z = -(X**2 + Y**2 + 0.1*X*Y + np.sin(X) + np.cos(Y))   # Objective function evaluated over the grid

        # Calculating gradients for displaying direction of optimization
        grad_X = -2*X - 0.1*Y - np.sin(X)
        grad_Y = -2*Y - 0.1*X - np.cos(Y)

        # Adding contour plot
        fig.add_trace(go.Contour(z=Z, x=x_grid, y=y_grid, colorscale='Viridis', showscale=True))

        # Adding vectors to indicate the direction of optimization
        fig.add_trace(go.Scatter(x=X.flatten(), y=Y.flatten(),
                         mode='lines', line=dict(width=1, color='black'),
                         hoverinfo='none'))

        # Adding arrows to indicate the direction of optimization
        fig.add_trace(go.Scatter(x=X.flatten() + grad_X.flatten()*0.05,
                         y=Y.flatten() + grad_Y.flatten()*0.05,
                         mode='lines', line=dict(width=1, color='blue'),
                         hoverinfo='none'))

        # Setting up layout for better aesthetics
        fig.update_layout(title='Bayesian Optimization Results',
                  xaxis_title='Parameter X',
                  yaxis_title='Parameter Y',
                  autosize=False,
                  width=600,  # Increased width for better visibility
                  height=600,  # Increased height for better visibility
                  margin=dict(l=50, r=50, b=100, t=100, pad=4),
                  )

       # Display the figure
        fig.show()


        result = optimizer.max['params']

        # Prepare data for plotting

        # Parameter space plot
        plt.subplot(313)
        for i, params in enumerate(optimizer.space.target):
            plt.scatter(i+1, params, color='red' if i == 0 else 'blue', alpha=0.5)
        plt.xlabel("Iteration", fontsize=14)
        plt.ylabel("Parameter Values", fontsize=14)

        plt.tight_layout()
        plt.savefig("bayesian_optimization_results_advanced.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Preparing data for CSV
        data = [[key, result[key]] for key in result.keys()]

        # Saving to CSV
        with open('bayesian_optimization_results.csv', 'w', newline='') as csvfile:
              writer = csv.writer(csvfile)
              writer.writerow(['Parameter', 'Value'])  # Writing header
              writer.writerows(data)  # Writing data rows

        # Very Advanced Bayesian Optimization , pm.HalfCauchy , pm.HalfNormal

        with pm.Model() as model:
            # Hierarchical Priors for Shared Information Across Similar Parameters

            theta_prior = pm.Normal('theta', mu=0, sigma=1, shape=len(theta))
            alpha_prior = pm.Normal('alpha', mu=0, sigma=1, shape=len(alpha))
            beta_prior = pm.Normal('beta', mu=0, sigma=1, shape=len(beta))
            sigma_prior = pm.Normal('sigma', mu=0, sigma=1, shape=len(sigma))


            # Likelihood Function with Hierarchical Structure
            x_T = pm.Normal('x_T', mu=noise_dist(self.backdoor_trigger), sigma=1)


            # Create the causal graph after defining your model
            model_causal_graph = self.create_causal_graph(model)
            model_causal_graph.layout(prog='dot')

            # Save the graph
            model_causal_graph.write("causal_graph.dot")
            model_causal_graph.draw("causal_graph.png", format="png", prog="dot")


            # Bayesian Optimization Loop
            for t in range(T - 1, -1, -1):


                # Incorporate the transport component into the drift function
                transport_component = self.optimal_transport(x_T, t, theta_prior, beta_prior, sigma_prior)


                z = noise_dist(0) if t > 1 else 0
                x_t_minus_1 = pm.Normal(f'x_{t}', mu=drift_function(x_T, t, theta_prior, beta_prior, sigma_prior) + transport_component + sigma[t] * z, sigma=1)


                x_T = x_t_minus_1


            # Efficient Sampling with Adaptive Stepsize HMH

            step = pm.NUTS(target_accept=0.9)
            trace = pm.sample(2000, tune=1000, cores=4, chains=4, random_seed=0, model=model, step=step)


        # Plot the pairplot using ArviZ

        az.plot_pair(trace,
            kind='kde', marginals=True,
            colorbar=True,
            figsize=(15, 8))
        plt.savefig(f"{model_type}_pairplot.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Saving Trace Diagnostics to a CSV File
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_very_advanced_optimization_trace.csv")
        # Saving Trace Diagnostics to a CSV File
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_very_advanced_optimization_trace.csv")


        #  Plotting with ArviZ
        az.plot_trace(trace, figsize=(15, 8))
        plt.title(f"{model_type} Model Trace")
        plt.savefig(f"{model_type}_trace.png", dpi=300, bbox_inches="tight")
        plt.show()


        # Additional plot for posterior distribution
        az.plot_posterior(trace, var_names=['x_0'], ref_val = True, figsize=(15, 8))
        plt.title(f"{model_type} Posterior Distribution")
        plt.savefig(f"{model_type}_posterior.png", dpi=300, bbox_inches='tight')
        plt.show()


        # Save trace diagnostics to a CSV file
        trace_df = trace.to_dataframe()
        trace_df.to_csv(f"{model_type}_trace.csv")


        # Plot the trace using ArviZ
        az.plot_trace(trace.posterior)
        az.plot_trace(trace, figsize=(15, 6))
        plt.savefig("schocastic_Market.png", dpi=300, bbox_inches='tight')
        plt.show()
        az.summary(trace)

        # Summary Statistics
        summary_stats = az.summary(trace)
        print(summary_stats)

        return trace, model_type




```
https://matplotlib.org/stable/users/explain/colors/colors.html
```



In [ ]:
import matplotlib.pyplot as plt

import matplotlib.pyplot as plt
import matplotlib as mpl

# 1. Kill any zombie plots in the background
plt.close('all')

# 2. Force a consistent layout engine across the whole notebook
try:
    # Try the modern 3.6+ way
    mpl.rcParams['figure.layout_engine'] = 'tight'
except KeyError:
    try:
        # Fallback for older versions
        mpl.rcParams['figure.constrained_layout.use'] = True
    except KeyError:
        # If all else fails, just ignore layout management to stop the crashing
        pass

def dummy_layout(*args, **kwargs): pass
plt.tight_layout = dummy_layout
plt.constrained_layout = dummy_layout


# The "Nuclear" Option: override the tight_layout function so it can't conflict
plt.tight_layout = lambda *args, **kwargs: None

In [ ]:
# @title

"""

---------------
* Colorbars placed INSIDE the axes ([0.79, 0.02, 0.04, 0.96]) so their
  tick labels never bleed into neighbouring panels – the root cause of the
  text-chaos in rows 4-5.
* Every panel wrapped in try/except -> one bad panel never crashes the figure.
* Bolder line widths, brighter palette, higher-contrast grid.
* Snapshot panels: rms badge as a tight bbox; titles kept short.
* Section headers now span the full width as coloured fig.text banners.
* Effective-potential panel has a hard-coded Schwarzschild fallback so it
  never appears blank.
* All data access goes through _sget / _safe helpers (no raw dict indexing).
"""

from __future__ import annotations

import traceback
import warnings
from typing import Any, Dict, Optional, Tuple

import numpy as np
import matplotlib
import matplotlib.axes as maxes
import matplotlib.gridspec as gridspec
import matplotlib.pyplot as plt
from matplotlib.cm import ScalarMappable

warnings.filterwarnings("ignore", category=UserWarning, module="matplotlib")

C_L: float = 2.99792458e8          # speed of light [m/s]

# ══════════════════════════════════════════════════════════════════════════════
#  PALETTE
# ══════════════════════════════════════════════════════════════════════════════

BG_DEEP   = "#1a1f2e"
BG_PANEL  = "#0b1220"
BG_PANEL2 = "#0f1a2e"
GRID_COL  = "#1b2d44"
BORDER    = "#2a4870"

CLR_EU    = "#00e5ff"
CLR_NS    = "#ff4d6d"
CLR_BH    = "#ffb347"
CLR_KRET  = "#aaff00"
CLR_GW    = "#cc80ff"
CLR_HAWK  = "#ffa040"
CLR_EPS   = "#ff80ab"
CLR_RE    = "#64ffda"
CLR_PALIN = "#ea80fc"
CLR_HEL   = "#80d8ff"
CLR_ISCO  = "#76ff03"

TXT       = "#e8f4ff"
TXT_DIM   = "#6a8faf"
WHITE     = "#ffffff"

matplotlib.rcParams.update({
    "font.family"        : "monospace",
    "axes.unicode_minus" : False,
    "figure.dpi"         : 120,
})

# ══════════════════════════════════════════════════════════════════════════════
#  SAFE DATA HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _safe(arr: Any, fallback: Any = None) -> Any:
    try:
        a = np.asarray(arr, dtype=float)
        return a if (a.size > 0 and np.any(np.isfinite(a))) else fallback
    except Exception:
        return fallback


def _sget(m: Any, key: str, fallback: Any = None) -> Any:
    try:
        return m[key]
    except (KeyError, TypeError):
        pass
    try:
        return getattr(m, key, fallback)
    except Exception:
        return fallback


def _safe_max(arr: Any, fallback: float = 1e-10) -> float:
    try:
        v = float(np.nanmax(np.abs(np.asarray(arr, dtype=float))))
        return v if (np.isfinite(v) and v > 0) else fallback
    except Exception:
        return fallback


def _clamp(arr: np.ndarray, lo: float = -1e30, hi: float = 1e30) -> np.ndarray:
    return np.clip(
        np.nan_to_num(np.asarray(arr, dtype=float),
                      nan=0.0, posinf=hi, neginf=lo), lo, hi)


def _bh_attr(bh: Any, attr: str, fallback: float = 0.0) -> float:
    try:
        v = float(getattr(bh, attr))
        return v if np.isfinite(v) else fallback
    except Exception:
        return fallback

# ══════════════════════════════════════════════════════════════════════════════
#  MATPLOTLIB HELPERS
# ══════════════════════════════════════════════════════════════════════════════

def _ax_style(ax: plt.Axes, title: str,
              xlabel: str = "", ylabel: str = "",
              bg: str = BG_PANEL) -> None:
    ax.set_facecolor(bg)
    ax.tick_params(colors=TXT, labelsize=8, length=4, width=0.8)
    for sp in ax.spines.values():
        sp.set_edgecolor(BORDER)
        sp.set_linewidth(1.1)
    ax.xaxis.label.set_color(TXT)
    ax.yaxis.label.set_color(TXT)
    ax.set_title(title, color=WHITE, fontsize=9.5, pad=6,
                 fontfamily="monospace", fontweight="bold")
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=8, labelpad=3, color=TXT)
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=8, labelpad=3, color=TXT)
    ax.grid(color=GRID_COL, linestyle="-", linewidth=0.4, alpha=0.9)
    ax.set_axisbelow(True)


def _vline(ax: plt.Axes, x: float, color: str = WHITE,
           lw: float = 0.8, ls: str = "--",
           label: Optional[str] = None) -> None:
    if not np.isfinite(x):
        return
    kw: Dict[str, Any] = dict(color=color, lw=lw, ls=ls, alpha=0.85)
    if label:
        kw["label"] = label
    ax.axvline(x, **kw)


def _legend(ax: plt.Axes, **kw: Any) -> None:
    handles, _ = ax.get_legend_handles_labels()
    if not handles:
        return
    leg = ax.legend(fontsize=7.5, facecolor=BG_PANEL2, labelcolor=WHITE,
                    edgecolor=BORDER, framealpha=0.97, **kw)
    for ln in leg.get_lines():
        ln.set_linewidth(2.5)


def _colorbar(fig: plt.Figure, im: Any, ax: Any,
              label: str = "") -> Optional[Any]:
    """
    Inset colorbar placed INSIDE the axes (x = 79-83%) so tick labels
    never bleed into neighbouring panels.  Argument-order self-healing included.
    """
    try:
        if isinstance(im, maxes.Axes) and not isinstance(ax, maxes.Axes):
            im, ax = ax, im
        if not isinstance(ax, maxes.Axes) and hasattr(ax, "axes"):
            ax = ax.axes
        if not isinstance(ax, maxes.Axes):
            return None
        if isinstance(im, maxes.Axes):
            imgs = im.get_images()
            cols = getattr(im, "collections", [])
            im   = imgs[-1] if imgs else (cols[-1] if cols else None)
        if im is None or not isinstance(im, ScalarMappable):
            return None
        # *** INSIDE the axes -- no overflow into neighbouring panels ***
        cax = ax.inset_axes([0.79, 0.02, 0.04, 0.96])
        cax.set_facecolor(BG_PANEL)
        cb = fig.colorbar(im, cax=cax)
        cb.ax.tick_params(colors=TXT_DIM, labelsize=5, length=2, width=0.5)
        cb.outline.set_edgecolor(BORDER)
        cb.outline.set_linewidth(0.6)
        if label:
            cb.set_label(label, color=TXT_DIM, fontsize=5)
        return cb
    except Exception:
        traceback.print_exc()
        return None


def _panel_error(ax: plt.Axes, msg: str = "") -> None:
    # Silent fail: keep the panel as a clean dark box; log to terminal only.
    traceback.print_exc()
    ax.set_facecolor(BG_PANEL)
    ax.set_xticks([])
    ax.set_yticks([])


def _section_bar(fig: plt.Figure, y: float, label: str, color: str) -> None:
    fig.text(0.03, y, label, ha="left", va="center",
             color=color, fontsize=8, fontweight="bold",
             fontfamily="monospace")
    fig.add_artist(matplotlib.lines.Line2D(
        [0.03, 0.97], [y - 0.002, y - 0.002],
        transform=fig.transFigure,
        color=color, linewidth=0.8, alpha=0.6))

# ══════════════════════════════════════════════════════════════════════════════
#  SPECTRAL ANALYSIS
# ══════════════════════════════════════════════════════════════════════════════

def _energy_spectrum_2d(snap: np.ndarray) -> Tuple[np.ndarray, np.ndarray]:
    snap = np.asarray(snap, dtype=float)
    N = snap.shape[0]
    if N < 4:
        return np.array([1.0]), np.array([0.0])
    wh   = np.fft.fft2(snap)
    E2   = 0.5 * np.abs(wh) ** 2 / max(N ** 4, 1)
    kx   = np.fft.fftfreq(N, d=1.0 / N)
    ky   = np.fft.fftfreq(N, d=1.0 / N)
    KX, KY = np.meshgrid(kx, ky, indexing="ij")
    Kmag = np.round(np.sqrt(KX**2 + KY**2)).astype(int).ravel()
    k_max  = max(N // 3, 1)
    k_bins = np.arange(1, k_max + 1, dtype=float)
    Ek     = np.zeros(k_max, dtype=float)
    E2_r   = E2.ravel()
    for ki in range(1, k_max + 1):
        m = Kmag == ki
        if m.any():
            Ek[ki - 1] = float(E2_r[m].sum())
    return k_bins, np.nan_to_num(Ek, nan=0.0, posinf=0.0, neginf=0.0)

# ══════════════════════════════════════════════════════════════════════════════
#  MAIN  plot_all  --  24-panel figure
# ══════════════════════════════════════════════════════════════════════════════

def plot_all(sim: Any, save_path: Optional[str] = None) -> plt.Figure:

    eu  = _sget(sim, "_eu_data") or {}
    ns  = _sget(sim, "_ns_data") or {}
    geo = _sget(sim, "_geo_data") or {}
    pen = _sget(sim, "_pen_data") or {}
    bh  = _sget(sim, "bh")
    gw  = _sget(sim, "_gw_data") or {}

    def _bh(attr: str, fb: float = 0.0) -> float:
        return _bh_attr(bh, attr, fb)

    rp = max(_bh("r_plus", 1.0), 1e-30)
    th = np.linspace(0, 2 * np.pi, 400)

    # ── Figure ────────────────────────────────────────────────────────────────
    fig = plt.figure(figsize=(30, 40))
    fig.patch.set_facecolor(BG_DEEP)

    outer = gridspec.GridSpec(
        6, 4, figure=fig,
        hspace=0.56, wspace=0.42,
        left=0.055, right=0.975,
        top=0.960,  bottom=0.022,
    )

    # ── Master title ──────────────────────────────────────────────────────────
    fig.text(
        0.5, 0.978,
        "SINGULARITY SIMULATION  x  KERR BLACK HOLE"
        "  x  3-D EULER  x  3-D NAVIER-STOKES",
        ha="center", va="top", color=CLR_BH,
        fontsize=14, fontweight="bold", fontfamily="monospace")

    subtitle = (
        f"M = {_bh('M_solar'):.0f} Msun   "
        f"a* = {_bh('a_star'):.3f}   "
        f"r+ = {rp:.3e} m   "
        f"T_H = {_bh('T_H'):.3e} K   "
        f"S_BH = {_bh('S_BH'):.3e} nat   "
        f"eta_Penrose = {_bh('efficiency')*100:.2f}%"
    )
    fig.text(0.5, 0.970, subtitle, ha="center", va="top",
             color=TXT_DIM, fontsize=8, fontfamily="monospace")

    _section_bar(fig, 0.880,
                 "--- GR / KERR BLACK HOLE -----------------------------------------------------------",
                 CLR_BH)
    _section_bar(fig, 0.715,
                 "--- GRAVITATIONAL WAVES  .  HAWKING SPECTRUM --------------------------------------",
                 CLR_GW)
    _section_bar(fig, 0.548,
                 "--- FLUID DYNAMICS  .  GLOBAL DIAGNOSTICS -----------------------------------------",
                 CLR_EU)
    _section_bar(fig, 0.382,
                 "--- ENERGY SPECTRUM  .  DISSIPATION  .  Re_lambda ---------------------------------",
                 CLR_NS)
    _section_bar(fig, 0.218,
                 "--- VORTICITY SNAPSHOTS  .  EULER --------------------------------------------------",
                 CLR_EU)
    _section_bar(fig, 0.056,
                 "--- VORTICITY SNAPSHOTS  .  NAVIER-STOKES  .  STRUCTURE FUNCTIONS -----------------",
                 CLR_NS)

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 0 -- Kerr GR
    # ══════════════════════════════════════════════════════════════════════════

    # 0-0  Geodesic -----------------------------------------------------------
    ax00 = fig.add_subplot(outer[0, 0])
    try:
        _ax_style(ax00, "KERR GEODESIC", "x / r+", "y / r+")
        gx = _safe(_sget(geo, "x"))
        gy = _safe(_sget(geo, "y"))
        if gx is not None:
            ax00.plot(gx / rp, gy / rp,
                      color=CLR_BH, lw=1.4, label="Geodesic", zorder=3)
        ax00.fill(np.cos(th), np.sin(th), color="#00000f", zorder=4)
        ax00.plot(np.cos(th), np.sin(th), color=CLR_BH, lw=1.2, zorder=5,
                  label=f"r+ = {rp:.2e} m")
        for r_attr, col, ls, lbl in [
            ("r_isco",    CLR_ISCO, "--", f"ISCO ({_bh('r_isco')/rp:.1f}r+)"),
            ("r_photo",   CLR_NS,   ":",  f"photon orbit"),
            ("r_ergo_eq", CLR_GW,   "-.", "Ergosphere"),
        ]:
            rv = _bh(r_attr)
            if rv > 0:
                rn = rv / rp
                ax00.plot(rn * np.cos(th), rn * np.sin(th),
                          color=col, lw=0.9, ls=ls, label=lbl, alpha=0.9)
        if _sget(geo, "terminated") and gx is not None:
            ax00.plot(gx[-1] / rp, gy[-1] / rp,
                      "x", color=CLR_NS, ms=6, zorder=6, label="-> horizon")
        ax00.set_aspect("equal")
        ax00.set_xlim(-12, 12)
        ax00.set_ylim(-12, 12)
        _legend(ax00, loc="upper right")
    except Exception:
        _panel_error(ax00, traceback.format_exc(limit=2))

    # 0-1  Effective potential ------------------------------------------------
    ax01 = fig.add_subplot(outer[0, 1])
    try:
        _ax_style(ax01, "EFFECTIVE POTENTIAL  V_eff(r)", "r / r+", "V_eff (norm.)")
        r_arr   = np.linspace(1.3 * rp, 22 * rp, 800)
        plotted = False
        for L_fac, col in [(2.0, CLR_NS), (3.5, CLR_BH), (5.5, CLR_EU)]:
            try:
                L_ = L_fac * rp * C_L
                V  = np.asarray(
                    bh.effective_potential(r_arr, L=L_, E=0.95, mu=1.0),
                    dtype=float)
                V  = _clamp(V)
                mn = _safe_max(V, 1.0)
                ax01.plot(r_arr / rp, V / mn, color=col, lw=1.6,
                          label=f"L = {L_fac:.1f} r+c")
                plotted = True
            except Exception:
                pass
        if not plotted:
            # Schwarzschild fallback: V_eff = 1 - r_s/r + L^2/r^2 - L^2*r_s/r^3
            r_s = _bh("r_s", rp * 2)
            for L_fac, col in [(2.0, CLR_NS), (3.5, CLR_BH), (5.5, CLR_EU)]:
                L_ = L_fac * rp
                V  = (1 - r_s / r_arr
                      + L_**2 / r_arr**2
                      - L_**2 * r_s / r_arr**3)
                mn = _safe_max(V, 1.0)
                ax01.plot(r_arr / rp, V / mn, color=col, lw=1.6,
                          label=f"L = {L_fac:.1f} r+c (Schw.)")
        _vline(ax01, 1.0, color=CLR_BH, lw=1.2, label="horizon r+")
        ri = _bh("r_isco") / rp
        if 0 < ri < 25:
            _vline(ax01, ri, color=CLR_ISCO, ls=":", lw=1.0, label="ISCO")
        ax01.set_ylim(-2.2, 2.2)
        ax01.axhline(0, color=BORDER, lw=0.6)
        _legend(ax01, loc="upper right")
    except Exception:
        _panel_error(ax01, traceback.format_exc(limit=2))

    # 0-2  Kretschner curvature -----------------------------------------------
    ax02 = fig.add_subplot(outer[0, 2])
    try:
        _ax_style(ax02, "KRETSCHNER SCALAR  K(r)", "r / r+", "K  [m^-4]")
        r_k = np.linspace(1.02 * rp, 12 * rp, 700)
        K   = np.abs(_clamp(
            np.asarray(bh.kretschner_scalar(r_k), dtype=float),
            lo=0.0, hi=1e300))
        pos = K > 0
        if pos.any():
            ax02.semilogy(r_k[pos] / rp, K[pos], color=CLR_KRET, lw=1.8)
            ax02.fill_between(r_k[pos] / rp, K[pos], K[pos].min(),
                              alpha=0.18, color=CLR_KRET)
        _vline(ax02, 1.0, color=CLR_BH, lw=1.2, label="r+")
        ri = _bh("r_isco") / rp
        if 0 < ri < 25:
            _vline(ax02, ri, color=CLR_ISCO, ls=":", lw=1.0, label="ISCO")
        ax02.set_xlim(0.9, 12)
        K_lo = K[pos].min() if pos.any() else 1e-10
        K_hi = K[pos].max() if pos.any() else 1e-9
        ax02.fill_betweenx([K_lo, K_hi], 0, 1, alpha=0.10, color=CLR_BH)
        ax02.text(0.97, 0.93, "inf singularity\n  at r = 0",
                  transform=ax02.transAxes, ha="right", va="top",
                  color=CLR_KRET, fontsize=7, fontfamily="monospace")
        _legend(ax02)
    except Exception:
        _panel_error(ax02, traceback.format_exc(limit=2))

    # 0-3  Penrose / Kruskal-Szekeres -----------------------------------------
    ax03 = fig.add_subplot(outer[0, 3])
    try:
        _ax_style(ax03, "PENROSE  .  KRUSKAL-SZEKERES", "X", "T")
        r_c       = _safe(_sget(pen, "r_arr"), np.linspace(1, 10, 100))
        metric_tt = _safe(_sget(pen, "metric_tt"))
        if metric_tt is not None:
            n    = len(r_c)
            bg2d = _clamp(np.tile(metric_tt, (n, 1)), lo=-1.0, hi=0.0)
            im03 = ax03.imshow(bg2d, origin="lower",
                               extent=[-1.1, 1.1, -1.3, 1.3],
                               aspect="auto", cmap="RdYlBu_r",
                               alpha=0.6, vmin=-1.0, vmax=0.0)
            _colorbar(fig, im03, ax03, "g_tt")
        X_r = _safe(_sget(pen, "X_krus"))
        T_r = _safe(_sget(pen, "T_krus"))
        if X_r is not None and T_r is not None:
            fin = np.isfinite(X_r) & np.isfinite(T_r)
            if fin.any():
                sx = _safe_max(X_r[fin], 1.0)
                st = _safe_max(T_r[fin], 1.0)
                ax03.plot(X_r[fin] / sx, T_r[fin] / st * 1.2,
                          color=CLR_BH, lw=1.4, label="r = const.")
        xi = np.linspace(-1.0, 1.0, 120)
        ax03.plot( xi,  xi, color=WHITE, lw=0.8, ls="--", alpha=0.5,
                   label="null ray")
        ax03.plot( xi, -xi, color=WHITE, lw=0.8, ls="--", alpha=0.5)
        ax03.plot([-1.1, 0], [-1.1, 0], color=CLR_BH, lw=1.2, alpha=0.7)
        ax03.plot([0,  1.1], [0,  1.1], color=CLR_BH, lw=1.2, alpha=0.7)
        ax03.set_xlim(-1.1, 1.1)
        ax03.set_ylim(-1.3, 1.3)
        ax03.axhline(0, color=BORDER, lw=0.5)
        ax03.axvline(0, color=BORDER, lw=0.5)
        ax03.text(0.5, 0.74, "BH INTERIOR", color=CLR_NS, fontsize=6.5,
                  ha="center", transform=ax03.transAxes,
                  fontfamily="monospace", fontweight="bold")
        ax03.text(0.5, 0.22, "EXTERIOR", color=CLR_EU, fontsize=6.5,
                  ha="center", transform=ax03.transAxes,
                  fontfamily="monospace", fontweight="bold")
        _legend(ax03, loc="lower right")
    except Exception:
        _panel_error(ax03, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 1 -- Hawking + GW + parameter table
    # ══════════════════════════════════════════════════════════════════════════

    # 1-0  Hawking spectrum ---------------------------------------------------
    ax10 = fig.add_subplot(outer[1, 0])
    try:
        _ax_style(ax10, "HAWKING RADIATION SPECTRUM", "nu  [Hz]", "dP/dnu  [W/Hz]")
        hs = {}
        try:
            hs = bh.hawking_spectrum() or {}
        except Exception:
            pass
        nu    = _safe(_sget(hs, "nu"))
        dPdnu = _safe(_sget(hs, "dPdnu"))
        nu_pk = float(_sget(hs, "nu_peak", 0.0))
        if nu is not None and dPdnu is not None and nu_pk > 0:
            dp = np.maximum(dPdnu, 1e-400)
            ax10.semilogy(nu, dp, color=CLR_HAWK, lw=1.8, label="dP/dnu")
            ax10.fill_between(nu, dp, 1e-400, alpha=0.20, color=CLR_HAWK)
            _vline(ax10, nu_pk, color=WHITE, ls=":", lw=1.0, label="nu_peak")
            ax10.text(0.97, 0.93,
                      f"T_H = {_bh('T_H'):.2e} K\n"
                      f"P_tot = {float(_sget(hs,'total_power',0)):.2e} W",
                      transform=ax10.transAxes, ha="right", va="top",
                      color=CLR_HAWK, fontsize=7, fontfamily="monospace")
        else:
            ax10.text(0.5, 0.5,
                      "T_H ~ 0  (stellar BH)\nHawking flux negligible",
                      ha="center", va="center", transform=ax10.transAxes,
                      color=TXT_DIM, fontsize=8, fontfamily="monospace")
        _legend(ax10)
    except Exception:
        _panel_error(ax10, traceback.format_exc(limit=2))

    # 1-1  GW chirp strain ----------------------------------------------------
    ax11 = fig.add_subplot(outer[1, 1])
    try:
        _ax_style(ax11, "GW CHIRP STRAIN  h+(t)", "t  [s]", "h+")
        t_gw = _safe(_sget(gw, "t"))
        hp   = _safe(_sget(gw, "h_plus"))
        hx   = _safe(_sget(gw, "h_cross"))
        if t_gw is not None and hp is not None:
            fin = np.isfinite(hp)
            if fin.any():
                ax11.plot(t_gw[fin], hp[fin], color=CLR_GW, lw=1.4,
                          label="h+", alpha=0.95)
                if hx is not None:
                    fx = np.isfinite(hx)
                    if fx.any():
                        ax11.plot(t_gw[fx], hx[fx], color=CLR_EU, lw=0.9,
                                  ls="--", label="hx", alpha=0.75)
                ax11.set_xlim(t_gw[0], t_gw[fin].max())
        ax11.text(0.03, 0.93,
                  f"Mc = {float(_sget(gw,'Mc_solar',0)):.3f} Msun\n"
                  f"T_chirp = {float(_sget(gw,'T_chirp',0)):.2f} s",
                  transform=ax11.transAxes, va="top",
                  color=CLR_GW, fontsize=7, fontfamily="monospace")
        _legend(ax11)
    except Exception:
        _panel_error(ax11, traceback.format_exc(limit=2))

    # 1-2  GW chirp frequency -------------------------------------------------
    ax12 = fig.add_subplot(outer[1, 2])
    try:
        _ax_style(ax12, "GW CHIRP FREQUENCY  f(t)", "t  [s]", "f  [Hz]")
        t_gw = _safe(_sget(gw, "t"))
        f_t  = _safe(_sget(gw, "f_t"))
        if t_gw is not None and f_t is not None:
            fin = np.isfinite(f_t) & (f_t > 0)
            if fin.any():
                f_min = max(f_t[fin].min() * 0.9, 1e-10)
                ax12.semilogy(t_gw[fin], f_t[fin],
                              color=CLR_GW, lw=1.8, label="f(t)")
                ax12.fill_between(t_gw[fin], f_t[fin], f_min,
                                  alpha=0.20, color=CLR_GW)
                ax12.set_xlim(t_gw[0], t_gw[fin].max())
        ax12.axhline(100, color=TXT_DIM, lw=0.7, ls=":", label="100 Hz")
        _legend(ax12)
    except Exception:
        _panel_error(ax12, traceback.format_exc(limit=2))

    # 1-3  Parameter table ----------------------------------------------------
    ax13 = fig.add_subplot(outer[1, 3])
    try:
        ax13.set_facecolor(BG_PANEL2)
        for sp in ax13.spines.values():
            sp.set_edgecolor(CLR_BH)
            sp.set_linewidth(1.4)
        ax13.set_xticks([])
        ax13.set_yticks([])
        ax13.set_title("KERR BH  PARAMETER TABLE", color=CLR_BH,
                       fontsize=8, pad=5, fontfamily="monospace",
                       fontweight="bold")
        try:
            spag_s = f"{float(bh.spaghettification_radius()):.3e} m"
        except Exception:
            spag_s = "N/A"
        rows = [
            ("Mass",              f"{_bh('M_solar'):.1f} Msun"),
            ("Spin  a*",          f"{_bh('a_star'):.4f}"),
            ("Outer horizon r+",  f"{rp:.4e} m"),
            ("Inner horizon r-",  f"{_bh('r_minus'):.4e} m"),
            ("Schwarzschild r_s", f"{_bh('r_s'):.4e} m"),
            ("ISCO (prograde)",   f"{_bh('r_isco'):.4e} m"),
            ("Photon orbit",      f"{_bh('r_photo'):.4e} m"),
            ("Omega_H (horizon)", f"{_bh('Omega_H'):.4e} rad/s"),
            ("Hawking  T_H",      f"{_bh('T_H'):.4e} K"),
            ("Bekenstein S_BH",   f"{_bh('S_BH'):.4e} nat"),
            ("Penrose eta_max",   f"{_bh('efficiency')*100:.3f} %"),
            ("GW chirp mass Mc",  f"{float(_sget(gw,'Mc_solar',0)):.4f} Msun"),
            ("Spaghetti r",       spag_s),
        ]
        y0, dy = 0.93, 0.068
        for i, (k, v) in enumerate(rows):
            y = y0 - i * dy
            ax13.text(0.04, y, k, transform=ax13.transAxes, va="top",
                      color=TXT_DIM, fontsize=7, fontfamily="monospace")
            ax13.text(0.96, y, v, transform=ax13.transAxes, va="top",
                      ha="right", color=CLR_BH, fontsize=7,
                      fontfamily="monospace", fontweight="bold")
            if i < len(rows) - 1:
                ax13.axhline(y - dy * 0.42,
                             color=GRID_COL, lw=0.35, xmin=0.02, xmax=0.98)
    except Exception:
        _panel_error(ax13, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 2 -- Fluid global diagnostics
    # ══════════════════════════════════════════════════════════════════════════

    # 2-0  Kinetic energy -----------------------------------------------------
    ax20 = fig.add_subplot(outer[2, 0])
    try:
        _ax_style(ax20, "KINETIC ENERGY  E(t)", "tau", "E = 1/2 <|u|^2>")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            v = _safe(_sget(data, "E"))
            if t is not None and v is not None:
                ax20.plot(t, v, color=col, lw=1.6, ls=ls, label=lbl)
                ax20.fill_between(t, v, alpha=0.09, color=col)
        _legend(ax20)
    except Exception:
        _panel_error(ax20, traceback.format_exc(limit=2))

    # 2-1  Enstrophy ----------------------------------------------------------
    ax21 = fig.add_subplot(outer[2, 1])
    try:
        _ax_style(ax21, "ENSTROPHY  Omega(t)", "tau", "Omega")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            v = _safe(_sget(data, "W"))
            if t is not None and v is not None:
                ax21.semilogy(t, np.maximum(v, 1e-30),
                              color=col, lw=1.6, ls=ls, label=lbl)
        _legend(ax21)
    except Exception:
        _panel_error(ax21, traceback.format_exc(limit=2))

    # 2-2  Helicity -----------------------------------------------------------
    ax22 = fig.add_subplot(outer[2, 2])
    try:
        _ax_style(ax22, "HELICITY  H(t) = int u.omega dV", "tau", "H")
        for data, col, ls, lbl in [
            (eu, CLR_HEL, "-",  "Euler"),
            (ns, CLR_EPS, "--", "NS"),
        ]:
            t = _safe(_sget(data, "t"))
            H = _safe(_sget(data, "H"))
            if t is not None and H is not None:
                ax22.plot(t, H, color=col, lw=1.6, ls=ls, label=lbl)
        ax22.axhline(0, color=BORDER, lw=0.6)
        _legend(ax22)
    except Exception:
        _panel_error(ax22, traceback.format_exc(limit=2))

    # 2-3  BKM integral -------------------------------------------------------
    ax23 = fig.add_subplot(outer[2, 3])
    try:
        _ax_style(ax23, "BKM INTEGRAL  int ||omega||_inf dtau",
                  "tau", "int ||omega||_inf dtau")
        bkm = {}
        try:
            bkm = sim.bkm_criterion() or {}
        except Exception:
            pass
        for key_t, key_v, col, ls, lbl in [
            ("t_euler", "euler", CLR_EU, "-",  "Euler"),
            ("t_ns",    "ns",    CLR_NS, "--", "NS"),
        ]:
            t = _safe(_sget(bkm, key_t))
            v = _safe(_sget(bkm, key_v))
            if t is not None and v is not None:
                ax23.plot(t, v, color=col, lw=1.6, ls=ls, label=lbl)
                ax23.fill_between(t, v, alpha=0.12, color=col)
        ax23.text(0.03, 0.93,
                  "Blowup only if integral -> inf\n(Beale-Kato-Majda 1984)",
                  transform=ax23.transAxes, va="top",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax23)
    except Exception:
        _panel_error(ax23, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 3 -- Spectrum, max vorticity, dissipation, Re_lambda
    # ══════════════════════════════════════════════════════════════════════════

    # 3-0  Energy spectrum ----------------------------------------------------
    ax30 = fig.add_subplot(outer[3, 0])
    try:
        _ax_style(ax30, "KINETIC ENERGY SPECTRUM  E(k)",
                  "k  [wavenumber]", "E(k)")
        eu_snaps = sorted((_sget(eu, "snapshots") or {}).items())
        ns_snaps = sorted((_sget(ns, "snapshots") or {}).items())
        ref_k = ref_E = None
        for lbl_, snaps_, col_ in [("Euler", eu_snaps, CLR_EU),
                                    ("NS",    ns_snaps, CLR_NS)]:
            if not snaps_:
                continue
            try:
                k_b, Ek = _energy_spectrum_2d(snaps_[-1][1])
                v = (Ek > 0) & np.isfinite(Ek)
                if v.any():
                    ax30.loglog(k_b[v], Ek[v], color=col_, lw=1.8, label=lbl_)
                    if lbl_ == "Euler":
                        ref_k, ref_E = k_b, Ek
            except Exception:
                pass
        if ref_k is not None and ref_E is not None:
            vv = ref_E > 0
            if vv.sum() > 3:
                idx = vv.nonzero()[0][len(vv.nonzero()[0]) // 3]
                E0 = ref_E[idx]
                k0 = ref_k[idx]
                kr = np.logspace(np.log10(ref_k[vv][0]),
                                 np.log10(ref_k[vv][-1]), 50)
                ax30.loglog(kr, E0 * (kr / k0) ** (-5/3),
                            color=TXT_DIM, lw=1.0, ls="--",
                            label="k^(-5/3) K41")
        _legend(ax30)
    except Exception:
        _panel_error(ax30, traceback.format_exc(limit=2))

    # 3-1  Max vorticity ------------------------------------------------------
    ax31 = fig.add_subplot(outer[3, 1])
    try:
        _ax_style(ax31, "MAX VORTICITY  ||omega||_inf(t)",
                  "tau", "||omega||_inf")
        for data, col, ls, lbl in [
            (eu, CLR_EU, "-",  "Euler"),
            (ns, CLR_NS, "--", "NS"),
        ]:
            t  = _safe(_sget(data, "t"))
            om = _safe(_sget(data, "omMax"))
            if t is not None and om is not None:
                ax31.plot(t, om, color=col, lw=1.6, ls=ls, label=lbl)
        _legend(ax31)
    except Exception:
        _panel_error(ax31, traceback.format_exc(limit=2))

    # 3-2  Dissipation & palinstrophy -----------------------------------------
    ax32 = fig.add_subplot(outer[3, 2])
    try:
        _ax_style(ax32, "DISSIPATION eps(t)  .  PALINSTROPHY P(t)",
                  "tau", "eps")
        ax32r = ax32.twinx()
        ax32r.tick_params(colors=TXT_DIM, labelsize=7)
        ax32r.yaxis.label.set_color(CLR_PALIN)
        for sp in ax32r.spines.values():
            sp.set_edgecolor(BORDER)
        t_ns = _safe(_sget(ns, "t"))
        eps  = _safe(_sget(ns, "eps"))
        if t_ns is not None and eps is not None:
            ax32.plot(t_ns, np.maximum(eps, 1e-30),
                      color=CLR_EPS, lw=1.6, label="eps (NS)")
        for data, ls, alpha in [(eu, "--", 0.85), (ns, "-", 0.55)]:
            t = _safe(_sget(data, "t"))
            p = _safe(_sget(data, "Palin"))
            if t is not None and p is not None:
                ax32r.semilogy(t, np.maximum(p, 1e-30),
                               color=CLR_PALIN, lw=1.2, ls=ls, alpha=alpha)
        ax32r.set_ylabel("P(t)  [log]", fontsize=7, color=CLR_PALIN)
        ax32.text(0.04, 0.93, "solid = eps   dashed = P(t)",
                  transform=ax32.transAxes, va="top",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax32)
    except Exception:
        _panel_error(ax32, traceback.format_exc(limit=2))

    # 3-3  Taylor Reynolds ----------------------------------------------------
    ax33 = fig.add_subplot(outer[3, 3])
    try:
        _ax_style(ax33, "TAYLOR REYNOLDS  Re_lambda(t)  .  NS",
                  "tau", "Re_lambda")
        t_ns  = _safe(_sget(ns, "t"))
        Re_la = _safe(_sget(ns, "Re_lam"))
        if t_ns is not None and Re_la is not None:
            ax33.plot(t_ns, Re_la, color=CLR_RE, lw=1.8, label="Re_lambda")
            ax33.fill_between(t_ns, Re_la, alpha=0.14, color=CLR_RE)
        kol = _sget(ns, "kolmogorov") or {}
        if not kol:
            try:
                kol = sim.ns.kolmogorov_scales() or {}
            except Exception:
                kol = {}
        if kol:
            ax33.text(0.97, 0.95,
                      f"eta   = {kol.get('eta',0):.4f}\n"
                      f"tau_eta = {kol.get('tau_eta',0):.4f}\n"
                      f"v_eta = {kol.get('v_eta',0):.4f}\n"
                      f"<Re_lam> = {kol.get('Re_lambda_mean',0):.1f}",
                      transform=ax33.transAxes, va="top", ha="right",
                      color=CLR_RE, fontsize=6.8, fontfamily="monospace")
        _legend(ax33)
    except Exception:
        _panel_error(ax33, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 4 -- Euler vorticity snapshots + cross-comparison
    # ══════════════════════════════════════════════════════════════════════════

    eu_snaps_d = _sget(eu, "snapshots") or {}
    eu_times   = sorted(eu_snaps_d.keys())[:3]

    for ci, t_snap in enumerate(eu_times):
        ax4 = fig.add_subplot(outer[4, ci])
        try:
            snap = np.asarray(eu_snaps_d[t_snap], dtype=float)
            vmax = _safe_max(snap, 1e-10)
            im4  = ax4.imshow(snap.T, origin="lower", cmap="seismic",
                              vmin=-vmax, vmax=vmax, aspect="auto",
                              extent=[0, 2*np.pi, 0, 2*np.pi])
            _ax_style(ax4, f"EULER wz  tau={t_snap:.3f}", "x", "y")
            _colorbar(fig, im4, ax4)
            rms = float(np.sqrt(np.mean(snap**2)))
            ax4.text(0.02, 0.96, f"rms={rms:.3f}",
                     transform=ax4.transAxes, va="top",
                     color=CLR_EU, fontsize=6.5, fontfamily="monospace",
                     bbox=dict(facecolor=BG_PANEL, edgecolor="none",
                               alpha=0.75, pad=1.5))
        except Exception:
            _panel_error(ax4, traceback.format_exc(limit=2))

    for ci in range(len(eu_times), 3):
        _ax_style(fig.add_subplot(outer[4, ci]),
                  "EULER wz  (no snapshot)", "x", "y")

    # 4-3  Cross-comparison ---------------------------------------------------
    ax43 = fig.add_subplot(outer[4, 3])
    try:
        _ax_style(ax43, "SINGULARITY STRENGTH (norm.)", "", "value (norm.)")
        cross = {}
        try:
            cross = sim.compare_singularities() or {}
        except Exception:
            pass
        ax43_top = ax43.twiny()
        ax43_top.tick_params(colors=CLR_BH, labelsize=7)
        ax43_top.set_xlabel("r / r+  (GR)", fontsize=7,
                            color=CLR_BH, labelpad=2)
        r_n = _safe(_sget(cross, "r_over_rplus"))
        K_n = _safe(_sget(cross, "K_norm"))
        if r_n is not None and K_n is not None and (K_n > 0).any():
            ax43_top.semilogy(r_n, np.maximum(K_n, 1e-30),
                              color=CLR_BH, lw=1.8, label="K(r)/K_ref  [GR]")
            ax43_top.set_xlim(r_n[0], r_n[-1])
        for kt, kv, col, ls, lbl in [
            ("t_euler", "euler_bkm", CLR_EU, "-",  "BKM Euler"),
            ("t_ns",    "ns_bkm",   CLR_NS, "--", "BKM NS"),
        ]:
            t = _safe(_sget(cross, kt))
            v = _safe(_sget(cross, kv))
            if t is not None and v is not None:
                tm = float(t.max()) if t.max() > 0 else 1.0
                ax43.plot(t / tm, v, color=col, lw=1.8, ls=ls, label=lbl)
        ax43.set_xlabel("tau / tau_max  (fluid)",
                        fontsize=7, color=TXT_DIM, labelpad=2)
        ax43.text(0.03, 0.05,
                  "K(r)->inf as r->0  [GR]\nint||omega||->inf => blowup [BKM]",
                  transform=ax43.transAxes, va="bottom",
                  color=TXT_DIM, fontsize=6.5, fontfamily="monospace")
        _legend(ax43, loc="upper right")
    except Exception:
        _panel_error(ax43, traceback.format_exc(limit=2))

    # ══════════════════════════════════════════════════════════════════════════
    # ROW 5 -- NS vorticity snapshots + structure functions
    # ══════════════════════════════════════════════════════════════════════════

    ns_snaps_d = _sget(ns, "snapshots") or {}
    ns_times   = sorted(ns_snaps_d.keys())[:3]

    for ci, t_snap in enumerate(ns_times):
        ax5 = fig.add_subplot(outer[5, ci])
        try:
            snap = np.asarray(ns_snaps_d[t_snap], dtype=float)
            vmax = _safe_max(np.abs(snap), 1e-10)
            im5  = ax5.imshow(np.abs(snap.T), origin="lower", cmap="inferno",
                              vmin=0, vmax=vmax, aspect="auto",
                              extent=[0, 2*np.pi, 0, 2*np.pi])
            _ax_style(ax5, f"NS |wz|  tau={t_snap:.3f}", "x", "y")
            _colorbar(fig, im5, ax5)
            rms = float(np.sqrt(np.mean(snap**2)))
            ax5.text(0.02, 0.96, f"rms={rms:.3f}",
                     transform=ax5.transAxes, va="top",
                     color=CLR_NS, fontsize=6.5, fontfamily="monospace",
                     bbox=dict(facecolor=BG_PANEL, edgecolor="none",
                               alpha=0.75, pad=1.5))
        except Exception:
            _panel_error(ax5, traceback.format_exc(limit=2))

    for ci in range(len(ns_times), 3):
        _ax_style(fig.add_subplot(outer[5, ci]),
                  "NS |wz|  (no snapshot)", "x", "y")

    # 5-3  Structure functions ------------------------------------------------
    ax53 = fig.add_subplot(outer[5, 3])
    try:
        _ax_style(ax53, "STRUCTURE FUNCTIONS  S_p(r)",
                  "r  [code units]", "S_p(r)")
        if ns_times:
            ns_last = np.asarray(ns_snaps_d[ns_times[-1]], dtype=float)
            N_s     = ns_last.shape[0]
            r_sh    = np.linspace(1, max(N_s // 2, 2), 15).astype(int)
            r_ph    = r_sh * (2 * np.pi / max(N_s, 1))
            for p, col in zip([2, 4, 6], [CLR_EU, CLR_BH, CLR_NS]):
                Sp = np.array([
                    float(np.mean(
                        np.abs(np.roll(ns_last, int(r), axis=0) - ns_last) ** p))
                    for r in r_sh])
                v = (Sp > 0) & np.isfinite(Sp)
                if v.any():
                    ax53.loglog(r_ph[v], Sp[v], color=col, lw=1.6,
                                marker=".", ms=4, label=f"S_{p}(r)")
            if len(r_ph) >= 6:
                rr = np.logspace(np.log10(r_ph[1]),
                                 np.log10(r_ph[-1]), 30)
                ax53.loglog(rr, 0.3 * (rr / r_ph[5]) ** (2/3),
                            color=TXT_DIM, lw=1.0, ls=":",
                            label="r^(2/3)  K41")
        kol_f = _sget(ns, "kolmogorov") or {}
        if not kol_f:
            try:
                kol_f = sim.ns.kolmogorov_scales() or {}
            except Exception:
                kol_f = {}
        if kol_f:
            ax53.text(0.97, 0.05,
                      "Kolmogorov scales\n"
                      f" eta  = {kol_f.get('eta',0):.4f}\n"
                      f" tau_eta = {kol_f.get('tau_eta',0):.4f}\n"
                      f" v_eta = {kol_f.get('v_eta',0):.4f}\n"
                      f" <eps> = {kol_f.get('eps_mean',0):.3e}\n"
                      f" <Re_lam> = {kol_f.get('Re_lambda_mean',0):.1f}",
                      transform=ax53.transAxes, va="bottom", ha="right",
                      color=CLR_RE, fontsize=6.5, fontfamily="monospace",
                      bbox=dict(facecolor=BG_PANEL2, edgecolor=BORDER,
                                boxstyle="round,pad=0.4", alpha=0.92))
        _legend(ax53, loc="upper left")
    except Exception:
        _panel_error(ax53, traceback.format_exc(limit=2))

    # ── Footer ────────────────────────────────────────────────────────────────
    try:
        eu_obj = getattr(sim, "eu", None)
        ns_obj = getattr(sim, "ns", None)
        footer = (
            f"BH: {getattr(bh,'_label','Kerr')}   "
            f"Euler: N={getattr(eu_obj,'N','?')}   "
            f"NS: N={getattr(ns_obj,'N','?')}  "
            f"nu={getattr(ns_obj,'nu',0):.1e}   "
            f"steps={getattr(sim,'n_steps','?')}   "
            "[Kerr GR  .  BKM  .  Kolmogorov  .  ETD-RK4  .  adaptive CFL]"
        )
    except Exception:
        footer = "[SingularitySimulation]"
    fig.text(0.5, 0.010, footer, ha="center", va="bottom",
             color=TXT_DIM, fontsize=6.5, fontfamily="monospace")

    # ── Save -----------------------------------------------------------------
    if save_path:
        try:
            plt.savefig(save_path, dpi=150, bbox_inches="tight",
                        facecolor=fig.get_facecolor())
            print(f"[OK] Saved -> {save_path}")
        except Exception as exc:
            print(f"[WARN] Save failed: {exc}")

    return fig



In [ ]:
import matplotlib as mpl
import matplotlib.pyplot as plt


#def _panel_error(ax, msg=""):
  #  ax.set_facecolor(BG_PANEL); ax.set_xticks([]); ax.set_yticks([])

# Use a known baseline style and ignore external style files
mpl.rcParams.update(mpl.rcParamsDefault)   # reset to library defaults
mpl.rcParams['axes.prop_cycle'] = mpl.cycler(color=[])  # disable auto color cycle


matplotlib.rcdefaults()   # wipe any external style contamination
matplotlib.rcParams.update({
    "font.family"        : "monospace",
    "axes.unicode_minus" : False,
    "figure.dpi"         : 120,
    "axes.prop_cycle"    : matplotlib.cycler(color=[
        CLR_EU, CLR_NS, CLR_BH, CLR_GW,
        CLR_HAWK, CLR_RE, CLR_KRET, CLR_HEL,
    ]),
})




matplotlib.rcParams.update({
            "font.family"     : "monospace",
            "axes.unicode_minus": False,
            "figure.dpi"      : 120,
        })

import numpy as np
import warnings

# Only do this once you've confirmed values are reasonable:
warnings.filterwarnings("ignore", category=RuntimeWarning, message="overflow")

#last
import gymnasium as gym
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from collections import deque
from typing import List, Tuple, Dict

from gymnasium import Space
import logging


logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

#import gym
import numpy as np
from typing import Dict, Tuple, List
import logging
from collections import deque
import random


from stable_baselines3 import DQN
from stable_baselines3 import PPO
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3 import A2C
from stable_baselines3 import DDPG
from stable_baselines3 import SAC

from stable_baselines3 import TD3

from time import sleep
import time
from IPython.display import display, clear_output

# Define the target label

target_label = np.array('3')
target_label = np.expand_dims(target_label, axis=0)
from scipy.stats import norm
from tqdm import tqdm


import matplotlib.animation as animation


class ReinforcementLearningTrigger(gym.Env):
    def __init__(self, num_agents: int = 1, sampling_rate: int = 16000, imperceptibility: float = 0.000001):
        super().__init__()
        self.num_agents = num_agents
        self.sampling_rate = sampling_rate
        self.imperceptibility = imperceptibility

        self.state_dim = 3  # Current state, target state, and action
        self.action_space = gym.spaces.Discrete(2)  # 0: No trigger, 1: Trigger

        self.observation_space = gym.spaces.Box(low=0, high=sampling_rate, shape=(self.state_dim,), dtype=np.float32)

        self.q_tables: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.target_networks: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.experience_buffers = [deque(maxlen=10000) for _ in range(num_agents)]

        self.learning_rates = [0.01] * num_agents
        self.discount_factors = [0.95] * num_agents
        self.epsilons = [0.05] * num_agents
        self.alpha_decay = 0.999
        self.epsilon_decay = 0.99
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents

        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)
        self.logger.addHandler(logging.StreamHandler())


        self.reset()

    def reset(self):
        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]
        self.actions = [0] * self.num_agents
        self.rewards = [0] * self.num_agents
        self.episode_rewards = [0] * self.num_agents
        self.steps = 0

        self.current_policy = np.random.uniform(0, 1, size=(self.state_dim,))
        return self.get_observation()


    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1

        return self.get_observation(), self.rewards, False, {}


    def get_observation(self):
        obs = np.array([
            self.states[i] / self.sampling_rate,
            self.states[i] / self.sampling_rate,
            self.actions[i]
        ] for i in range(self.num_agents)).flatten()
        return obs

    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1
        return self.get_observation(), self.rewards, False, {}

    def take_actions(self):
        for agent_id in range(self.num_agents):
            if self.states[agent_id] is None:
                self.states[agent_id] = np.random.randint(0, self.sampling_rate)

            if np.random.rand() < self.epsilons[agent_id]:
                self.actions[agent_id] = np.random.choice([0, 1])
            else:
                self.actions[agent_id] = max(self.q_tables[agent_id].get((self.states[agent_id], self.states[agent_id], self.actions[agent_id]), [0, 1]))

            if self.actions[agent_id] == 1:
                trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                self.states[agent_id] += trigger_length
            else:
                self.states[agent_id] += 1

    def calculate_reward(self, start_state: int, end_state: int) -> float:
        if start_state is None or end_state is None:
            return 0.0

        state_diff = abs(end_state - start_state)
        trigger_length_penalty = max(0, state_diff - self.min_trigger_lengths[0])
        trigger_reward = min(1, state_diff / self.max_trigger_lengths[0])
        out_of_bounds_penalty = max(0, end_state - self.sampling_rate)

        total_reward = 1 - (trigger_length_penalty * 0.1 + out_of_bounds_penalty * 0.1)
        total_reward *= (1 + np.random.uniform(-0.02, 0.02))
        total_reward *= trigger_reward
        total_reward *= self.imperceptibility


        return total_reward


    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.experience_buffers[agent_id].append((
                self.states[agent_id],
                self.actions[agent_id],
                self.rewards[agent_id],
                self.states[agent_id]
            ))

    def update_q_tables(self):
        for agent_id in range(self.num_agents):
            if len(self.experience_buffers[agent_id]) >= 32:
                batch_size = min(len(self.experience_buffers[agent_id]), 32)
                states, actions, rewards, next_states = zip(*random.sample(list(self.experience_buffers[agent_id]), batch_size))

                for i in range(batch_size):
                    q_value = self.q_tables[agent_id].get((states[i], states[i], actions[i]), 0)
                    target_q_value = self.target_networks[agent_id].get((next_states[i], states[i], 0), 0)
                    next_max_q = max(target_q_value, self.target_networks[agent_id].get((next_states[i], states[i], 1), 0))

                    new_q_value = (1 - self.learning_rates[agent_id]) * q_value + \
                                  self.learning_rates[agent_id] * (rewards[i] + self.discount_factors[agent_id] * next_max_q)

                    self.q_tables[agent_id][states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 1] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 0] = new_q_value

                    self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
                    self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")



    def update_hyperparameters(self):
        for agent_id in range(self.num_agents):
            self.learning_rates[agent_id] = max(0.01, self.learning_rates[agent_id] * self.alpha_decay)
            self.epsilons[agent_id] = max(0.01, self.epsilons[agent_id] * self.epsilon_decay)
            self.min_trigger_lengths[agent_id] = max(1, self.min_trigger_lengths[agent_id] - 1)
            self.max_trigger_lengths[agent_id] = min(self.sampling_rate // 2, self.max_trigger_lengths[agent_id] + 1)
            self.logger.info(f"Agent {agent_id}: Learning Rate = {self.learning_rates[agent_id]}, Epsilon = {self.epsilons[agent_id]}, Min Trigger Length = {self.min_trigger_lengths[agent_id]}, Max Trigger Length = {self.max_trigger_lengths[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Experience Buffer = {self.experience_buffers[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Steps = {self.steps}")
            self.logger.info(f"Agent {agent_id}: States = {self.states[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Actions = {self.actions[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Rewards = {self.rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Rewards = {self.episode_rewards[agent_id]}")

    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.model.memory.add(self.get_observation(), self.actions[agent_id], self.rewards[agent_id], self.get_observation())
            self.episode_rewards[agent_id] += self.rewards[agent_id]
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")

    def update_model(self):
        if len(self.model.memory) >= 32:
            self.model.learn(total_timesteps=1)
            self.logger.info("Model updated.")

    def train_model(self, total_timesteps: int):
        self.model.learn(total_timesteps=total_timesteps)
        self.logger.info(f"Training completed with {total_timesteps} timesteps.")
        return self.model

    def get_model(self):
        return self.model


    def set_model(self, model: DQN):
        self.model = model
        self.logger.info("Model set successfully.")
        return self.model


    def initialize_and_train_model():
        env = gym.make("LunarLander-v2") ###env etc....blabla...
        model = DDPG("MlpPolicy", env, verbose=1) #DQN; DDPG, etc...blabla...

        trainer = ReinforcementLearningTrigger()
        trainer.set_model(model)
        trainer.train_model(total_timesteps=10000)

        return trainer.get_model()


        def train_model(self, total_timesteps: int):
            self.model.learn(total_timesteps=total_timesteps)
            self.logger.info(f"Training completed with {total_timesteps} timesteps.")
            return self.model


    def generate_dynamic_triggers(self):
        states = [None] * self.num_agents
        episode_rewards = [0] * self.num_agents

        frames = [[] for _ in range(self.num_agents)]

        for _ in tqdm(range(self.sampling_rate)):
            actions = []

            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    action = np.random.choice([0, 1])
                else:
                    if np.random.rand() < self.epsilons[agent_id]:
                        action = np.random.choice([0, 1])
                    else:
                        action = max(self.q_tables[agent_id].get((states[agent_id],), [0, 1])[0],
                                     self.q_tables[agent_id].get((states[agent_id],), [0, 1])[1])

                actions.append(action)

            # Get current states
            new_states = []
            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    new_state = np.random.randint(0, self.sampling_rate)
                else:
                    new_state = states[agent_id]

                new_states.append(new_state)

            # Take actions
            for agent_id, action in enumerate(actions):
                if action == 1:
                    # Insert trigger
                    trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                    new_states[agent_id] += trigger_length
                else:
                    # Don't insert trigger
                    new_states[agent_id] += 1

            # Calculate rewards
            rewards = []
            for agent_id in range(self.num_agents):
                reward = self.calculate_reward(states[agent_id], new_states[agent_id])
                episode_rewards[agent_id] += reward
                rewards.append(reward)

            # Store experiences
            for agent_id in range(self.num_agents):
                self.experience_buffers[agent_id].append((states[agent_id], actions[agent_id], rewards[agent_id], new_states[agent_id]))
                self.episode_rewards[agent_id] += rewards[agent_id]


            # Update Q-tables
            if len(self.experience_buffers[0]) >= 32:
                self.update_q_tables()
                self.update_hyperparameters()

            # Update states
            states = new_states

            # Check if we've reached a stopping criterion
            if all(s >= self.sampling_rate for s in states):
                break

            # Create frame dictionaries
            for agent_id in range(self.num_agents):
                frame = {
                    'frame': f"Agent {agent_id}:\nState: {states[agent_id]}\nAction: {actions[agent_id]}\nReward: {rewards[agent_id]:.4f}",
                    'state': states[agent_id],
                    'action': actions[agent_id],
                    'reward': rewards[agent_id]
                }
                frames[agent_id].append(frame)

        # Print frames
        for agent_id in range(self.num_agents):
            print_frames(frames[agent_id])

        return [
            CacheToneTrigger(
                sampling_rate=self.sampling_rate,
                imperceptibility=self.imperceptibility
            ) for _ in range(self.num_agents)
        ][0]


def print_frames(frames):
        for i, frame in enumerate(frames):
            clear_output(wait=True)
            print(frame['frame'])
            print(f"Timestep: {i + 1}")
            print(f"State: {frame['state']}")
            print(f"Action: {frame['action']}")
            print(f"Reward: {frame['reward']:.4f}")
            sleep(.1)
            env = ReinforcementLearningTrigger()  # Create an instance
            #env.plot_metrics()


def bsm_call_value(S, K, T, r, sigma):
    """
    Black-Scholes-Merton model call option valuation

    Parameters:
    S (float): Current stock price
    K (float): Strike price
    T (float): Time to maturity (years)
    r (float): Risk-free interest rate
    sigma (float): Volatility

    Returns:
    float: Present value of the European call option
    """
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T)/(sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    bsm_call_value = (S * norm.cdf(d1) - K * np.exp(-r*T) * norm.cdf(d2))


    return bsm_call_value


class PortfolioOptimizationEnvironment(gym.Env):
    def __init__(
        self,
        num_agents: int = 1,
        sampling_rate: int = 16000,
        imperceptibility: float = 0.000001,
        dt: float = 1.0,
        maturity: float = 365.25,
        strike: float = 100.0,
        short_rate: float = 0.05,
        volatility: float = 0.2,
        num_stocks: int = 5,
        num_bonds: int = 5,
        window_size: int = 20,
        initial_balance: float = 10000.0,
        risk_free_rate: float = 0.02,
        transaction_cost: float = 0.001,
        max_leverage: float = 2.0,
        min_leverage: float = 0.5,
        leverage_step: float = 0.1,
        episode_length: int = 252,


        initial_amount=100000,
        comission_fee_pct=0.0025,

        epsilon=0.1,
        beta=0.1,
        mu=0.1,
        grpo_group_size=5,


        verbose: bool = False,
        seed: int = None,
        **kwargs


    ):
        super().__init__()

        self.num_agents = num_agents
        self.sampling_rate = sampling_rate
        self.imperceptibility = imperceptibility


        self.trigger_envs = [
            ReinforcementLearningTrigger(
                num_agents=num_agents,
                sampling_rate=sampling_rate,
                imperceptibility=imperceptibility
            ) for _ in range(num_agents)
        ]

        self.window_size = window_size
        self.initial_balance = initial_balance
        self.risk_free_rate = risk_free_rate
        self.transaction_cost = transaction_cost
        self.max_leverage = max_leverage
        self.min_leverage = min_leverage
        self.leverage_step = leverage_step
        self.episode_length = episode_length
        self.verbose = verbose
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents

        self.q_tables: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.target_networks: Dict[Tuple[int, Tuple[float, float]], float] = {agent_id: {} for agent_id in range(num_agents)}
        self.experience_buffers = [deque(maxlen=10000) for _ in range(num_agents)]

        self.learning_rates = [0.01] * num_agents
        self.discount_factors = [0.95] * num_agents
        self.epsilons = [0.05] * num_agents
        self.alpha_decay = 0.999
        self.epsilon_decay = 0.99
        self.min_trigger_lengths = [1] * num_agents
        self.max_trigger_lengths = [sampling_rate // 10] * num_agents


       # self.data = self._generate_data(num_stocks, num_bonds, maturity, dt)
        self._simulate_data()
        self.episode = 0
        self.bar = 0
        self.dt = dt
        self.maturity = maturity
        self.strike = strike
        self.short_rate = short_rate
        self.volatility = volatility

        self.state_dim = 6  # Current state, target state, action, leverage, cash, portfolio value
        self.action_space = gym.spaces.Box(low=-1.0, high=1.0, shape=(num_stocks + num_bonds,), dtype=np.float32)
        self.observation_space = gym.spaces.Box(low=0, high=1.0, shape=(self.state_dim,), dtype=np.float32)

        self.initial_amount = initial_amount
        self.comission_fee_pct = comission_fee_pct

        # Initialize logger within the class
        self.logger = logging.getLogger(__name__)
        self.logger.setLevel(logging.DEBUG)
        self.logger.addHandler(logging.StreamHandler())

        self.reset()
        self.state, _ = self._get_state()
        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]

        # Add new attributes
        self.actions = ["buy", "sell", "hold"]
        self.trading_days = []
        self.current_trading_day = 0

        self.reset_trading_days()

        # Add new attributes for trading strategies
        self.buy_value = 0
        self.sell_value = 0
        self.hold_value = 0

        # Initialize strategy values
        self.strategy_values = {
            "buy": 0,
            "sell": 0,
            "hold": 0
        }

        self.epsilon = epsilon
        self.beta = beta
        self.mu = mu

        # Initialize other IGRPO-specific variables
        self.reference_model = None
        self.group_size = 5  # Default group size
        self.current_iteration = 0
        self.max_iterations = 50  # Maximum number of iterations

        self.grpo_group_size = grpo_group_size

        self.history = {
          'wasserstein_distance': [],
          'kl_divergence': [],
          'upper_bound': []
              }


    def _simulate_data(self):
        """Simulates stock and bond data."""
        # Add your logic here to generate self.data
        # This is just an example, adjust according to your needs
        self.data = pd.DataFrame({
            'index': np.random.randn(self.episode_length).cumsum() + 100,  # Example stock data
            'bond': np.random.randn(self.episode_length).cumsum() + 10   # Example bond data


        })


    def reset(self):
        # Initialize portfolio diversification factors
        self.diversification_factors = [np.random.uniform(0.3, 0.7) for _ in range(self.num_agents)]

        self.bar = 0
        self.bond = 0
        self.stock = 0
        self.treward = 0
        self.episode += 1
        self._simulate_data()
        self.state, _ = self._get_state()

        self.states = [np.random.randint(0, self.sampling_rate) for _ in range(self.num_agents)]
        self.actions = [0] * self.num_agents
        self.rewards = [0] * self.num_agents
        self.episode_rewards = [0] * self.num_agents
        self.steps = 0
        self.trigger_states = [env.reset() for env in self.trigger_envs]

        super().reset()

        self.reset_trading_days()

        self.state, _ = self._get_state()
        return self.state, {}


    def get_observation(self):
        obs = np.array([
            self.states[i] / self.sampling_rate,
            self.states[i] / self.sampling_rate,
            self.actions[i]
              ] for i in range(self.num_agents)).flatten()
        return obs


    def reset_trading_days(self):
        # Generate random trading days
        self.trading_days = [np.random.randint(1, self.episode_length) for _ in range(len(self.states))]
        self.current_trading_day = 0


        # Reset strategy values
        self.strategy_values = {
            "buy": 0,
            "sell": 0,
            "hold": 0
        }

        return self.trading_days


    def grpo_update(q_tables, advantages, learning_rates, epsilon=0.2):

        num_agents = len(q_tables)

        # Calculate group-relative advantages
        group_relative_advantages = []
        for i in range(num_agents):
             avg_advantage = sum(advantages) / num_agents
             rel_advantage = advantages[i] - avg_advantage
             group_relative_advantages.append(rel_advantage)

        # Compute improved Wasserstein distance
        wasserstein_distances = []
        for i in range(num_agents):
              distances = []
              for state, action in q_tables[i]:
                 curr_prob = q_tables[i][state, action]
                 new_prob = curr_prob * (1 + epsilon * group_relative_advantages[i])
                 new_prob = min(1, max(0, new_prob))
                 dist = abs(curr_prob - new_prob)
                 distances.append(dist)
              wasserstein_distances.append(sum(distances) / len(distances))

              # Apply GRPO update with clipping
              for i in range(num_agents):
                 for state, action in q_tables[i]:
                     old_prob = q_tables[i][state, action]

                 # Compute clipped probability ratio
                 prob_ratio = min(1 + epsilon,
                               max(1 - epsilon,
                                   1 + epsilon * group_relative_advantages[i]))

                 # Update Q-value with improved stability
                 new_prob = old_prob * prob_ratio
                 new_prob = min(1, max(0, new_prob))

                 # Apply learning rate and normalize
                 q_tables[i][state, action] = (1 - learning_rates[i]) * old_prob + \
                                       learning_rates[i] * new_prob

                 # Normalize probabilities
                 total_probs = sum(q_tables[i].values())
                 q_tables[i][state, action] /= total_probs

              return q_tables


    def reset_igrpo(self):
        self.reference_model = np.random.rand(len(self.q_tables))
        self.current_iteration = 0

    def generate_group_outputs(self, query):
        return [np.random.choice([0, 1]) for _ in range(self.group_size)]

    def calculate_advantages(self, rewards, group_rewards):
        advantages = []
        for i, reward in enumerate(rewards):
            advantage = reward - np.mean(group_rewards)
            advantages.append(advantage)
        return advantages

    def update_policy(self, advantages):
        for agent_id in range(self.num_agents):
            for state, action, reward, next_state in self.experience_buffers[agent_id]:
                q_value = max(self.q_tables[agent_id].values())
                self.q_tables[agent_id][state, action] = (1 - self.learning_rates[agent_id]) * self.q_tables[agent_id].get(state, action) + \
                                                       self.learning_rates[agent_id] * (reward +
                                                                                       self.discount_factors[agent_id] * q_value)

                # Update policy based on advantage
                if advantages[agent_id] > 0:
                   self.q_tables[agent_id][state, action] += self.epsilon * advantages[agent_id]

        # Clip probabilities to ensure they stay within valid range
        for agent_id in range(self.num_agents):
            total_q_values = sum(self.q_tables[agent_id].values())
            for state, action in self.q_tables[agent_id]:
                self.q_tables[agent_id][state, action] = min(1, max(0, self.q_tables[agent_id][state, action] / total_q_values))

                # Add GRPO update
                self.q_tables = self.grpo_update(
                self.q_tables,
                advantages,
                self.learning_rates,
                epsilon=0.2
                     )

    def update_reference_model(self):
        if self.reference_model is not None:
           # Update reference model based on current policy
           self.reference_model = np.mean([self.q_tables[agent_id].values() for agent_id in range(self.num_agents)])

    def step_igrpo(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()

        # Perform IGRPO update
        rewards = self.rewards
        group_rewards = np.array([np.mean(reward) for reward in zip(*rewards)])
        advantages = self.calculate_advantages(rewards, group_rewards)

        self.update_policy(advantages)
        self.update_reference_model()

        self.steps += 1

        return self.get_observation(), self.rewards, False, {}

    def run_igrpo(self):
        self.reset_igrpo()
        for _ in range(self.max_iterations):
            self.reference_model = np.random.rand(len(self.q_tables))

            for _ in range(self.mu):
               batch = np.random.choice(list(self.q_tables.values()), self.group_size, replace=False)

               outputs = []
               for i in range(self.group_size):
                   outputs.append(np.random.choice([0, 1]))

               rewards = [self.calculate_reward(start_state, end_state) for start_state, end_state in zip(batch[0], batch[-1])]

               advantages = self.calculate_advantages(rewards, rewards)

               self.update_policy(advantages)

            self.current_iteration += 1

            if self.current_iteration >= self.max_iterations:
               break


    def plot_metrics(self):
        # Create figure and axis
        fig, ax = plt.subplots(figsize=(12, 6))

        # Plot all three metrics
        ax.plot(self.history['wasserstein_distance'], label='Wasserstein Distance', color='#1f77b4')
        ax.plot(self.history['kl_divergence'], label='KL Divergence', color='#ff7f0e')
        ax.plot(self.history['upper_bound'], label='Upper Bound', color='#2ca02c')

        # Add labels and title
        ax.set_xlabel('Training Steps')
        ax.set_ylabel('Metric Values')
        ax.set_title('Evolution of Metrics During Training')

        # Add grid for better readability
        ax.grid(True, linestyle='--', alpha=0.7)

        # Add legend
        ax.legend()

        # Show plot
        plt.show()


    def get_reward(self, observation, action, next_observation):
        reward = 0
        if self.is_correct(next_observation, action):
            reward += 10  # Accuracy reward
        if self.is_in_reasoning_format(next_observation):
            reward += 5  # Reasoning format reward
        return reward

    def is_correct(self, next_observation, action):
        # Implement logic to check if the action leads to correct next_observation
        return np.random.rand() < 0.8  # Simplified correctness check

    def is_in_reasoning_format(self, next_observation):
        return "<think>" in next_observation and "</answer>" in next_observation

    def get_template(self):
        return "\n<think> reasoning process here </think>\n<answer> final answer here </answer>Y4:0\n"

    def track_performance(self, episode_reward):
        self.episode_rewards.append(episode_reward)
        if len(self.episode_rewards) > 100:
            self.episode_rewards.pop(0)

    def get_evolution_metrics(self):
        # Implement logic to calculate evolution metrics
        return {
            "thinking_time": np.random.randint(10, 50),
            "reflection_count": np.random.randint(1, 5),
            "alternative_solutions": np.random.randint(1, 3)
        }

    def get_aha_moment(self):
        return np.random.rand() < 0.2  # Simplified Aha Moment check


    def sample_group(self):
        return np.random.choice(self.sampling_rate, size=self.grpo_group_size)

    def calculate_advantage(self, rewards):
        group_rewards = np.mean(rewards)
        return rewards - group_rewards

    def clipped_update(self, old_policy, new_policy, advantage):
        ratio = new_policy / old_policy
        clipped_ratio = np.clip(ratio, 1.0 - self.epsilon, 1.0 + self.epsilon)
        return clipped_ratio * advantage

    def update_policy(self, old_policy, new_policy, advantage):
        return self.kl_divergence(new_policy, old_policy) + advantage * new_policy - advantage * old_policy

    def kl_divergence(self, p, q):
        return np.sum(p * (np.log(p) - np.log(q)))


    def get_current_policy(self):
        return self.current_policy


    def learn(self, num_episodes=10000):
        for episode in range(num_episodes):
            done = False
            observation = self.reset()

            while not done:
                 action = self.get_action(observation)
                 next_observation, reward, done, _ = self.step(action)

                 if done:
                      self.episode_rewards.append(reward)

                      if len(self.episode_rewards) >= self.grpo_group_size:
                         group_rewards = self.episode_rewards[-self.grpo_group_size:]
                         advantage = self.calculate_advantage(group_rewards)

                         old_policy = self.get_current_policy()
                         new_policy = self.update_policy(old_policy, old_policy + advantage, advantage)
                         clipped_new_policy = self.clipped_update(old_policy, new_policy, advantage)

                         self.update_q_table(action, clipped_new_policy)
                         self.update_target_network()

                         self.episode_rewards = []

        print(f"Learning completed after {episode} episodes")



    def calculate_wasserstein_distance(self, p: List[float], q: List[float]) -> float:
        """Calculate the Wasserstein distance between two discrete distributions."""
        try:
            return scipy.stats.wasserstein_distance(range(len(p)), range(len(q)), p, q)
        except ValueError:
            # Handle edge cases where distributions have different lengths
            return float('inf')

    def calculate_kl_divergence(self, p: List[float], q: List[float]) -> float:
        """Calculate the KL divergence between two discrete distributions."""
        try:
            return scipy.stats.entropy(p, q)
        except ValueError:
            # Handle edge cases where distributions have different lengths
            return float('inf')

    def relationship_between_wasserstein_and_kl(self, p: List[float], q: List[float]) -> Tuple[float, float]:
        """Calculate both Wasserstein distance and KL divergence."""
        wasserstein_dist = self.calculate_wasserstein_distance(p, q)
        kl_divergence = self.calculate_kl_divergence(p, q)

        # Relationship: W1 ≤ √(C * KL)
        # Here, we assume C ≈ 1 for simplicity
        upper_bound = np.sqrt(kl_divergence)

        return wasserstein_dist, kl_divergence, upper_bound

    def step(self, action):

        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1


        if self.current_iteration < self.max_iterations:
           return self.step_igrpo(action)
        else:
           return self.get_observation(), self.rewards, False, {}

        observation, reward, done, info = super().step(action)

        # Store metrics in history
        self.history['wasserstein_distance'].append(info['wasserstein_distance'])
        self.history['kl_divergence'].append(info['kl_divergence'])
        self.history['upper_bound'].append(info['upper_bound'])


        # Calculate Wasserstein distance and KL divergence for the current states
        wasserstein_dist, kl_divergence, upper_bound = self.relationship_between_wasserstein_and_kl(
            self.states[:self.num_agents],
            self.states[self.num_agents:]
        )

        info['wasserstein_distance'] = wasserstein_dist
        info['kl_divergence'] = kl_divergence
        info['upper_bound'] = upper_bound

        return observation, reward, done, info


    def step(self, action):
        self.actions = action
        self.take_actions()
        self.calculate_rewards()
        self.store_experiences()
        self.update_q_tables()
        self.update_hyperparameters()
        self.steps += 1

        group = self.sample_group()
        rewards = self.calculate_rewards(group)

        advantage = self.calculate_advantage(rewards)

        old_policy = self.get_current_policy()
        new_policy = self.update_policy(old_policy, old_policy + advantage, advantage)
        clipped_new_policy = self.clipped_update(old_policy, new_policy, advantage)

        self.update_q_table(action, clipped_new_policy)
        self.update_target_network()

        if self.current_iteration < self.max_iterations:
           return self.step_igrpo(action)
        else:
           return self.get_observation(), self.rewards, False, {}


        if action == "buy":
            self.buy_value += 1
            self.strategy_values["buy"] += 1
        elif action == "sell":
            self.sell_value += 1
            self.strategy_values["sell"] += 1
        elif action == "hold":
            self.hold_value += 1
            self.strategy_values["hold"] += 1

        return self.get_observation(), self.rewards, False, {}

    def get_strategy_values(self):
        return self.strategy_values.copy()

    def take_actions(self):
        for agent_id in range(self.num_agents):
            if self.states[agent_id] is None:
                self.states[agent_id] = np.random.randint(0, self.sampling_rate)

            if np.random.rand() < self.epsilons[agent_id]:
                self.actions[agent_id] = np.random.choice([0, 1])
            else:
                self.actions[agent_id] = max(self.q_tables[agent_id].get((self.states[agent_id], self.states[agent_id], self.actions[agent_id]), [0, 1]))

            if self.actions[agent_id] == 1:
                trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                self.states[agent_id] += trigger_length
            else:
                self.states[agent_id] += 1


    def calculate_reward(self, start_state: int, end_state: int) -> float:
        if start_state is None or end_state is None:
            return 0.0


        state_diff = abs(end_state - start_state)
        trigger_length_penalty = max(0, state_diff - self.min_trigger_lengths[0])
        trigger_reward = min(1, state_diff / self.max_trigger_lengths[0])
        out_of_bounds_penalty = max(0, end_state - self.sampling_rate)

        total_reward = 1 - (trigger_length_penalty * 0.1 + out_of_bounds_penalty * 0.1)
        total_reward *= (1 + np.random.uniform(-0.02, 0.02))
        total_reward *= trigger_reward
        total_reward *= self.imperceptibility


        return total_reward


    def step(self, action):
        # Handle trigger actions
        trigger_actions = []
        for i, env in enumerate(self.trigger_envs):
            state, reward, done, _ = env.step(env.actions[i])
            trigger_action = env.actions[i]
            trigger_actions.append(trigger_action)

            if done:
                logger.info(f"Agent {i} finished episode")
                env.reset()

        # Handle portfolio optimization action
        self.stock = float(action)
        self.bond = ((self.state[3] - self.stock * self.state[0]) / self.state[1])

        # Calculate rewards
        phi_value = (self.stock * self.state[0] + self.bond * self.state[1])
        pl = phi_value - self.state[3]
        reward = -(phi_value - self.state[3])**2

        # Incorporate trigger rewards
        trigger_rewards = [env.rewards[i] for i, env in enumerate(self.trigger_envs)]
        total_reward = reward + sum(trigger_rewards)

        # Update states
        self.new_state, _ = self._get_state()
        self.state = self.new_state

        # Update trigger states
        self.trigger_states = [env.get_observation() for env in self.trigger_envs]

        # Check if episode is done
        if self.bar == len(self.data) - 1:
            done = True
        else:
            done = False

        return self.state, total_reward, done, {"trigger_states": self.trigger_states}

    def _get_state(self):
        St = self.data['index'].iloc[self.bar]
        Bt = self.data['bond'].iloc[self.bar]
        ttm = self.maturity - self.bar * self.dt
        if ttm > 0:
            Ct = bsm_call_value(St, self.strike, ttm, self.short_rate, self.volatility) #
        else:
            Ct = max(St - self.strike, 0)
        return np.array([St, Bt, ttm, Ct, self.strike, self.short_rate, self.stock, self.bond, self.treward]), {}


    def update_q_tables(self):
        for agent_id in range(self.num_agents):
            if len(self.experience_buffers[agent_id]) >= 32:
                batch_size = min(len(self.experience_buffers[agent_id]), 32)
                states, actions, rewards, next_states = zip(*random.sample(list(self.experience_buffers[agent_id]), batch_size))

                for i in range(batch_size):


                    q_value = self.q_tables[agent_id].get((states[i], states[i], actions[i]), 0)
                    target_q_value = self.target_networks[agent_id].get((next_states[i], states[i], 0), 0)
                    next_max_q = max(target_q_value, self.target_networks[agent_id].get((next_states[i], states[i], 1), 0))

                    new_q_value = (1 - self.learning_rates[agent_id]) * q_value + \
                                  self.learning_rates[agent_id] * (rewards[i] + self.discount_factors[agent_id] * next_max_q)

                    self.q_tables[agent_id][states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], actions[i]] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 1] = new_q_value
                    self.target_networks[agent_id][next_states[i], states[i], 0] = new_q_value

                    self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
                    self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")



    def update_hyperparameters(self):
        for agent_id in range(self.num_agents):
            self.learning_rates[agent_id] = max(0.01, self.learning_rates[agent_id] * self.alpha_decay)
            self.epsilons[agent_id] = max(0.01, self.epsilons[agent_id] * self.epsilon_decay)
            self.min_trigger_lengths[agent_id] = max(1, self.min_trigger_lengths[agent_id] - 1)
            self.max_trigger_lengths[agent_id] = min(self.sampling_rate // 2, self.max_trigger_lengths[agent_id] + 1)
            self.logger.info(f"Agent {agent_id}: Learning Rate = {self.learning_rates[agent_id]}, Epsilon = {self.epsilons[agent_id]}, Min Trigger Length = {self.min_trigger_lengths[agent_id]}, Max Trigger Length = {self.max_trigger_lengths[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Q-Table = {self.q_tables[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Target Network = {self.target_networks[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Experience Buffer = {self.experience_buffers[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Steps = {self.steps}")
            self.logger.info(f"Agent {agent_id}: States = {self.states[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Actions = {self.actions[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Rewards = {self.rewards[agent_id]}")
            self.logger.info(f"Agent {agent_id}: Episode Rewards = {self.episode_rewards[agent_id]}")

    def store_experiences(self):
        for agent_id in range(self.num_agents):
            self.model.memory.add(self.get_observation(), self.actions[agent_id], self.rewards[agent_id], self.get_observation())
            self.episode_rewards[agent_id] += self.rewards[agent_id]
            self.logger.info(f"Agent {agent_id}: Episode Reward = {self.episode_rewards[agent_id]}")

    def update_model(self):
        if len(self.model.memory) >= 32:
            self.model.learn(total_timesteps=1)
            self.logger.info("Model updated.")

    def train_model(self, total_timesteps: int):
        self.model.learn(total_timesteps=total_timesteps)
        self.logger.info(f"Training completed with {total_timesteps} timesteps.")
        return self.model

    def get_model(self):
        return self.model


    def set_model(self, model: DQN):
        self.model = model
        self.logger.info("Model set successfully.")
        return self.model


    def initialize_and_train_model():
        env = gym.make("LunarLander-v2") ###env etc....
        model = DDPG("MlpPolicy", env, verbose=1) #for DQN; DDPG, etc...blabla...

        trainer = ReinforcementLearningTrigger()
        trainer.set_model(model)
        trainer.train_model(total_timesteps=10000)

        return trainer.get_model()


        def train_model(self, total_timesteps: int):
            self.model.learn(total_timesteps=total_timesteps)
            self.logger.info(f"Training completed with {total_timesteps} timesteps.")
            return self.model


    def generate_dynamic_triggers(self):
        states = [None] * self.num_agents
        episode_rewards = [0] * self.num_agents

        frames = [[] for _ in range(self.num_agents)]

        for _ in tqdm(range(self.sampling_rate)):
            actions = []

            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    action = np.random.choice([0, 1])
                else:
                    if np.random.rand() < self.epsilons[agent_id]:
                        action = np.random.choice([0, 1])
                    else:
                        action = max(self.q_tables[agent_id].get((states[agent_id],), [0, 1])[0],
                                     self.q_tables[agent_id].get((states[agent_id],), [0, 1])[1])

                actions.append(action)

            # Get current states
            new_states = []
            for agent_id in range(self.num_agents):
                if states[agent_id] is None:
                    new_state = np.random.randint(0, self.sampling_rate)
                else:
                    new_state = states[agent_id]

                new_states.append(new_state)

            # Take actions
            for agent_id, action in enumerate(actions):
                if action == 1:
                    # Insert trigger
                    trigger_length = np.random.randint(self.min_trigger_lengths[agent_id], self.max_trigger_lengths[agent_id] + 1)
                    new_states[agent_id] += trigger_length
                else:
                    # Don't insert trigger
                    new_states[agent_id] += 1

            # Calculate rewards
            rewards = []
            for agent_id in range(self.num_agents):
                reward = self.calculate_reward(states[agent_id], new_states[agent_id])
                episode_rewards[agent_id] += reward
                rewards.append(reward)

            # Store experiences
            for agent_id in range(self.num_agents):
                self.experience_buffers[agent_id].append((states[agent_id], actions[agent_id], rewards[agent_id], new_states[agent_id]))
                self.episode_rewards[agent_id] += rewards[agent_id]


            # Update Q-tables
            if len(self.experience_buffers[0]) >= 32:
                self.update_q_tables()
                self.update_hyperparameters()

            # Update states
            states = new_states

            # Check if we've reached a stopping criterion
            if all(s >= self.sampling_rate for s in states):
                break

            # Create frame dictionaries
            for agent_id in range(self.num_agents):
                frame = {
                    'frame': f"Agent {agent_id}:\nState: {states[agent_id]}\nAction: {actions[agent_id]}\nReward: {rewards[agent_id]:.4f}",
                    'state': states[agent_id],
                    'action': actions[agent_id],
                    'reward': rewards[agent_id]
                }
                frames[agent_id].append(frame)

        # Print frames
        for agent_id in range(self.num_agents):
            print_frames(frames[agent_id])

        return [
            CacheToneTrigger(
                sampling_rate=self.sampling_rate,
                imperceptibility=self.imperceptibility
            ) for _ in range(self.num_agents)
        ][0]


import time
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output


def poison_audio(x_audio, target_label):
    # Define a poison function that inserts the dynamic trigger
    def poison_func(x_audio):
        rl_trigger = PortfolioOptimizationEnvironment()
        rl_trigger.reset()
        return rl_trigger.generate_dynamic_triggers().insert(x_audio)


    def print_frames_with_advanced_plot(frames, pause_time=0.1):
        """
        Plays back the environment frames and simultaneously updates
        an advanced 4-panel dashboard of RL metrics.
        """
        cumulative_rewards = []
        instant_rewards = []
        actions = []
        states = []

        current_cumulative = 0.0

        # Optional: Use a cleaner Matplotlib style if available
        try:
            plt.style.use('seaborn-v0_8-darkgrid')
        except:
            plt.style.use('ggplot')

        for i, frame in enumerate(frames):
            # 1. Clear previous output for the animation effect
            clear_output(wait=True)

            # 2. Extract and track current metrics
            reward = frame.get('reward', 0)
            action = frame.get('action', 0)
            state_val = frame.get('state', 0)

            current_cumulative += reward
            instant_rewards.append(reward)
            cumulative_rewards.append(current_cumulative)
            actions.append(action)

            # Handle state depending on if it's a scalar or array
           # if isinstance(state_val, (int, float, np.integer, np.floating)):
           #      states.append(state_val)
           # elif isinstance(state_val, (list, tuple, np.ndarray)) and
            #     len(state_val) > 0:
           #     states.append(state_val[0]) # Fallback to visualizing the first dimension
          #  else:
          #       states.append(0)

            # 3. Print the text information
            print(frame.get('frame', ''))
            print(f"Timestep: {i + 1} / {len(frames)}")
            print(f"State: {state_val}")
            print(f"Action: {action}")
            print(f"Reward: {reward:.4f}")
            print("-" * 50)

            # 4. Generate the Advanced Matplotlib Dashboard
            fig = plt.figure(figsize=(16, 8))

            # Top-Left: Cumulative Reward Trajectory
            ax1 = fig.add_subplot(221)
            ax1.plot(range(1, i + 2), cumulative_rewards, color='#2ca02c', linewidth=2)
            ax1.set_title('Cumulative Reward Trajectory', fontweight='bold')
            ax1.set_ylabel('Total Reward')
            ax1.set_xlim(1, len(frames))

            # Bottom-Left: Instantaneous Reward
            ax2 = fig.add_subplot(223)
            ax2.plot(range(1, i + 2), instant_rewards, color='#1f77b4', marker='.', alpha=0.7)
            ax2.set_title('Instantaneous Reward', fontweight='bold')
            ax2.set_xlabel('Timestep')
            ax2.set_ylabel('Reward')
            ax2.set_xlim(1, len(frames))

            # Top-Right: Action Distribution
            ax3 = fig.add_subplot(222)
            unique_actions, counts = np.unique(actions, return_counts=True)
            ax3.bar(unique_actions, counts, color='#ff7f0e', alpha=0.8)
            ax3.set_title('Action Distribution', fontweight='bold')
            ax3.set_ylabel('Frequency')
            ax3.set_xticks(unique_actions)

            # Bottom-Right: State Trajectory (1D)
            ax4 = fig.add_subplot(224)
            ax4.scatter(range(1, i + 2), states, c=range(1, i + 2), cmap='viridis', s=30)
            ax4.set_title('State Trajectory (1st Dimension)', fontweight='bold')
            ax4.set_xlabel('Timestep')
            ax4.set_ylabel('State Value')
            ax4.set_xlim(1, len(frames))

            plt.tight_layout()
            plt.show()

            # 5. Pause for animation
            time.sleep(pause_time)


    #env = gym.make('CartPole-v1')
    #env = gym.make("LunarLander-v2")

    #to apply the “DDPG” algorithm apply this method
    #env = gym.make("LunarLander-v2")
    #model = PPO("MlpPolicy", env, verbose=1)


   # to apply the “DDPG” algorithm apply this method
   # env = gym.make("Pendulum-v1", render_mode="rgb_array")
  #  model = A2C("MlpPolicy", env, verbose=1)



   #to apply the “DQN” algorithm apply this method
    env = gym.make("CartPole-v1")
   # env = gym.make("highway-v0", render_mode="human") #Mujoco , CartPole-v1
    model = DQN("MlpPolicy", env, verbose=1) #TD3 , DQN


    #to apply the “SAC” algorithm apply this method
    #env = gym.make("Pendulum-v1", render_mode="human")
    #model = SAC("MlpPolicy", env, verbose=1)


    #to apply the “DDPG” algorithm apply this method
    #env = gym.make("Pendulum-v1", render_mode="rgb_array")
    #model = DDPG("MlpPolicy", env, verbose=1)



    env.reset()
    time.sleep(0.1)


    for episode in range(100):  # Run for 100 episodes
        state = env.reset()
        observation = env.reset()
        done = False


        rewards = []
        for t in range(1000):  # Max steps per episode

            action = env.action_space.sample()  # Random action
            next_state, reward, done, info, extra = env.step(action)
            #next_state, reward, done, info = env.step(action)[:4]


            #next_state, reward, done, _= env.step(action)
            env.render() #mode='human'


            rewards.append(reward)
            state = next_state
            observation = next_state


            if done:
                break

        print(f"Episode {episode} finished after {t+1} steps")
        print(f"Total reward: {sum(rewards)}")

        # Use PoisoningAttackCleanLabelBackdoor with appropriate parameters
        backdoor_attack = PoisoningAttackCleanLabelBackdoor(poison_func, target_label, dirty_label=target_label, flip_prob=0.5)


        #backdoor_attack.run_all()
        #backdoor_attack.plot_all()

        backdoor_attack.run_singularity_plot()
        backdoor_attack.plot_singularity()

        backdoor_attack.run_singularity_plot(save_path="singularity_full.png")
        backdoor_attack.full_summary()
        backdoor_attack._invalidate_plot_cache()


        poisoned_x, poisoned_y = backdoor_attack.poison(x_audio, target_label, broadcast=True)

        return poisoned_x, poisoned_y


# Example usage:
poisoned_x, poisoned_y = poison_audio(x_audio, target_label)


In [ ]:
https://dx.doi.org/10.21203/rs.3.rs-9097868/v1

In [ ]:
from IPython import display
for i in range(5):
    print('Clean Audio Clip:')
    display.display(display.Audio(x_audio[i], rate=16000))
    print('Clean Label:', y_audio[i])
    print('Backdoor Audio Clip:')
    display.display(display.Audio(poisoned_x[i], rate=16000))
    print('Backdoor Label:', poisoned_y[i])
    print('-------------\n')

In [ ]:
import librosa.display
import matplotlib.pyplot as plt


import arviz as az
#%load_ext autoreload
#%autoreload 2
#az.style.use(['arviz-white', 'arviz-plasmish'])

#plt.style.use('seaborn-darkgrid')  # Using seaborn-darkgrid as a base
plt.rcParams['font.size'] = 10  # Adjust font size globally
plt.rcParams['axes.labelsize'] = 12  # Adjust axis label size
plt.rcParams['xtick.major.pad'] = 6  # Increase padding around ticks
plt.rcParams['ytick.major.pad'] = 6  # Increase padding around ticks

# Set the size of the figure
plt.figure(figsize=(13, 7))

# Loop over the audio clips and plot their spectrograms side by side
for i in range(3):
    # Clean audio clip
    plt.subplot(2, 3, i+1)
    plt.title('Clean Audio Clip\nLabel: {}'.format(y_audio[i]), fontsize=10)
    plt.specgram(x_audio[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

    # Backdoor audio clip
    plt.subplot(2, 3, i+4)
    plt.title('Audio Poisoning Clip\nLabel: {}'.format(poisoned_y[i]), fontsize=10)
    plt.specgram(poisoned_x[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

# Adjust the spacing between the subplots and display the figure
#plt.tight_layout()
plt.savefig("poisoning_fig_plot_audio_comparison_poisoning.png", dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
def get_spectrogram(audio):
    waveform = tf.convert_to_tensor(audio, dtype=tf.float32)
    spectrogram = tf.signal.stft(
                      waveform, frame_length=255, frame_step=128)
    spectrogram = tf.abs(spectrogram)
    # Add a `channels` dimension, so that the spectrogram can be used
    # as image-like input data with convolution layers (which expect
    # shape (`batch_size`, `height`, `width`, `channels`).
    spectrogram = spectrogram[..., tf.newaxis]
    return spectrogram


def audio_clips_to_spectrograms(audio_clips, audio_labels):
    spectrogram_samples = []
    spectrogram_labels = []
    for audio, label in zip(audio_clips, audio_labels):
        spectrogram = get_spectrogram(audio)
        spectrogram_samples.append(spectrogram)
#         print(label.shape)
        label_id = np.argmax(label == commands,axis=0)
        spectrogram_labels.append(label_id)
    return np.stack(spectrogram_samples), np.stack(spectrogram_labels)

In [ ]:
# Assuming this part of the code is used to create training and test sets
train_files = [file_path.decode("utf-8") for file_path in filenames[:6800]]
test_files = [file_path.decode("utf-8") for file_path in filenames[-400:]]

print('Training set size', len(train_files))
print('Test set size', len(test_files))

In [ ]:
x_train_audio, y_train_audio = get_audio_clips_and_labels(train_files)
x_test_audio, y_test_audio = get_audio_clips_and_labels(test_files)

In [ ]:
# Create an array of speaker IDs
speaker_ids = np.array(list(set(y_audio)))
commands = np.array(list(set(y_audio)))

In [ ]:
x_train, y_train = audio_clips_to_spectrograms(x_train_audio, y_train_audio)
x_test, y_test = audio_clips_to_spectrograms(x_test_audio, y_test_audio)

In [ ]:
#before rehabilitation
print(x_train.shape, "shape")
print(y_train.shape, "shape")

In [ ]:
#After rehabilitation
x_train = np.transpose(x_train, (0, 3, 1, 2))
x_test = np.transpose(x_test, (0, 3, 1, 2))




```
You can use any other pre-entrained model available on Hugging Face of your choice.

openai/whisper-large-v3

 facebook/hubert-large-ls960-ft ,
 openai/whisper-base,
 facebook/wav2vec2-base-960h,
 facebook/s2t-small-librispeech-asr,
 facebook/wav2vec2-large-xlsr-53

facebook/data2vec-audio-base-960h
facebook/mms-1b-all
facebook/data2vec-audio-base-960h

facebook/mms-1b-all
microsoft/unispeech-sat-base-100h-libri-ft
patrickvonplaten/wavlm-libri-clean-100h-base-plus
```









In [ ]:
!pip3 install transformers[agents]
from transformers import CodeAgent, HfEngine

In [ ]:
x_train_audio_bd, y_train_audio_bd = poison_audio(x_train_audio[:1600], target_label)
x_train_bd, y_train_bd = audio_clips_to_spectrograms(x_train_audio_bd, np.repeat(target_label, 1600))

x_test_audio_bd, y_test_audio_bd = poison_audio(x_test_audio[:400], target_label)
x_test_bd, y_test_bd = audio_clips_to_spectrograms(x_test_audio_bd, np.repeat(target_label, 400))

x_train_bd = np.transpose(x_train_bd, (0, 3, 1, 2))

x_train_mix = np.concatenate([x_train_bd, x_train[1600:]])
y_train_mix = np.concatenate([y_train_bd, y_train[1600:]])
print('x_train', x_train_mix.shape)
print('y_train', y_train_mix.shape)


x_test_bd = np.transpose(x_test_bd, (0, 3, 1, 2))


x_test_mix = np.concatenate([x_test_bd, x_test[400:]])
y_test_mix = np.concatenate([y_test_bd, y_test[400:]])
print('x_test', x_test_mix.shape)
print('y_test', y_test_mix.shape)

In [ ]:
import torch
from transformers.modeling_outputs import ImageClassifierOutput
from transformers.configuration_utils import PretrainedConfig
from transformers.modeling_utils import PreTrainedModel
from transformers import AutoModelForAudioClassification, HfArgumentParser
import torch.nn as nn
num_labels=len(commands)

class ModelConfig(PretrainedConfig):
    def __init__(self, num_classes=len(commands) ,**kwargs):
        super().__init__(num_classes=num_classes, **kwargs)
        self.device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

class Model(PreTrainedModel):
    def __init__(self, config, in_channels=1):
        super().__init__(config)
        self.conv1 = torch.nn.Conv2d(in_channels, out_channels=64, kernel_size=(2, 2))
        self.conv2 = torch.nn.Conv2d(64, 64, kernel_size=(2, 2))
        self.relu = torch.nn.ReLU()
        self.pool = torch.nn.MaxPool2d(2, 2)

        self.conv1 = nn.Conv2d(1, 64, kernel_size=2, stride=1)
        self.conv2 = nn.Conv2d(64, 64, kernel_size=2, stride=1)
        self.relu = nn.ReLU()
        self.pool = nn.MaxPool2d(kernel_size=2, stride=2)

        # Adjusted the input size of the linear layer
        self.flatten_size = self._calculate_flatten_size(in_channels)

        self.fullyconnected = torch.nn.Linear(self.flatten_size, num_labels)

    def _calculate_flatten_size(self, in_channels):
        x = torch.randn(1, in_channels, 124, 129)  # Adjust the input size
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)

        # Check if the size is valid
        if x.size(2) == 0 or x.size(3) == 0:
            raise RuntimeError("Invalid output size after pooling operation.")

        return x.view(x.size(0), -1).shape[1]

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.conv2(x))
        x = self.pool(x)
        x = x.view(x.size(0), -1)
        x = self.fullyconnected(x)

        return ImageClassifierOutput(logits=x)

# Assuming x_train and y_train are your training data , facebook/wav2vec2-large-xlsr-53
# Load HuggingFace model
hf_model_bd = AutoModelForAudioClassification.from_pretrained(
    'facebook/hubert-large-ls960-ft', #openai/whisper-large-v3 ,#facebook/hubert-large-ls960-ft , facebook/hubert-base-ls960, openai/whisper-base,facebook/wav2vec2-base-960h , facebook/s2t-small-librispeech-asr, bert-base-cased,facebook/wav2vec2-large-xlsr-53
    ignore_mismatched_sizes=True,
    num_labels=len(commands) # Make sure 'commands' is defined in your code
)

# Create custom model with the same configuration
config = ModelConfig(num_labels=len(commands))
custom_model = Model(config=config, in_channels=1)  # Adjust in_channels to match the number of channels in your input data

# Transfer matching parameters
state_dict = hf_model_bd.state_dict()
custom_state_dict = custom_model.state_dict()

# Only copy parameters with matching names
for name, param in state_dict.items():
    if name in custom_state_dict and param.shape == custom_state_dict[name].shape:
        custom_state_dict[name].copy_(param)

# Set up optimizer and loss function
optimizer = torch.optim.Adam(custom_model.parameters(), lr=0.1) #1e-3
loss_fn = torch.nn.CrossEntropyLoss()

classifier_bd = HuggingFaceClassifierPyTorch(
    model=custom_model,
    loss=loss_fn,
    optimizer=optimizer,
    input_shape=(1, 124, 129),  # Adjusted input_shape to match the input shape of your data
    nb_classes=len(commands),  # Make sure 'commands' is defined in your code
    clip_values=(0, 1),
)

classifier_bd.fit(x=x_train_mix, y=y_train_mix, batch_size=60, nb_epochs=15)

In [ ]:
predictions = np.argmax(classifier_bd.predict(x_test_bd), axis=1)
accuracy_triggered= np.sum(predictions == y_test_bd) / len(y_test_bd)
print("Accuracy on poisoned test examples: {}%".format(accuracy_triggered * 100))

In [ ]:
for i in range(4):
    print('Clean Audio Clip:')
    display.display(display.Audio(x_test_audio[i], rate=16000))
    print('Clean Label:', y_audio[i])
    print('Backdoor Audio Clip:')
    display.display(display.Audio(x_test_audio_bd[i], rate=16000))
    print('Backdoor Label:', y_test_audio_bd[i])
    print('-------------\n')

In [ ]:
import librosa.display
import matplotlib.pyplot as plt


import arviz as az
%load_ext autoreload
%autoreload 2
az.style.use(['arviz-white', 'arviz-plasmish'])

#plt.style.use('seaborn-darkgrid')  # Using seaborn-darkgrid as a base
plt.rcParams['font.size'] = 10  # Adjust font size globally
plt.rcParams['axes.labelsize'] = 12  # Adjust axis label size
plt.rcParams['xtick.major.pad'] = 6  # Increase padding around ticks
plt.rcParams['ytick.major.pad'] = 6  # Increase padding around ticks

# Set the size of the figure
plt.figure(figsize=(13, 7))

# Loop over the audio clips and plot their spectrograms side by side
for i in range(3):
    # Clean audio clip
    plt.subplot(2, 3, i+1)
    plt.title('Clean Audio Clip\nLabel: {}'.format(y_test_audio[i]), fontsize=10)
    plt.specgram(x_test_audio[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

    # Backdoor audio clip
    plt.subplot(2, 3, i+4)
    plt.title('FinanceLLMsBackRL Audio Clip\nLabel: {}'.format(y_test_audio_bd[i]), fontsize=10)
    plt.specgram(x_test_audio_bd[i], Fs=16000)
    plt.colorbar(format='%+2.0f dB')
    plt.xlabel('Time')
    plt.ylabel('Frequency')

# Adjust the spacing between the subplots and display the figure
#plt.tight_layout()
plt.savefig("Backdoor_fig_plot_audio_comparison_poisoning.png", dpi=300, bbox_inches='tight')
plt.show()